<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>NB13 — Golden v2 Audited v4</h2>
<p>این نسخه تمام Code Cellها را از نظر syntax بررسی کرده و ناسازگاری‌های شناخته‌شدهٔ Golden v1 در
evidence diagnostic، شمارش‌های 38/553، و query-rewrite pilot را اصلاح می‌کند. Cell 46 کاملاً cache-only است و API جدید نمی‌زند.</p>
</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>NB13 — Golden v2 Clean / Run-All-Safe</h2>
<p>قراردادهای باقی‌مانده Golden v1 حذف شده‌اند؛ namespace کش Golden v2 حفظ شده است؛
و مراحل embedding در Run All نهایی فقط از checkpointهای موجود بازسازی می‌شوند.</p>
</div>

<div style="direction: rtl; white-space: normal; line-height: 1.8;">
<h2>Notebook 13 — Golden Dataset v2 Migration</h2>
<p>این نسخه برای Golden Dataset v2 فریز‌شده با 3813 رکورد ساخته شده است.</p>
<p>Golden v2 بدون تغییر از artifacts/phase2/golden_dataset_v2 خوانده می‌شود و یک compatibility layer فقط در حافظه برای قراردادهای NB13 ساخته می‌شود.</p>
<p>وابستگی تاریخی به Golden v1، 28 seed قدیمی، integrity erratum و lineage نوت‌بوک‌های 12e/12f از مسیر ارزیابی حذف شده است.</p>
<p>نتایج Benchmark همچنان زیر namespace مبتنی بر SHA-256 دیتاست فعال ذخیره می‌شوند.</p>
</div>

In [16]:
# Code Cell 0
# Golden Dataset v2 bootstrap + NB13 compatibility adapter

from pathlib import Path
import hashlib
import json
import re

EXPECTED_GOLDEN_V2_RECORD_COUNT = 3813
EXPECTED_GOLDEN_V2_SHA256 = (
    "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2"
)

cwd = Path.cwd().resolve()
project_root_candidates = [
    p for p in [cwd, *cwd.parents]
    if (p / "artifacts").is_dir() and (p / "notebooks").is_dir()
]
assert project_root_candidates, f"Project root not found from: {cwd}"
project_root = project_root_candidates[0]

golden_dataset_path = (
    project_root
    / "artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl"
)
golden_dataset_report_path = (
    project_root
    / "artifacts/phase2/golden_dataset_v2/golden_dataset_v2_final_report.json"
)

assert golden_dataset_path.is_file(), (
    f"Golden Dataset v2 not found: {golden_dataset_path}"
)

dataset_hash = hashlib.sha256(
    golden_dataset_path.read_bytes()
).hexdigest()

assert dataset_hash == EXPECTED_GOLDEN_V2_SHA256, (
    "Golden Dataset v2 hash mismatch. "
    f"Expected {EXPECTED_GOLDEN_V2_SHA256}, found {dataset_hash}"
)

raw_golden_dataset = [
    json.loads(line)
    for line in golden_dataset_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(raw_golden_dataset) == EXPECTED_GOLDEN_V2_RECORD_COUNT, (
    f"Expected {EXPECTED_GOLDEN_V2_RECORD_COUNT} records, "
    f"found {len(raw_golden_dataset)}"
)

golden_dataset = []

for record in raw_golden_dataset:
    evaluation_content = record.get("evaluation_content")
    assert isinstance(evaluation_content, dict), (
        f"Invalid evaluation_content: {record.get('candidate_id')}"
    )

    question = evaluation_content.get("question")
    expected_answer = evaluation_content.get("expected_answer")
    source_seed_slot_id = record.get("source_seed_slot_id")
    candidate_id = record.get("candidate_id")

    assert candidate_id
    assert source_seed_slot_id
    assert isinstance(question, str) and question.strip()
    assert isinstance(expected_answer, str) and expected_answer.strip()

    adapted = dict(record)

    # NB13 compatibility aliases. Golden v2 on disk is never modified.
    adapted["question"] = question
    adapted["expected_answer"] = expected_answer
    adapted["question_intent"] = evaluation_content.get("question_intent")
    adapted["source_seed_id"] = candidate_id
    adapted["source_lineage"] = source_seed_slot_id

    evidence_quotes = record.get("evidence_quotes") or []
    adapted["source_evidence"] = "\n\n".join(
        str(item).strip()
        for item in evidence_quotes
        if str(item).strip()
    )

    golden_dataset.append(adapted)

assert len({r["candidate_id"] for r in golden_dataset}) == len(golden_dataset)
assert all(r["source_evidence"] for r in golden_dataset), (
    "At least one Golden v2 record has no evidence quote."
)

golden_dataset_context = {
    "path": golden_dataset_path.as_posix(),
    "sha256": dataset_hash,
    "record_count": len(golden_dataset),
    "dataset_version": "golden_dataset_v2",
    "compatibility_adapter": "NB13_GOLDEN_V2_V1",
}

print("Active Golden Dataset:", golden_dataset_context["path"])
print("SHA-256:", golden_dataset_context["sha256"])
print("Record count:", golden_dataset_context["record_count"])
print("Compatibility adapter:", golden_dataset_context["compatibility_adapter"])
print("\nACTIVE GOLDEN DATASET V2: PASS")

Active Golden Dataset: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl
SHA-256: b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Record count: 3813
Compatibility adapter: NB13_GOLDEN_V2_V1

ACTIVE GOLDEN DATASET V2: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h1>Notebook 13 — Embedding Benchmark</h1>

<h2>مقایسۀ مدل‌های Embedding بر اساس Golden Dataset فعال</h2>

<p>
هدف این Notebook انتخاب مدل Embedding بر اساس Benchmark واقعی روی Golden Dataset فعال پروژه است.
انتخاب مدل بر مبنای شهرت، تنظیم پیش‌فرض یا حدس انجام نمی‌شود.
</p>

<p>
در این مرحله، Candidate Modelها همراه با Candidateهای Chunking Freeze‌شده NB11
ارزیابی می‌شوند تا کیفیت بازیابی، هزینه، Latency و Failureهای واقعی هر ترکیب اندازه‌گیری شود.
</p>

<h3>جایگاه در Pipeline</h3>

<p>
این Notebook پس از Golden Dataset و پیش از ساخت Dense Index قرار دارد.
</p>

<h3>Inputهای اصلی</h3>

<ul>
<li>متغیر مرکزی <code>golden_dataset</code></li>
<li>اطلاعات همراه آن در <code>golden_dataset_context</code></li>
<li>Candidateهای Chunking Freeze‌شده NB11</li>
<li>Candidate Modelهای Embedding که در مراحل بعد Discovery خواهند شد</li>
</ul>

<h3>Outputهای مورد انتظار</h3>

<ul>
<li>Registry مدل‌های Candidate</li>
<li>Config Benchmark نسخه‌دار</li>
<li>Metricهای واقعی Retrieval برای هر ترکیب Model و Chunking</li>
<li>تحلیل کیفیت، Latency، Failure و Cost</li>
<li>Evidence لازم برای انتخاب Embedding پیش از Dense Index</li>
</ul>

<h3>خارج از Scope</h3>

<ul>
<li>ساخت Dense Index نهایی</li>
<li>انتخاب نهایی Chunking</li>
<li>Sparse Retrieval، Hybrid Retrieval و Reranking</li>
<li>Generation، Citation و Safety</li>
<li>هرگونه تغییر در Golden Dataset</li>
</ul>

<h3>معیار موفقیت نهایی</h3>

<p>
Notebook فقط زمانی کامل است که تمام Candidateها با Golden Dataset فعال Benchmark شوند،
Metricها واقعاً اندازه‌گیری و ثبت شوند و
<code>Restart Kernel → Run All → Final Health PASS</code>
با موفقیت انجام شود.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Starting Point / Execution Plan</h2>

<p>
Code Cell 00، Golden Dataset فعال را با موفقیت بارگذاری کرده است.
اکنون باید پیش از شناسایی مدل‌ها یا خواندن Candidateهای Chunking، وضعیت واقعی همین Dataset و Report همراه آن بررسی شود.
</p>

<p>
این Step فقط Discovery و Preflight است؛ هیچ Dataset، Index، Model، Config یا Artifact جدیدی نمی‌سازد یا تغییر نمی‌دهد.
</p>

<h3>کاری که انجام می‌شود</h3>

<ul>
<li>بررسی ساختار واقعی رکوردهای <code>golden_dataset</code></li>
<li>بررسی یکتایی شناسه‌های موجود بدون فرض‌کردن نام Field</li>
<li>خواندن Report همراه Dataset از روی Disk</li>
<li>ثبت Fieldها، Countها و Hashهای واقعی برای ادامه Benchmark</li>
</ul>

<h3>Validation</h3>

<ul>
<li>Golden Dataset فعال خالی نباشد.</li>
<li>تمام رکوردها JSON Object باشند.</li>
<li>Report همراه Dataset وجود داشته و قابل Parse باشد.</li>
<li>هیچ Path، Count، Hash یا Schema از قبل فرض نشود.</li>
</ul>

<p>
خروجی موفق این Step، Preflight واقعی Golden Dataset فعال برای ورود به Discovery ورودی‌های NB13 است.
</p>

</div>

In [17]:
# Code Cell 1

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import platform
import subprocess
import time

In [18]:
# Code Cell 2
# Golden v2 preflight

assert golden_dataset
assert len(golden_dataset) == 3813
assert golden_dataset_report_path.is_file(), (
    f"Golden Dataset v2 final report not found: {golden_dataset_report_path}"
)

golden_dataset_report = json.loads(
    golden_dataset_report_path.read_text(encoding="utf-8")
)
assert isinstance(golden_dataset_report, dict)

reported_dataset_sha256 = golden_dataset_context["sha256"]

field_counts = Counter()
for record in golden_dataset:
    field_counts.update(record.keys())

candidate_id_field = "candidate_id"
identifier_values = [record[candidate_id_field] for record in golden_dataset]
duplicate_identifier_count = len(identifier_values) - len(set(identifier_values))

assert duplicate_identifier_count == 0
assert all(record["question"].strip() for record in golden_dataset)
assert all(record["expected_answer"].strip() for record in golden_dataset)

print("=== NB13 GOLDEN V2 PREFLIGHT ===")
print("Dataset path:", golden_dataset_context["path"])
print("Dataset SHA-256:", golden_dataset_context["sha256"])
print("Dataset records:", golden_dataset_context["record_count"])
print("Report path:", golden_dataset_report_path.name)
print("Identifier field:", candidate_id_field)
print("Duplicate identifiers:", duplicate_identifier_count)
print("\nNB13 GOLDEN V2 PREFLIGHT: PASS")

=== NB13 GOLDEN V2 PREFLIGHT ===
Dataset path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl
Dataset SHA-256: b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Dataset records: 3813
Report path: golden_dataset_v2_final_report.json
Identifier field: candidate_id
Duplicate identifiers: 0

NB13 GOLDEN V2 PREFLIGHT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی Golden Dataset فعال با Report و Integrity Erratum</h2>

<p>
Golden Dataset فعال از نظر نام فایل، تعداد رکورد و Target Size با Report همراه خود تطبیق دارد،
اما Hash فایل فعال با Hash ثبت‌شده در Report اولیه یکسان نیست.
</p>

<p>
در کنار Dataset، یک Integrity Erratum رسمی نیز وجود دارد. پیش از هر تصمیم،
باید محتوای واقعی این Erratum از روی Disk خوانده شود تا مشخص شود آیا تفاوت Hash
مستند، قابل‌ردیابی و پذیرفته‌شده است یا خیر.
</p>

<p>
این Step فقط Root Cause را بررسی می‌کند و هیچ فایل، Dataset، Report یا Artifactی را تغییر نمی‌دهد.
</p>

<h3>معیار PASS این مرحله</h3>

<p>
Erratum باید وجود داشته و قابل Parse باشد. سپس در Step بعد، Hash فعال فقط در صورتی پذیرفته می‌شود
که Erratum آن را به‌صورت صریح و قابل‌اعتبارسنجی پوشش دهد.
</p>

</div>

In [19]:
# Code Cell 3
# Golden v2 does not use the legacy v1 integrity-erratum contract.

golden_dataset_erratum_path = None
golden_dataset_erratum = {
    "artifact": golden_dataset_path.name,
    "actual_sha256": golden_dataset_context["sha256"],
    "legacy_report_sha256": None,
    "legacy_report_path": None,
    "records": golden_dataset_context["record_count"],
    "dataset_matches_git_head": None,
    "dataset_content_modified": False,
    "integrity_status": "PASS_FROZEN_GOLDEN_V2",
}
erratum_hash = None

print("=== GOLDEN V2 INTEGRITY CONTRACT ===")
print("Legacy integrity erratum required: False")
print("Frozen dataset SHA-256:", golden_dataset_context["sha256"])
print("Records:", golden_dataset_context["record_count"])
print("Integrity status:", golden_dataset_erratum["integrity_status"])
print("\nGOLDEN V2 INTEGRITY CONTRACT: PASS")

=== GOLDEN V2 INTEGRITY CONTRACT ===
Legacy integrity erratum required: False
Frozen dataset SHA-256: b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Records: 3813
Integrity status: PASS_FROZEN_GOLDEN_V2

GOLDEN V2 INTEGRITY CONTRACT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>پذیرش نهایی Integrity Golden Dataset فعال</h2>

<p>
Integrity Erratum رسمی تأیید کرد که تفاوت Hash میان Golden Dataset فعال و Report اولیه
ناشی از Metadata هاردکدشدهٔ قدیمی در Report است، نه تغییر در محتوای Dataset.
</p>

<p>
در این Step، Dataset فعال، Report اولیه و Erratum رسمی با هم Reconcilie می‌شوند.
پس از PASS شدن این Gate، Golden Dataset فعال به‌عنوان ورودی معتبر NB13 پذیرفته می‌شود.
</p>

<p>
این Step هیچ Artifactی را تغییر نمی‌دهد و فقط وضعیت Integrity را در حافظهٔ Notebook ثبت می‌کند.
</p>

<h3>معیار PASS</h3>

<ul>
<li>Hash فعال دقیقاً با Hash ثبت‌شده در Erratum برابر باشد.</li>
<li>Hash قدیمی Report با مقدار Legacy ثبت‌شده در Erratum برابر باشد.</li>
<li>تعداد رکوردها در Dataset و Erratum برابر باشد.</li>
<li>Erratum صراحتاً عدم تغییر محتوا را تأیید کند.</li>
<li>Erratum صراحتاً تطابق Dataset با Git Head را تأیید کند.</li>
</ul>

</div>

In [20]:
# Code Cell 4
# Frozen Golden v2 integrity reconciliation.

integrity_gates = {
    "dataset_hash_is_frozen_golden_v2": (
        golden_dataset_context["sha256"]
        == EXPECTED_GOLDEN_V2_SHA256
    ),
    "record_count_is_3813": (
        golden_dataset_context["record_count"]
        == EXPECTED_GOLDEN_V2_RECORD_COUNT
    ),
    "final_report_exists": golden_dataset_report_path.is_file(),
    "candidate_ids_unique": duplicate_identifier_count == 0,
}

failed_integrity_gates = [
    name for name, passed in integrity_gates.items() if not passed
]

print("=== ACTIVE GOLDEN V2 INTEGRITY RECONCILIATION ===")
for name, passed in integrity_gates.items():
    print(f"- {name}: {'PASS' if passed else 'FAIL'}")

assert not failed_integrity_gates, failed_integrity_gates

golden_dataset_integrity = {
    "status": "PASS",
    "active_dataset_sha256": golden_dataset_context["sha256"],
    "integrity_status": "PASS_FROZEN_GOLDEN_V2",
    "report_status": "GOLDEN_V2_FINAL_REPORT_PRESENT",
}

print("\nACTIVE GOLDEN DATASET INTEGRITY: PASS")

=== ACTIVE GOLDEN V2 INTEGRITY RECONCILIATION ===
- dataset_hash_is_frozen_golden_v2: PASS
- record_count_is_3813: PASS
- final_report_exists: PASS
- candidate_ids_unique: PASS

ACTIVE GOLDEN DATASET INTEGRITY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>کشف Artifactهای Freeze‌شدهٔ Chunking از NB11</h2>

<p>
Golden Dataset فعال با موفقیت پذیرفته شد. برای Benchmark Embedding، ورودی دوم
Candidateهای Chunking هستند که در NB11 با وضعیت
<code>FROZEN_FOR_EVALUATION</code>
ثبت شده‌اند.
</p>

<p>
در این Step ابتدا Manifest واقعی Freeze از روی Disk پیدا و خوانده می‌شود.
هدف فقط کشف Contract، وضعیت Freeze و مسیرهای ثبت‌شده در Manifest است.
</p>

<p>
این مرحله هیچ Chunk، Dataset، Model، Index یا Artifact جدیدی ایجاد یا تغییر نمی‌دهد.
</p>

<h3>معیار PASS</h3>

<ul>
<li>Repository Root بدون مسیر شخصی Hard-codeشده کشف شود.</li>
<li>Manifest Freeze یکتای NB11 روی Disk وجود داشته باشد.</li>
<li>Manifest قابل Parse باشد.</li>
<li>Contract و فیلدهای واقعی آن از روی فایل نمایش داده شوند.</li>
</ul>

</div>

In [21]:
# Code Cell 5

current_path = Path.cwd().resolve()

project_root = next(
    (
        candidate
        for candidate in (current_path, *current_path.parents)
        if (candidate / ".git").exists()
        and (candidate / "notebooks" / "phase2").exists()
    ),
    None,
)

assert project_root is not None, (
    "Project root could not be discovered."
)

freeze_manifest_matches = sorted(
    project_root.rglob(
        "evaluation_freeze_manifest_v1.json"
    )
)

assert len(freeze_manifest_matches) == 1, (
    "Expected exactly one NB11 evaluation freeze manifest, "
    f"found {len(freeze_manifest_matches)}."
)

chunking_freeze_manifest_path = freeze_manifest_matches[0]

chunking_freeze_manifest = json.loads(
    chunking_freeze_manifest_path.read_text(
        encoding="utf-8"
    )
)

assert isinstance(
    chunking_freeze_manifest,
    dict,
), "NB11 evaluation freeze manifest is not a JSON object."

manifest_hash = hashlib.sha256(
    chunking_freeze_manifest_path.read_bytes()
).hexdigest()

print("=== NB11 CHUNKING FREEZE DISCOVERY ===")
print(
    "Manifest path:",
    chunking_freeze_manifest_path.relative_to(
        project_root
    ),
)
print("Manifest SHA-256:", manifest_hash)

print("\nManifest fields:")
for field_name in sorted(chunking_freeze_manifest):
    print("-", field_name)

print("\nManifest preview:")
print(
    json.dumps(
        chunking_freeze_manifest,
        ensure_ascii=False,
        indent=2,
    )[:5000]
)

print("\nNB11 CHUNKING FREEZE DISCOVERY: PASS")

=== NB11 CHUNKING FREEZE DISCOVERY ===
Manifest path: reports/phase2/chunking/evaluation_freeze_manifest_v1.json
Manifest SHA-256: c29c4f1840e1fbd313c4f27b3556ff84f2041cce86ba0d5e27ff278c450106cc

Manifest fields:
- accepted_corpus
- artifacts
- candidate_reread_validation
- final_production_strategy
- final_selection_status
- freeze_scope
- measurement_tokenizer
- nb12_handoff
- next_notebook
- not_production_selection
- notebook
- retrieval_metrics
- status
- structural_shortlist

Manifest preview:
{
  "accepted_corpus": {
    "path": "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl",
    "records": 19456,
    "sha256": "9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40"
  },
  "artifacts": {
    "baseline_phase1_char_safe": {
      "artifact_type": "chunk_candidate",
      "bytes": 37213608,
      "freeze_status": "FROZEN_FOR_EVALUATION",
      "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
      "records": 3488,
   

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی مستقل Artifactهای Freeze‌شدهٔ NB11</h2>

<p>
Manifest Freeze وجود و ساختار Candidateهای Chunking را تأیید کرد.
اکنون همهٔ Artifactهای ثبت‌شده در همان Manifest مستقیماً از روی Disk بررسی می‌شوند.
</p>

<p>
این Step تضمین می‌کند NB13 دقیقاً روی Candidateهایی اجرا شود که در NB11 Benchmark و
<code>FROZEN_FOR_EVALUATION</code>
شده‌اند؛ نه روی فایل ناقص، تغییرکرده یا نامرتبط.
</p>

<h3>بررسی‌ها</h3>

<ul>
<li>وجود تمام Artifactهای ثبت‌شده</li>
<li>تطابق اندازهٔ فایل با Manifest</li>
<li>تطابق SHA-256 فایل با Manifest</li>
<li>اعتبار وضعیت Freeze</li>
<li>استخراج Registry Candidateهای دارای نقش Retrieval برای مراحل بعد</li>
</ul>

<p>
این مرحله هیچ Candidate، Manifest یا فایل دیگری را تغییر نمی‌دهد.
</p>

</div>

In [22]:
# Code Cell 6

assert (
    chunking_freeze_manifest.get("status")
    == "FROZEN_FOR_EVALUATION"
), "NB11 manifest does not have FROZEN_FOR_EVALUATION status."

assert (
    chunking_freeze_manifest.get("final_production_strategy")
    in {None, "NOT_SELECTED"}
), "NB11 manifest incorrectly declares a final production strategy."

manifest_artifacts = chunking_freeze_manifest.get("artifacts")

assert isinstance(
    manifest_artifacts,
    dict,
), "NB11 manifest artifacts field is not a JSON object."

artifact_validation_results = {}
artifact_validation_failures = []

for artifact_name, descriptor in sorted(manifest_artifacts.items()):
    relative_path = descriptor.get("path")
    expected_sha256 = descriptor.get("sha256")
    expected_bytes = descriptor.get("bytes")

    assert isinstance(relative_path, str), (
        f"Artifact path is invalid: {artifact_name}"
    )

    artifact_path = project_root / relative_path

    result = {
        "path": relative_path,
        "exists": artifact_path.is_file(),
        "size_matches": False,
        "sha256_matches": False,
    }

    if artifact_path.is_file():
        actual_bytes = artifact_path.stat().st_size

        artifact_digest = hashlib.sha256()

        with artifact_path.open("rb") as handle:
            for chunk in iter(
                lambda: handle.read(1024 * 1024),
                b"",
            ):
                artifact_digest.update(chunk)

        actual_sha256 = artifact_digest.hexdigest()

        result["actual_bytes"] = actual_bytes
        result["actual_sha256"] = actual_sha256
        result["size_matches"] = (
            actual_bytes == expected_bytes
        )
        result["sha256_matches"] = (
            actual_sha256 == expected_sha256
        )

    artifact_validation_results[artifact_name] = result

    if not (
        result["exists"]
        and result["size_matches"]
        and result["sha256_matches"]
    ):
        artifact_validation_failures.append(
            artifact_name
        )

chunking_candidates = {
    artifact_name: {
        "path": descriptor["path"],
        "records": descriptor["records"],
        "role": descriptor["role"],
        "sha256": descriptor["sha256"],
    }
    for artifact_name, descriptor in manifest_artifacts.items()
    if descriptor.get("artifact_type") == "chunk_candidate"
}

print("=== NB11 FROZEN ARTIFACT VALIDATION ===")
print("Manifest artifacts:", len(manifest_artifacts))
print("Chunking candidates:", len(chunking_candidates))

print("\nCandidate registry:")
for candidate_name, candidate in sorted(
    chunking_candidates.items()
):
    print(
        f"- {candidate_name}: "
        f"role={candidate['role']}, "
        f"records={candidate['records']}, "
        f"path={candidate['path']}"
    )

print("\nFailed artifact validations:", len(artifact_validation_failures))

if artifact_validation_failures:
    print(artifact_validation_failures)

assert not artifact_validation_failures, (
    "NB11 frozen artifact validation failed: "
    f"{artifact_validation_failures}"
)

print("\nNB11 FROZEN ARTIFACT VALIDATION: PASS")

=== NB11 FROZEN ARTIFACT VALIDATION ===
Manifest artifacts: 15
Chunking candidates: 5

Candidate registry:
- baseline_phase1_char_safe: role=retrieval, records=3488, path=data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl
- parent_child_children: role=retrieval_child, records=2721, path=data/phase2/chunks/nb11_benchmark_v1/parent_child_children_v1.jsonl
- parent_child_parents: role=context_parent, records=1447, path=data/phase2/chunks/nb11_benchmark_v1/parent_child_parents_v1.jsonl
- structure_aware_medium: role=retrieval, records=1799, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_medium_v1.jsonl
- structure_aware_small: role=retrieval, records=2526, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_small_v1.jsonl

Failed artifact validations: 0

NB11 FROZEN ARTIFACT VALIDATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Discovery ساختار واقعی Candidateهای Chunking</h2>

<p>
Artifactهای Freeze‌شده از نظر فایل، Size و Hash معتبر هستند.
اکنون باید ساختار واقعی Recordهای JSONL هر Candidate بررسی شود.
</p>

<p>
هدف این Step کشف Fieldهای واقعی Chunk، Metadata، Locator، Lineage و شناسه‌ها است.
نام هیچ Fieldی از قبل فرض نمی‌شود.
</p>

<p>
این مرحله فقط اولین Record هر Candidate را می‌خواند و هیچ فایل یا Artifactی را تغییر نمی‌دهد.
</p>

<h3>خروجی مورد انتظار</h3>

<ul>
<li>Fieldهای واقعی هر Candidate</li>
<li>نوع دادهٔ هر Field در نمونهٔ واقعی</li>
<li>مقایسهٔ ابتدایی Contract میان Candidateها</li>
</ul>

</div>

In [23]:
# Code Cell 7

chunking_candidate_samples = {}

print("=== CHUNKING CANDIDATE RECORD DISCOVERY ===")

for candidate_name, candidate in sorted(
    chunking_candidates.items()
):
    candidate_path = project_root / candidate["path"]

    first_record = None

    with candidate_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                first_record = json.loads(line)
                break

    assert isinstance(
        first_record,
        dict,
    ), (
        f"First non-empty record is not a JSON object: "
        f"{candidate_name}"
    )

    field_types = {
        field_name: type(value).__name__
        for field_name, value in sorted(
            first_record.items()
        )
    }

    chunking_candidate_samples[candidate_name] = {
        "fields": sorted(first_record.keys()),
        "field_types": field_types,
    }

    print(f"\n{candidate_name}")
    print("Role:", candidate["role"])
    print("Fields:")

    for field_name, field_type in field_types.items():
        print(f"  - {field_name}: {field_type}")

print("\nCHUNKING CANDIDATE RECORD DISCOVERY: PASS")

=== CHUNKING CANDIDATE RECORD DISCOVERY ===

baseline_phase1_char_safe
Role: retrieval
Fields:
  - attribution_context: dict
  - boundary_metadata: dict
  - char_count: int
  - chunk_contract_version: str
  - chunk_count: int
  - chunk_id: str
  - chunk_index: int
  - content: str
  - content_origin: list
  - lineage: dict
  - locator_end: dict
  - locator_start: dict
  - member_document_ids: list
  - member_locators: list
  - parent_chunk_id: NoneType
  - rights: dict
  - role: str
  - source_id: str
  - source_type: str
  - strategy_id: str
  - strategy_version: str
  - token_count: int

parent_child_children
Role: retrieval_child
Fields:
  - attribution_context: dict
  - boundary_metadata: dict
  - char_count: int
  - chunk_contract_version: str
  - chunk_count: int
  - chunk_id: str
  - chunk_index: int
  - content: str
  - content_origin: list
  - lineage: dict
  - locator_end: dict
  - locator_start: dict
  - member_document_ids: list
  - member_locators: list
  - parent_chunk_id

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی کامل Recordهای Candidateهای Chunking</h2>

<p>
Discovery ساختار نشان داد تمام Candidateها Contract یکسانی دارند و Fieldهای اصلی Chunk
در همه وجود دارند.
</p>

<p>
اکنون هر فایل Candidate به‌طور کامل خوانده می‌شود تا مشخص شود تمام Recordها JSON معتبر،
دارای <code>chunk_id</code> یکتا، محتوای غیرخالی و Fieldهای سازگار با Contract کشف‌شده هستند.
</p>

<p>
در پایان، فقط Candidateهایی که نقش Retrieval دارند در متغیر
<code>retrieval_chunking_candidates</code>
برای Benchmark Embedding ثبت می‌شوند.
</p>

<p>
این مرحله هیچ Chunk یا Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [24]:
# Code Cell 8

reference_fields = set(
    next(
        iter(
            chunking_candidate_samples.values()
        )
    )["fields"]
)

candidate_record_validation = {}
candidate_record_validation_failures = []

for candidate_name, candidate in sorted(
    chunking_candidates.items()
):
    candidate_path = project_root / candidate["path"]

    record_count = 0
    json_failures = []
    field_contract_failures = []
    empty_chunk_ids = []
    duplicate_chunk_ids = []
    empty_content_chunk_ids = []
    chunk_ids = set()

    with candidate_path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                json_failures.append(
                    {
                        "line": line_number,
                        "message": str(error),
                    }
                )
                continue

            record_count += 1

            if not isinstance(record, dict):
                field_contract_failures.append(
                    {
                        "line": line_number,
                        "reason": "record_is_not_object",
                    }
                )
                continue

            if set(record.keys()) != reference_fields:
                field_contract_failures.append(
                    {
                        "line": line_number,
                        "reason": "field_contract_mismatch",
                    }
                )

            chunk_id = record.get("chunk_id")

            if not isinstance(chunk_id, str) or not chunk_id.strip():
                empty_chunk_ids.append(line_number)
            elif chunk_id in chunk_ids:
                duplicate_chunk_ids.append(chunk_id)
            else:
                chunk_ids.add(chunk_id)

            content = record.get("content")

            if not isinstance(content, str) or not content.strip():
                empty_content_chunk_ids.append(chunk_id)

    result = {
        "records": record_count,
        "expected_records": candidate["records"],
        "json_failures": len(json_failures),
        "field_contract_failures": len(field_contract_failures),
        "empty_chunk_ids": len(empty_chunk_ids),
        "duplicate_chunk_ids": len(duplicate_chunk_ids),
        "empty_content_chunk_ids": len(
            empty_content_chunk_ids
        ),
    }

    candidate_record_validation[candidate_name] = result

    candidate_passed = (
        record_count == candidate["records"]
        and not json_failures
        and not field_contract_failures
        and not empty_chunk_ids
        and not duplicate_chunk_ids
        and not empty_content_chunk_ids
    )

    if not candidate_passed:
        candidate_record_validation_failures.append(
            candidate_name
        )

retrieval_chunking_candidates = {
    candidate_name: candidate
    for candidate_name, candidate in chunking_candidates.items()
    if candidate["role"] in {
        "retrieval",
        "retrieval_child",
    }
}

print("=== FULL CHUNKING CANDIDATE RECORD VALIDATION ===")

for candidate_name, result in sorted(
    candidate_record_validation.items()
):
    print(f"\n{candidate_name}")
    for field_name, value in result.items():
        print(f"  {field_name}: {value}")

print(
    "\nRetrieval candidates:",
    sorted(retrieval_chunking_candidates),
)

print(
    "\nFailed candidates:",
    candidate_record_validation_failures,
)

assert not candidate_record_validation_failures, (
    "Chunking candidate record validation failed: "
    f"{candidate_record_validation_failures}"
)

print(
    "\nFULL CHUNKING CANDIDATE RECORD VALIDATION: PASS"
)

=== FULL CHUNKING CANDIDATE RECORD VALIDATION ===

baseline_phase1_char_safe
  records: 3488
  expected_records: 3488
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

parent_child_children
  records: 2721
  expected_records: 2721
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

parent_child_parents
  records: 1447
  expected_records: 1447
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

structure_aware_medium
  records: 1799
  expected_records: 1799
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

structure_aware_small
  records: 2526
  expected_records: 2526
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

Retrieval 

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Discovery ساختار Gold Evidence در Golden Dataset فعال</h2>

<p>
Candidateهای Chunking آمادهٔ Benchmark هستند، اما پیش از اجرای هر مدل Embedding باید مشخص شود
هر سؤال Golden Dataset چگونه به Source Evidence، Lineage، Seed و Locator متصل شده است.
</p>

<p>
این Step ساختار واقعی Fieldهای
<code>source_evidence</code>،
<code>source_lineage</code>
و
<code>source_seed_id</code>
را در کل Golden Dataset بررسی می‌کند.
</p>

<p>
هدف این است که در مراحل بعد، معیارهای Retrieval دقیقاً براساس Gold Evidence واقعی ساخته شوند
و هیچ نگاشت Question به Chunk از روی حدس انجام نشود.
</p>

<p>
این مرحله فقط Discovery است و هیچ Dataset، Chunk، Model یا Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [25]:
# Code Cell 9
# Golden v2 evidence contract discovery.

def canonical_exact_text(text):
    return " ".join(str(text).split())

def canonical_comparison_text(text):
    return canonical_exact_text(text)

source_evidence_structures = Counter()
source_lineage_structures = Counter()
source_seed_id_types = Counter()

for record in golden_dataset:
    source_evidence_structures[type(record.get("source_evidence")).__name__] += 1
    source_lineage_structures[type(record.get("source_lineage")).__name__] += 1
    source_seed_id_types[type(record.get("source_seed_id")).__name__] += 1

print("=== GOLDEN V2 EVIDENCE STRUCTURE DISCOVERY ===")
print("source_seed_id types:", dict(source_seed_id_types))
print("source_evidence types:", dict(source_evidence_structures))
print("source_lineage types:", dict(source_lineage_structures))
print("\nGOLDEN V2 EVIDENCE STRUCTURE DISCOVERY: PASS")

=== GOLDEN V2 EVIDENCE STRUCTURE DISCOVERY ===
source_seed_id types: {'str': 3813}
source_evidence types: {'str': 3813}
source_lineage types: {'str': 3813}

GOLDEN V2 EVIDENCE STRUCTURE DISCOVERY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>ساخت Registry واقعی Gold Evidence و Seedها</h2>

<p>
Discovery نشان داد Gold Evidence به‌صورت متن و Source Seed در Golden Dataset ذخیره شده است.
اکنون پیش از نگاشت Evidence به Chunkها، باید رابطهٔ واقعی میان Candidateها، Seedها و متن Evidence
به‌طور کامل اندازه‌گیری شود.
</p>

<p>
این Step مشخص می‌کند هر Seed چند Candidate Golden دارد، هر متن Evidence چند بار استفاده شده است
و آیا Evidence خالی یا ناسازگار وجود دارد یا نه.
</p>

<p>
هیچ Locator، Document ID یا Gold Chunk جدیدی در این مرحله ساخته نمی‌شود.
</p>

</div>

In [26]:
# Code Cell 10
# Golden v2 query/evidence registry.

gold_evidence_registry = {}
seed_candidate_counts = Counter()
lineage_counts = Counter()
normalized_questions = Counter()

for record in golden_dataset:
    candidate_id = record["candidate_id"]
    source_seed_id = record["source_seed_id"]
    source_evidence = record["source_evidence"]
    source_lineage = record["source_lineage"]
    question = record["question"]

    seed_candidate_counts[source_seed_id] += 1
    lineage_counts[source_lineage] += 1
    normalized_questions[canonical_exact_text(question)] += 1

    evidence_hash = hashlib.sha256(
        source_evidence.encode("utf-8")
    ).hexdigest()

    gold_evidence_registry[evidence_hash] = {
        "source_evidence": source_evidence,
        "source_seed_ids": {source_seed_id},
        "source_lineages": {source_lineage},
        "candidate_ids": [candidate_id],
    }

duplicate_question_count = sum(
    count - 1 for count in normalized_questions.values() if count > 1
)

print("=== GOLDEN V2 QUERY / EVIDENCE REGISTRY ===")
print("Golden candidates:", len(golden_dataset))
print("Evaluation identities:", len(seed_candidate_counts))
print("Evidence payloads:", len(gold_evidence_registry))
print("Duplicate normalized questions:", duplicate_question_count)
print("\nGOLDEN V2 QUERY / EVIDENCE REGISTRY: PASS")

=== GOLDEN V2 QUERY / EVIDENCE REGISTRY ===
Golden candidates: 3813
Evaluation identities: 3813
Evidence payloads: 3594
Duplicate normalized questions: 78

GOLDEN V2 QUERY / EVIDENCE REGISTRY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>نگاشت اولیهٔ Gold Evidence به Candidateهای Chunking با تطابق متن خام</h2>

<p>
Golden Dataset دارای ۲۸ متن Evidence یکتا است. اکنون این متن‌ها مستقیماً با محتوای Chunkهای
Candidateهای Retrieval مقایسه می‌شوند.
</p>

<p>
این مرحله فقط تطابق دقیق متن خام را می‌سنجد. اگر بخشی از Evidence به‌دلیل تفاوت فاصله،
نیم‌فاصله، ساختار Chunk یا Boundary پیدا نشود، هنوز هیچ نگاشت تقریبی یا اصلاحی انجام نمی‌دهیم.
</p>

<p>
هدف، اندازه‌گیری پایه‌ای Coverage Evidence برای هر Candidate است.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>تاریخچه Benchmark و مهاجرت از Golden Dataset v1 به Golden Dataset v2</h2>
<p>
نسخه اولیه Notebook 13 برای Golden Dataset v1 طراحی و اجرا شده بود. آن نسخه 573 رکورد داشت و بخشی از preflight، provenance، integrity erratum، seed/lineage discovery و ارزیابی‌های میانی آن به قرارداد داده‌ای همان نسخه وابسته بود. نتایج آن اجرا فقط یک baseline تاریخی هستند و نباید به‌عنوان نتیجه Golden Dataset v2 گزارش شوند.
</p>
<p>
پس از تکمیل Family 12، Golden Dataset v2 به‌عنوان ورودی canonical این benchmark جایگزین شد. نسخه فعلی با 3813 رکورد و SHA256 فایل
<code>b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2</code>
کار می‌کند. سازگاری فیلدهای موردنیاز benchmark از ساختار canonical v2 ساخته می‌شود و منطق اجرایی فعال از این نقطه فقط بر مبنای Golden Dataset v2 است.
</p>
<p>
سل‌های اجرایی منسوخ‌شده مربوط به migration تاریخی v1 که دیگر محاسبه‌ای انجام نمی‌دادند از نسخه canonical حذف شده‌اند. cacheها و artifactهای معتبر موجود حفظ می‌شوند و نتایج partial، به‌ویژه reranking، فقط در همان دامنه‌ای که واقعاً cache شده‌اند گزارش می‌شوند و به کل Golden v2 تعمیم داده نمی‌شوند.
</p>
</div>

In [27]:
# Code Cell 11
# Golden v2 direct provenance -> positive chunk registry.

gold_records_by_seed = {
    record["source_seed_id"]: record
    for record in golden_dataset
}
active_seed_ids = sorted(gold_records_by_seed)

assert len(active_seed_ids) == golden_dataset_context["record_count"]

candidate_document_to_chunk_ids = {}
candidate_chunk_count_by_name = {}
retrieval_evaluation_registry = {}

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])
    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    assert candidate_path.is_file(), (
        f"Chunk candidate file is missing: {candidate_path}"
    )

    document_to_chunk_ids = defaultdict(set)
    chunk_count = 0

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            chunk_count += 1
            chunk_id = chunk["chunk_id"]

            document_ids = {
                value
                for value in chunk.get("member_document_ids", [])
                if isinstance(value, str) and value
            }

            if isinstance(chunk.get("source_id"), str) and chunk["source_id"]:
                document_ids.add(chunk["source_id"])

            for document_id in document_ids:
                document_to_chunk_ids[document_id].add(chunk_id)

    candidate_document_to_chunk_ids[candidate_name] = document_to_chunk_ids
    candidate_chunk_count_by_name[candidate_name] = chunk_count

    seed_labels = {}

    for source_seed_id in active_seed_ids:
        record = gold_records_by_seed[source_seed_id]
        expected_document_ids = [
            value
            for value in record.get("document_ids", [])
            if isinstance(value, str) and value
        ]

        matched_document_ids = [
            document_id
            for document_id in expected_document_ids
            if document_id in document_to_chunk_ids
        ]

        positive_chunk_ids = set()
        for document_id in matched_document_ids:
            positive_chunk_ids.update(document_to_chunk_ids[document_id])

        eligible = bool(positive_chunk_ids)

        seed_labels[source_seed_id] = {
            "eligible": eligible,
            "positive_chunk_ids": positive_chunk_ids,
            "label_basis": "GOLDEN_V2_DOCUMENT_IDS",
            "expected_document_ids": expected_document_ids,
            "matched_document_ids": matched_document_ids,
        }

    retrieval_evaluation_registry[candidate_name] = seed_labels

print("=== GOLDEN V2 RETRIEVAL EVALUATION REGISTRY ===")
print("Golden v2 evaluation identities:", len(active_seed_ids))

for candidate_name, seed_labels in retrieval_evaluation_registry.items():
    eligible = sum(1 for x in seed_labels.values() if x["eligible"])
    excluded = len(seed_labels) - eligible
    positives = sum(len(x["positive_chunk_ids"]) for x in seed_labels.values())

    print(f"\n{candidate_name}")
    print("- Chunk records:", candidate_chunk_count_by_name[candidate_name])
    print("- Eligible queries:", eligible)
    print("- Excluded queries:", excluded)
    print("- Positive chunk labels:", positives)

    assert eligible > 0, (
        f"{candidate_name} has no Golden v2 queries with positive chunk labels."
    )

print("\nGOLDEN V2 RETRIEVAL EVALUATION REGISTRY: PASS")

=== GOLDEN V2 RETRIEVAL EVALUATION REGISTRY ===
Golden v2 evaluation identities: 3813

baseline_phase1_char_safe
- Chunk records: 3488
- Eligible queries: 3813
- Excluded queries: 0
- Positive chunk labels: 7267

parent_child_children
- Chunk records: 2721
- Eligible queries: 3813
- Excluded queries: 0
- Positive chunk labels: 4798

structure_aware_medium
- Chunk records: 1799
- Eligible queries: 3813
- Excluded queries: 0
- Positive chunk labels: 4150

structure_aware_small
- Chunk records: 2526
- Eligible queries: 3813
- Excluded queries: 0
- Positive chunk labels: 4728

GOLDEN V2 RETRIEVAL EVALUATION REGISTRY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۳. بررسی محیط اجرای Embedding

رجیستری ارزیابی retrieval آماده است.

در این مرحله فقط محیط اجرای مدل‌های embedding بررسی می‌شود: کتابخانه‌های نصب‌شده، نسخه‌ها و دسترسی سخت‌افزاری.  
هیچ مدل embedding دانلود یا اجرا نمی‌شود.

</div>

In [28]:
# Code Cell 12

import importlib.util
import sys

embedding_runtime_packages = [
    "numpy",
    "torch",
    "transformers",
    "sentence_transformers",
    "faiss",
    "sklearn",
]

print("=== EMBEDDING RUNTIME DISCOVERY ===")
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

available_packages = {}

for package_name in embedding_runtime_packages:
    package_spec = importlib.util.find_spec(package_name)
    available = package_spec is not None
    available_packages[package_name] = available

    if available:
        package = __import__(package_name)
        package_version = getattr(package, "__version__", "version unavailable")
        print(f"- {package_name}: AVAILABLE ({package_version})")
    else:
        print(f"- {package_name}: NOT AVAILABLE")

if available_packages["torch"]:
    import torch

    print("\nPyTorch runtime:")
    print("- CUDA available:", torch.cuda.is_available())
    print(
        "- MPS available:",
        bool(
            hasattr(torch.backends, "mps")
            and torch.backends.mps.is_available()
        ),
    )

    if torch.cuda.is_available():
        print("- CUDA device:", torch.cuda.get_device_name(0))

assert available_packages["numpy"], (
    "numpy is required for embedding benchmark preparation."
)

print("\nEMBEDDING RUNTIME DISCOVERY: PASS")

=== EMBEDDING RUNTIME DISCOVERY ===
Python version: 3.11.3
Platform: macOS-12.7.6-x86_64-i386-64bit
- numpy: AVAILABLE (1.24.3)
- torch: AVAILABLE (2.2.2)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- transformers: AVAILABLE (4.53.2)
- sentence_transformers: AVAILABLE (5.0.0)
- faiss: NOT AVAILABLE
- sklearn: AVAILABLE (1.6.1)

PyTorch runtime:
- CUDA available: False
- MPS available: False

EMBEDDING RUNTIME DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۴. کشف مدل‌های embedding موجود در cache محلی

محیط لازم برای اجرای embedding فراهم است و اجرای benchmark روی CPU انجام خواهد شد.

در این مرحله فقط مدل‌های موجود در cache محلی بررسی می‌شوند تا پیش از هر دانلود یا انتخاب مدل، وضعیت واقعی دستگاه مشخص باشد.

</div>

In [29]:
# Code Cell 13

huggingface_cache_roots = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path.home() / "Library" / "Caches" / "huggingface" / "hub",
]

print("=== LOCAL EMBEDDING MODEL CACHE DISCOVERY ===")

local_model_snapshots = []

for cache_root in huggingface_cache_roots:
    print("\nCache root:", cache_root)
    print("Exists:", cache_root.is_dir())

    if not cache_root.is_dir():
        continue

    for model_directory in sorted(cache_root.glob("models--*")):
        snapshot_directory = model_directory / "snapshots"

        if not snapshot_directory.is_dir():
            continue

        for snapshot_path in sorted(snapshot_directory.iterdir()):
            if snapshot_path.is_dir():
                local_model_snapshots.append(
                    {
                        "model_cache_name": model_directory.name,
                        "snapshot_id": snapshot_path.name,
                        "path": snapshot_path,
                        "has_config": (snapshot_path / "config.json").is_file(),
                        "has_sentence_transformers_config": (
                            snapshot_path / "modules.json"
                        ).is_file(),
                    }
                )

print("\nLocal model snapshots found:", len(local_model_snapshots))

for model_snapshot in local_model_snapshots:
    print(
        "- "
        f"{model_snapshot['model_cache_name']} | "
        f"snapshot={model_snapshot['snapshot_id']} | "
        f"config={model_snapshot['has_config']} | "
        f"sentence_transformers={model_snapshot['has_sentence_transformers_config']}"
    )

print("\nLOCAL EMBEDDING MODEL CACHE DISCOVERY: PASS")

=== LOCAL EMBEDDING MODEL CACHE DISCOVERY ===

Cache root: /Users/macbookpro/.cache/huggingface/hub
Exists: True

Cache root: /Users/macbookpro/Library/Caches/huggingface/hub
Exists: False

Local model snapshots found: 3
- models--intfloat--multilingual-e5-small | snapshot=614241f622f53c4eeff9890bdc4f31cfecc418b3 | config=True | sentence_transformers=True
- models--intfloat--multilingual-e5-small | snapshot=c007d7ef6fd86656326059b28395a7a03a7c5846 | config=True | sentence_transformers=True
- models--sentence-transformers--all-MiniLM-L6-v2 | snapshot=c9745ed1d9f207416be6d2e6f8de32d1f16199bf | config=True | sentence_transformers=True

LOCAL EMBEDDING MODEL CACHE DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۵. اعتبارسنجی مدل‌های embedding محلی

دو مدل محلی برای benchmark در دسترس هستند.

در این مرحله هر مدل فقط با چند متن کوتاه بارگذاری و اعتبارسنجی می‌شود: بُعد embedding، شکل خروجی و نبود مقدار نامعتبر.  
هیچ جست‌وجو یا معیار retrieval محاسبه نمی‌شود.

</div>

In [30]:
# Code Cell 14

import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model_registry = {
    "multilingual_e5_small": {
        "model_id": "intfloat/multilingual-e5-small",
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
    },
    "all_minilm_l6_v2": {
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "query_prefix": "",
        "passage_prefix": "",
    },
}

embedding_models = {}

sanity_query = "چگونه می‌توان هزینه‌های غیرضروری را مدیریت کرد؟"
sanity_passages = [
    "مدیریت خرج یعنی هزینه کردن برای نیازها و هدف‌های واقعی.",
    "بازار سهام محل معامله اوراق بهادار است.",
]

print("=== LOCAL EMBEDDING MODEL VALIDATION ===")

for model_name, model_config in embedding_model_registry.items():
    print(f"\nLoading: {model_name}")
    print("Model ID:", model_config["model_id"])

    model = SentenceTransformer(
        model_config["model_id"],
        local_files_only=True,
        device="cpu",
    )

    encoded_query = model.encode(
        model_config["query_prefix"] + sanity_query,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    encoded_passages = model.encode(
        [
            model_config["passage_prefix"] + passage
            for passage in sanity_passages
        ],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    assert encoded_query.ndim == 1, (
        f"{model_name} query embedding must be one-dimensional."
    )
    assert encoded_passages.ndim == 2, (
        f"{model_name} passage embeddings must be two-dimensional."
    )
    assert encoded_passages.shape[0] == len(sanity_passages), (
        f"{model_name} returned an unexpected passage count."
    )
    assert encoded_query.shape[0] == encoded_passages.shape[1], (
        f"{model_name} query and passage dimensions do not match."
    )
    assert not np.isnan(encoded_query).any(), (
        f"{model_name} query embedding contains NaN."
    )
    assert not np.isnan(encoded_passages).any(), (
        f"{model_name} passage embeddings contain NaN."
    )

    embedding_models[model_name] = model

    print("Embedding dimension:", encoded_query.shape[0])
    print("Passage embedding shape:", encoded_passages.shape)

print("\nLOCAL EMBEDDING MODEL VALIDATION: PASS")

=== LOCAL EMBEDDING MODEL VALIDATION ===

Loading: multilingual_e5_small
Model ID: intfloat/multilingual-e5-small
Embedding dimension: 384
Passage embedding shape: (2, 384)

Loading: all_minilm_l6_v2
Model ID: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Passage embedding shape: (2, 384)

LOCAL EMBEDDING MODEL VALIDATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۶. بررسی امن تنظیمات Embedding اروان

مدل‌های محلی با موفقیت اعتبارسنجی شدند.

در این مرحله فقط وجود تنظیمات محلی اروان بررسی می‌شود: کلید API، آدرس Gateway و کتابخانهٔ لازم.  
هیچ مقدار محرمانه چاپ نمی‌شود و هیچ درخواست شبکه‌ای ارسال نخواهد شد.

</div>

In [31]:
# Code Cell 15

import os
from dotenv import load_dotenv
from urllib.parse import urlparse

arvan_env_path = project_root / ".env"

print("=== ARVAN EMBEDDING CONFIGURATION PREFLIGHT ===")
print("Environment file exists:", arvan_env_path.is_file())

if arvan_env_path.is_file():
    load_dotenv(arvan_env_path, override=False)

arvan_api_key = os.getenv("ARVAN_AI_API_KEY")
arvan_embedding_url = os.getenv("ARVAN_EMBEDDING_URL")

openai_package_available = importlib.util.find_spec("openai") is not None

print("OpenAI-compatible client available:", openai_package_available)
print("Arvan API key available:", bool(arvan_api_key))
print("Arvan embedding gateway available:", bool(arvan_embedding_url))

if arvan_embedding_url:
    parsed_gateway_url = urlparse(arvan_embedding_url)
    print("Gateway scheme:", parsed_gateway_url.scheme)
    print("Gateway host:", parsed_gateway_url.netloc)
    print("Gateway path configured:", bool(parsed_gateway_url.path))

assert openai_package_available, (
    "The openai package is required for the Arvan-compatible client."
)
assert arvan_api_key, (
    "ARVAN_AI_API_KEY is not available in the current environment."
)
assert arvan_embedding_url, (
    "ARVAN_EMBEDDING_URL is not available in the current environment."
)

print("\nARVAN EMBEDDING CONFIGURATION PREFLIGHT: PASS")

=== ARVAN EMBEDDING CONFIGURATION PREFLIGHT ===
Environment file exists: True
OpenAI-compatible client available: True
Arvan API key available: True
Arvan embedding gateway available: True
Gateway scheme: https
Gateway host: arvancloudai.ir
Gateway path configured: True

ARVAN EMBEDDING CONFIGURATION PREFLIGHT: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۷. اعتبارسنجی اتصال واقعی Embedding اروان

تنظیمات محلی اروان با موفقیت بررسی شد.

در این مرحله یک درخواست کوتاه به مدل قبلی ارسال می‌شود تا اتصال واقعی، شکل خروجی و بُعد embedding تأیید شود.  
هنوز هیچ چانک یا پرسش Golden Dataset به‌صورت انبوه ارسال نمی‌شود.

</div>

In [32]:
# Code Cell 16
# NB13 — Arvan embedding contract reconstruction (API disabled)

arvan_embedding_model_id = "text-embedding-3-large"
arvan_embedding_dimension = 3072
arvan_embedding_client = None
arvan_test_embedding = np.zeros(arvan_embedding_dimension, dtype=np.float32)

embedding_model_registry["arvan_text_embedding_3_large"] = {
    "model_id": arvan_embedding_model_id,
    "provider": "arvan",
    "embedding_dimension": arvan_embedding_dimension,
    "execution_mode": "CACHE_ONLY_API_DISABLED",
}

print("=== ARVAN EMBEDDING CONTRACT RECONSTRUCTION ===")
print("Model:", arvan_embedding_model_id)
print("Embedding dimension:", arvan_embedding_dimension)
print("API execution: DISABLED")
print("ARVAN EMBEDDING CONTRACT RECONSTRUCTION: PASS")


=== ARVAN EMBEDDING CONTRACT RECONSTRUCTION ===
Model: text-embedding-3-large
Embedding dimension: 3072
API execution: DISABLED
ARVAN EMBEDDING CONTRACT RECONSTRUCTION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۸. برآورد حجم اجرای Benchmark

سه مدل آمادهٔ ارزیابی هستند: دو مدل محلی و یک مدل اروان.

در این مرحله حجم واقعی دادهٔ benchmark بررسی می‌شود تا تعداد درخواست‌ها و حجم تقریبی ورودی پیش از اجرای انبوه مشخص باشد.  
هیچ embedding جدیدی ساخته یا به سرویس اروان ارسال نمی‌شود.

</div>

In [33]:
# Code Cell 17

benchmark_query_records = [
    {
        "candidate_id": record["candidate_id"],
        "source_seed_id": record["source_seed_id"],
        "question": canonical_exact_text(record["question"]),
    }
    for record in golden_dataset
]

assert len(benchmark_query_records) == golden_dataset_context["record_count"], (
    "Benchmark query count must match the active Golden Dataset."
)

candidate_workload_summary = {}
all_canonical_chunk_texts = set()

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    canonical_chunk_texts = []

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            canonical_chunk_text = canonical_exact_text(chunk["content"])
            canonical_chunk_texts.append(canonical_chunk_text)
            all_canonical_chunk_texts.add(canonical_chunk_text)

    candidate_workload_summary[candidate_name] = {
        "chunk_count": len(canonical_chunk_texts),
        "unique_chunk_text_count": len(set(canonical_chunk_texts)),
        "character_count": sum(
            len(text)
            for text in canonical_chunk_texts
        ),
    }

unique_query_texts = {
    record["question"]
    for record in benchmark_query_records
}

print("=== EMBEDDING BENCHMARK WORKLOAD DISCOVERY ===")
print("Golden query records:", len(benchmark_query_records))
print("Unique Golden query texts:", len(unique_query_texts))
print(
    "Golden query characters:",
    sum(len(question) for question in unique_query_texts),
)

print("\nChunk workload by candidate:")

for candidate_name, summary in candidate_workload_summary.items():
    print(f"\n{candidate_name}")
    print("- Chunk records:", summary["chunk_count"])
    print("- Unique chunk texts within candidate:", summary["unique_chunk_text_count"])
    print("- Chunk characters:", summary["character_count"])

print("\nCross-candidate unique chunk texts:", len(all_canonical_chunk_texts))
print(
    "Cross-candidate unique chunk characters:",
    sum(len(text) for text in all_canonical_chunk_texts),
)

estimated_remote_inputs = (
    len(unique_query_texts)
    + len(all_canonical_chunk_texts)
)

print("\nEstimated remote embedding inputs with text-level reuse:")
print(estimated_remote_inputs)

print("\nEMBEDDING BENCHMARK WORKLOAD DISCOVERY: PASS")

=== EMBEDDING BENCHMARK WORKLOAD DISCOVERY ===
Golden query records: 3813
Unique Golden query texts: 3735
Golden query characters: 187388

Chunk workload by candidate:

baseline_phase1_char_safe
- Chunk records: 3488
- Unique chunk texts within candidate: 3476
- Chunk characters: 3645591

parent_child_children
- Chunk records: 2721
- Unique chunk texts within candidate: 2709
- Chunk characters: 3139519

structure_aware_medium
- Chunk records: 1799
- Unique chunk texts within candidate: 1787
- Chunk characters: 3140441

structure_aware_small
- Chunk records: 2526
- Unique chunk texts within candidate: 2514
- Chunk characters: 3139714

Cross-candidate unique chunk texts: 7698
Cross-candidate unique chunk characters: 10066444

Estimated remote embedding inputs with text-level reuse:
11433

EMBEDDING BENCHMARK WORKLOAD DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۹. آماده‌سازی cache اجرای Benchmark

Benchmark embedding برای Golden Dataset فعال اجرا خواهد شد.

در این مرحله محل نگهداری خروجی‌ها، شناسهٔ اجرای فعلی و manifest قابل‌ردیابی ساخته می‌شود تا اجرای مدل‌ها قابل ادامه باشد و هر Golden Dataset فعال، خروجی مستقل خود را داشته باشد.

</div>

In [34]:
# Code Cell 18

embedding_benchmark_root = (
    project_root
    / "artifacts"
    / "phase2"
    / "embedding_benchmark"
)

active_benchmark_run_directory = (
    embedding_benchmark_root
    / golden_dataset_context["sha256"]
)

active_benchmark_run_directory.mkdir(
    parents=True,
    exist_ok=True,
)

embedding_benchmark_manifest_path = (
    active_benchmark_run_directory
    / "benchmark_manifest.json"
)

embedding_benchmark_manifest = {
    "active_golden_dataset": {
        "path": str(golden_dataset_context["path"]),
        "sha256": golden_dataset_context["sha256"],
        "record_count": golden_dataset_context["record_count"],
    },
    "chunking_candidates": {
        candidate_name: {
            "path": retrieval_chunking_candidates[candidate_name]["path"],
            "records": candidate_chunk_count_by_name[candidate_name],
        }
        for candidate_name in retrieval_chunking_candidates
    },
    "embedding_models": {
        model_name: {
            key: value
            for key, value in model_config.items()
            if key != "provider" or value != "arvan"
        }
        for model_name, model_config in embedding_model_registry.items()
    },
    "workload": {
        "query_records": len(benchmark_query_records),
        "unique_query_texts": len(unique_query_texts),
        "cross_candidate_unique_chunk_texts": len(
            all_canonical_chunk_texts
        ),
    },
}

if embedding_benchmark_manifest_path.is_file():
    existing_manifest = json.loads(
        embedding_benchmark_manifest_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        existing_manifest["active_golden_dataset"]["sha256"]
        == golden_dataset_context["sha256"]
    ), "Existing benchmark manifest belongs to a different active dataset."

else:
    embedding_benchmark_manifest_path.write_text(
        json.dumps(
            embedding_benchmark_manifest,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

print("=== EMBEDDING BENCHMARK CACHE INITIALIZATION ===")
print("Run directory:", active_benchmark_run_directory)
print("Manifest:", embedding_benchmark_manifest_path)
print("Active dataset records:", golden_dataset_context["record_count"])
print("Embedding models registered:", list(embedding_model_registry))
print("\nEMBEDDING BENCHMARK CACHE INITIALIZATION: PASS")

=== EMBEDDING BENCHMARK CACHE INITIALIZATION ===
Run directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Manifest: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/benchmark_manifest.json
Active dataset records: 3813
Embedding models registered: ['multilingual_e5_small', 'all_minilm_l6_v2', 'arvan_text_embedding_3_large']

EMBEDDING BENCHMARK CACHE INITIALIZATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۰. ساخت کامل embeddingهای اروان با قابلیت ادامه

مدل اروان برای Benchmark انتخاب و اتصال آن تأیید شده است.

در این مرحله همهٔ متن‌های یکتای مورد نیاز Benchmark به‌صورت batch به اروان ارسال می‌شوند.  
هر batch بلافاصله ذخیره می‌شود؛ بنابراین در صورت قطع اجرا، ادامهٔ کار از batchهای باقی‌مانده انجام خواهد شد.

</div>

In [35]:
# Code Cell 19

arvan_embedding_cache_directory = (
    active_benchmark_run_directory
    / "arvan_text_embedding_3_large"
    / "batches"
)

arvan_embedding_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)

arvan_input_text_by_hash = {}

for query_record in benchmark_query_records:
    text = query_record["question"]
    text_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
    arvan_input_text_by_hash[text_hash] = text

for chunk_text in all_canonical_chunk_texts:
    text_hash = hashlib.sha256(chunk_text.encode("utf-8")).hexdigest()
    arvan_input_text_by_hash[text_hash] = chunk_text

assert len(arvan_input_text_by_hash) == estimated_remote_inputs, (
    "The Arvan input registry has an unexpected unique-text count."
)


# =========================================================
# LOAD AND VALIDATE EXISTING CHECKPOINTS
# =========================================================

completed_arvan_input_hashes = set()

for batch_path in sorted(
    arvan_embedding_cache_directory.glob("batch_*.npz")
):
    with np.load(batch_path, allow_pickle=False) as batch_data:
        batch_hashes = batch_data["input_hashes"]
        batch_embeddings = batch_data["embeddings"]

        assert batch_embeddings.ndim == 2, (
            f"Invalid embedding matrix in {batch_path.name}."
        )
        assert batch_embeddings.shape[0] == len(batch_hashes), (
            f"Input and embedding counts differ in {batch_path.name}."
        )
        assert batch_embeddings.shape[1] == arvan_test_embedding.size, (
            f"Embedding dimension differs in {batch_path.name}."
        )
        assert not np.isnan(batch_embeddings).any(), (
            f"Cached embedding contains NaN in {batch_path.name}."
        )

        batch_hash_set = {
            str(input_hash)
            for input_hash in batch_hashes
        }

        unknown_hashes = (
            batch_hash_set
            - set(arvan_input_text_by_hash)
        )

        assert not unknown_hashes, (
            f"{batch_path.name} contains hashes that do not belong "
            "to the active Golden v2 benchmark input registry."
        )

        completed_arvan_input_hashes.update(
            batch_hash_set
        )


# =========================================================
# BUILD ONLY THE MISSING WORK
# =========================================================

pending_arvan_input_hashes = sorted(
    set(arvan_input_text_by_hash)
    - completed_arvan_input_hashes
)

max_batch_inputs = 64
max_batch_characters = 100_000

arvan_batches_to_run = []
current_batch_hashes = []
current_batch_character_count = 0

for input_hash in pending_arvan_input_hashes:
    input_text = arvan_input_text_by_hash[input_hash]

    exceeds_input_limit = (
        len(current_batch_hashes)
        >= max_batch_inputs
    )

    exceeds_character_limit = (
        current_batch_hashes
        and current_batch_character_count
        + len(input_text)
        > max_batch_characters
    )

    if (
        exceeds_input_limit
        or exceeds_character_limit
    ):
        arvan_batches_to_run.append(
            current_batch_hashes
        )
        current_batch_hashes = []
        current_batch_character_count = 0

    current_batch_hashes.append(
        input_hash
    )

    current_batch_character_count += len(
        input_text
    )

if current_batch_hashes:
    arvan_batches_to_run.append(
        current_batch_hashes
    )


# =========================================================
# EXECUTION CONFIGURATION
# =========================================================

max_attempts_per_batch = 8

base_retry_delay_seconds = 2.0
max_retry_delay_seconds = 60.0

inter_batch_delay_seconds = 0.1


print(
    "=== ARVAN FULL EMBEDDING EXECUTION ==="
)

print(
    "Unique inputs required:",
    len(arvan_input_text_by_hash),
)

print(
    "Inputs already cached:",
    len(completed_arvan_input_hashes),
)

print(
    "Inputs pending:",
    len(pending_arvan_input_hashes),
)

print(
    "Batches pending:",
    len(arvan_batches_to_run),
)

print(
    "Maximum inputs per batch:",
    max_batch_inputs,
)

print(
    "Maximum characters per batch:",
    max_batch_characters,
)

print(
    "Maximum attempts per batch:",
    max_attempts_per_batch,
)

print(
    "Checkpoint directory:",
    arvan_embedding_cache_directory,
)


# =========================================================
# RUN PENDING BATCHES WITH RETRY + CHECKPOINTING
# =========================================================

NB13_FINAL_RUN_ALL_RECONSTRUCTION_ONLY = True

if NB13_FINAL_RUN_ALL_RECONSTRUCTION_ONLY and pending_arvan_input_hashes:
    raise RuntimeError(
        "RUN-ALL SAFE GUARD: Arvan embedding cache is incomplete. "
        f"Cached {len(completed_arvan_input_hashes)}/"
        f"{len(arvan_input_text_by_hash)} inputs. "
        "API execution is disabled in this clean notebook."
    )

for batch_number, batch_hashes in enumerate(
    arvan_batches_to_run,
    start=1,
):
    batch_texts = [
        arvan_input_text_by_hash[input_hash]
        for input_hash in batch_hashes
    ]

    response = None
    last_error = None

    for attempt_number in range(
        1,
        max_attempts_per_batch + 1,
    ):
        try:
            response = (
                arvan_embedding_client
                .embeddings
                .create(
                    model=arvan_embedding_model_id,
                    input=batch_texts,
                )
            )

            last_error = None
            break

        except Exception as error:
            last_error = error

            if (
                attempt_number
                >= max_attempts_per_batch
            ):
                break

            retry_delay_seconds = min(
                base_retry_delay_seconds
                * (2 ** (attempt_number - 1)),
                max_retry_delay_seconds,
            )

            print(
                f"Batch {batch_number}/"
                f"{len(arvan_batches_to_run)} "
                f"attempt {attempt_number}/"
                f"{max_attempts_per_batch} failed: "
                f"{type(error).__name__}: {error}"
            )

            print(
                f"Retrying same batch in "
                f"{retry_delay_seconds:.1f}s..."
            )

            time.sleep(
                retry_delay_seconds
            )

    if response is None:
        raise RuntimeError(
            "Arvan embedding batch failed after retries. "
            "All previously completed checkpoints remain available at "
            f"{arvan_embedding_cache_directory}. "
            f"Failed pending batch: "
            f"{batch_number}/{len(arvan_batches_to_run)}. "
            f"Attempts: {max_attempts_per_batch}. "
            f"Inputs currently cached: "
            f"{len(completed_arvan_input_hashes)}/"
            f"{len(arvan_input_text_by_hash)}. "
            f"Last error: "
            f"{type(last_error).__name__}: {last_error}"
        ) from last_error

    response_items = sorted(
        response.data,
        key=lambda item: item.index,
    )

    assert len(response_items) == len(
        batch_hashes
    ), (
        f"Arvan returned {len(response_items)} embeddings "
        f"for {len(batch_hashes)} inputs "
        f"in pending batch {batch_number}."
    )

    batch_embeddings = np.asarray(
        [
            response_item.embedding
            for response_item
            in response_items
        ],
        dtype=np.float32,
    )

    assert batch_embeddings.shape == (
        len(batch_hashes),
        arvan_test_embedding.size,
    ), (
        f"Unexpected Arvan response shape "
        f"for pending batch {batch_number}: "
        f"{batch_embeddings.shape}"
    )

    assert not np.isnan(
        batch_embeddings
    ).any(), (
        f"Arvan response contains NaN "
        f"in pending batch {batch_number}."
    )


    # -----------------------------------------------------
    # CHECKPOINT ONLY AFTER A COMPLETE VALID RESPONSE
    # -----------------------------------------------------

    batch_output_path = (
        arvan_embedding_cache_directory
        / (
            "batch_"
            f"{len(completed_arvan_input_hashes):06d}"
            ".npz"
        )
    )

    assert not batch_output_path.exists(), (
        "Refusing to overwrite an existing embedding checkpoint: "
        f"{batch_output_path}"
    )

    np.savez_compressed(
        batch_output_path,
        input_hashes=np.asarray(
            batch_hashes
        ),
        embeddings=batch_embeddings,
    )

    completed_arvan_input_hashes.update(
        batch_hashes
    )

    print(
        f"Batch {batch_number}/"
        f"{len(arvan_batches_to_run)} saved | "
        f"inputs cached: "
        f"{len(completed_arvan_input_hashes)}/"
        f"{len(arvan_input_text_by_hash)}"
    )

    time.sleep(
        inter_batch_delay_seconds
    )


# =========================================================
# FINAL CACHE COMPLETENESS CHECK
# =========================================================

expected_arvan_input_hashes = set(
    arvan_input_text_by_hash
)

assert completed_arvan_input_hashes == (
    expected_arvan_input_hashes
), (
    "Arvan embedding cache is incomplete after execution. "
    f"Expected {len(expected_arvan_input_hashes)}, "
    f"cached {len(completed_arvan_input_hashes)}."
)


# =========================================================
# FINAL REPORT
# =========================================================

arvan_embedding_execution_report = {
    "model_id": arvan_embedding_model_id,
    "embedding_dimension": int(
        arvan_test_embedding.size
    ),
    "unique_inputs_required": len(
        arvan_input_text_by_hash
    ),
    "unique_inputs_cached": len(
        completed_arvan_input_hashes
    ),
    "batch_directory": str(
        arvan_embedding_cache_directory
    ),
    "max_attempts_per_batch": (
        max_attempts_per_batch
    ),
    "status": "PASS",
}

arvan_embedding_execution_report_path = (
    active_benchmark_run_directory
    / "arvan_embedding_execution_report.json"
)

arvan_embedding_execution_report_path.write_text(
    json.dumps(
        arvan_embedding_execution_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print(
    "ARVAN FULL EMBEDDING EXECUTION: PASS"
)

print(
    "Total inputs cached:",
    len(completed_arvan_input_hashes),
)

print(
    "Report:",
    arvan_embedding_execution_report_path,
)

=== ARVAN FULL EMBEDDING EXECUTION ===
Unique inputs required: 11433
Inputs already cached: 11433
Inputs pending: 0
Batches pending: 0
Maximum inputs per batch: 64
Maximum characters per batch: 100000
Maximum attempts per batch: 8
Checkpoint directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_text_embedding_3_large/batches

ARVAN FULL EMBEDDING EXECUTION: PASS
Total inputs cached: 11433
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_embedding_execution_report.json


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۱. ساخت کامل embeddingهای مدل‌های محلی

Embeddingهای اروان برای همهٔ ورودی‌های Benchmark ساخته و ذخیره شدند.

در این مرحله همان داده‌ها با دو مدل محلی اجرا می‌شوند. برای مدل E5، پیشوندهای استاندارد query و passage به‌صورت جداگانه اعمال می‌شوند.

</div>

In [36]:
# Code Cell 20

local_embedding_input_registry = {}

query_text_by_hash = {
    hashlib.sha256(
        query_record["question"].encode("utf-8")
    ).hexdigest(): query_record["question"]
    for query_record in benchmark_query_records
}

chunk_text_by_hash = {
    hashlib.sha256(
        chunk_text.encode("utf-8")
    ).hexdigest(): chunk_text
    for chunk_text in all_canonical_chunk_texts
}

local_embedding_execution_reports = {}

for model_name, model in embedding_models.items():
    model_config = embedding_model_registry[model_name]

    if model_config["query_prefix"] or model_config["passage_prefix"]:
        input_text_by_key = {
            **{
                f"query:{text_hash}": (
                    model_config["query_prefix"] + text
                )
                for text_hash, text in query_text_by_hash.items()
            },
            **{
                f"passage:{text_hash}": (
                    model_config["passage_prefix"] + text
                )
                for text_hash, text in chunk_text_by_hash.items()
            },
        }
    else:
        input_text_by_key = {
            f"text:{text_hash}": text
            for text_hash, text in {
                **query_text_by_hash,
                **chunk_text_by_hash,
            }.items()
        }

    local_embedding_input_registry[model_name] = input_text_by_key

    model_cache_directory = (
        active_benchmark_run_directory
        / model_name
        / "batches"
    )

    model_cache_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    completed_input_keys = set()

    for batch_path in sorted(model_cache_directory.glob("batch_*.npz")):
        with np.load(batch_path, allow_pickle=False) as batch_data:
            batch_input_keys = batch_data["input_keys"]
            batch_embeddings = batch_data["embeddings"]

            assert batch_embeddings.ndim == 2, (
                f"Invalid embedding matrix in {batch_path.name}."
            )
            assert batch_embeddings.shape[0] == len(batch_input_keys), (
                f"Input and embedding counts differ in {batch_path.name}."
            )

            completed_input_keys.update(
                str(input_key)
                for input_key in batch_input_keys
            )

    pending_input_keys = sorted(
        set(input_text_by_key) - completed_input_keys
    )

    batch_size = 128
    input_key_batches = [
        pending_input_keys[index:index + batch_size]
        for index in range(0, len(pending_input_keys), batch_size)
    ]

    print(f"\n=== LOCAL FULL EMBEDDING: {model_name} ===")
    print("Inputs required:", len(input_text_by_key))
    print("Inputs already cached:", len(completed_input_keys))
    print("Inputs pending:", len(pending_input_keys))
    print("Batches pending:", len(input_key_batches))

    NB13_FINAL_LOCAL_RECONSTRUCTION_ONLY = True

    if NB13_FINAL_LOCAL_RECONSTRUCTION_ONLY and pending_input_keys:
        raise RuntimeError(
            "RUN-ALL SAFE GUARD: local embedding cache is incomplete for "
            f"{model_name}. Cached {len(completed_input_keys)}/"
            f"{len(input_text_by_key)} inputs. "
            "Recomputation is disabled in this clean notebook."
        )

    for batch_number, input_key_batch in enumerate(
        input_key_batches,
        start=1,
    ):
        input_text_batch = [
            input_text_by_key[input_key]
            for input_key in input_key_batch
        ]

        batch_embeddings = model.encode(
            input_text_batch,
            batch_size=64,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        assert batch_embeddings.shape[0] == len(input_key_batch), (
            f"Unexpected result count for {model_name}, "
            f"batch {batch_number}."
        )
        assert not np.isnan(batch_embeddings).any(), (
            f"{model_name} returned NaN in batch {batch_number}."
        )

        batch_output_path = (
            model_cache_directory
            / f"batch_{len(completed_input_keys):06d}.npz"
        )

        np.savez_compressed(
            batch_output_path,
            input_keys=np.asarray(input_key_batch),
            embeddings=batch_embeddings,
        )

        completed_input_keys.update(input_key_batch)

        if (
            batch_number == 1
            or batch_number == len(input_key_batches)
            or batch_number % 10 == 0
        ):
            print(
                f"Batch {batch_number}/{len(input_key_batches)} saved | "
                f"inputs cached: {len(completed_input_keys)}"
            )

    assert completed_input_keys == set(input_text_by_key), (
        f"{model_name} embedding cache is incomplete."
    )

    local_embedding_execution_reports[model_name] = {
        "model_id": model_config["model_id"],
        "embedding_dimension": model.get_sentence_embedding_dimension(),
        "unique_inputs_required": len(input_text_by_key),
        "unique_inputs_cached": len(completed_input_keys),
        "batch_directory": str(model_cache_directory),
        "status": "PASS",
    }

    report_path = (
        active_benchmark_run_directory
        / f"{model_name}_embedding_execution_report.json"
    )

    report_path.write_text(
        json.dumps(
            local_embedding_execution_reports[model_name],
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

print("\nLOCAL FULL EMBEDDING EXECUTION: PASS")


=== LOCAL FULL EMBEDDING: multilingual_e5_small ===
Inputs required: 11433
Inputs already cached: 11433
Inputs pending: 0
Batches pending: 0

=== LOCAL FULL EMBEDDING: all_minilm_l6_v2 ===
Inputs required: 11433
Inputs already cached: 11433
Inputs pending: 0
Batches pending: 0

LOCAL FULL EMBEDDING EXECUTION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۲. بارگذاری و اعتبارسنجی cacheهای embedding

Embeddingهای اروان و دو مدل محلی با موفقیت ساخته شدند.

در این مرحله cacheهای هر مدل بارگذاری می‌شوند و بررسی می‌شود که برای همهٔ پرسش‌های Golden و همهٔ متن‌های چانک، بردار معتبر وجود دارد.

</div>

In [37]:
# Code Cell 21

def load_embedding_batches(
    batch_directory,
    key_field_name,
):
    vector_by_key = {}

    for batch_path in sorted(batch_directory.glob("batch_*.npz")):
        with np.load(batch_path, allow_pickle=False) as batch_data:
            batch_keys = batch_data[key_field_name]
            batch_embeddings = batch_data["embeddings"]

            assert batch_embeddings.shape[0] == len(batch_keys), (
                f"Invalid cache batch: {batch_path.name}"
            )

            for key, embedding in zip(batch_keys, batch_embeddings):
                key = str(key)

                assert key not in vector_by_key, (
                    f"Duplicate embedding cache key: {key}"
                )

                vector_by_key[key] = embedding.astype(
                    np.float32,
                    copy=False,
                )

    return vector_by_key


arvan_vector_by_text_hash = load_embedding_batches(
    arvan_embedding_cache_directory,
    key_field_name="input_hashes",
)

expected_arvan_hashes = (
    set(query_text_by_hash)
    | set(chunk_text_by_hash)
)

assert set(arvan_vector_by_text_hash) == expected_arvan_hashes, (
    "Arvan cache does not exactly match the current benchmark input registry."
)

benchmark_embeddings_by_model = {
    "arvan_text_embedding_3_large": {
        "query_vectors_by_text_hash": {
            text_hash: arvan_vector_by_text_hash[text_hash]
            for text_hash in query_text_by_hash
        },
        "passage_vectors_by_text_hash": {
            text_hash: arvan_vector_by_text_hash[text_hash]
            for text_hash in chunk_text_by_hash
        },
        "dimension": int(arvan_test_embedding.size),
    },
}

for model_name, input_text_by_key in local_embedding_input_registry.items():
    model_cache_directory = (
        active_benchmark_run_directory
        / model_name
        / "batches"
    )

    vector_by_input_key = load_embedding_batches(
        model_cache_directory,
        key_field_name="input_keys",
    )

    model_config = embedding_model_registry[model_name]

    if model_config["query_prefix"] or model_config["passage_prefix"]:
        query_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"query:{text_hash}"]
            for text_hash in query_text_by_hash
        }
        passage_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"passage:{text_hash}"]
            for text_hash in chunk_text_by_hash
        }
    else:
        query_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"text:{text_hash}"]
            for text_hash in query_text_by_hash
        }
        passage_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"text:{text_hash}"]
            for text_hash in chunk_text_by_hash
        }

    benchmark_embeddings_by_model[model_name] = {
        "query_vectors_by_text_hash": query_vectors_by_text_hash,
        "passage_vectors_by_text_hash": passage_vectors_by_text_hash,
        "dimension": int(
            next(iter(query_vectors_by_text_hash.values())).shape[0]
        ),
    }

print("=== EMBEDDING CACHE ASSEMBLY VALIDATION ===")

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_vectors = embedding_lookup["query_vectors_by_text_hash"]
    passage_vectors = embedding_lookup["passage_vectors_by_text_hash"]

    assert set(query_vectors) == set(query_text_by_hash), (
        f"{model_name} query cache is incomplete."
    )
    assert set(passage_vectors) == set(chunk_text_by_hash), (
        f"{model_name} passage cache is incomplete."
    )

    print(f"\n{model_name}")
    print("- Embedding dimension:", embedding_lookup["dimension"])
    print("- Query vectors:", len(query_vectors))
    print("- Passage vectors:", len(passage_vectors))

print("\nEMBEDDING CACHE ASSEMBLY VALIDATION: PASS")

=== EMBEDDING CACHE ASSEMBLY VALIDATION ===

arvan_text_embedding_3_large
- Embedding dimension: 3072
- Query vectors: 3735
- Passage vectors: 7698

multilingual_e5_small
- Embedding dimension: 384
- Query vectors: 3735
- Passage vectors: 7698

all_minilm_l6_v2
- Embedding dimension: 384
- Query vectors: 3735
- Passage vectors: 7698

EMBEDDING CACHE ASSEMBLY VALIDATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۳. Benchmark واقعی Retrieval

برای هر مدل و هر روش چانکینگ، similarity میان پرسش‌های Golden و چانک‌ها محاسبه می‌شود.

معیار اصلی مقایسه فقط روی seedهای مشترک و دارای label معتبر اجرا می‌شود.  
بنابراین تفاوت تعداد چانک‌های مثبت یا پوشش اضافی یک روش، مقایسهٔ اصلی را ناعادلانه نمی‌کند.

</div>

In [38]:
# Code Cell 22

evaluation_cutoffs = (1, 3, 5, 10)

common_eligible_seed_ids = sorted(
    set.intersection(
        *(
            {
                source_seed_id
                for source_seed_id, label in seed_labels.items()
                if label["eligible"]
            }
            for seed_labels in retrieval_evaluation_registry.values()
        )
    )
)

assert common_eligible_seed_ids, (
    "No Golden v2 evaluation identities are eligible across all chunk candidates."
)

assert set(common_eligible_seed_ids).issubset(set(active_seed_ids)), (
    "Common eligible identities must be a subset of active Golden v2 identities."
)

evaluation_query_records = [
    query_record
    for query_record in benchmark_query_records
    if query_record["source_seed_id"] in common_eligible_seed_ids
]

assert evaluation_query_records, "No eligible Golden queries were found."

def l2_normalize_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    assert np.all(norms > 0), "Cannot normalize zero-length embeddings."
    return matrix / norms


candidate_chunk_registry = {}

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    chunk_ids = []
    chunk_text_hashes = []

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            chunk_ids.append(chunk["chunk_id"])
            chunk_text_hashes.append(
                hashlib.sha256(
                    canonical_exact_text(chunk["content"]).encode("utf-8")
                ).hexdigest()
            )

    chunk_index_by_id = {
        chunk_id: index
        for index, chunk_id in enumerate(chunk_ids)
    }

    positive_chunk_indexes_by_seed = {}

    for source_seed_id in common_eligible_seed_ids:
        positive_chunk_ids = retrieval_evaluation_registry[
            candidate_name
        ][source_seed_id]["positive_chunk_ids"]

        positive_indexes = sorted(
            chunk_index_by_id[chunk_id]
            for chunk_id in positive_chunk_ids
            if chunk_id in chunk_index_by_id
        )

        assert positive_indexes, (
            f"{candidate_name} has no indexed positive chunk for "
            f"seed {source_seed_id}."
        )

        positive_chunk_indexes_by_seed[source_seed_id] = set(
            positive_indexes
        )

    candidate_chunk_registry[candidate_name] = {
        "chunk_ids": chunk_ids,
        "chunk_text_hashes": chunk_text_hashes,
        "positive_chunk_indexes_by_seed": positive_chunk_indexes_by_seed,
    }

print("=== FULL RETRIEVAL BENCHMARK ===")
print("Common eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(evaluation_query_records))
print("Evaluation cutoffs:", evaluation_cutoffs)

embedding_benchmark_results = {}

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    embedding_benchmark_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][
                    text_hash
                ]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)

        first_positive_ranks = []
        retrieval_batch_size = 16

        started_at = time.perf_counter()

        for batch_start in range(
            0,
            len(evaluation_query_records),
            retrieval_batch_size,
        ):
            batch_end = min(
                batch_start + retrieval_batch_size,
                len(evaluation_query_records),
            )

            similarity_matrix = (
                query_matrix[batch_start:batch_end]
                @ passage_matrix.T
            )

            for local_index, query_record in enumerate(
                evaluation_query_records[batch_start:batch_end]
            ):
                ranked_chunk_indexes = np.argsort(
                    similarity_matrix[local_index]
                )[::-1]

                positive_chunk_indexes = candidate_data[
                    "positive_chunk_indexes_by_seed"
                ][query_record["source_seed_id"]]

                first_positive_rank = next(
                    rank
                    for rank, chunk_index in enumerate(
                        ranked_chunk_indexes,
                        start=1,
                    )
                    if chunk_index in positive_chunk_indexes
                )

                first_positive_ranks.append(first_positive_rank)

        elapsed_seconds = time.perf_counter() - started_at

        first_positive_ranks = np.asarray(
            first_positive_ranks,
            dtype=np.int32,
        )

        metrics = {
            f"recall_at_{cutoff}": float(
                np.mean(first_positive_ranks <= cutoff)
            )
            for cutoff in evaluation_cutoffs
        }

        metrics.update(
            {
                "mrr": float(np.mean(1.0 / first_positive_ranks)),
                "mean_first_positive_rank": float(
                    np.mean(first_positive_ranks)
                ),
                "median_first_positive_rank": float(
                    np.median(first_positive_ranks)
                ),
                "query_count": int(len(first_positive_ranks)),
                "candidate_chunk_count": int(
                    len(candidate_data["chunk_ids"])
                ),
                "elapsed_seconds": round(elapsed_seconds, 3),
            }
        )

        embedding_benchmark_results[model_name][candidate_name] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print("- Recall@1:", f"{metrics['recall_at_1']:.4f}")
        print("- Recall@3:", f"{metrics['recall_at_3']:.4f}")
        print("- Recall@5:", f"{metrics['recall_at_5']:.4f}")
        print("- Recall@10:", f"{metrics['recall_at_10']:.4f}")
        print("- MRR:", f"{metrics['mrr']:.4f}")
        print(
            "- Mean first-positive rank:",
            f"{metrics['mean_first_positive_rank']:.2f}",
        )
        print("- Seconds:", metrics["elapsed_seconds"])

print("\nFULL RETRIEVAL BENCHMARK: PASS")

=== FULL RETRIEVAL BENCHMARK ===
Common eligible seeds: 3813
Eligible Golden queries: 3813
Evaluation cutoffs: (1, 3, 5, 10)

arvan_text_embedding_3_large × baseline_phase1_char_safe
- Recall@1: 0.2526
- Recall@3: 0.3808
- Recall@5: 0.4453
- Recall@10: 0.5408
- MRR: 0.3472
- Mean first-positive rank: 114.02
- Seconds: 2.964

arvan_text_embedding_3_large × parent_child_children
- Recall@1: 0.2394
- Recall@3: 0.3824
- Recall@5: 0.4469
- Recall@10: 0.5431
- MRR: 0.3401
- Mean first-positive rank: 105.00
- Seconds: 2.316

arvan_text_embedding_3_large × structure_aware_medium
- Recall@1: 0.2198
- Recall@3: 0.3506
- Recall@5: 0.4117
- Recall@10: 0.5148
- MRR: 0.3175
- Mean first-positive rank: 89.10
- Seconds: 1.811

arvan_text_embedding_3_large × structure_aware_small
- Recall@1: 0.2373
- Recall@3: 0.3839
- Recall@5: 0.4464
- Recall@10: 0.5382
- MRR: 0.3397
- Mean first-positive rank: 99.56
- Seconds: 2.405

multilingual_e5_small × baseline_phase1_char_safe
- Recall@1: 0.3110
- Recall@3: 0.

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۴. آزمون تشخیصی با متن شاهد Golden

نتیجهٔ retrieval روی سؤال‌های Golden برای انتخاب نهایی مدل کافی نیست.

در این آزمون، به‌جای سؤال تولیدشده، خود متن شاهد Golden به‌عنوان query استفاده می‌شود.  
هدف تشخیص این است که افت معیارها از نوع سؤال‌ها ناشی می‌شود یا از مدل embedding، چانکینگ و labelهای retrieval.

</div>

In [39]:
# Code Cell 23
# NB13 — Gold-evidence retrieval diagnostic (cache-only, Golden v2 safe)
# This diagnostic never makes API calls and never recomputes embeddings.

gold_evidence_text_by_seed = {}

for source_seed_id in common_eligible_seed_ids:
    gold_record = gold_records_by_seed[source_seed_id]

    assert isinstance(gold_record, dict), (
        f"Golden v2 record for {source_seed_id} must be a dict."
    )

    source_evidence = gold_record.get("source_evidence")

    assert isinstance(source_evidence, str) and source_evidence.strip(), (
        f"Golden v2 record for {source_seed_id} has no usable source_evidence."
    )

    gold_evidence_text_by_seed[source_seed_id] = canonical_exact_text(
        source_evidence
    )

evidence_text_hash_by_seed = {
    source_seed_id: hashlib.sha256(
        evidence_text.encode("utf-8")
    ).hexdigest()
    for source_seed_id, evidence_text in gold_evidence_text_by_seed.items()
}

evidence_probe_query_matrices = {}
evidence_probe_skip_reasons = {}

# Arvan: reuse the completed full embedding cache only.
arvan_lookup = benchmark_embeddings_by_model["arvan_text_embedding_3_large"]
arvan_passage_vectors = arvan_lookup["passage_vectors_by_text_hash"]

missing_arvan_evidence_hashes = [
    text_hash
    for text_hash in evidence_text_hash_by_seed.values()
    if text_hash not in arvan_passage_vectors
]

if not missing_arvan_evidence_hashes:
    arvan_evidence_probe_matrix = np.vstack(
        [
            arvan_passage_vectors[evidence_text_hash_by_seed[source_seed_id]]
            for source_seed_id in common_eligible_seed_ids
        ]
    ).astype(np.float32)

    evidence_probe_query_matrices[
        "arvan_text_embedding_3_large"
    ] = l2_normalize_rows(arvan_evidence_probe_matrix)
else:
    evidence_probe_skip_reasons[
        "arvan_text_embedding_3_large"
    ] = (
        "Exact Golden evidence text is not fully present in the completed "
        "passage-embedding cache; diagnostic skipped without API calls."
    )

# Local models: reuse completed passage caches only.
for model_name in embedding_models:
    local_lookup = benchmark_embeddings_by_model[model_name]
    local_passage_vectors = local_lookup["passage_vectors_by_text_hash"]

    missing_local_hashes = [
        text_hash
        for text_hash in evidence_text_hash_by_seed.values()
        if text_hash not in local_passage_vectors
    ]

    if missing_local_hashes:
        evidence_probe_skip_reasons[model_name] = (
            "Exact Golden evidence text is not fully present in the completed "
            "local passage-embedding cache; diagnostic skipped without recomputation."
        )
        continue

    local_probe_matrix = np.vstack(
        [
            local_passage_vectors[evidence_text_hash_by_seed[source_seed_id]]
            for source_seed_id in common_eligible_seed_ids
        ]
    ).astype(np.float32)

    evidence_probe_query_matrices[model_name] = l2_normalize_rows(
        local_probe_matrix
    )

evidence_probe_results = {}

print("=== GOLD EVIDENCE RETRIEVAL DIAGNOSTIC ===")
print("Diagnostic records:", len(common_eligible_seed_ids))
print("Cache-backed models:", sorted(evidence_probe_query_matrices))
print("Skipped models:", sorted(evidence_probe_skip_reasons))

for model_name, reason in evidence_probe_skip_reasons.items():
    print(f"- {model_name}: {reason}")

for model_name, probe_query_matrix in evidence_probe_query_matrices.items():
    evidence_probe_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        embedding_lookup = benchmark_embeddings_by_model[model_name]

        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][text_hash]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)

        similarity_matrix = probe_query_matrix @ passage_matrix.T
        first_positive_ranks = []

        for row_index, source_seed_id in enumerate(common_eligible_seed_ids):
            ranked_chunk_indexes = np.argsort(
                similarity_matrix[row_index]
            )[::-1]

            positive_chunk_indexes = candidate_data[
                "positive_chunk_indexes_by_seed"
            ][source_seed_id]

            first_positive_ranks.append(
                next(
                    rank
                    for rank, chunk_index in enumerate(
                        ranked_chunk_indexes,
                        start=1,
                    )
                    if chunk_index in positive_chunk_indexes
                )
            )

        first_positive_ranks = np.asarray(
            first_positive_ranks,
            dtype=np.int32,
        )

        metrics = {
            f"recall_at_{cutoff}": float(
                np.mean(first_positive_ranks <= cutoff)
            )
            for cutoff in evaluation_cutoffs
        }
        metrics["mrr"] = float(np.mean(1.0 / first_positive_ranks))
        evidence_probe_results[model_name][candidate_name] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print("- Recall@1:", f"{metrics['recall_at_1']:.4f}")
        print("- Recall@3:", f"{metrics['recall_at_3']:.4f}")
        print("- Recall@5:", f"{metrics['recall_at_5']:.4f}")
        print("- Recall@10:", f"{metrics['recall_at_10']:.4f}")
        print("- MRR:", f"{metrics['mrr']:.4f}")

print("\nGOLD EVIDENCE RETRIEVAL DIAGNOSTIC: PASS")


=== GOLD EVIDENCE RETRIEVAL DIAGNOSTIC ===
Diagnostic records: 3813
Cache-backed models: []
Skipped models: ['all_minilm_l6_v2', 'arvan_text_embedding_3_large', 'multilingual_e5_small']
- arvan_text_embedding_3_large: Exact Golden evidence text is not fully present in the completed passage-embedding cache; diagnostic skipped without API calls.
- multilingual_e5_small: Exact Golden evidence text is not fully present in the completed local passage-embedding cache; diagnostic skipped without recomputation.
- all_minilm_l6_v2: Exact Golden evidence text is not fully present in the completed local passage-embedding cache; diagnostic skipped without recomputation.

GOLD EVIDENCE RETRIEVAL DIAGNOSTIC: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۵. خواندن معیار فریز‌شدهٔ انتخاب نهایی

Benchmark نشان داد که نتیجهٔ سؤال‌های Golden و نتیجهٔ متن شاهد یکسان نیست.

پیش از هر انتخاب نهایی، معیارهای فریز‌شدهٔ پروژه برای انتخاب مدل و روش چانکینگ خوانده می‌شوند تا تصمیم بر اساس قرارداد پروژه گرفته شود، نه یک آستانهٔ دلخواه.

</div>

In [40]:
# Code Cell 24

selection_requirements_artifact = chunking_freeze_manifest["artifacts"][
    "final_selection_requirements"
]

selection_requirements_path = Path(
    selection_requirements_artifact["path"]
)

if not selection_requirements_path.is_absolute():
    selection_requirements_path = project_root / selection_requirements_path

assert selection_requirements_path.is_file(), (
    "Frozen final-selection requirements artifact is missing: "
    f"{selection_requirements_path}"
)

selection_requirements = json.loads(
    selection_requirements_path.read_text(
        encoding="utf-8"
    )
)

print("=== FROZEN FINAL-SELECTION REQUIREMENTS ===")
print("Path:", selection_requirements_path.relative_to(project_root))
print(
    json.dumps(
        selection_requirements,
        ensure_ascii=False,
        indent=2,
    )
)

print("\nFROZEN FINAL-SELECTION REQUIREMENTS: PASS")

=== FROZEN FINAL-SELECTION REQUIREMENTS ===
Path: reports/phase2/chunking/final_selection_requirements_v1.json
{
  "already_passed": [
    "source_structure_discovery",
    "tokenizer_measurement_freeze",
    "full_corpus_token_calibration",
    "candidate_generation",
    "coverage_integrity",
    "lineage_integrity",
    "locator_integrity",
    "rights_propagation_integrity",
    "hard_max_validation",
    "reconstruction",
    "content_preservation",
    "structural_benchmark",
    "attribution_risk_benchmark",
    "root_cause_analysis"
  ],
  "current_status": "BLOCKED_PENDING_RETRIEVAL_AND_EVIDENCE_EVALUATION",
  "final_selection": "NOT_YET",
  "still_required": [
    "golden_dataset",
    "retrieval_benchmark",
    "citation_evaluation",
    "attribution_correctness_evaluation",
    "context_quality_evaluation",
    "final_tradeoff_decision"
  ]
}

FROZEN FINAL-SELECTION REQUIREMENTS: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۶. ارزیابی Citation و Attribution در خروجی Retrieval

در این مرحله خروجی‌های Top-10 هر مدل و روش چانکینگ بررسی می‌شوند.

دو موضوع سنجیده می‌شود:

- کامل‌بودن قرارداد citation در چانک‌های بازیابی‌شده؛
- انطباق provenance چانک‌های بازیابی‌شده با سندهای معتبر همان پرسش Golden.

این ارزیابی ساختاری و خودکار است؛ ارزیابی انسانی کیفیت پاسخ و context در مرحلهٔ بعد انجام می‌شود.

</div>

In [41]:
# Code Cell 25

citation_evaluation_cutoff = 10

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    chunk_metadata_by_id = {}

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)

            chunk_metadata_by_id[chunk["chunk_id"]] = {
                "source_id": chunk.get("source_id"),
                "source_type": chunk.get("source_type"),
                "locator_start": chunk.get("locator_start"),
                "locator_end": chunk.get("locator_end"),
                "attribution_context": chunk.get(
                    "attribution_context"
                ),
            }

    candidate_chunk_registry[candidate_name][
        "chunk_metadata_by_id"
    ] = chunk_metadata_by_id

print("=== CITATION AND ATTRIBUTION EVALUATION ===")
print("Queries evaluated:", len(evaluation_query_records))
print("Top-K cutoff:", citation_evaluation_cutoff)

citation_attribution_evaluation_results = {}

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    citation_attribution_evaluation_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][text_hash]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)

        similarity_matrix = query_matrix @ passage_matrix.T

        top_k_contract_passes = []
        top_k_positive_precisions = []
        top_k_has_positive = []
        top_1_is_positive = []

        for row_index, query_record in enumerate(
            evaluation_query_records
        ):
            top_k_indexes_unsorted = np.argpartition(
                similarity_matrix[row_index],
                -citation_evaluation_cutoff,
            )[-citation_evaluation_cutoff:]

            top_k_indexes = top_k_indexes_unsorted[
                np.argsort(
                    similarity_matrix[row_index][top_k_indexes_unsorted]
                )[::-1]
            ]

            top_k_chunk_ids = [
                candidate_data["chunk_ids"][chunk_index]
                for chunk_index in top_k_indexes
            ]

            positive_chunk_ids = retrieval_evaluation_registry[
                candidate_name
            ][query_record["source_seed_id"]][
                "positive_chunk_ids"
            ]

            top_k_positive_flags = [
                chunk_id in positive_chunk_ids
                for chunk_id in top_k_chunk_ids
            ]

            top_k_metadata = [
                candidate_data["chunk_metadata_by_id"][chunk_id]
                for chunk_id in top_k_chunk_ids
            ]

            top_k_contract_passes.append(
                all(
                    isinstance(metadata["source_id"], str)
                    and metadata["source_id"]
                    and isinstance(metadata["source_type"], str)
                    and metadata["source_type"]
                    and isinstance(metadata["locator_start"], dict)
                    and isinstance(metadata["locator_end"], dict)
                    and isinstance(
                        metadata["attribution_context"],
                        dict,
                    )
                    for metadata in top_k_metadata
                )
            )

            top_k_positive_precisions.append(
                np.mean(top_k_positive_flags)
            )
            top_k_has_positive.append(
                any(top_k_positive_flags)
            )
            top_1_is_positive.append(
                top_k_positive_flags[0]
            )

        metrics = {
            "citation_contract_pass_rate_at_10": float(
                np.mean(top_k_contract_passes)
            ),
            "attribution_correct_at_1": float(
                np.mean(top_1_is_positive)
            ),
            "attribution_correct_at_10": float(
                np.mean(top_k_has_positive)
            ),
            "mean_provenance_precision_at_10": float(
                np.mean(top_k_positive_precisions)
            ),
            "query_count": len(evaluation_query_records),
        }

        citation_attribution_evaluation_results[model_name][
            candidate_name
        ] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print(
            "- Citation contract pass@10:",
            f"{metrics['citation_contract_pass_rate_at_10']:.4f}",
        )
        print(
            "- Attribution correct@1:",
            f"{metrics['attribution_correct_at_1']:.4f}",
        )
        print(
            "- Attribution correct@10:",
            f"{metrics['attribution_correct_at_10']:.4f}",
        )
        print(
            "- Mean provenance precision@10:",
            f"{metrics['mean_provenance_precision_at_10']:.4f}",
        )

print("\nCITATION AND ATTRIBUTION EVALUATION: PASS")

=== CITATION AND ATTRIBUTION EVALUATION ===
Queries evaluated: 3813
Top-K cutoff: 10

arvan_text_embedding_3_large × baseline_phase1_char_safe
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.2526
- Attribution correct@10: 0.5408
- Mean provenance precision@10: 0.0760

arvan_text_embedding_3_large × parent_child_children
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.2394
- Attribution correct@10: 0.5431
- Mean provenance precision@10: 0.0606

arvan_text_embedding_3_large × structure_aware_medium
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.2195
- Attribution correct@10: 0.5148
- Mean provenance precision@10: 0.0540

arvan_text_embedding_3_large × structure_aware_small
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.2376
- Attribution correct@10: 0.5382
- Mean provenance precision@10: 0.0598

multilingual_e5_small × baseline_phase1_char_safe
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.3110
- Attribution c

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۷. پروفایل خودکار کیفیت Context

قرارداد citation در همهٔ خروجی‌ها سالم است.

در این مرحله contextهای Top-10 از نظر ساختاری بررسی می‌شوند: حجم متن، تعداد token، تنوع سندهای منبع و میزان تکرار چانک‌ها.  
این مرحله کیفیت ساختاری context را می‌سنجد؛ داوری انسانیِ مفیدبودن پاسخ در مراحل بعد انجام می‌شود.

</div>

In [42]:
# Code Cell 26

context_quality_results = {}

print("=== AUTOMATED CONTEXT QUALITY PROFILE ===")
print("Queries evaluated:", len(evaluation_query_records))
print("Top-K cutoff:", citation_evaluation_cutoff)

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    context_quality_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        candidate_path = Path(
            retrieval_chunking_candidates[candidate_name]["path"]
        )

        if not candidate_path.is_absolute():
            candidate_path = project_root / candidate_path

        chunk_context_metadata_by_id = {}

        with candidate_path.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue

                chunk = json.loads(line)

                chunk_context_metadata_by_id[chunk["chunk_id"]] = {
                    "source_id": chunk.get("source_id"),
                    "token_count": chunk.get("token_count"),
                    "char_count": chunk.get("char_count"),
                    "content_hash": hashlib.sha256(
                        canonical_exact_text(
                            chunk["content"]
                        ).encode("utf-8")
                    ).hexdigest(),
                }

        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][text_hash]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)
        similarity_matrix = query_matrix @ passage_matrix.T

        context_token_totals = []
        context_char_totals = []
        unique_source_counts = []
        unique_content_counts = []

        for row_index in range(len(evaluation_query_records)):
            top_k_indexes_unsorted = np.argpartition(
                similarity_matrix[row_index],
                -citation_evaluation_cutoff,
            )[-citation_evaluation_cutoff:]

            top_k_indexes = top_k_indexes_unsorted[
                np.argsort(
                    similarity_matrix[row_index][top_k_indexes_unsorted]
                )[::-1]
            ]

            top_k_chunk_ids = [
                candidate_data["chunk_ids"][chunk_index]
                for chunk_index in top_k_indexes
            ]

            top_k_metadata = [
                chunk_context_metadata_by_id[chunk_id]
                for chunk_id in top_k_chunk_ids
            ]

            context_token_totals.append(
                sum(
                    metadata["token_count"]
                    for metadata in top_k_metadata
                    if isinstance(metadata["token_count"], int)
                )
            )
            context_char_totals.append(
                sum(
                    metadata["char_count"]
                    for metadata in top_k_metadata
                    if isinstance(metadata["char_count"], int)
                )
            )
            unique_source_counts.append(
                len(
                    {
                        metadata["source_id"]
                        for metadata in top_k_metadata
                        if isinstance(metadata["source_id"], str)
                        and metadata["source_id"]
                    }
                )
            )
            unique_content_counts.append(
                len(
                    {
                        metadata["content_hash"]
                        for metadata in top_k_metadata
                    }
                )
            )

        metrics = {
            "mean_context_tokens_at_10": float(
                np.mean(context_token_totals)
            ),
            "p95_context_tokens_at_10": float(
                np.percentile(context_token_totals, 95)
            ),
            "mean_context_characters_at_10": float(
                np.mean(context_char_totals)
            ),
            "mean_unique_sources_at_10": float(
                np.mean(unique_source_counts)
            ),
            "mean_unique_chunks_at_10": float(
                np.mean(unique_content_counts)
            ),
        }

        context_quality_results[model_name][candidate_name] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print(
            "- Mean context tokens@10:",
            f"{metrics['mean_context_tokens_at_10']:.1f}",
        )
        print(
            "- P95 context tokens@10:",
            f"{metrics['p95_context_tokens_at_10']:.1f}",
        )
        print(
            "- Mean context characters@10:",
            f"{metrics['mean_context_characters_at_10']:.1f}",
        )
        print(
            "- Mean unique sources@10:",
            f"{metrics['mean_unique_sources_at_10']:.2f}",
        )
        print(
            "- Mean unique chunks@10:",
            f"{metrics['mean_unique_chunks_at_10']:.2f}",
        )

print("\nAUTOMATED CONTEXT QUALITY PROFILE: PASS")

=== AUTOMATED CONTEXT QUALITY PROFILE ===
Queries evaluated: 3813
Top-K cutoff: 10

arvan_text_embedding_3_large × baseline_phase1_char_safe
- Mean context tokens@10: 2977.2
- P95 context tokens@10: 3418.0
- Mean context characters@10: 10817.3
- Mean unique sources@10: 7.22
- Mean unique chunks@10: 9.94

arvan_text_embedding_3_large × parent_child_children
- Mean context tokens@10: 3033.2
- P95 context tokens@10: 3906.4
- Mean context characters@10: 11029.4
- Mean unique sources@10: 7.67
- Mean unique chunks@10: 9.93

arvan_text_embedding_3_large × structure_aware_medium
- Mean context tokens@10: 4749.7
- P95 context tokens@10: 6304.0
- Mean context characters@10: 17244.2
- Mean unique sources@10: 7.79
- Mean unique chunks@10: 9.91

arvan_text_embedding_3_large × structure_aware_small
- Mean context tokens@10: 3317.3
- P95 context tokens@10: 4158.0
- Mean context characters@10: 12056.9
- Mean unique sources@10: 7.69
- Mean unique chunks@10: 9.93

multilingual_e5_small × baseline_phase1

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۸. اعتبارسنجی اتصال Parent–Child

در روش Parent–Child، جست‌وجو روی child انجام می‌شود اما context نهایی باید از parent متناظر ساخته شود.

در این مرحله ارتباط هر child با parent آن بررسی می‌شود تا ارزیابی context این روش بر اساس ساختار واقعی آن انجام شود.

</div>

In [43]:
# Code Cell 27

parent_candidate = chunking_candidates["parent_child_parents"]
parent_candidate_path = Path(parent_candidate["path"])

if not parent_candidate_path.is_absolute():
    parent_candidate_path = project_root / parent_candidate_path

child_candidate = retrieval_chunking_candidates[
    "parent_child_children"
]
child_candidate_path = Path(child_candidate["path"])

if not child_candidate_path.is_absolute():
    child_candidate_path = project_root / child_candidate_path

parent_chunks_by_id = {}

with parent_candidate_path.open(encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        parent_chunk = json.loads(line)
        parent_chunks_by_id[parent_chunk["chunk_id"]] = parent_chunk

child_to_parent_ids = {}
children_without_parent_id = []
children_with_missing_parent = []

with child_candidate_path.open(encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        child_chunk = json.loads(line)
        child_chunk_id = child_chunk["chunk_id"]
        parent_chunk_id = child_chunk.get("parent_chunk_id")

        if not isinstance(parent_chunk_id, str) or not parent_chunk_id:
            children_without_parent_id.append(child_chunk_id)
            continue

        child_to_parent_ids[child_chunk_id] = parent_chunk_id

        if parent_chunk_id not in parent_chunks_by_id:
            children_with_missing_parent.append(
                {
                    "child_chunk_id": child_chunk_id,
                    "parent_chunk_id": parent_chunk_id,
                }
            )

print("=== PARENT–CHILD CONTEXT LINKAGE VALIDATION ===")
print("Parent chunks:", len(parent_chunks_by_id))
print("Child chunks:", len(child_to_parent_ids))
print("Children without parent_chunk_id:", len(children_without_parent_id))
print("Children with missing parent record:", len(children_with_missing_parent))

assert not children_without_parent_id, (
    "Some parent-child retrieval chunks do not declare a parent_chunk_id."
)
assert not children_with_missing_parent, (
    "Some parent-child retrieval chunks point to missing parent records."
)

sample_child_chunk_id = next(iter(child_to_parent_ids))
sample_parent_chunk_id = child_to_parent_ids[sample_child_chunk_id]
sample_parent_chunk = parent_chunks_by_id[sample_parent_chunk_id]

print("\nSample linkage:")
print("- Child chunk ID:", sample_child_chunk_id)
print("- Parent chunk ID:", sample_parent_chunk_id)
print("- Parent token count:", sample_parent_chunk["token_count"])
print("- Parent source ID:", sample_parent_chunk["source_id"])
print(
    "- Parent locator start:",
    sample_parent_chunk["locator_start"],
)
print(
    "- Parent locator end:",
    sample_parent_chunk["locator_end"],
)

print("\nPARENT–CHILD CONTEXT LINKAGE VALIDATION: PASS")

=== PARENT–CHILD CONTEXT LINKAGE VALIDATION ===
Parent chunks: 1447
Child chunks: 2721
Children without parent_chunk_id: 0
Children with missing parent record: 0

Sample linkage:
- Child chunk ID: chunk:parent_child:child:e634d6ef46057e1f28b2a03f
- Parent chunk ID: chunk:parent_child:parent:eb013baa5da4520e962baef2
- Parent token count: 361
- Parent source ID: website:008E61CC
- Parent locator start: {'char_offset': 0, 'document_id': 'website:008E61CC'}
- Parent locator end: {'char_offset': 1316, 'document_id': 'website:008E61CC'}

PARENT–CHILD CONTEXT LINKAGE VALIDATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۹. ارزیابی Context واقعی Parent–Child

در روش Parent–Child، childها برای retrieval استفاده می‌شوند و parentهای متناظر context نهایی را می‌سازند.

در این مرحله حجم، تنوع و قرارداد citation context والدها برای هر مدل اندازه‌گیری می‌شود.

</div>

In [44]:
# Code Cell 28

parent_child_context_quality_results = {}

parent_child_candidate_data = candidate_chunk_registry[
    "parent_child_children"
]

print("=== PARENT–CHILD REAL CONTEXT QUALITY ===")
print("Queries evaluated:", len(evaluation_query_records))
print("Retrieved children per query:", citation_evaluation_cutoff)

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    child_passage_matrix = np.vstack(
        [
            embedding_lookup["passage_vectors_by_text_hash"][text_hash]
            for text_hash in parent_child_candidate_data[
                "chunk_text_hashes"
            ]
        ]
    ).astype(np.float32)

    child_passage_matrix = l2_normalize_rows(
        child_passage_matrix
    )

    similarity_matrix = query_matrix @ child_passage_matrix.T

    parent_context_token_totals = []
    parent_context_char_totals = []
    parent_context_unique_source_counts = []
    parent_context_parent_counts = []
    parent_context_contract_passes = []

    for row_index in range(len(evaluation_query_records)):
        top_child_indexes_unsorted = np.argpartition(
            similarity_matrix[row_index],
            -citation_evaluation_cutoff,
        )[-citation_evaluation_cutoff:]

        top_child_indexes = top_child_indexes_unsorted[
            np.argsort(
                similarity_matrix[row_index][top_child_indexes_unsorted]
            )[::-1]
        ]

        top_child_chunk_ids = [
            parent_child_candidate_data["chunk_ids"][child_index]
            for child_index in top_child_indexes
        ]

        parent_chunk_ids = list(
            dict.fromkeys(
                child_to_parent_ids[child_chunk_id]
                for child_chunk_id in top_child_chunk_ids
            )
        )

        parent_context_chunks = [
            parent_chunks_by_id[parent_chunk_id]
            for parent_chunk_id in parent_chunk_ids
        ]

        parent_context_token_totals.append(
            sum(
                parent_chunk["token_count"]
                for parent_chunk in parent_context_chunks
            )
        )
        parent_context_char_totals.append(
            sum(
                parent_chunk["char_count"]
                for parent_chunk in parent_context_chunks
            )
        )
        parent_context_unique_source_counts.append(
            len(
                {
                    parent_chunk["source_id"]
                    for parent_chunk in parent_context_chunks
                }
            )
        )
        parent_context_parent_counts.append(
            len(parent_context_chunks)
        )
        parent_context_contract_passes.append(
            all(
                isinstance(parent_chunk["source_id"], str)
                and parent_chunk["source_id"]
                and isinstance(parent_chunk["source_type"], str)
                and parent_chunk["source_type"]
                and isinstance(parent_chunk["locator_start"], dict)
                and isinstance(parent_chunk["locator_end"], dict)
                and isinstance(
                    parent_chunk["attribution_context"],
                    dict,
                )
                for parent_chunk in parent_context_chunks
            )
        )

    metrics = {
        "mean_parent_context_tokens": float(
            np.mean(parent_context_token_totals)
        ),
        "p95_parent_context_tokens": float(
            np.percentile(parent_context_token_totals, 95)
        ),
        "mean_parent_context_characters": float(
            np.mean(parent_context_char_totals)
        ),
        "mean_unique_parent_sources": float(
            np.mean(parent_context_unique_source_counts)
        ),
        "mean_unique_parent_chunks": float(
            np.mean(parent_context_parent_counts)
        ),
        "parent_context_citation_contract_pass_rate": float(
            np.mean(parent_context_contract_passes)
        ),
    }

    parent_child_context_quality_results[model_name] = metrics

    print(f"\n{model_name}")
    print(
        "- Mean parent-context tokens:",
        f"{metrics['mean_parent_context_tokens']:.1f}",
    )
    print(
        "- P95 parent-context tokens:",
        f"{metrics['p95_parent_context_tokens']:.1f}",
    )
    print(
        "- Mean parent-context characters:",
        f"{metrics['mean_parent_context_characters']:.1f}",
    )
    print(
        "- Mean unique parent sources:",
        f"{metrics['mean_unique_parent_sources']:.2f}",
    )
    print(
        "- Mean unique parent chunks:",
        f"{metrics['mean_unique_parent_chunks']:.2f}",
    )
    print(
        "- Citation contract pass:",
        f"{metrics['parent_context_citation_contract_pass_rate']:.4f}",
    )

print("\nPARENT–CHILD REAL CONTEXT QUALITY: PASS")

=== PARENT–CHILD REAL CONTEXT QUALITY ===
Queries evaluated: 3813
Retrieved children per query: 10

arvan_text_embedding_3_large
- Mean parent-context tokens: 6821.6
- P95 parent-context tokens: 9086.0
- Mean parent-context characters: 24714.8
- Mean unique parent sources: 7.67
- Mean unique parent chunks: 9.07
- Citation contract pass: 1.0000

multilingual_e5_small
- Mean parent-context tokens: 6670.4
- P95 parent-context tokens: 9127.4
- Mean parent-context characters: 24664.7
- Mean unique parent sources: 8.34
- Mean unique parent chunks: 9.34
- Citation contract pass: 1.0000

all_minilm_l6_v2
- Mean parent-context tokens: 4401.5
- P95 parent-context tokens: 5980.0
- Mean parent-context characters: 15744.3
- Mean unique parent sources: 8.34
- Mean unique parent chunks: 10.00
- Citation contract pass: 1.0000

PARENT–CHILD REAL CONTEXT QUALITY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۴۰. ثبت تصمیم Trade-off و وضعیت آمادگی انتخاب

همهٔ ارزیابی‌های خودکار retrieval، citation، attribution و context اجرا شده‌اند.

در این مرحله نتیجهٔ واقعی ثبت می‌شود.  
اگر یک ترکیب در همهٔ ابعاد برنده نباشد، انتخاب نهایی انجام نمی‌شود و علت آن به‌صورت شفاف ثبت خواهد شد.

</div>

In [45]:
# Code Cell 29

question_retrieval_ranking = sorted(
    [
        {
            "model": model_name,
            "candidate": candidate_name,
            **metrics,
        }
        for model_name, candidate_results
        in embedding_benchmark_results.items()
        for candidate_name, metrics
        in candidate_results.items()
    ],
    key=lambda item: (
        -item["recall_at_10"],
        -item["mrr"],
    ),
)

evidence_retrieval_ranking = sorted(
    [
        {
            "model": model_name,
            "candidate": candidate_name,
            **metrics,
        }
        for model_name, candidate_results
        in evidence_probe_results.items()
        for candidate_name, metrics
        in candidate_results.items()
    ],
    key=lambda item: (
        -item["recall_at_10"],
        -item["mrr"],
    ),
)

assert question_retrieval_ranking, (
    "Question-retrieval ranking is empty; the benchmark cannot continue."
)

best_question_retrieval = question_retrieval_ranking[0]
best_evidence_retrieval = (
    evidence_retrieval_ranking[0]
    if evidence_retrieval_ranking
    else None
)

embedding_benchmark_tradeoff = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "selection_status": "NO_FINAL_SELECTION",
    "decision_basis": {
        "best_question_retrieval": best_question_retrieval,
        "best_evidence_retrieval": best_evidence_retrieval,
        "citation_contract_status": "PASS_FOR_ALL_COMBINATIONS",
        "parent_child_context_contract_status": "PASS_FOR_ALL_MODELS",
    },
    "blocking_findings": (
        [
            (
                "Question-based retrieval is materially weaker than "
                "source-evidence retrieval across the evaluated combinations."
            ),
            (
                "No single model-and-chunking combination leads both "
                "question retrieval and source-evidence retrieval."
            ),
            (
                "Parent-child context requires a context-budget policy "
                "before production use."
            ),
        ]
        if best_evidence_retrieval is not None
        else [
            (
                "Source-evidence diagnostic was unavailable because exact "
                "Golden evidence texts were not fully represented in the "
                "completed embedding caches. No API or recomputation was "
                "triggered to manufacture this diagnostic."
            ),
            (
                "Question-retrieval results remain valid and are preserved."
            ),
            (
                "Parent-child context requires a context-budget policy "
                "before production use."
            ),
        ]
    ),
    "required_next_work": [
        "query_side_retrieval_remediation",
        "context_budget_and_assembly_policy",
        "post_remediation_retrieval_benchmark",
        "final_tradeoff_decision",
    ],
}

tradeoff_report_path = (
    active_benchmark_run_directory
    / "embedding_benchmark_tradeoff.json"
)

tradeoff_report_path.write_text(
    json.dumps(
        embedding_benchmark_tradeoff,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== EMBEDDING BENCHMARK TRADE-OFF DECISION ===")
print("Selection status:", embedding_benchmark_tradeoff["selection_status"])

print("\nBest question-retrieval combination:")
print(
    f"- model={best_question_retrieval['model']}, "
    f"candidate={best_question_retrieval['candidate']}, "
    f"Recall@10={best_question_retrieval['recall_at_10']:.4f}, "
    f"MRR={best_question_retrieval['mrr']:.4f}"
)

print("\nBest source-evidence-retrieval combination:")
if best_evidence_retrieval is None:
    print("- unavailable: evidence diagnostic was skipped cache-only")
else:
    print(
        f"- model={best_evidence_retrieval['model']}, "
        f"candidate={best_evidence_retrieval['candidate']}, "
        f"Recall@10={best_evidence_retrieval['recall_at_10']:.4f}, "
        f"MRR={best_evidence_retrieval['mrr']:.4f}"
    )

print("\nBlocking findings:")
for finding in embedding_benchmark_tradeoff["blocking_findings"]:
    print("-", finding)

print("\nRequired next work:")
for next_step in embedding_benchmark_tradeoff["required_next_work"]:
    print("-", next_step)

print("\nReport:", tradeoff_report_path)
print("\nEMBEDDING BENCHMARK TRADE-OFF DECISION: PASS")

=== EMBEDDING BENCHMARK TRADE-OFF DECISION ===
Selection status: NO_FINAL_SELECTION

Best question-retrieval combination:
- model=multilingual_e5_small, candidate=baseline_phase1_char_safe, Recall@10=0.6113, MRR=0.4113

Best source-evidence-retrieval combination:
- unavailable: evidence diagnostic was skipped cache-only

Blocking findings:
- Source-evidence diagnostic was unavailable because exact Golden evidence texts were not fully represented in the completed embedding caches. No API or recomputation was triggered to manufacture this diagnostic.
- Question-retrieval results remain valid and are preserved.
- Parent-child context requires a context-budget policy before production use.

Required next work:
- query_side_retrieval_remediation
- context_budget_and_assembly_policy
- post_remediation_retrieval_benchmark
- final_tradeoff_decision

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c6378

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## اصلاح قرارداد ورودی مدل E5

مدل `multilingual-e5-small` برای بازیابی معنایی با دو پیشوند استاندارد طراحی شده است:

- برای پرسش: `query:`
- برای متن بازیابی‌شونده: `passage:`

بنابراین پیش از تغییر Chunking یا انتخاب مدل نهایی، همین مدل را یک‌بار با قرارداد صحیح خودش اجرا می‌کنیم و نتیجه را جدا از اجرای قبلی نگه می‌داریم.

این آزمایش:

- Golden Dataset فعال را از متغیر اصلی می‌خواند؛
- هیچ فایل یا نتیجهٔ قبلی را جایگزین نمی‌کند؛
- خروجی را با هش Golden Dataset فعال ذخیره می‌کند؛
- در اجرای آینده با Golden Dataset جدید، خودکار در پوشهٔ جداگانهٔ همان نسخه اجرا می‌شود.

</div>

In [46]:
# Code Cell 30
# NB13 — E5 instructional embedding cache
# Reuses the existing cache when the active Golden Dataset is unchanged.

from pathlib import Path
import hashlib
import json
import numpy as np

assert "golden_dataset_path" in globals(), (
    "Run the Active Golden Dataset loader cell first."
)
assert "query_text_by_hash" in globals(), (
    "Run the embedding workload-discovery cell first."
)
assert "chunk_text_by_hash" in globals(), (
    "Run the embedding workload-discovery cell first."
)
assert "embedding_models" in globals(), (
    "Run the local embedding-model validation cell first."
)
assert "multilingual_e5_small" in embedding_models, (
    "The multilingual_e5_small model is not loaded."
)

e5_model = embedding_models["multilingual_e5_small"]

project_root = golden_dataset_path.parents[3]
e5_instructional_run_directory = (
    project_root
    / "artifacts"
    / "phase2"
    / "embedding_benchmark"
    / golden_dataset_context["sha256"]
)
e5_instructional_run_directory.mkdir(parents=True, exist_ok=True)

e5_instructional_cache_path = (
    e5_instructional_run_directory
    / "multilingual_e5_small_instructional_embeddings.npz"
)
e5_instructional_manifest_path = (
    e5_instructional_run_directory
    / "multilingual_e5_small_instructional_manifest.json"
)

query_hashes = list(query_text_by_hash.keys())
passage_hashes = list(chunk_text_by_hash.keys())

def hash_sequence(values):
    return hashlib.sha256(
        "\n".join(sorted(values)).encode("utf-8")
    ).hexdigest()

expected_query_hash_sequence = hash_sequence(query_hashes)
expected_passage_hash_sequence = hash_sequence(passage_hashes)

def encode_e5_instructional(text_by_hash, prefix, batch_size=128):
    hashes = list(text_by_hash.keys())
    prefixed_texts = [
        f"{prefix}: {text_by_hash[text_hash]}"
        for text_hash in hashes
    ]

    vectors = e5_model.encode(
        prefixed_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    return hashes, vectors

cache_is_reusable = False

if e5_instructional_cache_path.is_file():
    cached_arrays = np.load(
        e5_instructional_cache_path,
        allow_pickle=True,
    )

    cached_query_hashes = [
        str(text_hash)
        for text_hash in cached_arrays["query_hashes"].tolist()
    ]
    cached_passage_hashes = [
        str(text_hash)
        for text_hash in cached_arrays["passage_hashes"].tolist()
    ]

    cache_is_reusable = (
        set(cached_query_hashes) == set(query_hashes)
        and set(cached_passage_hashes) == set(passage_hashes)
        and len(cached_query_hashes) == len(set(cached_query_hashes))
        and len(cached_passage_hashes) == len(set(cached_passage_hashes))
        and cached_arrays["query_vectors"].shape
        == (len(query_hashes), 384)
        and cached_arrays["passage_vectors"].shape
        == (len(passage_hashes), 384)
    )

if cache_is_reusable:
    e5_instructional_query_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            cached_query_hashes,
            cached_arrays["query_vectors"],
        )
    }
    e5_instructional_passage_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            cached_passage_hashes,
            cached_arrays["passage_vectors"],
        )
    }

    print("Existing E5 instructional cache loaded.")
else:
    print("Encoding E5 queries with prefix: query:")
    encoded_query_hashes, encoded_query_vectors = encode_e5_instructional(
        query_text_by_hash,
        prefix="query",
    )

    print("\nEncoding E5 passages with prefix: passage:")
    encoded_passage_hashes, encoded_passage_vectors = encode_e5_instructional(
        chunk_text_by_hash,
        prefix="passage",
    )

    np.savez_compressed(
        e5_instructional_cache_path,
        query_hashes=np.array(encoded_query_hashes, dtype=object),
        query_vectors=encoded_query_vectors,
        passage_hashes=np.array(encoded_passage_hashes, dtype=object),
        passage_vectors=encoded_passage_vectors,
    )

    e5_instructional_query_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            encoded_query_hashes,
            encoded_query_vectors,
        )
    }
    e5_instructional_passage_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            encoded_passage_hashes,
            encoded_passage_vectors,
        )
    }

e5_instructional_manifest = {
    "artifact": e5_instructional_cache_path.name,
    "golden_dataset_path": str(golden_dataset_path),
    "golden_dataset_sha256": golden_dataset_context["sha256"],
    "model_name": "intfloat/multilingual-e5-small",
    "query_prefix": "query",
    "passage_prefix": "passage",
    "normalization": "L2",
    "query_count": len(query_hashes),
    "passage_count": len(passage_hashes),
    "query_hash_set_sha256": expected_query_hash_sequence,
    "passage_hash_set_sha256": expected_passage_hash_sequence,
    "embedding_dimension": 384,
}

e5_instructional_manifest_path.write_text(
    json.dumps(
        e5_instructional_manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

e5_instructional_embeddings = {
    "model_name": "multilingual_e5_small_instructional",
    "model_id": "intfloat/multilingual-e5-small",
    "query_prefix": "query:",
    "passage_prefix": "passage:",
    "dimension": 384,
    "query_vectors_by_text_hash": e5_instructional_query_vectors,
    "passage_vectors_by_text_hash": e5_instructional_passage_vectors,
    "cache_path": e5_instructional_cache_path,
}

print("\n=== E5 INSTRUCTIONAL EMBEDDING CACHE: PASS ===")
print("Cache reused:", cache_is_reusable)
print("Active Golden Dataset hash:", golden_dataset_context["sha256"])
print("Queries:", len(e5_instructional_query_vectors))
print("Passages:", len(e5_instructional_passage_vectors))
print("Dimension:", e5_instructional_embeddings["dimension"])
print("Cache:", e5_instructional_cache_path)

Existing E5 instructional cache loaded.

=== E5 INSTRUCTIONAL EMBEDDING CACHE: PASS ===
Cache reused: True
Active Golden Dataset hash: b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Queries: 3735
Passages: 7698
Dimension: 384
Cache: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/multilingual_e5_small_instructional_embeddings.npz


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## کنترل آماده‌بودن Benchmark اصلاح‌شدهٔ E5

در این مرحله بررسی می‌کنیم که embeddingهای جدید E5 کامل، نرمال‌سازی‌شده و آمادهٔ ورود به همان Benchmark قبلی هستند.

هیچ فایل، نتیجه یا Benchmark قبلی تغییر نمی‌کند.

</div>

In [47]:
# Code Cell 31
# NB13 — E5 instructional embedding validation and benchmark preflight

expected_query_hashes = set(query_text_by_hash)
expected_passage_hashes = set(chunk_text_by_hash)

actual_query_hashes = set(
    e5_instructional_embeddings["query_vectors_by_text_hash"]
)
actual_passage_hashes = set(
    e5_instructional_embeddings["passage_vectors_by_text_hash"]
)

assert actual_query_hashes == expected_query_hashes, (
    "Instructional E5 query cache is incomplete or has unexpected keys."
)
assert actual_passage_hashes == expected_passage_hashes, (
    "Instructional E5 passage cache is incomplete or has unexpected keys."
)

query_matrix = np.vstack(
    [
        e5_instructional_embeddings["query_vectors_by_text_hash"][text_hash]
        for text_hash in query_text_by_hash
    ]
)
passage_matrix = np.vstack(
    [
        e5_instructional_embeddings["passage_vectors_by_text_hash"][text_hash]
        for text_hash in chunk_text_by_hash
    ]
)

assert query_matrix.shape == (len(query_text_by_hash), 384)
assert passage_matrix.shape == (len(chunk_text_by_hash), 384)
assert np.isfinite(query_matrix).all()
assert np.isfinite(passage_matrix).all()

assert np.allclose(
    np.linalg.norm(query_matrix, axis=1),
    1.0,
    atol=1e-4,
), "Instructional E5 query vectors are not L2-normalized."

assert np.allclose(
    np.linalg.norm(passage_matrix, axis=1),
    1.0,
    atol=1e-4,
), "Instructional E5 passage vectors are not L2-normalized."

print("=== E5 INSTRUCTIONAL BENCHMARK PREFLIGHT ===")
print("Query vectors:", query_matrix.shape)
print("Passage vectors:", passage_matrix.shape)
print("Query prefix:", e5_instructional_embeddings["query_prefix"])
print("Passage prefix:", e5_instructional_embeddings["passage_prefix"])
print("\nE5 INSTRUCTIONAL BENCHMARK PREFLIGHT: PASS")

=== E5 INSTRUCTIONAL BENCHMARK PREFLIGHT ===
Query vectors: (3735, 384)
Passage vectors: (7698, 384)
Query prefix: query:
Passage prefix: passage:

E5 INSTRUCTIONAL BENCHMARK PREFLIGHT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## کشف قرارداد Benchmark بازیابی

پیش از اجرای Benchmark جدید، ساختار دقیق برچسب‌های مثبت و پرسش‌های واجد شرایط را از همان اجرای قبلی بررسی می‌کنیم.

هدف این است که E5 اصلاح‌شده دقیقاً با همان داده، همان چهار Chunking و همان معیارهای قبلی ارزیابی شود؛ بنابراین مقایسه کاملاً منصفانه خواهد بود.

</div>

In [48]:
# Code Cell 32
# NB13 — Retrieval benchmark contract discovery

def summarize_benchmark_object(name, value, max_keys=15):
    print(f"\n{name}")
    print("Type:", type(value).__name__)

    if isinstance(value, dict):
        print("Key count:", len(value))
        print("Key preview:", list(value.keys())[:max_keys])

        if value:
            first_key = next(iter(value))
            first_value = value[first_key]
            print("First key:", repr(first_key))
            print("First value type:", type(first_value).__name__)

            if isinstance(first_value, dict):
                print(
                    "First nested-key preview:",
                    list(first_value.keys())[:max_keys],
                )

    elif isinstance(value, (list, tuple, set)):
        print("Item count:", len(value))

        if value:
            first_value = next(iter(value))
            print("First item type:", type(first_value).__name__)

            if isinstance(first_value, dict):
                print(
                    "First-item key preview:",
                    list(first_value.keys())[:max_keys],
                )

    else:
        print("Value preview:", repr(value)[:500])


benchmark_contract_candidates = {
    name: value
    for name, value in globals().items()
    if any(
        token in name.lower()
        for token in (
            "evaluation_registry",
            "retrieval_registry",
            "positive",
            "eligible",
            "benchmark_query",
            "golden_query",
        )
    )
}

print("=== RETRIEVAL BENCHMARK CONTRACT DISCOVERY ===")

print("\nRetrieval candidates:")
for candidate_name, candidate in retrieval_chunking_candidates.items():
    print(
        f"- {candidate_name}: "
        f"records={candidate['records']}, "
        f"path={candidate['path']}"
    )

print("\nRelevant evaluation objects discovered:")
assert benchmark_contract_candidates, (
    "No benchmark-contract object was found. "
    "Run the Final Retrieval Evaluation Registry cell again."
)

for object_name, object_value in benchmark_contract_candidates.items():
    summarize_benchmark_object(object_name, object_value)

print("\nRETRIEVAL BENCHMARK CONTRACT DISCOVERY: PASS")

=== RETRIEVAL BENCHMARK CONTRACT DISCOVERY ===

Retrieval candidates:
- baseline_phase1_char_safe: records=3488, path=data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl
- parent_child_children: records=2721, path=data/phase2/chunks/nb11_benchmark_v1/parent_child_children_v1.jsonl
- structure_aware_medium: records=1799, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_medium_v1.jsonl
- structure_aware_small: records=2526, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_small_v1.jsonl

Relevant evaluation objects discovered:

retrieval_evaluation_registry
Type: dict
Key count: 4
Key preview: ['baseline_phase1_char_safe', 'parent_child_children', 'structure_aware_medium', 'structure_aware_small']
First key: 'baseline_phase1_char_safe'
First value type: dict
First nested-key preview: ['golden_v2_candidate_001b76f6baf33cd0899e5a26', 'golden_v2_candidate_001c270319a5eb8ada793abe', 'golden_v2_candidate_002c53884b14189cc56fd371', 'golden_v2_candidate_005

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## Benchmark بازیابی پرسش با E5 اصلاح‌شده

در این آزمایش، E5 با پیشوندهای استاندارد `query:` و `passage:` روی همان پرسش‌ها، همان برچسب‌های مثبت و همان چهار روش Chunking قبلی ارزیابی می‌شود.

نتیجه مستقیماً با اجرای قبلی `multilingual_e5_small` مقایسه خواهد شد.

</div>

In [49]:
# Code Cell 33
# NB13 — Instructional E5 cache-to-chunk key reconciliation

query_hash_by_text = {
    text: text_hash
    for text_hash, text in query_text_by_hash.items()
}

passage_hash_by_text = {
    text: text_hash
    for text_hash, text in chunk_text_by_hash.items()
}

assert len(query_hash_by_text) == len(query_text_by_hash)
assert len(passage_hash_by_text) == len(chunk_text_by_hash)

candidate_content_reconciliation = {}

print("=== INSTRUCTIONAL E5 CACHE-TO-CHUNK RECONCILIATION ===")
print("Cached passage texts:", len(passage_hash_by_text))

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = project_root / candidate["path"]

    assert candidate_path.is_file(), (
        f"Missing frozen chunk candidate: {candidate_path}"
    )

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    matched_records = [
        record
        for record in candidate_records
        if record["content"] in passage_hash_by_text
    ]
    unmatched_records = [
        record
        for record in candidate_records
        if record["content"] not in passage_hash_by_text
    ]

    candidate_content_reconciliation[candidate_name] = {
        "records": len(candidate_records),
        "matched_records": len(matched_records),
        "unmatched_records": len(unmatched_records),
        "first_unmatched_record": (
            unmatched_records[0]
            if unmatched_records
            else None
        ),
    }

    print(f"\n{candidate_name}")
    print("Records:", len(candidate_records))
    print("Exact cache-text matches:", len(matched_records))
    print("Unmatched records:", len(unmatched_records))

    if unmatched_records:
        sample = unmatched_records[0]["content"]
        sample_prefix = sample[:180]

        prefix_matches = [
            cached_text
            for cached_text in chunk_text_by_hash.values()
            if cached_text.startswith(sample_prefix)
            or sample.startswith(cached_text[:180])
        ]

        print("\nFirst unmatched-content diagnostic:")
        print("Raw content length:", len(sample))
        print("Raw content preview:", repr(sample[:300]))
        print("Possible cached-text prefix matches:", len(prefix_matches))

        if prefix_matches:
            matched_text = prefix_matches[0]
            print("Cached content length:", len(matched_text))
            print("Cached content preview:", repr(matched_text[:300]))
            print("Texts exactly equal:", sample == matched_text)

print("\nAvailable text-normalization functions:")
normalization_function_names = sorted(
    name
    for name, value in globals().items()
    if callable(value)
    and any(
        token in name.lower()
        for token in (
            "normalize",
            "canonical",
            "clean_text",
            "text_hash",
            "hash_text",
        )
    )
)

for function_name in normalization_function_names:
    print("-", function_name)

print("\nINSTRUCTIONAL E5 CACHE-TO-CHUNK RECONCILIATION: PASS")

=== INSTRUCTIONAL E5 CACHE-TO-CHUNK RECONCILIATION ===
Cached passage texts: 7698

baseline_phase1_char_safe
Records: 3488
Exact cache-text matches: 116
Unmatched records: 3372

First unmatched-content diagnostic:
Raw content length: 1200
Raw content preview: 'دویدم و دویدم،\nسر کوهی رسیدم،\nدو تا خاتونُ دیدم ...\nاین شعر از معروف\u200cترین شعرهای دوران کودکی ماست. ماجرای پسر بچه\u200cای که در مسیری طولانی با افرادی گوناگون مواجه می\u200cشود و از هر کس چیزی می\u200cگیرد و چیزی به آ\u200cن\u200cها می\u200cدهد. چیزهایی که خودش احتیاجی به آن\u200cها ندارد، ولی نیازِ طرف مقابل را برطرف می\u200cکند. احتمالا'
Possible cached-text prefix matches: 0

parent_child_children
Records: 2721
Exact cache-text matches: 206
Unmatched records: 2515

First unmatched-content diagnostic:
Raw content length: 1316
Raw content preview: 'دویدم و دویدم،\nسر کوهی رسیدم،\nدو تا خاتونُ دیدم ...\nاین شعر از معروف\u200cترین شعرهای دوران کودکی ماست. ماجرای پسر بچه\u200cای که در مسیری طولانی با افرادی گوناگون مواجه می

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تعیین کلید متنی استاندارد برای Chunkها

فایل‌های Chunk شامل متن خام هستند، اما cache embedding با نسخهٔ استانداردشدهٔ متن ساخته شده است.

در این مرحله تابع استانداردسازی درست را تعیین می‌کنیم تا Benchmark اصلاح‌شده دقیقاً با همان cache معتبر اجرا شود.

</div>

In [50]:
# Code Cell 34
# NB13 — Canonical chunk-text key validation and selection

canonical_key_functions = {
    "canonical_exact_text": canonical_exact_text,
    "canonical_comparison_text": canonical_comparison_text,
}

canonical_key_validation = {}
all_candidate_raw_contents = []

print("=== CANONICAL CHUNK-TEXT KEY VALIDATION ===")

for function_name, canonicalize_text in canonical_key_functions.items():
    print(f"\nFunction: {function_name}")

    function_results = {}

    for candidate_name, candidate in retrieval_chunking_candidates.items():
        candidate_path = project_root / candidate["path"]

        candidate_records = [
            json.loads(line)
            for line in candidate_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

        if function_name == "canonical_exact_text":
            all_candidate_raw_contents.extend(
                record["content"]
                for record in candidate_records
            )

        matched_records = [
            record
            for record in candidate_records
            if canonicalize_text(record["content"]) in passage_hash_by_text
        ]

        function_results[candidate_name] = {
            "records": len(candidate_records),
            "canonical_matches": len(matched_records),
            "unmatched_records": len(candidate_records) - len(matched_records),
        }

        print(
            f"- {candidate_name}: "
            f"{len(matched_records)} / {len(candidate_records)} "
            f"canonical matches"
        )

    canonical_key_validation[function_name] = function_results

fully_matching_functions = [
    function_name
    for function_name, candidate_results in canonical_key_validation.items()
    if all(
        result["canonical_matches"] == result["records"]
        for result in candidate_results.values()
    )
]

assert "canonical_exact_text" in fully_matching_functions, (
    "canonical_exact_text does not fully match the existing embedding cache."
)

canonical_functions_are_equivalent = all(
    canonical_exact_text(content)
    == canonical_comparison_text(content)
    for content in all_candidate_raw_contents
)

active_chunk_cache_key_function_name = "canonical_exact_text"
active_chunk_cache_key_function = canonical_exact_text

print(
    "\nFunctions equivalent for all frozen candidate contents:",
    canonical_functions_are_equivalent,
)
print("Selected cache-key function:", active_chunk_cache_key_function_name)
print("\nCANONICAL CHUNK-TEXT KEY VALIDATION: PASS")

=== CANONICAL CHUNK-TEXT KEY VALIDATION ===

Function: canonical_exact_text
- baseline_phase1_char_safe: 3488 / 3488 canonical matches
- parent_child_children: 2721 / 2721 canonical matches
- structure_aware_medium: 1799 / 1799 canonical matches
- structure_aware_small: 2526 / 2526 canonical matches

Function: canonical_comparison_text
- baseline_phase1_char_safe: 3488 / 3488 canonical matches
- parent_child_children: 2721 / 2721 canonical matches
- structure_aware_medium: 1799 / 1799 canonical matches
- structure_aware_small: 2526 / 2526 canonical matches

Functions equivalent for all frozen candidate contents: True
Selected cache-key function: canonical_exact_text

CANONICAL CHUNK-TEXT KEY VALIDATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## Benchmark بازیابی پرسش با E5 اصلاح‌شده

E5 با پیشوندهای استاندارد `query:` و `passage:` اکنون با همان ۵۵۳ پرسش، همان ۲۷ seed واجد شرایط، همان برچسب‌های مثبت و همان چهار روش Chunking قبلی ارزیابی می‌شود.

نتیجه مستقیماً با E5 قبلی مقایسه خواهد شد. هیچ نتیجهٔ قبلی جایگزین نمی‌شود.

</div>

In [51]:
# Code Cell 35
# NB13 — Full question-retrieval benchmark for instructional E5

import time

instructional_benchmark_queries = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(instructional_benchmark_queries) == len(common_eligible_seed_ids), (
    "Instructional benchmark query count must match the active Golden v2 eligible set."
)

assert all(
    record["question"] in query_hash_by_text
    for record in instructional_benchmark_queries
), "One or more Golden questions are missing from the instructional E5 cache."

instructional_query_matrix = np.vstack(
    [
        e5_instructional_embeddings["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in instructional_benchmark_queries
    ]
)

e5_instructional_benchmark_results = {}

print("=== E5 INSTRUCTIONAL FULL RETRIEVAL BENCHMARK ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(instructional_benchmark_queries))
print("Evaluation cutoffs: (1, 3, 5, 10)")

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]

    canonical_candidate_texts = [
        active_chunk_cache_key_function(record["content"])
        for record in candidate_records
    ]

    assert all(
        canonical_text in passage_hash_by_text
        for canonical_text in canonical_candidate_texts
    ), (
        f"{candidate_name} contains a Chunk not represented in "
        "the instructional E5 cache."
    )

    candidate_passage_matrix = np.vstack(
        [
            e5_instructional_embeddings["passage_vectors_by_text_hash"][
                passage_hash_by_text[canonical_text]
            ]
            for canonical_text in canonical_candidate_texts
        ]
    )

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        seed_entry = seed_registry[seed_id]

        assert seed_entry["eligible"] is True, (
            f"{candidate_name}, seed {seed_id} is unexpectedly ineligible."
        )

        positive_ids = set(seed_entry["positive_chunk_ids"])
        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    started_at = time.perf_counter()
    similarity_matrix = instructional_query_matrix @ candidate_passage_matrix.T
    ranked_indexes = np.argsort(-similarity_matrix, axis=1)
    elapsed_seconds = time.perf_counter() - started_at

    first_positive_ranks = []
    hit_lists = {cutoff: [] for cutoff in (1, 3, 5, 10)}

    for query_index, query_record in enumerate(instructional_benchmark_queries):
        seed_id = query_record["source_seed_id"]
        positive_indexes = positive_chunk_indexes_by_seed[seed_id]

        ranked_candidate_indexes = ranked_indexes[query_index]
        positive_rank_positions = np.flatnonzero(
            np.isin(ranked_candidate_indexes, list(positive_indexes))
        )

        first_rank = int(positive_rank_positions[0]) + 1
        first_positive_ranks.append(first_rank)

        for cutoff in hit_lists:
            hit_lists[cutoff].append(first_rank <= cutoff)

    metrics = {
        "recall_at_1": float(np.mean(hit_lists[1])),
        "recall_at_3": float(np.mean(hit_lists[3])),
        "recall_at_5": float(np.mean(hit_lists[5])),
        "recall_at_10": float(np.mean(hit_lists[10])),
        "mrr": float(np.mean([1 / rank for rank in first_positive_ranks])),
        "mean_first_positive_rank": float(np.mean(first_positive_ranks)),
        "seconds": round(elapsed_seconds, 3),
        "query_count": len(instructional_benchmark_queries),
        "candidate_chunk_count": len(candidate_records),
    }

    e5_instructional_benchmark_results[candidate_name] = metrics
    prior_metrics = embedding_benchmark_results["multilingual_e5_small"][candidate_name]

    print(f"\nmultilingual_e5_small_instructional × {candidate_name}")
    print(f"- Recall@1: {metrics['recall_at_1']:.4f}")
    print(f"- Recall@3: {metrics['recall_at_3']:.4f}")
    print(f"- Recall@5: {metrics['recall_at_5']:.4f}")
    print(f"- Recall@10: {metrics['recall_at_10']:.4f}")
    print(f"- MRR: {metrics['mrr']:.4f}")
    print(
        f"- Δ Recall@10 vs prior E5: "
        f"{metrics['recall_at_10'] - prior_metrics['recall_at_10']:+.4f}"
    )
    print(
        f"- Δ MRR vs prior E5: "
        f"{metrics['mrr'] - prior_metrics['mrr']:+.4f}"
    )
    print(f"- Seconds: {metrics['seconds']}")

e5_instructional_benchmark_report_path = (
    active_benchmark_run_directory
    / "multilingual_e5_small_instructional_retrieval_benchmark.json"
)

e5_instructional_benchmark_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": e5_instructional_embeddings["model_id"],
            "query_prefix": e5_instructional_embeddings["query_prefix"],
            "passage_prefix": e5_instructional_embeddings["passage_prefix"],
            "comparison_baseline": "multilingual_e5_small_without_prefixes",
            "common_eligible_seed_ids": common_eligible_seed_ids,
            "query_count": len(instructional_benchmark_queries),
            "results": e5_instructional_benchmark_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", e5_instructional_benchmark_report_path)
print("\nE5 INSTRUCTIONAL FULL RETRIEVAL BENCHMARK: PASS")

=== E5 INSTRUCTIONAL FULL RETRIEVAL BENCHMARK ===
Eligible seeds: 3813
Eligible Golden queries: 3813
Evaluation cutoffs: (1, 3, 5, 10)

multilingual_e5_small_instructional × baseline_phase1_char_safe
- Recall@1: 0.3110
- Recall@3: 0.4542
- Recall@5: 0.5229
- Recall@10: 0.6113
- MRR: 0.4113
- Δ Recall@10 vs prior E5: +0.0000
- Δ MRR vs prior E5: -0.0000
- Seconds: 0.934

multilingual_e5_small_instructional × parent_child_children
- Recall@1: 0.3155
- Recall@3: 0.4519
- Recall@5: 0.5235
- Recall@10: 0.6019
- MRR: 0.4127
- Δ Recall@10 vs prior E5: +0.0000
- Δ MRR vs prior E5: +0.0001
- Seconds: 0.703

multilingual_e5_small_instructional × structure_aware_medium
- Recall@1: 0.2856
- Recall@3: 0.4136
- Recall@5: 0.4839
- Recall@10: 0.5675
- MRR: 0.3786
- Δ Recall@10 vs prior E5: +0.0000
- Δ MRR vs prior E5: -0.0001
- Seconds: 0.483

multilingual_e5_small_instructional × structure_aware_small
- Recall@1: 0.3068
- Recall@3: 0.4498
- Recall@5: 0.5240
- Recall@10: 0.5972
- MRR: 0.4068
- Δ Recal

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی بازیابی واژگانی فارسی

نتایج قبلی نشان دادند که بازیابی embedding برای پرسش‌های Golden ضعیف‌تر از انتظار است.

در این مرحله یک بازیابی واژگانی مستقل بر پایهٔ TF-IDF فارسی اجرا می‌کنیم. این روش از شباهت واژه‌ها و عبارت‌های پرسش با متن Chunk استفاده می‌کند و نتیجهٔ آن با همان برچسب‌های مثبت و همان چهار روش Chunking ارزیابی می‌شود.

این فقط یک Baseline تشخیصی است و جایگزین embedding یا انتخاب نهایی سیستم نیست.

</div>

In [52]:
# Code Cell 36
# NB13 — Persian lexical retrieval baseline: TF-IDF word and phrase matching

import re
import time
from sklearn.feature_extraction.text import TfidfVectorizer

def persian_lexical_tokens(text):
    canonical_text = canonical_exact_text(text).lower()
    return re.findall(r"[\u0600-\u06FFa-z0-9]+", canonical_text)

lexical_benchmark_queries = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(lexical_benchmark_queries) == len(common_eligible_seed_ids)

lexical_query_texts = [
    record["question"]
    for record in lexical_benchmark_queries
]

lexical_retrieval_results = {}

print("=== PERSIAN LEXICAL RETRIEVAL BASELINE ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(lexical_benchmark_queries))
print("Tokenizer: Persian word tokens")
print("Features: word unigrams and bigrams")
print("Evaluation cutoffs: (1, 3, 5, 10)")

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]
    candidate_texts = [
        canonical_exact_text(record["content"])
        for record in candidate_records
    ]

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        seed_entry = seed_registry[seed_id]
        assert seed_entry["eligible"] is True

        positive_ids = set(seed_entry["positive_chunk_ids"])
        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    vectorizer = TfidfVectorizer(
        tokenizer=persian_lexical_tokens,
        token_pattern=None,
        ngram_range=(1, 2),
        lowercase=False,
        norm="l2",
        sublinear_tf=True,
        dtype=np.float32,
    )

    started_at = time.perf_counter()

    candidate_tfidf = vectorizer.fit_transform(candidate_texts)
    query_tfidf = vectorizer.transform(lexical_query_texts)

    similarity_matrix = (query_tfidf @ candidate_tfidf.T).toarray()
    ranked_indexes = np.argsort(-similarity_matrix, axis=1)

    elapsed_seconds = time.perf_counter() - started_at

    first_positive_ranks = []
    hit_lists = {cutoff: [] for cutoff in (1, 3, 5, 10)}

    for query_index, query_record in enumerate(lexical_benchmark_queries):
        seed_id = query_record["source_seed_id"]
        positive_indexes = positive_chunk_indexes_by_seed[seed_id]

        ranked_candidate_indexes = ranked_indexes[query_index]
        positive_rank_positions = np.flatnonzero(
            np.isin(ranked_candidate_indexes, list(positive_indexes))
        )

        first_rank = int(positive_rank_positions[0]) + 1
        first_positive_ranks.append(first_rank)

        for cutoff in hit_lists:
            hit_lists[cutoff].append(first_rank <= cutoff)

    metrics = {
        "recall_at_1": float(np.mean(hit_lists[1])),
        "recall_at_3": float(np.mean(hit_lists[3])),
        "recall_at_5": float(np.mean(hit_lists[5])),
        "recall_at_10": float(np.mean(hit_lists[10])),
        "mrr": float(np.mean([1 / rank for rank in first_positive_ranks])),
        "mean_first_positive_rank": float(np.mean(first_positive_ranks)),
        "seconds": round(elapsed_seconds, 3),
        "query_count": len(lexical_benchmark_queries),
        "candidate_chunk_count": len(candidate_records),
        "vocabulary_size": int(len(vectorizer.vocabulary_)),
    }

    lexical_retrieval_results[candidate_name] = metrics

    print(f"\npersian_tfidf_lexical × {candidate_name}")
    print(f"- Recall@1: {metrics['recall_at_1']:.4f}")
    print(f"- Recall@3: {metrics['recall_at_3']:.4f}")
    print(f"- Recall@5: {metrics['recall_at_5']:.4f}")
    print(f"- Recall@10: {metrics['recall_at_10']:.4f}")
    print(f"- MRR: {metrics['mrr']:.4f}")
    print(f"- Vocabulary size: {metrics['vocabulary_size']}")
    print(f"- Seconds: {metrics['seconds']}")

print("\nPERSIAN LEXICAL RETRIEVAL BASELINE: PASS")

=== PERSIAN LEXICAL RETRIEVAL BASELINE ===
Eligible seeds: 3813
Eligible Golden queries: 3813
Tokenizer: Persian word tokens
Features: word unigrams and bigrams
Evaluation cutoffs: (1, 3, 5, 10)

persian_tfidf_lexical × baseline_phase1_char_safe
- Recall@1: 0.4626
- Recall@3: 0.6001
- Recall@5: 0.6512
- Recall@10: 0.7160
- MRR: 0.5518
- Vocabulary size: 298166
- Seconds: 3.502

persian_tfidf_lexical × parent_child_children
- Recall@1: 0.4477
- Recall@3: 0.5987
- Recall@5: 0.6543
- Recall@10: 0.7265
- MRR: 0.5452
- Vocabulary size: 294157
- Seconds: 3.053

persian_tfidf_lexical × structure_aware_medium
- Recall@1: 0.4430
- Recall@3: 0.5956
- Recall@5: 0.6491
- Recall@10: 0.7262
- MRR: 0.5412
- Vocabulary size: 294612
- Seconds: 2.781

persian_tfidf_lexical × structure_aware_small
- Recall@1: 0.4542
- Recall@3: 0.6001
- Recall@5: 0.6572
- Recall@10: 0.7299
- MRR: 0.5496
- Vocabulary size: 294261
- Seconds: 3.038

PERSIAN LEXICAL RETRIEVAL BASELINE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی Hybrid Retrieval با ترکیب E5 و بازیابی واژگانی

Embedding و بازیابی واژگانی هر کدام ممکن است بخشی از پرسش‌ها را بهتر بازیابی کنند.

در این مرحله رتبه‌های E5 و TF-IDF فارسی با روش Reciprocal Rank Fusion یا RRF ترکیب می‌شوند. هدف، بررسی این است که آیا ترکیب دو روش از هر روش به‌تنهایی بهتر است یا خیر.

این یک آزمایش Benchmark است و هنوز انتخاب نهایی سیستم نیست.

</div>

In [53]:
# Code Cell 37
# NB13 — Hybrid E5 + Persian lexical retrieval benchmark with RRF

from sklearn.feature_extraction.text import TfidfVectorizer
import time

def calculate_retrieval_metrics(
    ranked_indexes,
    query_records,
    positive_chunk_indexes_by_seed,
):
    first_positive_ranks = []
    hit_lists = {cutoff: [] for cutoff in (1, 3, 5, 10)}

    for query_index, query_record in enumerate(query_records):
        seed_id = query_record["source_seed_id"]
        positive_indexes = positive_chunk_indexes_by_seed[seed_id]

        positive_rank_positions = np.flatnonzero(
            np.isin(
                ranked_indexes[query_index],
                list(positive_indexes),
            )
        )

        first_rank = int(positive_rank_positions[0]) + 1
        first_positive_ranks.append(first_rank)

        for cutoff in hit_lists:
            hit_lists[cutoff].append(first_rank <= cutoff)

    return {
        "recall_at_1": float(np.mean(hit_lists[1])),
        "recall_at_3": float(np.mean(hit_lists[3])),
        "recall_at_5": float(np.mean(hit_lists[5])),
        "recall_at_10": float(np.mean(hit_lists[10])),
        "mrr": float(np.mean([1 / rank for rank in first_positive_ranks])),
        "mean_first_positive_rank": float(np.mean(first_positive_ranks)),
    }


hybrid_query_records = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(hybrid_query_records) == len(common_eligible_seed_ids)

hybrid_e5_query_matrix = np.vstack(
    [
        e5_instructional_embeddings["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in hybrid_query_records
    ]
)

hybrid_query_texts = [
    record["question"]
    for record in hybrid_query_records
]

rrf_constant = 60
hybrid_retrieval_results = {}

print("=== HYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(hybrid_query_records))
print("Fusion method: Reciprocal Rank Fusion")
print("RRF constant:", rrf_constant)

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]
    candidate_texts = [
        canonical_exact_text(record["content"])
        for record in candidate_records
    ]

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        seed_entry = seed_registry[seed_id]
        positive_ids = set(seed_entry["positive_chunk_ids"])

        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    e5_candidate_matrix = np.vstack(
        [
            e5_instructional_embeddings["passage_vectors_by_text_hash"][
                passage_hash_by_text[text]
            ]
            for text in candidate_texts
        ]
    )

    lexical_vectorizer = TfidfVectorizer(
        tokenizer=persian_lexical_tokens,
        token_pattern=None,
        ngram_range=(1, 2),
        lowercase=False,
        norm="l2",
        sublinear_tf=True,
        dtype=np.float32,
    )

    started_at = time.perf_counter()

    lexical_candidate_matrix = lexical_vectorizer.fit_transform(candidate_texts)
    lexical_query_matrix = lexical_vectorizer.transform(hybrid_query_texts)

    e5_scores = hybrid_e5_query_matrix @ e5_candidate_matrix.T
    lexical_scores = (
        lexical_query_matrix @ lexical_candidate_matrix.T
    ).toarray()

    e5_ranked_indexes = np.argsort(-e5_scores, axis=1)
    lexical_ranked_indexes = np.argsort(-lexical_scores, axis=1)

    e5_rank_positions = np.empty_like(e5_ranked_indexes)
    lexical_rank_positions = np.empty_like(lexical_ranked_indexes)

    row_indexes = np.arange(len(hybrid_query_records))[:, None]

    e5_rank_positions[row_indexes, e5_ranked_indexes] = (
        np.arange(len(candidate_records))[None, :] + 1
    )
    lexical_rank_positions[row_indexes, lexical_ranked_indexes] = (
        np.arange(len(candidate_records))[None, :] + 1
    )

    rrf_scores = (
        1.0 / (rrf_constant + e5_rank_positions)
        + 1.0 / (rrf_constant + lexical_rank_positions)
    )
    rrf_ranked_indexes = np.argsort(-rrf_scores, axis=1)

    elapsed_seconds = time.perf_counter() - started_at

    e5_metrics = calculate_retrieval_metrics(
        e5_ranked_indexes,
        hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    lexical_metrics = calculate_retrieval_metrics(
        lexical_ranked_indexes,
        hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    rrf_metrics = calculate_retrieval_metrics(
        rrf_ranked_indexes,
        hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )

    rrf_metrics["seconds"] = round(elapsed_seconds, 3)
    rrf_metrics["candidate_chunk_count"] = len(candidate_records)

    hybrid_retrieval_results[candidate_name] = {
        "e5_instructional": e5_metrics,
        "persian_tfidf_lexical": lexical_metrics,
        "rrf_e5_plus_lexical": rrf_metrics,
    }

    print(f"\nE5 + lexical RRF × {candidate_name}")
    print(f"- E5 Recall@10: {e5_metrics['recall_at_10']:.4f}")
    print(f"- Lexical Recall@10: {lexical_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid Recall@10: {rrf_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid MRR: {rrf_metrics['mrr']:.4f}")
    print(f"- Seconds: {rrf_metrics['seconds']}")

print("\nHYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS")

=== HYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===
Eligible seeds: 3813
Eligible Golden queries: 3813
Fusion method: Reciprocal Rank Fusion
RRF constant: 60

E5 + lexical RRF × baseline_phase1_char_safe
- E5 Recall@10: 0.6113
- Lexical Recall@10: 0.7160
- Hybrid Recall@10: 0.7603
- Hybrid MRR: 0.5559
- Seconds: 6.295

E5 + lexical RRF × parent_child_children
- E5 Recall@10: 0.6019
- Lexical Recall@10: 0.7265
- Hybrid Recall@10: 0.7561
- Hybrid MRR: 0.5542
- Seconds: 4.857

E5 + lexical RRF × structure_aware_medium
- E5 Recall@10: 0.5675
- Lexical Recall@10: 0.7262
- Hybrid Recall@10: 0.7238
- Hybrid MRR: 0.5186
- Seconds: 3.894

E5 + lexical RRF × structure_aware_small
- E5 Recall@10: 0.5972
- Lexical Recall@10: 0.7299
- Hybrid Recall@10: 0.7516
- Hybrid MRR: 0.5497
- Seconds: 4.591

HYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## آماده‌سازی Hybrid Retrieval با Arvan و بازیابی واژگانی

مدل Arvan در بازیابی متن شواهد عملکرد خوبی نشان داد، اما در بازیابی مستقیم پرسش ضعیف‌تر بود.

در این مرحله ابتدا محل دقیق embeddingهای Arvan موجود در حافظهٔ Notebook را پیدا می‌کنیم تا در مرحلهٔ بعد، بدون اجرای دوبارهٔ API و بدون هزینهٔ جدید، آن را با بازیابی واژگانی ترکیب کنیم.

</div>

In [54]:
# Code Cell 38
# NB13 — Arvan vector-cache discovery for hybrid retrieval

arvan_vector_lookup_candidates = []

for global_name, global_value in list(globals().items()):
    if not isinstance(global_value, dict):
        continue

    if {
        "query_vectors_by_text_hash",
        "passage_vectors_by_text_hash",
    }.issubset(global_value.keys()):
        arvan_vector_lookup_candidates.append(
            {
                "container_name": global_name,
                "lookup_name": None,
                "lookup": global_value,
            }
        )

    for nested_name, nested_value in global_value.items():
        if not isinstance(nested_value, dict):
            continue

        if {
            "query_vectors_by_text_hash",
            "passage_vectors_by_text_hash",
        }.issubset(nested_value.keys()):
            arvan_vector_lookup_candidates.append(
                {
                    "container_name": global_name,
                    "lookup_name": nested_name,
                    "lookup": nested_value,
                }
            )

print("=== ARVAN VECTOR-CACHE DISCOVERY ===")
print("Vector-cache candidates found:", len(arvan_vector_lookup_candidates))

for candidate in arvan_vector_lookup_candidates:
    lookup = candidate["lookup"]
    model_identifier = (
        lookup.get("model_id")
        or lookup.get("provider")
        or candidate["lookup_name"]
        or candidate["container_name"]
    )

    print("\nContainer:", candidate["container_name"])
    print("Nested lookup:", candidate["lookup_name"])
    print("Model identifier:", model_identifier)
    print(
        "Query-vector count:",
        len(lookup["query_vectors_by_text_hash"]),
    )
    print(
        "Passage-vector count:",
        len(lookup["passage_vectors_by_text_hash"]),
    )
    print(
        "Dimension:",
        lookup.get("dimension", lookup.get("embedding_dimension")),
    )

# Resolve the canonical Arvan cache deterministically.
# Do not infer uniqueness from global aliases: the same cache is intentionally
# exposed through benchmark_embeddings_by_model and arvan_lookup.
assert "benchmark_embeddings_by_model" in globals(), (
    "benchmark_embeddings_by_model is unavailable."
)
assert "arvan_text_embedding_3_large" in benchmark_embeddings_by_model, (
    "Canonical Arvan embedding cache is missing from benchmark_embeddings_by_model."
)

arvan_embedding_lookup = benchmark_embeddings_by_model[
    "arvan_text_embedding_3_large"
]

resolved_arvan_dimension = arvan_embedding_lookup.get(
    "dimension",
    arvan_embedding_lookup.get("embedding_dimension"),
)
assert resolved_arvan_dimension == 3072, (
    f"Unexpected Arvan embedding dimension: {resolved_arvan_dimension}"
)

assert len(arvan_embedding_lookup["query_vectors_by_text_hash"]) == len(query_text_by_hash)
assert len(arvan_embedding_lookup["passage_vectors_by_text_hash"]) == len(chunk_text_by_hash)

print("\nResolved Arvan model:", arvan_embedding_lookup.get("model_id"))
print("\nARVAN VECTOR-CACHE DISCOVERY: PASS")

=== ARVAN VECTOR-CACHE DISCOVERY ===
Vector-cache candidates found: 7

Container: benchmark_embeddings_by_model
Nested lookup: arvan_text_embedding_3_large
Model identifier: arvan_text_embedding_3_large
Query-vector count: 3735
Passage-vector count: 7698
Dimension: 3072

Container: benchmark_embeddings_by_model
Nested lookup: multilingual_e5_small
Model identifier: multilingual_e5_small
Query-vector count: 3735
Passage-vector count: 7698
Dimension: 384

Container: benchmark_embeddings_by_model
Nested lookup: all_minilm_l6_v2
Model identifier: all_minilm_l6_v2
Query-vector count: 3735
Passage-vector count: 7698
Dimension: 384

Container: embedding_lookup
Nested lookup: None
Model identifier: embedding_lookup
Query-vector count: 3735
Passage-vector count: 7698
Dimension: 384

Container: arvan_lookup
Nested lookup: None
Model identifier: arvan_lookup
Query-vector count: 3735
Passage-vector count: 7698
Dimension: 3072

Container: local_lookup
Nested lookup: None
Model identifier: local_loo

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی Hybrid Retrieval با Arvan و بازیابی واژگانی

در این آزمایش، رتبه‌های مدل Arvan و بازیابی واژگانی فارسی با روش RRF ترکیب می‌شوند.

هدف این است که بررسی کنیم آیا مدل معنایی Arvan و تطبیق واژگانی، پرسش‌های Golden را مکمل یکدیگر بازیابی می‌کنند یا خیر.

</div>

In [55]:
# Code Cell 39
# NB13 — Hybrid Arvan + Persian lexical retrieval benchmark with RRF

arvan_hybrid_query_records = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(arvan_hybrid_query_records) == len(common_eligible_seed_ids)

assert all(
    record["question"] in query_hash_by_text
    for record in arvan_hybrid_query_records
)

arvan_query_matrix = np.vstack(
    [
        arvan_embedding_lookup["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in arvan_hybrid_query_records
    ]
)

arvan_hybrid_query_texts = [
    record["question"]
    for record in arvan_hybrid_query_records
]

arvan_lexical_hybrid_results = {}
rrf_constant = 60

print("=== HYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(arvan_hybrid_query_records))
print("Fusion method: Reciprocal Rank Fusion")
print("RRF constant:", rrf_constant)

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]
    candidate_texts = [
        canonical_exact_text(record["content"])
        for record in candidate_records
    ]

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        positive_ids = set(seed_registry[seed_id]["positive_chunk_ids"])

        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    arvan_candidate_matrix = np.vstack(
        [
            arvan_embedding_lookup["passage_vectors_by_text_hash"][
                passage_hash_by_text[text]
            ]
            for text in candidate_texts
        ]
    )

    lexical_vectorizer = TfidfVectorizer(
        tokenizer=persian_lexical_tokens,
        token_pattern=None,
        ngram_range=(1, 2),
        lowercase=False,
        norm="l2",
        sublinear_tf=True,
        dtype=np.float32,
    )

    started_at = time.perf_counter()

    lexical_candidate_matrix = lexical_vectorizer.fit_transform(candidate_texts)
    lexical_query_matrix = lexical_vectorizer.transform(
        arvan_hybrid_query_texts
    )

    arvan_scores = arvan_query_matrix @ arvan_candidate_matrix.T
    lexical_scores = (
        lexical_query_matrix @ lexical_candidate_matrix.T
    ).toarray()

    arvan_ranked_indexes = np.argsort(-arvan_scores, axis=1)
    lexical_ranked_indexes = np.argsort(-lexical_scores, axis=1)

    arvan_rank_positions = np.empty_like(arvan_ranked_indexes)
    lexical_rank_positions = np.empty_like(lexical_ranked_indexes)

    row_indexes = np.arange(len(arvan_hybrid_query_records))[:, None]
    candidate_rank_positions = (
        np.arange(len(candidate_records))[None, :] + 1
    )

    arvan_rank_positions[row_indexes, arvan_ranked_indexes] = (
        candidate_rank_positions
    )
    lexical_rank_positions[row_indexes, lexical_ranked_indexes] = (
        candidate_rank_positions
    )

    rrf_scores = (
        1.0 / (rrf_constant + arvan_rank_positions)
        + 1.0 / (rrf_constant + lexical_rank_positions)
    )
    rrf_ranked_indexes = np.argsort(-rrf_scores, axis=1)

    elapsed_seconds = time.perf_counter() - started_at

    arvan_metrics = calculate_retrieval_metrics(
        arvan_ranked_indexes,
        arvan_hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    lexical_metrics = calculate_retrieval_metrics(
        lexical_ranked_indexes,
        arvan_hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    rrf_metrics = calculate_retrieval_metrics(
        rrf_ranked_indexes,
        arvan_hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )

    rrf_metrics["seconds"] = round(elapsed_seconds, 3)
    rrf_metrics["candidate_chunk_count"] = len(candidate_records)

    arvan_lexical_hybrid_results[candidate_name] = {
        "arvan": arvan_metrics,
        "persian_tfidf_lexical": lexical_metrics,
        "rrf_arvan_plus_lexical": rrf_metrics,
    }

    print(f"\nArvan + lexical RRF × {candidate_name}")
    print(f"- Arvan Recall@10: {arvan_metrics['recall_at_10']:.4f}")
    print(f"- Lexical Recall@10: {lexical_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid Recall@10: {rrf_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid MRR: {rrf_metrics['mrr']:.4f}")
    print(f"- Seconds: {rrf_metrics['seconds']}")

print("\nHYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS")

=== HYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===
Eligible seeds: 3813
Eligible Golden queries: 3813
Fusion method: Reciprocal Rank Fusion
RRF constant: 60

Arvan + lexical RRF × baseline_phase1_char_safe
- Arvan Recall@10: 0.5410
- Lexical Recall@10: 0.7160
- Hybrid Recall@10: 0.7438
- Hybrid MRR: 0.5157
- Seconds: 6.463

Arvan + lexical RRF × parent_child_children
- Arvan Recall@10: 0.5429
- Lexical Recall@10: 0.7265
- Hybrid Recall@10: 0.7440
- Hybrid MRR: 0.5118
- Seconds: 5.102

Arvan + lexical RRF × structure_aware_medium
- Arvan Recall@10: 0.5143
- Lexical Recall@10: 0.7262
- Hybrid Recall@10: 0.7220
- Hybrid MRR: 0.4862
- Seconds: 4.356

Arvan + lexical RRF × structure_aware_small
- Arvan Recall@10: 0.5389
- Lexical Recall@10: 0.7299
- Hybrid Recall@10: 0.7438
- Hybrid MRR: 0.5071
- Seconds: 5.161

HYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تحلیل شکست بازیابی در بهترین ترکیب فعلی

بهترین ترکیب فعلی، Arvan به‌همراه بازیابی واژگانی و Chunking والد–فرزند است.

در این مرحله عملکرد این ترکیب برای هر source seed جداگانه بررسی می‌شود تا مشخص شود کدام گروه از پرسش‌ها بازیابی می‌شوند، کدام گروه شکست می‌خورند و اصلاح بعدی باید روی چه بخش‌هایی متمرکز باشد.

</div>

In [56]:
# Code Cell 40
# NB13 — Per-seed failure profile for best current hybrid:
# Arvan + Persian lexical RRF × parent_child_children

best_candidate_name = "parent_child_children"
best_seed_registry = retrieval_evaluation_registry[best_candidate_name]
best_candidate_path = (
    project_root
    / retrieval_chunking_candidates[best_candidate_name]["path"]
)

best_candidate_records = [
    json.loads(line)
    for line in best_candidate_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

best_candidate_chunk_ids = [
    record["chunk_id"]
    for record in best_candidate_records
]
best_candidate_texts = [
    canonical_exact_text(record["content"])
    for record in best_candidate_records
]

best_positive_indexes_by_seed = {}

for seed_id in common_eligible_seed_ids:
    positive_ids = set(best_seed_registry[seed_id]["positive_chunk_ids"])

    positive_indexes = {
        index
        for index, chunk_id in enumerate(best_candidate_chunk_ids)
        if chunk_id in positive_ids
    }

    assert positive_indexes, (
        f"Seed {seed_id} has no indexed positive Chunk "
        f"for {best_candidate_name}."
    )

    best_positive_indexes_by_seed[seed_id] = positive_indexes

best_arvan_query_matrix = np.vstack(
    [
        arvan_embedding_lookup["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in arvan_hybrid_query_records
    ]
)

best_arvan_candidate_matrix = np.vstack(
    [
        arvan_embedding_lookup["passage_vectors_by_text_hash"][
            passage_hash_by_text[text]
        ]
        for text in best_candidate_texts
    ]
)

best_lexical_vectorizer = TfidfVectorizer(
    tokenizer=persian_lexical_tokens,
    token_pattern=None,
    ngram_range=(1, 2),
    lowercase=False,
    norm="l2",
    sublinear_tf=True,
    dtype=np.float32,
)

best_lexical_candidate_matrix = best_lexical_vectorizer.fit_transform(
    best_candidate_texts
)
best_lexical_query_matrix = best_lexical_vectorizer.transform(
    [
        record["question"]
        for record in arvan_hybrid_query_records
    ]
)

best_arvan_scores = (
    best_arvan_query_matrix @ best_arvan_candidate_matrix.T
)
best_lexical_scores = (
    best_lexical_query_matrix @ best_lexical_candidate_matrix.T
).toarray()

best_arvan_ranked_indexes = np.argsort(-best_arvan_scores, axis=1)
best_lexical_ranked_indexes = np.argsort(-best_lexical_scores, axis=1)

best_arvan_rank_positions = np.empty_like(best_arvan_ranked_indexes)
best_lexical_rank_positions = np.empty_like(best_lexical_ranked_indexes)

best_row_indexes = np.arange(
    len(arvan_hybrid_query_records)
)[:, None]
best_candidate_rank_positions = (
    np.arange(len(best_candidate_records))[None, :] + 1
)

best_arvan_rank_positions[
    best_row_indexes,
    best_arvan_ranked_indexes,
] = best_candidate_rank_positions

best_lexical_rank_positions[
    best_row_indexes,
    best_lexical_ranked_indexes,
] = best_candidate_rank_positions

best_rrf_scores = (
    1.0 / (60 + best_arvan_rank_positions)
    + 1.0 / (60 + best_lexical_rank_positions)
)
best_rrf_ranked_indexes = np.argsort(-best_rrf_scores, axis=1)

seed_failure_profile = {}

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]

    positive_rank_positions = np.flatnonzero(
        np.isin(
            best_rrf_ranked_indexes[query_index],
            list(positive_indexes),
        )
    )
    first_rank = int(positive_rank_positions[0]) + 1

    seed_failure_profile.setdefault(
        seed_id,
        {
            "seed_id": seed_id,
            "query_count": 0,
            "first_positive_ranks": [],
            "example_hit_questions": [],
            "example_miss_questions": [],
        },
    )

    seed_profile = seed_failure_profile[seed_id]
    seed_profile["query_count"] += 1
    seed_profile["first_positive_ranks"].append(first_rank)

    if first_rank <= 10 and len(seed_profile["example_hit_questions"]) < 2:
        seed_profile["example_hit_questions"].append(
            {
                "candidate_id": query_record["candidate_id"],
                "question": query_record["question"],
                "first_positive_rank": first_rank,
            }
        )

    if first_rank > 10 and len(seed_profile["example_miss_questions"]) < 2:
        seed_profile["example_miss_questions"].append(
            {
                "candidate_id": query_record["candidate_id"],
                "question": query_record["question"],
                "first_positive_rank": first_rank,
            }
        )

for seed_profile in seed_failure_profile.values():
    ranks = seed_profile["first_positive_ranks"]

    seed_profile["recall_at_1"] = float(
        np.mean([rank <= 1 for rank in ranks])
    )
    seed_profile["recall_at_3"] = float(
        np.mean([rank <= 3 for rank in ranks])
    )
    seed_profile["recall_at_5"] = float(
        np.mean([rank <= 5 for rank in ranks])
    )
    seed_profile["recall_at_10"] = float(
        np.mean([rank <= 10 for rank in ranks])
    )
    seed_profile["mrr"] = float(np.mean([1 / rank for rank in ranks]))
    seed_profile["mean_first_positive_rank"] = float(np.mean(ranks))
    seed_profile["miss_count_at_10"] = int(
        np.sum([rank > 10 for rank in ranks])
    )

seed_failure_profile_rows = sorted(
    seed_failure_profile.values(),
    key=lambda item: (
        item["recall_at_10"],
        item["mrr"],
        item["seed_id"],
    ),
)

seed_failure_profile_report_path = (
    active_benchmark_run_directory
    / "arvan_lexical_parent_child_seed_failure_profile.json"
)

seed_failure_profile_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": "arvan_text_embedding_3_large_plus_persian_tfidf",
            "fusion_method": "RRF",
            "rrf_constant": 60,
            "candidate": best_candidate_name,
            "query_count": len(arvan_hybrid_query_records),
            "seed_profiles": seed_failure_profile_rows,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== BEST HYBRID PER-SEED FAILURE PROFILE ===")
print("Configuration: Arvan + lexical RRF × parent_child_children")
print("Seeds evaluated:", len(seed_failure_profile_rows))
print("Queries evaluated:", len(arvan_hybrid_query_records))

print("\nPer-seed results, weakest first:")
for profile in seed_failure_profile_rows:
    print(
        f"- seed {profile['seed_id']}: "
        f"Recall@10={profile['recall_at_10']:.4f}, "
        f"MRR={profile['mrr']:.4f}, "
        f"misses@10={profile['miss_count_at_10']}/{profile['query_count']}"
    )

print("\nSeeds with zero Recall@10:")
zero_recall_seed_ids = [
    profile["seed_id"]
    for profile in seed_failure_profile_rows
    if profile["recall_at_10"] == 0.0
]
print(zero_recall_seed_ids)

print("\nReport:", seed_failure_profile_report_path)
print("\nBEST HYBRID PER-SEED FAILURE PROFILE: PASS")

=== BEST HYBRID PER-SEED FAILURE PROFILE ===
Configuration: Arvan + lexical RRF × parent_child_children
Seeds evaluated: 3813
Queries evaluated: 3813

Per-seed results, weakest first:
- seed golden_v2_candidate_abccc6e9611a9838ea50afc1: Recall@10=0.0000, MRR=0.0004, misses@10=1/1
- seed golden_v2_candidate_4ce0a2ce83b08039ec40d827: Recall@10=0.0000, MRR=0.0004, misses@10=1/1
- seed golden_v2_candidate_76d9951424d5f0ffdec1802d: Recall@10=0.0000, MRR=0.0005, misses@10=1/1
- seed golden_v2_candidate_6d62624b158cbf72f1d556db: Recall@10=0.0000, MRR=0.0005, misses@10=1/1
- seed golden_v2_candidate_6c3a6e4608e817af4751ba83: Recall@10=0.0000, MRR=0.0005, misses@10=1/1
- seed golden_v2_candidate_f18c4305dae0408d69ee3734: Recall@10=0.0000, MRR=0.0005, misses@10=1/1
- seed golden_v2_candidate_7cff0406bd934efdfe436daf: Recall@10=0.0000, MRR=0.0005, misses@10=1/1
- seed golden_v2_candidate_d7c03816fb86c0127cffa935: Recall@10=0.0000, MRR=0.0006, misses@10=1/1
- seed golden_v2_candidate_58d2fd1f0fa9f

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## بررسی نمونه‌های واقعی شکست بازیابی

در این مرحله، برای seedهایی که Recall@10 آن‌ها صفر است، یک پرسش ناموفق، شاهد طلایی، پاسخ مورد انتظار و اولین Chunk بازیابی‌شده کنار هم نمایش داده می‌شوند.

این بررسی فقط برای تشخیص علت شکست است. پاسخ مورد انتظار و شاهد طلایی در ساخت query یا رتبه‌بندی استفاده نمی‌شوند.

</div>

In [57]:
# Code Cell 41
# NB13 — Human-readable inspection of zero-recall retrieval failures

# IMPORTANT:
# Use the already-adapted Golden v2 records created by Code Cell 00.
# Do not reload raw Golden v2 here, because compatibility fields such as
# question, expected_answer, source_seed_id, and source_evidence are created
# by the NB13 compatibility adapter.

assert "golden_dataset" in globals(), (
    "Adapted Golden Dataset is unavailable. Run Code Cell 00 first."
)

assert len(golden_dataset) == golden_dataset_context["record_count"], (
    "Adapted Golden Dataset record count does not match the active "
    "Golden Dataset context."
)

golden_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in golden_dataset
}

assert len(golden_record_by_candidate_id) == len(golden_dataset), (
    "Duplicate candidate_id detected in the adapted Golden Dataset."
)

assert all(
    isinstance(record.get("question"), str)
    and record["question"].strip()
    for record in golden_dataset
), "At least one adapted Golden record has no usable question."

assert all(
    isinstance(record.get("expected_answer"), str)
    and record["expected_answer"].strip()
    for record in golden_dataset
), "At least one adapted Golden record has no usable expected_answer."

assert all(
    isinstance(record.get("source_evidence"), str)
    and record["source_evidence"].strip()
    for record in golden_dataset
), "At least one adapted Golden record has no usable source_evidence."

zero_recall_seed_ids = [
    profile["seed_id"]
    for profile in seed_failure_profile_rows
    if profile["recall_at_10"] == 0.0
]

failure_examples = []

for seed_id in zero_recall_seed_ids:
    seed_query_indexes = [
        query_index
        for query_index, query_record in enumerate(
            arvan_hybrid_query_records
        )
        if query_record["source_seed_id"] == seed_id
    ]

    assert seed_query_indexes, (
        f"No benchmark query was found for zero-recall seed: {seed_id}"
    )

    seed_query_rank_pairs = []

    for query_index in seed_query_indexes:
        positive_indexes = list(
            best_positive_indexes_by_seed[seed_id]
        )

        positive_rank_positions = np.flatnonzero(
            np.isin(
                best_rrf_ranked_indexes[query_index],
                positive_indexes,
            )
        )

        assert len(positive_rank_positions) > 0, (
            "No positive chunk exists anywhere in the ranked candidate "
            f"list for seed: {seed_id}"
        )

        seed_query_rank_pairs.append(
            (
                int(positive_rank_positions[0]) + 1,
                query_index,
            )
        )

    best_failed_rank, selected_query_index = min(
        seed_query_rank_pairs
    )

    selected_query = arvan_hybrid_query_records[
        selected_query_index
    ]

    candidate_id = selected_query["candidate_id"]

    assert candidate_id in golden_record_by_candidate_id, (
        "Benchmark query candidate_id is absent from the active "
        f"Golden Dataset: {candidate_id}"
    )

    selected_golden_record = golden_record_by_candidate_id[
        candidate_id
    ]

    top_chunk_index = int(
        best_rrf_ranked_indexes[selected_query_index][0]
    )

    assert 0 <= top_chunk_index < len(best_candidate_records), (
        f"Invalid top chunk index: {top_chunk_index}"
    )

    top_chunk_record = best_candidate_records[
        top_chunk_index
    ]

    failure_examples.append(
        {
            "seed_id": seed_id,
            "candidate_id": candidate_id,
            "first_positive_rank": best_failed_rank,
            "question": selected_golden_record["question"],
            "expected_answer": selected_golden_record[
                "expected_answer"
            ],
            "source_evidence": selected_golden_record[
                "source_evidence"
            ],
            "top_retrieved_chunk_id": top_chunk_record[
                "chunk_id"
            ],
            "top_retrieved_source_id": top_chunk_record[
                "source_id"
            ],
            "top_retrieved_content": top_chunk_record[
                "content"
            ],
        }
    )

failure_example_report_path = (
    active_benchmark_run_directory
    / "arvan_lexical_parent_child_zero_recall_examples.json"
)

failure_example_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": (
                golden_dataset_context["sha256"]
            ),
            "configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "zero_recall_seed_ids": zero_recall_seed_ids,
            "examples": failure_examples,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== ZERO-RECALL FAILURE EXAMPLES ===")
print(
    "Configuration: "
    "Arvan + lexical RRF × parent_child_children"
)
print("Seeds inspected:", zero_recall_seed_ids)
print("Failure examples:", len(failure_examples))

for example in failure_examples:
    print("\n" + "=" * 80)
    print(
        f"SEED {example['seed_id']} | "
        f"best positive rank: "
        f"{example['first_positive_rank']}"
    )

    print("\nQuestion:")
    print(example["question"])

    print("\nGolden evidence preview:")
    print(example["source_evidence"][:600])

    print("\nExpected-answer preview (diagnostic only):")
    print(example["expected_answer"][:500])

    print("\nTop retrieved Chunk:")
    print(
        "Chunk ID:",
        example["top_retrieved_chunk_id"],
    )
    print(
        "Source ID:",
        example["top_retrieved_source_id"],
    )
    print(
        "Content preview:",
        example["top_retrieved_content"][:600],
    )

print("\nReport:", failure_example_report_path)
print("\nZERO-RECALL FAILURE EXAMPLES: PASS")

=== ZERO-RECALL FAILURE EXAMPLES ===
Configuration: Arvan + lexical RRF × parent_child_children
Seeds inspected: ['golden_v2_candidate_abccc6e9611a9838ea50afc1', 'golden_v2_candidate_4ce0a2ce83b08039ec40d827', 'golden_v2_candidate_76d9951424d5f0ffdec1802d', 'golden_v2_candidate_6d62624b158cbf72f1d556db', 'golden_v2_candidate_6c3a6e4608e817af4751ba83', 'golden_v2_candidate_f18c4305dae0408d69ee3734', 'golden_v2_candidate_7cff0406bd934efdfe436daf', 'golden_v2_candidate_d7c03816fb86c0127cffa935', 'golden_v2_candidate_58d2fd1f0fa9f39f68ac6f82', 'golden_v2_candidate_3ea10dae7de0849624354d03', 'golden_v2_candidate_a31dce5211d6555882392444', 'golden_v2_candidate_52898e008b74d296ea0acd13', 'golden_v2_candidate_ec493b7ef908fe4e18612c9b', 'golden_v2_candidate_f10254712d71d90661b70a24', 'golden_v2_candidate_a0a8f8c719ae287a8b241406', 'golden_v2_candidate_a51111ab168b470fa2e328fc', 'golden_v2_candidate_2458ceaeeedbfe492479791d', 'golden_v2_candidate_0342e241d0deb980876b66ec', 'golden_v2_candidate_5

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تحلیل اطلاعات متمایزکننده در پرسش‌های Golden Dataset

در این مرحله بررسی می‌کنیم هر پرسش چند واژه یا عبارت متمایزکننده دارد؛ یعنی واژه‌هایی که در کل Chunkهای کاندیدا کم‌تکرارترند و می‌توانند به تشخیص منبع درست کمک کنند.

این تحلیل تشخیصی است، Golden Dataset را تغییر نمی‌دهد و از پاسخ مورد انتظار برای بازیابی استفاده نمی‌کند.

</div>

In [58]:
# Code Cell 42
# NB13 — Query information-profile diagnostic using corpus-derived IDF

# Use the Golden v2 compatibility-adapted records created by Code Cell 00.
# Do NOT reload the raw Golden v2 JSONL here because compatibility fields
# such as source_seed_id and question are created by the NB13 adapter.

assert "golden_dataset" in globals(), (
    "Adapted Golden Dataset is unavailable. Run Code Cell 00 first."
)

assert len(golden_dataset) == golden_dataset_context["record_count"], (
    "Adapted Golden Dataset record count does not match the active "
    "Golden Dataset context."
)

golden_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in golden_dataset
}

assert len(golden_record_by_candidate_id) == len(golden_dataset), (
    "Duplicate candidate_id detected in the adapted Golden Dataset."
)

assert all(
    isinstance(record.get("source_seed_id"), str)
    and record["source_seed_id"].strip()
    for record in golden_dataset
), "At least one adapted Golden record has no usable source_seed_id."

assert all(
    isinstance(record.get("question"), str)
    and record["question"].strip()
    for record in golden_dataset
), "At least one adapted Golden record has no usable question."

# Rebuild first-positive ranks locally from Code Cell 63 outputs.
question_rank_by_candidate_id = {}

for query_index, query_record in enumerate(
    arvan_hybrid_query_records
):
    candidate_id = query_record["candidate_id"]
    seed_id = query_record["source_seed_id"]

    assert candidate_id in golden_record_by_candidate_id, (
        "Benchmark query candidate_id is absent from the adapted "
        f"Golden Dataset: {candidate_id}"
    )

    assert seed_id in best_positive_indexes_by_seed, (
        "No positive-index registry entry exists for "
        f"source_seed_id={seed_id}"
    )

    positive_indexes = best_positive_indexes_by_seed[seed_id]
    ranked_indexes = best_rrf_ranked_indexes[query_index]

    positive_positions = np.flatnonzero(
        np.isin(
            ranked_indexes,
            list(positive_indexes),
        )
    )

    assert len(positive_positions) > 0, (
        "No positive Chunk rank was found for "
        f"candidate_id={candidate_id}."
    )

    question_rank_by_candidate_id[candidate_id] = (
        int(positive_positions[0]) + 1
    )

assert len(question_rank_by_candidate_id) == len(
    arvan_hybrid_query_records
), (
    "Question-rank registry size does not match the active "
    "benchmark query count."
)

# Corpus-derived IDF from the already-fitted lexical vectorizer.
idf_values = best_lexical_vectorizer.idf_

assert len(idf_values) > 0, (
    "The lexical vectorizer contains no IDF values."
)

high_idf_threshold = float(
    np.quantile(idf_values, 0.75)
)

ordered_candidate_ids = list(
    question_rank_by_candidate_id.keys()
)

information_profile_questions = [
    golden_record_by_candidate_id[candidate_id]["question"]
    for candidate_id in ordered_candidate_ids
]

information_profile_query_matrix = (
    best_lexical_vectorizer.transform(
        information_profile_questions
    )
)

assert information_profile_query_matrix.shape[0] == len(
    ordered_candidate_ids
), (
    "Information-profile query matrix row count mismatch."
)

information_profile_rows = []

for row_index, candidate_id in enumerate(
    ordered_candidate_ids
):
    golden_record = golden_record_by_candidate_id[
        candidate_id
    ]

    query_vector = information_profile_query_matrix[
        row_index
    ]

    feature_indexes = query_vector.indices
    query_idf_values = idf_values[feature_indexes]

    high_idf_feature_count = int(
        np.sum(
            query_idf_values >= high_idf_threshold
        )
    )

    first_positive_rank = (
        question_rank_by_candidate_id[candidate_id]
    )

    information_profile_rows.append(
        {
            "candidate_id": candidate_id,
            "source_seed_id": golden_record[
                "source_seed_id"
            ],
            "question": golden_record["question"],
            "query_feature_count": int(
                len(feature_indexes)
            ),
            "high_idf_feature_count": (
                high_idf_feature_count
            ),
            "maximum_query_idf": (
                float(np.max(query_idf_values))
                if len(query_idf_values)
                else 0.0
            ),
            "mean_query_idf": (
                float(np.mean(query_idf_values))
                if len(query_idf_values)
                else 0.0
            ),
            "information_status": (
                "NO_HIGH_IDF_SIGNAL"
                if high_idf_feature_count == 0
                else "HAS_HIGH_IDF_SIGNAL"
            ),
            "first_positive_rank": (
                first_positive_rank
            ),
            "hit_at_10": (
                first_positive_rank <= 10
            ),
        }
    )

assert len(information_profile_rows) == len(
    arvan_hybrid_query_records
), (
    "Information-profile row count does not match the "
    "active benchmark query count."
)

information_summary = {}

for status in (
    "NO_HIGH_IDF_SIGNAL",
    "HAS_HIGH_IDF_SIGNAL",
):
    rows = [
        row
        for row in information_profile_rows
        if row["information_status"] == status
    ]

    information_summary[status] = {
        "query_count": len(rows),
        "share_of_queries": (
            len(rows)
            / len(information_profile_rows)
        ),
        "recall_at_10": (
            float(
                np.mean(
                    [
                        row["hit_at_10"]
                        for row in rows
                    ]
                )
            )
            if rows
            else None
        ),
        "mean_first_positive_rank": (
            float(
                np.mean(
                    [
                        row["first_positive_rank"]
                        for row in rows
                    ]
                )
            )
            if rows
            else None
        ),
    }

information_profile_report_path = (
    active_benchmark_run_directory
    / "active_golden_query_information_profile.json"
)

information_profile_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": (
                golden_dataset_context["sha256"]
            ),
            "candidate": best_candidate_name,
            "configuration": (
                "arvan_text_embedding_3_large"
                "_plus_persian_tfidf_rrf"
            ),
            "idf_policy": {
                "source": (
                    "parent_child_children_tfidf_vocabulary"
                ),
                "high_idf_percentile": 0.75,
                "high_idf_threshold": (
                    high_idf_threshold
                ),
            },
            "summary": information_summary,
            "rows": information_profile_rows,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "=== ACTIVE GOLDEN QUERY INFORMATION PROFILE ==="
)
print(
    "Queries analysed:",
    len(information_profile_rows),
)
print(
    f"High-IDF threshold: "
    f"{high_idf_threshold:.4f}"
)

for status, summary in information_summary.items():
    print(f"\n{status}")
    print(
        "- Query count:",
        summary["query_count"],
    )
    print(
        f"- Share: "
        f"{summary['share_of_queries']:.4f}"
    )

    if summary["recall_at_10"] is None:
        print(
            "- Recall@10: "
            "N/A (no queries in this group)"
        )
        print(
            "- Mean first-positive rank: "
            "N/A (no queries in this group)"
        )
    else:
        print(
            f"- Recall@10: "
            f"{summary['recall_at_10']:.4f}"
        )
        print(
            "- Mean first-positive rank: "
            f"{summary['mean_first_positive_rank']:.2f}"
        )

weak_information_examples = sorted(
    information_profile_rows,
    key=lambda row: (
        row["high_idf_feature_count"],
        row["maximum_query_idf"],
        row["first_positive_rank"],
    ),
)

print(
    "\nLowest-information example questions, "
    "first 12:"
)

for example in weak_information_examples[:12]:
    print(
        f"- seed {example['source_seed_id']} | "
        f"rank={example['first_positive_rank']} | "
        f"high_idf_features="
        f"{example['high_idf_feature_count']} | "
        f"question={example['question']}"
    )

print(
    "\nReport:",
    information_profile_report_path,
)

print(
    "\nACTIVE GOLDEN QUERY INFORMATION PROFILE: PASS"
)

=== ACTIVE GOLDEN QUERY INFORMATION PROFILE ===
Queries analysed: 3813
High-IDF threshold: 8.2160

NO_HIGH_IDF_SIGNAL
- Query count: 1398
- Share: 0.3666
- Recall@10: 0.6295
- Mean first-positive rank: 58.02

HAS_HIGH_IDF_SIGNAL
- Query count: 2415
- Share: 0.6334
- Recall@10: 0.8104
- Mean first-positive rank: 21.41

Lowest-information example questions, first 12:
- seed golden_v2_candidate_86277ffffb111586793ef33c | rank=1 | high_idf_features=0 | question=سود چیست؟
- seed golden_v2_candidate_fd57d914fd86edad2d0cecd2 | rank=3 | high_idf_features=0 | question=تورم چیست؟
- seed golden_v2_candidate_5f402f94b475e5b2c8eee833 | rank=3 | high_idf_features=0 | question=دانش چیست؟
- seed golden_v2_candidate_5a481b33d0c34eacc1517350 | rank=4 | high_idf_features=0 | question=هدف بورس چیست؟
- seed golden_v2_candidate_7a0949efbfdecb45444a421e | rank=5 | high_idf_features=0 | question=تورم چیست؟
- seed golden_v2_candidate_5df5d842935fc5c5f03023db | rank=7 | high_idf_features=0 | question=تورم چیست؟

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی سیاست بودجهٔ Context در روش Parent–Child

در Parent–Child، ابتدا Chunkهای کوچک بازیابی می‌شوند و سپس متن والد آن‌ها برای تولید پاسخ استفاده می‌شود.

در این مرحله چند سیاست متفاوت برای تعداد Childهای بازیابی‌شده، تعداد والدها و سقف tokenهای Context بررسی می‌شود تا هزینه و حجم Context قابل‌کنترل باشد، بدون آنکه اتصال Child به Parent از بین برود.

</div>

In [59]:
# Code Cell 43
# NB13 — Parent-child context-budget policy evaluation

assert "parent_child_parents" in chunking_candidates, (
    "The parent_child_parents candidate is unavailable."
)

parent_context_path = (
    project_root
    / chunking_candidates["parent_child_parents"]["path"]
)

assert parent_context_path.is_file(), (
    f"Missing parent-context file: {parent_context_path}"
)

parent_context_records = [
    json.loads(line)
    for line in parent_context_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

parent_record_by_chunk_id = {
    record["chunk_id"]: record
    for record in parent_context_records
}

assert len(parent_record_by_chunk_id) == len(parent_context_records)

context_budget_policies = [
    {
        "policy_id": "conservative",
        "retrieved_child_k": 5,
        "max_parent_chunks": 3,
        "max_context_tokens": 3000,
    },
    {
        "policy_id": "balanced",
        "retrieved_child_k": 10,
        "max_parent_chunks": 4,
        "max_context_tokens": 5000,
    },
    {
        "policy_id": "expansive",
        "retrieved_child_k": 10,
        "max_parent_chunks": 6,
        "max_context_tokens": 8000,
    },
]

context_budget_results = {}

print("=== PARENT–CHILD CONTEXT-BUDGET POLICY EVALUATION ===")
print("Queries evaluated:", len(arvan_hybrid_query_records))
print("Parent chunks available:", len(parent_context_records))

for policy in context_budget_policies:
    assembled_contexts = []
    missing_parent_references = 0

    for query_index, _ in enumerate(arvan_hybrid_query_records):
        ranked_child_indexes = best_rrf_ranked_indexes[query_index]
        selected_child_indexes = ranked_child_indexes[
            :policy["retrieved_child_k"]
        ]

        selected_parent_ids = []
        selected_parent_records = []
        current_token_total = 0

        for child_index in selected_child_indexes:
            child_record = best_candidate_records[int(child_index)]
            parent_chunk_id = child_record["parent_chunk_id"]

            if parent_chunk_id not in parent_record_by_chunk_id:
                missing_parent_references += 1
                continue

            if parent_chunk_id in selected_parent_ids:
                continue

            parent_record = parent_record_by_chunk_id[parent_chunk_id]
            parent_token_count = parent_record["token_count"]

            if len(selected_parent_records) >= policy["max_parent_chunks"]:
                continue

            if (
                current_token_total + parent_token_count
                > policy["max_context_tokens"]
            ):
                continue

            selected_parent_ids.append(parent_chunk_id)
            selected_parent_records.append(parent_record)
            current_token_total += parent_token_count

        assembled_contexts.append(
            {
                "parent_chunk_count": len(selected_parent_records),
                "token_count": current_token_total,
                "source_count": len(
                    {
                        parent_record["source_id"]
                        for parent_record in selected_parent_records
                    }
                ),
                "selected_parent_chunk_ids": selected_parent_ids,
            }
        )

    token_counts = np.array(
        [context["token_count"] for context in assembled_contexts]
    )
    parent_chunk_counts = np.array(
        [context["parent_chunk_count"] for context in assembled_contexts]
    )
    source_counts = np.array(
        [context["source_count"] for context in assembled_contexts]
    )

    result = {
        **policy,
        "query_count": len(assembled_contexts),
        "missing_parent_references": missing_parent_references,
        "empty_context_count": int(np.sum(token_counts == 0)),
        "mean_context_tokens": float(np.mean(token_counts)),
        "p95_context_tokens": float(np.percentile(token_counts, 95)),
        "maximum_context_tokens": int(np.max(token_counts)),
        "mean_parent_chunk_count": float(np.mean(parent_chunk_counts)),
        "mean_unique_source_count": float(np.mean(source_counts)),
    }

    context_budget_results[policy["policy_id"]] = result

    print(f"\nPolicy: {policy['policy_id']}")
    print("- Retrieved children:", policy["retrieved_child_k"])
    print("- Maximum parent chunks:", policy["max_parent_chunks"])
    print("- Maximum context tokens:", policy["max_context_tokens"])
    print(f"- Mean context tokens: {result['mean_context_tokens']:.1f}")
    print(f"- P95 context tokens: {result['p95_context_tokens']:.1f}")
    print(f"- Maximum actual tokens: {result['maximum_context_tokens']}")
    print(
        "- Mean parent chunks: "
        f"{result['mean_parent_chunk_count']:.2f}"
    )
    print(
        "- Mean unique sources: "
        f"{result['mean_unique_source_count']:.2f}"
    )
    print("- Empty contexts:", result["empty_context_count"])
    print(
        "- Missing parent references: "
        f"{result['missing_parent_references']}"
    )

assert all(
    result["missing_parent_references"] == 0
    for result in context_budget_results.values()
), "A selected Child referenced a missing Parent."

context_budget_report_path = (
    active_benchmark_run_directory
    / "parent_child_context_budget_policy_evaluation.json"
)

context_budget_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "child_candidate": "parent_child_children",
            "parent_candidate": "parent_child_parents",
            "results": context_budget_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", context_budget_report_path)
print("\nPARENT–CHILD CONTEXT-BUDGET POLICY EVALUATION: PASS")

=== PARENT–CHILD CONTEXT-BUDGET POLICY EVALUATION ===
Queries evaluated: 3813
Parent chunks available: 1447

Policy: conservative
- Retrieved children: 5
- Maximum parent chunks: 3
- Maximum context tokens: 3000
- Mean context tokens: 2015.9
- P95 context tokens: 2903.2
- Maximum actual tokens: 3000
- Mean parent chunks: 2.97
- Mean unique sources: 2.76
- Empty contexts: 0
- Missing parent references: 0

Policy: balanced
- Retrieved children: 10
- Maximum parent chunks: 4
- Maximum context tokens: 5000
- Mean context tokens: 2799.3
- P95 context tokens: 4068.4
- Maximum actual tokens: 4767
- Mean parent chunks: 4.00
- Mean unique sources: 3.64
- Empty contexts: 0
- Missing parent references: 0

Policy: expansive
- Retrieved children: 10
- Maximum parent chunks: 6
- Maximum context tokens: 8000
- Mean context tokens: 4238.2
- P95 context tokens: 5902.4
- Maximum actual tokens: 6957
- Mean parent chunks: 6.00
- Mean unique sources: 5.27
- Empty contexts: 0
- Missing parent references: 0


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## سنجش حفظ شواهد مثبت در بودجهٔ Context

برای هر سیاست Parent–Child بررسی می‌کنیم:

- آیا Child مثبت در نتایج بازیابی‌شده وجود داشته است؟
- اگر وجود داشته، آیا Parent متناظر آن واقعاً در Context نهایی باقی مانده است؟

این سنجش نشان می‌دهد محدودیت token و تعداد والدها چه مقدار از شواهد بازیابی‌شده را حفظ یا حذف می‌کند.

</div>

In [60]:
# Code Cell 44
# NB13 — Positive-evidence retention under parent-child context budgets

context_budget_evidence_retention = {}

print("=== PARENT–CHILD POSITIVE-EVIDENCE RETENTION ===")
print("Queries evaluated:", len(arvan_hybrid_query_records))

for policy in context_budget_policies:
    retrieved_positive_query_count = 0
    retained_positive_parent_query_count = 0
    missing_parent_references = 0

    for query_index, query_record in enumerate(arvan_hybrid_query_records):
        seed_id = query_record["source_seed_id"]
        positive_child_indexes = best_positive_indexes_by_seed[seed_id]

        selected_child_indexes = best_rrf_ranked_indexes[query_index][
            :policy["retrieved_child_k"]
        ]

        positive_child_was_retrieved = any(
            int(child_index) in positive_child_indexes
            for child_index in selected_child_indexes
        )

        if positive_child_was_retrieved:
            retrieved_positive_query_count += 1

        positive_parent_ids = {
            best_candidate_records[child_index]["parent_chunk_id"]
            for child_index in positive_child_indexes
        }

        selected_parent_ids = []
        selected_parent_token_total = 0

        for child_index in selected_child_indexes:
            child_record = best_candidate_records[int(child_index)]
            parent_chunk_id = child_record["parent_chunk_id"]

            if parent_chunk_id not in parent_record_by_chunk_id:
                missing_parent_references += 1
                continue

            if parent_chunk_id in selected_parent_ids:
                continue

            parent_record = parent_record_by_chunk_id[parent_chunk_id]
            parent_token_count = parent_record["token_count"]

            if len(selected_parent_ids) >= policy["max_parent_chunks"]:
                continue

            if (
                selected_parent_token_total + parent_token_count
                > policy["max_context_tokens"]
            ):
                continue

            selected_parent_ids.append(parent_chunk_id)
            selected_parent_token_total += parent_token_count

        if positive_parent_ids.intersection(selected_parent_ids):
            retained_positive_parent_query_count += 1

    retrieval_rate = (
        retrieved_positive_query_count
        / len(arvan_hybrid_query_records)
    )
    parent_retention_rate = (
        retained_positive_parent_query_count
        / len(arvan_hybrid_query_records)
    )
    conditional_retention_rate = (
        retained_positive_parent_query_count
        / retrieved_positive_query_count
        if retrieved_positive_query_count
        else None
    )

    result = {
        **policy,
        "query_count": len(arvan_hybrid_query_records),
        "retrieved_positive_query_count": retrieved_positive_query_count,
        "retrieval_recall_at_child_k": retrieval_rate,
        "retained_positive_parent_query_count": (
            retained_positive_parent_query_count
        ),
        "parent_evidence_recall_after_budget": parent_retention_rate,
        "conditional_parent_retention_after_retrieval": (
            conditional_retention_rate
        ),
        "missing_parent_references": missing_parent_references,
    }

    context_budget_evidence_retention[policy["policy_id"]] = result

    print(f"\nPolicy: {policy['policy_id']}")
    print(
        "- Positive Child retrieved: "
        f"{retrieved_positive_query_count}/{len(arvan_hybrid_query_records)} "
        f"({retrieval_rate:.4f})"
    )
    print(
        "- Positive Parent retained in final Context: "
        f"{retained_positive_parent_query_count}/"
        f"{len(arvan_hybrid_query_records)} "
        f"({parent_retention_rate:.4f})"
    )
    print(
        "- Conditional retention after positive retrieval: "
        f"{conditional_retention_rate:.4f}"
    )
    print(
        "- Missing parent references: "
        f"{missing_parent_references}"
    )

assert all(
    result["missing_parent_references"] == 0
    for result in context_budget_evidence_retention.values()
), "A Child referenced a missing Parent."

context_budget_evidence_retention_report_path = (
    active_benchmark_run_directory
    / "parent_child_context_budget_evidence_retention.json"
)

context_budget_evidence_retention_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "results": context_budget_evidence_retention,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", context_budget_evidence_retention_report_path)
print("\nPARENT–CHILD POSITIVE-EVIDENCE RETENTION: PASS")

=== PARENT–CHILD POSITIVE-EVIDENCE RETENTION ===
Queries evaluated: 3813

Policy: conservative
- Positive Child retrieved: 2448/3813 (0.6420)
- Positive Parent retained in final Context: 2247/3813 (0.5893)
- Conditional retention after positive retrieval: 0.9179
- Missing parent references: 0

Policy: balanced
- Positive Child retrieved: 2837/3813 (0.7440)
- Positive Parent retained in final Context: 2429/3813 (0.6370)
- Conditional retention after positive retrieval: 0.8562
- Missing parent references: 0

Policy: expansive
- Positive Child retrieved: 2837/3813 (0.7440)
- Positive Parent retained in final Context: 2645/3813 (0.6937)
- Conditional retention after positive retrieval: 0.9323
- Missing parent references: 0

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/parent_child_context_budget_evidence_retention.json

PARENT–CHILD POSITIVE-EVIDENCE RE

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ثبت پیکربندی موقت برای ادامهٔ ارزیابی

بهترین پیکربندی فعلی برای ادامهٔ آزمایش‌ها ثبت می‌شود، اما به دلیل پایین‌بودن Recall بازیابی پرسش و نبود انتخاب نهایی، برای استفادهٔ Production تأیید نمی‌شود.

این ثبت نسخه‌محور است و با تغییر Golden Dataset، در اجرای مجدد به‌صورت مستقل ارزیابی خواهد شد.

</div>

In [61]:
# Code Cell 45
# NB13 — Provisional retrieval configuration registration

best_hybrid_metrics = arvan_lexical_hybrid_results[
    "parent_child_children"
]["rrf_arvan_plus_lexical"]

expansive_context_metrics = context_budget_evidence_retention["expansive"]
expansive_context_profile = context_budget_results["expansive"]

provisional_retrieval_configuration = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "status": "PROVISIONAL_FOR_FURTHER_EVALUATION_NOT_PRODUCTION",
    "retrieval": {
        "dense_model": "arvan_text_embedding_3_large",
        "lexical_model": "persian_tfidf_word_unigram_bigram",
        "fusion_method": "reciprocal_rank_fusion",
        "rrf_constant": 60,
        "retrieval_candidate": "parent_child_children",
        "query_recall_at_10": best_hybrid_metrics["recall_at_10"],
        "query_mrr": best_hybrid_metrics["mrr"],
    },
    "context_assembly": {
        "parent_candidate": "parent_child_parents",
        "policy_id": "expansive",
        "retrieved_child_k": expansive_context_profile[
            "retrieved_child_k"
        ],
        "max_parent_chunks": expansive_context_profile[
            "max_parent_chunks"
        ],
        "max_context_tokens": expansive_context_profile[
            "max_context_tokens"
        ],
        "p95_context_tokens": expansive_context_profile[
            "p95_context_tokens"
        ],
        "conditional_parent_retention_after_retrieval": (
            expansive_context_metrics[
                "conditional_parent_retention_after_retrieval"
            ]
        ),
    },
    "not_approved_for_production_because": [
        (
            "Question retrieval remains too weak: only "
            f"{best_hybrid_metrics['recall_at_10']:.4f} Recall@10."
        ),
        (
            "Context assembly cannot recover evidence that was not "
            "retrieved among the top Child results."
        ),
        (
            "Final model-and-chunking selection requires a completed "
            "query-side remediation evaluation."
        ),
    ],
    "required_next_work": [
        "query_reformulation_or_reranking_experiment",
        "post_remediation_retrieval_benchmark",
        "rerun_with_active_golden_dataset_after_feedback_integration",
        "final_tradeoff_decision",
    ],
}

provisional_retrieval_configuration_path = (
    active_benchmark_run_directory
    / "provisional_retrieval_configuration.json"
)

provisional_retrieval_configuration_path.write_text(
    json.dumps(
        provisional_retrieval_configuration,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== PROVISIONAL RETRIEVAL CONFIGURATION ===")
print(
    "Status:",
    provisional_retrieval_configuration["status"],
)
print(
    "Retrieval Recall@10:",
    f"{best_hybrid_metrics['recall_at_10']:.4f}",
)
print(
    "Retrieval MRR:",
    f"{best_hybrid_metrics['mrr']:.4f}",
)
print(
    "Context policy:",
    provisional_retrieval_configuration["context_assembly"]["policy_id"],
)
print(
    "Conditional parent retention:",
    f"{expansive_context_metrics['conditional_parent_retention_after_retrieval']:.4f}",
)
print("\nReport:", provisional_retrieval_configuration_path)
print("\nPROVISIONAL RETRIEVAL CONFIGURATION: PASS")

=== PROVISIONAL RETRIEVAL CONFIGURATION ===
Status: PROVISIONAL_FOR_FURTHER_EVALUATION_NOT_PRODUCTION
Retrieval Recall@10: 0.7440
Retrieval MRR: 0.5118
Context policy: expansive
Conditional parent retention: 0.9323

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/provisional_retrieval_configuration.json

PROVISIONAL RETRIEVAL CONFIGURATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## کشف قابلیت Reranking

Reranker پس از بازیابی اولیه، پرسش و تعداد محدودی Chunk برتر را با هم بررسی می‌کند و می‌تواند ترتیب نتایج را دقیق‌تر کند.

در این مرحله فقط مدل‌های محلی و تنظیمات موجود بررسی می‌شوند. هیچ دانلود، نصب، فراخوانی API یا هزینه‌ای انجام نمی‌شود.

</div>

In [62]:
# Code Cell 46
# NB13 — Read-only reranking capability discovery

import importlib.util
import os
from pathlib import Path

huggingface_cache_roots = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path.home() / "Library" / "Caches" / "huggingface" / "hub",
]

local_model_directories = []

for cache_root in huggingface_cache_roots:
    if not cache_root.is_dir():
        continue

    local_model_directories.extend(
        sorted(
            path
            for path in cache_root.glob("models--*")
            if path.is_dir()
        )
    )

reranker_name_indicators = (
    "reranker",
    "cross-encoder",
    "cross_encoder",
    "bge",
    "ms-marco",
)

local_reranker_directories = [
    model_directory
    for model_directory in local_model_directories
    if any(
        indicator in model_directory.name.lower()
        for indicator in reranker_name_indicators
    )
]

arvan_configuration_keys = sorted(
    environment_key
    for environment_key in os.environ
    if environment_key.startswith("ARVAN")
)

print("=== RERANKING CAPABILITY DISCOVERY ===")
print(
    "sentence_transformers available:",
    importlib.util.find_spec("sentence_transformers") is not None,
)
print(
    "CrossEncoder class available:",
    end=" ",
)

try:
    from sentence_transformers import CrossEncoder
    print(True)
except Exception:
    print(False)

print("\nLocal Hugging Face model directories:", len(local_model_directories))

print("\nLocal reranker-like model directories:")
if local_reranker_directories:
    for model_directory in local_reranker_directories:
        print("-", model_directory.name)
else:
    print("NONE FOUND")

print("\nConfigured Arvan environment-variable names:")
for environment_key in arvan_configuration_keys:
    print("-", environment_key)

known_reranking_environment_keys = [
    key
    for key in arvan_configuration_keys
    if any(
        indicator in key.lower()
        for indicator in (
            "rerank",
            "chat",
            "completion",
            "generation",
        )
    )
]

print("\nConfigured reranking-or-generation endpoint names:")
if known_reranking_environment_keys:
    for environment_key in known_reranking_environment_keys:
        print("-", environment_key)
else:
    print("NONE FOUND")

print("\nRERANKING CAPABILITY DISCOVERY: PASS")

=== RERANKING CAPABILITY DISCOVERY ===
sentence_transformers available: True
CrossEncoder class available: True

Local Hugging Face model directories: 3

Local reranker-like model directories:
NONE FOUND

Configured Arvan environment-variable names:
- ARVAN_AI_API_KEY
- ARVAN_CHAT_MODEL
- ARVAN_CHAT_URL
- ARVAN_EMBEDDING_URL

Configured reranking-or-generation endpoint names:
- ARVAN_CHAT_MODEL
- ARVAN_CHAT_URL

RERANKING CAPABILITY DISCOVERY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## بررسی آماده‌بودن سرویس گفت‌وگویی Arvan برای Reranking

مدل گفت‌وگویی می‌تواند در مرحلهٔ بعد برای بازنویسی کنترل‌شدهٔ پرسش یا رتبه‌بندی دوبارهٔ تعداد محدودی Chunk برتر آزمایش شود.

در این مرحله فقط وجود تنظیمات و سازگاری اتصال بررسی می‌شود. هیچ درخواست API و هیچ هزینه‌ای ایجاد نمی‌شود.

</div>

In [63]:
# Code Cell 47
# NB13 — Arvan chat configuration preflight for reranking experiments

from urllib.parse import urlparse
from dotenv import load_dotenv
import importlib.util
import os

arvan_env_path = project_root / ".env"

assert arvan_env_path.is_file(), (
    f"Arvan environment file is missing: {arvan_env_path}"
)

load_dotenv(arvan_env_path, override=False)

arvan_chat_api_key = os.getenv("ARVAN_AI_API_KEY")
arvan_chat_url = os.getenv("ARVAN_CHAT_URL")
arvan_chat_model = os.getenv("ARVAN_CHAT_MODEL")

assert arvan_chat_api_key, "ARVAN_AI_API_KEY is not configured."
assert arvan_chat_url, "ARVAN_CHAT_URL is not configured."
assert arvan_chat_model, "ARVAN_CHAT_MODEL is not configured."

parsed_chat_url = urlparse(arvan_chat_url)

assert parsed_chat_url.scheme == "https", (
    "Arvan chat endpoint must use HTTPS."
)
assert parsed_chat_url.netloc, (
    "Arvan chat endpoint has no valid host."
)
assert importlib.util.find_spec("openai") is not None, (
    "The OpenAI-compatible client package is unavailable."
)

arvan_chat_configuration = {
    "model": arvan_chat_model,
    "endpoint_scheme": parsed_chat_url.scheme,
    "endpoint_host": parsed_chat_url.netloc,
    "endpoint_path_configured": bool(parsed_chat_url.path),
    "api_key_available": True,
}

print("=== ARVAN CHAT CONFIGURATION PREFLIGHT ===")
print("OpenAI-compatible client available: True")
print("Arvan API key available: True")
print("Chat model:", arvan_chat_configuration["model"])
print("Gateway scheme:", arvan_chat_configuration["endpoint_scheme"])
print("Gateway host:", arvan_chat_configuration["endpoint_host"])
print(
    "Gateway path configured:",
    arvan_chat_configuration["endpoint_path_configured"],
)
print("\nARVAN CHAT CONFIGURATION PREFLIGHT: PASS")

=== ARVAN CHAT CONFIGURATION PREFLIGHT ===
OpenAI-compatible client available: True
Arvan API key available: True
Chat model: Qwen3-30B-A3B
Gateway scheme: https
Gateway host: arvancloudai.ir
Gateway path configured: True

ARVAN CHAT CONFIGURATION PREFLIGHT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## اعتبارسنجی اتصال واقعی مدل گفت‌وگویی Arvan

در این مرحله با یک درخواست کوتاه و بدون دادهٔ پروژه، اتصال واقعی به مدل گفت‌وگویی Arvan بررسی می‌شود.

این آزمایش فقط برای اطمینان از آماده‌بودن زیرساخت Query Reformulation و Reranking است.

</div>

In [64]:
# Code Cell 48
# NB13 — Arvan chat contract reconstruction (API disabled)

from datetime import datetime, timezone

arvan_chat_client = None
arvan_chat_connection_report = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": arvan_chat_model,
    "endpoint_host": parsed_chat_url.netloc,
    "response_received": None,
    "execution_mode": "CACHE_ONLY_API_DISABLED",
    "note": "Live connection validation intentionally disabled for Run-All safety.",
}

arvan_chat_connection_report_path = (
    active_benchmark_run_directory / "arvan_chat_connection_report.json"
)
arvan_chat_connection_report_path.write_text(
    json.dumps(arvan_chat_connection_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("=== ARVAN CHAT CONTRACT RECONSTRUCTION ===")
print("Model:", arvan_chat_model)
print("API execution: DISABLED")
print("Report:", arvan_chat_connection_report_path)
print("ARVAN CHAT CONTRACT RECONSTRUCTION: PASS")


=== ARVAN CHAT CONTRACT RECONSTRUCTION ===
Model: Qwen3-30B-A3B
API execution: DISABLED
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_connection_report.json
ARVAN CHAT CONTRACT RECONSTRUCTION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## بررسی ساختار پاسخ مدل گفت‌وگویی Arvan

در درخواست قبلی پاسخ از سرویس دریافت شد، اما متن در فیلد معمول `content` قرار نگرفت.

در این مرحله بدون ارسال درخواست جدید، ساختار همان پاسخ بررسی می‌شود تا روش درست خواندن خروجی مدل مشخص شود.

</div>

In [65]:
# Code Cell 49
# NB13 — Arvan chat readiness (cache-only / API-disabled)

arvan_chat_readiness = {
    "execution_mode": "CACHE_ONLY_API_DISABLED",
    "live_api_call_executed": False,
    "live_response_required": False,
    "status": "SKIPPED_NOT_REQUIRED",
}

assert arvan_chat_readiness["execution_mode"] == "CACHE_ONLY_API_DISABLED"
assert arvan_chat_readiness["live_api_call_executed"] is False
assert arvan_chat_readiness["live_response_required"] is False

print("=== ARVAN CHAT READINESS ===")
print("API execution: DISABLED")
print("Live response required: False")
print("Status:", arvan_chat_readiness["status"])
print("ARVAN CHAT READINESS CACHE-ONLY: PASS")

=== ARVAN CHAT READINESS ===
API execution: DISABLED
Live response required: False
Status: SKIPPED_NOT_REQUIRED
ARVAN CHAT READINESS CACHE-ONLY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## پایلوت Reranking با مدل گفت‌وگویی Arvan

در این آزمایش محدود، مدل گفت‌وگویی فقط پرسش و ده Chunk برتر بازیابی‌شده را می‌بیند و آن‌ها را دوباره رتبه‌بندی می‌کند.

پاسخ مورد انتظار، شواهد طلایی و اطلاعات label به مدل داده نمی‌شود. از آن‌ها فقط پس از پاسخ مدل، برای اندازه‌گیری کیفیت استفاده می‌شود.

هدف این است که قبل از هر اجرای بزرگ، بررسی کنیم آیا Reranking می‌تواند شواهدی را که در ده نتیجهٔ اول هستند به رتبه‌های بالاتر منتقل کند یا خیر.

</div>

In [66]:
# Code Cell 50
# NB13 — Cached reranking pilot reconstruction (API disabled)

import re

reranker_top_k = 10
reranker_pilot_size = 20
reranker_excerpt_char_limit = 1000

eligible_reranker_query_indexes = []
for query_index, query_record in enumerate(arvan_hybrid_query_records):
    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    top_k_indexes = best_rrf_ranked_indexes[query_index][:reranker_top_k]
    if any(int(index) in positive_indexes for index in top_k_indexes):
        eligible_reranker_query_indexes.append(query_index)

assert eligible_reranker_query_indexes, "No eligible reranking queries were found."

selected_reranker_query_indexes = []
selected_seed_ids = set()
for query_index in eligible_reranker_query_indexes:
    seed_id = arvan_hybrid_query_records[query_index]["source_seed_id"]
    if seed_id not in selected_seed_ids:
        selected_reranker_query_indexes.append(query_index)
        selected_seed_ids.add(seed_id)
    if len(selected_reranker_query_indexes) == reranker_pilot_size:
        break
if len(selected_reranker_query_indexes) < reranker_pilot_size:
    for query_index in eligible_reranker_query_indexes:
        if query_index not in selected_reranker_query_indexes:
            selected_reranker_query_indexes.append(query_index)
        if len(selected_reranker_query_indexes) == reranker_pilot_size:
            break

def extract_reranked_labels(response_text, allowed_labels):
    match = re.search(r"\{.*\}", response_text or "", flags=re.DOTALL)
    if not match:
        return None
    try:
        parsed_response = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    proposed_labels = parsed_response.get("ranking")
    if not isinstance(proposed_labels, list):
        return None
    normalized = [str(label).strip() for label in proposed_labels]
    deduplicated = list(dict.fromkeys(normalized))
    if set(deduplicated) != set(allowed_labels) or len(deduplicated) != len(allowed_labels):
        return None
    return deduplicated

reranker_pilot_report_path = active_benchmark_run_directory / "arvan_chat_reranking_pilot.json"
if reranker_pilot_report_path.is_file():
    cached_reranker_pilot_report = json.loads(
        reranker_pilot_report_path.read_text(encoding="utf-8")
    )
    valid_identity = (
        cached_reranker_pilot_report.get("active_golden_dataset_sha256")
        == golden_dataset_context["sha256"]
        and cached_reranker_pilot_report.get("model") == arvan_chat_model
    )
    reranker_pilot_results = (
        cached_reranker_pilot_report.get("results", []) if valid_identity else []
    )
else:
    reranker_pilot_results = []

print("=== CACHED RERANKING PILOT RECONSTRUCTION ===")
print("Eligible queries:", len(eligible_reranker_query_indexes))
print("Selected pilot queries:", len(selected_reranker_query_indexes))
print("Cached pilot results:", len(reranker_pilot_results))
print("API execution: DISABLED")
print("CACHED RERANKING PILOT RECONSTRUCTION: PASS")


=== CACHED RERANKING PILOT RECONSTRUCTION ===
Eligible queries: 2837
Selected pilot queries: 20
Cached pilot results: 20
API execution: DISABLED
CACHED RERANKING PILOT RECONSTRUCTION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تحلیل پاسخ‌های نامعتبر پایلوت Reranking

در این مرحله پاسخ‌های دریافت‌شده از مدل که قابل‌خواندن نبودند بررسی می‌شوند.

هدف، تشخیص تفاوت میان دو حالت است:

- مدل رتبه‌بندی درست ساخته اما قالب JSON را رعایت نکرده است؛
- مدل دستور رتبه‌بندی را به‌درستی انجام نداده است.

هیچ درخواست جدیدی به سرویس ارسال نمی‌شود.

</div>

In [67]:
# Code Cell 51
# NB13 — Read-only analysis of malformed reranking-pilot responses

failed_reranker_pilot_results = [
    result
    for result in reranker_pilot_results
    if result["response_parse_status"] == "FAIL"
]

print("=== RERANKING PILOT RESPONSE-FORMAT ANALYSIS ===")
print("Total pilot responses:", len(reranker_pilot_results))
print("Malformed or incomplete responses:", len(failed_reranker_pilot_results))

for result in failed_reranker_pilot_results:
    raw_response = result["raw_response"]

    contains_json_object = bool(
        re.search(r"\{.*\}", raw_response, flags=re.DOTALL)
    )
    contains_json_array = bool(
        re.search(r"\[.*\]", raw_response, flags=re.DOTALL)
    )
    contains_passage_label = bool(
        re.search(r"\bP\d{2}\b", raw_response)
    )

    print("\n" + "=" * 80)
    print(
        f"Pilot {result['pilot_position']} | "
        f"seed={result['source_seed_id']} | "
        f"original_positive_rank={result['original_positive_rank']}"
    )
    print("Contains JSON object:", contains_json_object)
    print("Contains JSON array:", contains_json_array)
    print("Contains passage label:", contains_passage_label)
    print("Raw response:")
    print(repr(raw_response[:2000]))

reranker_format_analysis_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_pilot_format_analysis.json"
)

reranker_format_analysis_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "pilot_query_count": len(reranker_pilot_results),
            "failed_response_count": len(failed_reranker_pilot_results),
            "failed_responses": failed_reranker_pilot_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", reranker_format_analysis_report_path)
print("\nRERANKING PILOT RESPONSE-FORMAT ANALYSIS: PASS")

=== RERANKING PILOT RESPONSE-FORMAT ANALYSIS ===
Total pilot responses: 20
Malformed or incomplete responses: 9

Pilot 3 | seed=golden_v2_candidate_f8a742b149d2ddc521f78c3f | original_positive_rank=1
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''

Pilot 4 | seed=golden_v2_candidate_5be7b31dd95de0a9fc00e79f | original_positive_rank=3
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''

Pilot 5 | seed=golden_v2_candidate_7a0949efbfdecb45444a421e | original_positive_rank=5
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''

Pilot 8 | seed=golden_v2_candidate_213a5d82a0e3b960fd59e599 | original_positive_rank=1
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''

Pilot 10 | seed=golden_v2_candidate_1f30ced9b8cae506e54374ff | original_positive_rank=1
Contains JSON object: False
Contains JS

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## آزمون کنترل‌شدهٔ سقف پاسخ Reranker

برخی پاسخ‌های پایلوت کاملاً خالی بودند. در این مرحله فقط یک نمونه از آن‌ها با سقف پاسخ بالاتر تکرار می‌شود تا مشخص شود آیا مدل پس از پایان reasoning داخلی، خروجی رتبه‌بندی قابل‌خواندن تولید می‌کند یا خیر.

</div>

In [68]:
# Code Cell 52
# NB13 — Controlled reranking retry inspection (API disabled)

retry_reranked_labels = None
retry_raw_response_text = None

print("=== CONTROLLED RERANKING RETRY — CACHE ONLY ===")
print("Failed pilot responses:", len(failed_reranker_pilot_results))
print("API execution: DISABLED")
if failed_reranker_pilot_results:
    print("Retry intentionally not executed during Run-All.")
else:
    print("No failed pilot response requires retry.")
print("CONTROLLED RERANKING RETRY INSPECTION: PASS")


=== CONTROLLED RERANKING RETRY — CACHE ONLY ===
Failed pilot responses: 9
API execution: DISABLED
Retry intentionally not executed during Run-All.
CONTROLLED RERANKING RETRY INSPECTION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## پایلوت کالیبره‌شدهٔ Reranking

پس از تأیید سقف مناسب پاسخ، Reranking روی ده پرسش از seedهای متفاوت اجرا می‌شود.

هدف این مرحله سنجش هم‌زمان سه معیار است:

- پایداری قالب خروجی؛
- توان مدل در حفظ Passage مثبت؛
- بهبود یا افت رتبهٔ Passage مثبت نسبت به رتبه‌بندی اولیه.

</div>

In [69]:
# Code Cell 53
# NB13 — Calibrated reranking pilot reconstruction (API disabled)

calibrated_reranker_pilot_size = min(10, len(eligible_reranker_query_indexes))
calibrated_query_indexes = []
calibrated_seed_ids = set()
for query_index in eligible_reranker_query_indexes:
    seed_id = arvan_hybrid_query_records[query_index]["source_seed_id"]
    if seed_id not in calibrated_seed_ids:
        calibrated_query_indexes.append(query_index)
        calibrated_seed_ids.add(seed_id)
    if len(calibrated_query_indexes) == calibrated_reranker_pilot_size:
        break

calibrated_reranker_report_path = (
    active_benchmark_run_directory / "arvan_chat_calibrated_reranking_pilot.json"
)
if calibrated_reranker_report_path.is_file():
    cached_calibrated_reranker_report = json.loads(
        calibrated_reranker_report_path.read_text(encoding="utf-8")
    )
    valid_identity = (
        cached_calibrated_reranker_report.get("active_golden_dataset_sha256")
        == golden_dataset_context["sha256"]
        and cached_calibrated_reranker_report.get("model") == arvan_chat_model
    )
    calibrated_reranker_results = (
        cached_calibrated_reranker_report.get("results", []) if valid_identity else []
    )
else:
    calibrated_reranker_results = []

print("=== CALIBRATED RERANKING PILOT — CACHE ONLY ===")
print("Selected queries:", len(calibrated_query_indexes))
print("Cached results:", len(calibrated_reranker_results))
print("API execution: DISABLED")
print("CALIBRATED RERANKING PILOT RECONSTRUCTION: PASS")


=== CALIBRATED RERANKING PILOT — CACHE ONLY ===
Selected queries: 10
Cached results: 10
API execution: DISABLED
CALIBRATED RERANKING PILOT RECONSTRUCTION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی کامل مشروط Reranking

Reranker نمی‌تواند Chunkی را که در ده نتیجهٔ اولیه وجود ندارد بازیابی کند. بنابراین در این ارزیابی فقط پرسش‌هایی بررسی می‌شوند که حداقل یک شاهد مثبت از قبل در Top-10 آن‌ها وجود دارد.

هدف، سنجش دقیق اثر Reranking بر رتبهٔ شاهدهای موجود است. شواهد طلایی فقط برای محاسبهٔ معیار پس از پاسخ مدل استفاده می‌شوند و به مدل داده نمی‌شوند.

خروجی هر درخواست به‌صورت checkpoint ذخیره می‌شود تا اجرای مجدد از نقطهٔ قبلی ادامه پیدا کند.

</div>

In [70]:
# Code Cell 54
# NB13 — Cache-only conditional reranking reconstruction
# FINAL RUN-ALL SAFETY: NO CHAT API CALLS ARE ALLOWED IN THIS CELL.

conditional_reranker_top_k = 10
conditional_reranker_max_tokens = 1536

conditional_reranker_cache_path = (
    active_benchmark_run_directory
    / "arvan_chat_conditional_reranking_cache.json"
)

conditional_reranker_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_conditional_reranking_evaluation.json"
)

conditional_query_indexes = list(
    eligible_reranker_query_indexes
)

eligible_candidate_ids = [
    arvan_hybrid_query_records[query_index]["candidate_id"]
    for query_index in conditional_query_indexes
]

assert len(eligible_candidate_ids) == len(
    set(eligible_candidate_ids)
), "Duplicate candidate_id detected among reranking-eligible queries."

eligible_candidate_id_set = set(
    eligible_candidate_ids
)

print(
    "=== CONDITIONAL RERANKING — CACHE-ONLY RECONSTRUCTION ==="
)
print(
    "Eligible queries:",
    len(conditional_query_indexes),
)
print(
    "API execution:",
    "DISABLED",
)
print(
    "Cache path:",
    conditional_reranker_cache_path,
)

# ------------------------------------------------------------
# Load cache only.
# Missing cache is valid: it means zero completed reranking calls.
# ------------------------------------------------------------

if conditional_reranker_cache_path.is_file():
    conditional_reranker_cache = json.loads(
        conditional_reranker_cache_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        conditional_reranker_cache.get(
            "active_golden_dataset_sha256"
        )
        == golden_dataset_context["sha256"]
    ), (
        "The existing conditional reranker cache belongs "
        "to a different Golden Dataset."
    )

    assert (
        conditional_reranker_cache.get("model")
        == arvan_chat_model
    ), (
        "The existing conditional reranker cache belongs "
        "to a different chat model."
    )

    assert (
        conditional_reranker_cache.get("candidate")
        == best_candidate_name
    ), (
        "The existing conditional reranker cache belongs "
        "to a different retrieval candidate."
    )

    assert (
        int(
            conditional_reranker_cache.get(
                "top_k",
                conditional_reranker_top_k,
            )
        )
        == conditional_reranker_top_k
    ), (
        "The existing conditional reranker cache uses "
        "a different top-k value."
    )

    cached_results_by_candidate_id = (
        conditional_reranker_cache.get(
            "results_by_candidate_id",
            {},
        )
    )

    assert isinstance(
        cached_results_by_candidate_id,
        dict,
    ), (
        "Conditional reranker cache has an invalid "
        "results_by_candidate_id structure."
    )

else:
    conditional_reranker_cache = {
        "active_golden_dataset_sha256": (
            golden_dataset_context["sha256"]
        ),
        "model": arvan_chat_model,
        "candidate": best_candidate_name,
        "top_k": conditional_reranker_top_k,
        "max_tokens": conditional_reranker_max_tokens,
        "results_by_candidate_id": {},
    }

    cached_results_by_candidate_id = {}

# ------------------------------------------------------------
# Validate cached records.
# Do NOT execute missing records.
# ------------------------------------------------------------

unknown_cached_candidate_ids = sorted(
    set(cached_results_by_candidate_id)
    - eligible_candidate_id_set
)

assert not unknown_cached_candidate_ids, (
    "Conditional reranker cache contains candidate IDs "
    "that are not part of the current eligible query set. "
    f"Examples: {unknown_cached_candidate_ids[:5]}"
)

conditional_reranker_results = []

for candidate_id in eligible_candidate_ids:
    if candidate_id not in cached_results_by_candidate_id:
        continue

    result = cached_results_by_candidate_id[
        candidate_id
    ]

    assert (
        result.get("candidate_id")
        == candidate_id
    ), (
        "Cached conditional reranker result has a "
        f"candidate_id mismatch: {candidate_id}"
    )

    assert "response_parse_status" in result, (
        "Cached conditional reranker result is missing "
        f"response_parse_status: {candidate_id}"
    )

    conditional_reranker_results.append(
        result
    )

completed_candidate_id_set = {
    result["candidate_id"]
    for result in conditional_reranker_results
}

missing_conditional_candidate_ids = [
    candidate_id
    for candidate_id in eligible_candidate_ids
    if candidate_id
    not in completed_candidate_id_set
]

parsed_conditional_results = [
    result
    for result in conditional_reranker_results
    if (
        result.get("response_parse_status")
        == "PASS"
        and result.get("reranked_positive_rank")
        is not None
        and result.get("rank_change")
        is not None
    )
]

failed_parse_conditional_results = [
    result
    for result in conditional_reranker_results
    if result.get("response_parse_status")
    != "PASS"
]

eligible_query_count = len(
    conditional_query_indexes
)

completed_query_count = len(
    conditional_reranker_results
)

missing_query_count = len(
    missing_conditional_candidate_ids
)

assert (
    completed_query_count
    + missing_query_count
    == eligible_query_count
), (
    "Conditional reranking completion accounting "
    "does not match the eligible query count."
)

# ------------------------------------------------------------
# Summarize ONLY actually cached results.
# Never present a partial cache as a complete evaluation.
# ------------------------------------------------------------

conditional_reranker_summary = {
    "execution_mode": "CACHE_ONLY_API_DISABLED",
    "evaluation_status": (
        "COMPLETE"
        if missing_query_count == 0
        else "PARTIAL_CACHE_ONLY"
    ),
    "eligible_query_count": (
        eligible_query_count
    ),
    "completed_query_count": (
        completed_query_count
    ),
    "missing_query_count": (
        missing_query_count
    ),
    "completion_rate": (
        completed_query_count
        / eligible_query_count
        if eligible_query_count
        else 1.0
    ),
    "parsed_query_count": len(
        parsed_conditional_results
    ),
    "failed_parse_query_count": len(
        failed_parse_conditional_results
    ),
    "parse_success_rate_among_completed": (
        len(parsed_conditional_results)
        / completed_query_count
        if completed_query_count
        else None
    ),
    "mean_original_positive_rank": (
        float(
            np.mean(
                [
                    result[
                        "original_positive_rank"
                    ]
                    for result
                    in parsed_conditional_results
                ]
            )
        )
        if parsed_conditional_results
        else None
    ),
    "mean_reranked_positive_rank": (
        float(
            np.mean(
                [
                    result[
                        "reranked_positive_rank"
                    ]
                    for result
                    in parsed_conditional_results
                ]
            )
        )
        if parsed_conditional_results
        else None
    ),
    "improved_query_count": int(
        np.sum(
            [
                result["rank_change"] > 0
                for result
                in parsed_conditional_results
            ]
        )
    ),
    "unchanged_query_count": int(
        np.sum(
            [
                result["rank_change"] == 0
                for result
                in parsed_conditional_results
            ]
        )
    ),
    "worsened_query_count": int(
        np.sum(
            [
                result["rank_change"] < 0
                for result
                in parsed_conditional_results
            ]
        )
    ),
    "total_observed_cost": float(
        np.sum(
            [
                attempt.get("cost") or 0.0
                for result
                in conditional_reranker_results
                for attempt
                in result.get("attempts", [])
            ]
        )
    ),
}

# ------------------------------------------------------------
# Write reconstruction report.
# This writes ONLY the report, never the API cache.
# ------------------------------------------------------------

conditional_reranker_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": (
                golden_dataset_context[
                    "sha256"
                ]
            ),
            "model": arvan_chat_model,
            "retrieval_configuration": (
                "arvan_text_embedding_3_large"
                "_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "top_k": conditional_reranker_top_k,
            "execution_mode": (
                "CACHE_ONLY_API_DISABLED"
            ),
            "summary": (
                conditional_reranker_summary
            ),
            "missing_candidate_ids": (
                missing_conditional_candidate_ids
            ),
            "results": (
                conditional_reranker_results
            ),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# Human-readable status
# ------------------------------------------------------------

print(
    "Cached completed:",
    completed_query_count,
)
print(
    "Missing / NOT executed:",
    missing_query_count,
)
print(
    "Completion rate:",
    f"{conditional_reranker_summary['completion_rate']:.4f}",
)
print(
    "Parsed cached results:",
    len(parsed_conditional_results),
)
print(
    "Failed-parse cached results:",
    len(failed_parse_conditional_results),
)

if (
    conditional_reranker_summary[
        "parse_success_rate_among_completed"
    ]
    is None
):
    print(
        "Parse success among completed:",
        "N/A",
    )
else:
    print(
        "Parse success among completed:",
        f"{conditional_reranker_summary['parse_success_rate_among_completed']:.4f}",
    )

print(
    "Mean original positive rank:",
    conditional_reranker_summary[
        "mean_original_positive_rank"
    ],
)

print(
    "Mean reranked positive rank:",
    conditional_reranker_summary[
        "mean_reranked_positive_rank"
    ],
)

print(
    "Improved / unchanged / worsened:",
    conditional_reranker_summary[
        "improved_query_count"
    ],
    "/",
    conditional_reranker_summary[
        "unchanged_query_count"
    ],
    "/",
    conditional_reranker_summary[
        "worsened_query_count"
    ],
)

print(
    "Total observed cached cost:",
    f"{conditional_reranker_summary['total_observed_cost']:.6f}",
)

print(
    "Evaluation status:",
    conditional_reranker_summary[
        "evaluation_status"
    ],
)

print(
    "Report:",
    conditional_reranker_report_path,
)

if missing_query_count:
    print(
        "\nNOTE: Conditional reranking is intentionally "
        "incomplete. Missing queries were NOT sent "
        "to the API."
    )

print(
    "\nCONDITIONAL RERANKING CACHE-ONLY "
    "RECONSTRUCTION: PASS"
)

=== CONDITIONAL RERANKING — CACHE-ONLY RECONSTRUCTION ===
Eligible queries: 2837
API execution: DISABLED
Cache path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_conditional_reranking_cache.json
Cached completed: 15
Missing / NOT executed: 2822
Completion rate: 0.0053
Parsed cached results: 15
Failed-parse cached results: 0
Parse success among completed: 1.0000
Mean original positive rank: 2.066666666666667
Mean reranked positive rank: 2.066666666666667
Improved / unchanged / worsened: 2 / 10 / 3
Total observed cached cost: 0.016493
Evaluation status: PARTIAL_CACHE_ONLY
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_conditional_reranking_evaluation.json

NOTE: Conditional reranking is intentionally in

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## محاسبهٔ اثر کلی Reranking

Reranking فقط می‌تواند ترتیب Chunkهایی را تغییر دهد که از قبل در Top-10 بازیابی اولیه حضور دارند.

در این مرحله معیارهای نهایی بازیابی اولیه و بازیابی پس از Reranking روی تمام ۵۵۳ پرسش مقایسه می‌شوند. هیچ درخواست جدیدی به سرویس ارسال نمی‌شود.

</div>

In [71]:
# Code Cell 55
# NB13 — Full-query metric impact of conditional chat reranking

reranked_positive_rank_by_candidate_id = {
    result["candidate_id"]: result["reranked_positive_rank"]
    for result in parsed_conditional_results
}


baseline_first_positive_rank_by_candidate_id = {}
final_first_positive_rank_by_candidate_id = {}

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    candidate_id = query_record["candidate_id"]
    seed_id = query_record["source_seed_id"]

    positive_rank_positions = np.flatnonzero(
        np.isin(
            best_rrf_ranked_indexes[query_index],
            list(best_positive_indexes_by_seed[seed_id]),
        )
    )

    baseline_first_rank = int(positive_rank_positions[0]) + 1
    baseline_first_positive_rank_by_candidate_id[candidate_id] = (
        baseline_first_rank
    )

    final_first_positive_rank_by_candidate_id[candidate_id] = (
        reranked_positive_rank_by_candidate_id.get(
            candidate_id,
            baseline_first_rank,
        )
    )

assert len(final_first_positive_rank_by_candidate_id) == len(arvan_hybrid_query_records)

def metrics_from_first_positive_ranks(rank_by_candidate_id):
    ranks = list(rank_by_candidate_id.values())

    return {
        "recall_at_1": float(np.mean([rank <= 1 for rank in ranks])),
        "recall_at_3": float(np.mean([rank <= 3 for rank in ranks])),
        "recall_at_5": float(np.mean([rank <= 5 for rank in ranks])),
        "recall_at_10": float(np.mean([rank <= 10 for rank in ranks])),
        "mrr": float(np.mean([1 / rank for rank in ranks])),
        "mean_first_positive_rank": float(np.mean(ranks)),
    }

baseline_hybrid_metrics = metrics_from_first_positive_ranks(
    baseline_first_positive_rank_by_candidate_id
)
reranked_hybrid_metrics = metrics_from_first_positive_ranks(
    final_first_positive_rank_by_candidate_id
)

reranking_impact_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "reranker_model": arvan_chat_model,
    "candidate": best_candidate_name,
    "reranker_scope": {
        "all_query_count": len(arvan_hybrid_query_records),
        "reranked_query_count": len(reranked_positive_rank_by_candidate_id),
        "reranking_input_top_k": 10,
    },
    "baseline_metrics": baseline_hybrid_metrics,
    "reranked_metrics": reranked_hybrid_metrics,
    "metric_deltas": {
        metric_name: (
            reranked_hybrid_metrics[metric_name]
            - baseline_hybrid_metrics[metric_name]
        )
        for metric_name in baseline_hybrid_metrics
    },
}

reranking_impact_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_full_metric_impact.json"
)

reranking_impact_report_path.write_text(
    json.dumps(
        reranking_impact_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== FULL-QUERY RERANKING METRIC IMPACT ===")
print("Queries evaluated:", len(final_first_positive_rank_by_candidate_id))
print("Queries reranked:", len(reranked_positive_rank_by_candidate_id))

for metric_name in (
    "recall_at_1",
    "recall_at_3",
    "recall_at_5",
    "recall_at_10",
    "mrr",
    "mean_first_positive_rank",
):
    print(
        f"\n{metric_name}"
        f"\n- baseline: {baseline_hybrid_metrics[metric_name]:.4f}"
        f"\n- reranked: {reranked_hybrid_metrics[metric_name]:.4f}"
        f"\n- delta: "
        f"{reranked_hybrid_metrics[metric_name] - baseline_hybrid_metrics[metric_name]:+.4f}"
    )

print("\nReport:", reranking_impact_report_path)
print("\nFULL-QUERY RERANKING METRIC IMPACT: PASS")

=== FULL-QUERY RERANKING METRIC IMPACT ===
Queries evaluated: 3813
Queries reranked: 15

recall_at_1
- baseline: 0.3968
- reranked: 0.3963
- delta: -0.0005

recall_at_3
- baseline: 0.5681
- reranked: 0.5681
- delta: +0.0000

recall_at_5
- baseline: 0.6420
- reranked: 0.6418
- delta: -0.0003

recall_at_10
- baseline: 0.7440
- reranked: 0.7440
- delta: +0.0000

mrr
- baseline: 0.5118
- reranked: 0.5115
- delta: -0.0003

mean_first_positive_rank
- baseline: 34.8345
- reranked: 34.8345
- delta: +0.0000

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_reranking_full_metric_impact.json

FULL-QUERY RERANKING METRIC IMPACT: PASS


<div dir="rtl" align="right">

## اثر Reranking بر نگه‌داشتن شواهد در کانتکست نهایی

در سل قبل دیدیم که Reranking می‌تواند جایگاه شواهد درست را در میان ده Chunk بازیابی‌شده بهتر کند، اما نمی‌تواند شواهدی را که اصلاً در ده نتیجهٔ اول نیستند پیدا کند.

در این سل بررسی می‌کنیم که آیا جابه‌جایی Chunkها توسط Reranking باعث می‌شود در سیاست انتخاب‌شدهٔ کانتکست والد–فرزند، Parent حاوی شواهد درست با احتمال بیشتری وارد کانتکست نهایی شود یا نه. این تحلیل فقط از خروجی‌های ذخیره‌شده استفاده می‌کند و هیچ فراخوانی جدیدی به Arvan ندارد.

</div>

In [72]:
# Code Cell 56
# NB13 — Reranking impact on expansive parent-context evidence retention
# Uses cached reranking responses only; makes no Arvan API requests.

selected_context_policy = next(
    policy
    for policy in context_budget_policies
    if policy["policy_id"] == "expansive"
)

reranking_result_by_candidate_id = {
    result["candidate_id"]: result
    for result in parsed_conditional_results
}


def assemble_parent_context_from_child_indexes(
    ranked_child_indexes,
    policy,
):
    selected_parent_ids = []
    current_token_total = 0
    missing_parent_references = 0

    for child_index in ranked_child_indexes[:policy["retrieved_child_k"]]:
        child_record = best_candidate_records[int(child_index)]
        parent_chunk_id = child_record["parent_chunk_id"]

        if parent_chunk_id not in parent_record_by_chunk_id:
            missing_parent_references += 1
            continue

        if parent_chunk_id in selected_parent_ids:
            continue

        parent_record = parent_record_by_chunk_id[parent_chunk_id]
        parent_token_count = parent_record["token_count"]

        if len(selected_parent_ids) >= policy["max_parent_chunks"]:
            continue

        if (
            current_token_total + parent_token_count
            > policy["max_context_tokens"]
        ):
            continue

        selected_parent_ids.append(parent_chunk_id)
        current_token_total += parent_token_count

    return {
        "selected_parent_chunk_ids": selected_parent_ids,
        "token_count": current_token_total,
        "missing_parent_references": missing_parent_references,
    }

reranking_context_rows = []

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    candidate_id = query_record["candidate_id"]
    seed_id = query_record["source_seed_id"]

    baseline_ranked_child_indexes = best_rrf_ranked_indexes[query_index]
    final_ranked_child_indexes = baseline_ranked_child_indexes.copy()

    reranking_applied = candidate_id in reranking_result_by_candidate_id

    if reranking_applied:
        reranking_result = reranking_result_by_candidate_id[candidate_id]

        successful_attempt = next(
            attempt
            for attempt in reranking_result["attempts"]
            if attempt["parsed"]
        )

        reranked_labels = extract_reranked_labels(
            successful_attempt["raw_response"],
            allowed_labels={
                f"P{rank:02d}"
                for rank in range(1, conditional_reranker_top_k + 1)
            },
        )

        assert reranked_labels is not None, (
            f"Cached reranking response cannot be parsed: {candidate_id}"
        )

        baseline_top_k_indexes = baseline_ranked_child_indexes[
            :conditional_reranker_top_k
        ]

        reranked_top_k_indexes = np.array(
            [
                int(
                    baseline_top_k_indexes[
                        int(label.removeprefix("P")) - 1
                    ]
                )
                for label in reranked_labels
            ],
            dtype=int,
        )

        assert len(set(reranked_top_k_indexes)) == conditional_reranker_top_k
        assert set(reranked_top_k_indexes) == set(baseline_top_k_indexes)

        final_ranked_child_indexes = np.concatenate(
            [
                reranked_top_k_indexes,
                baseline_ranked_child_indexes[
                    conditional_reranker_top_k:
                ],
            ]
        )

    positive_child_indexes = best_positive_indexes_by_seed[seed_id]
    positive_parent_ids = {
        best_candidate_records[int(child_index)]["parent_chunk_id"]
        for child_index in positive_child_indexes
    }

    baseline_context = assemble_parent_context_from_child_indexes(
        baseline_ranked_child_indexes,
        selected_context_policy,
    )
    reranked_context = assemble_parent_context_from_child_indexes(
        final_ranked_child_indexes,
        selected_context_policy,
    )

    baseline_parent_hit = bool(
        positive_parent_ids.intersection(
            baseline_context["selected_parent_chunk_ids"]
        )
    )
    reranked_parent_hit = bool(
        positive_parent_ids.intersection(
            reranked_context["selected_parent_chunk_ids"]
        )
    )

    reranking_context_rows.append(
        {
            "candidate_id": candidate_id,
            "source_seed_id": seed_id,
            "reranking_applied": reranking_applied,
            "baseline_parent_evidence_retained": baseline_parent_hit,
            "reranked_parent_evidence_retained": reranked_parent_hit,
            "baseline_parent_chunk_count": len(
                baseline_context["selected_parent_chunk_ids"]
            ),
            "reranked_parent_chunk_count": len(
                reranked_context["selected_parent_chunk_ids"]
            ),
            "baseline_context_tokens": baseline_context["token_count"],
            "reranked_context_tokens": reranked_context["token_count"],
            "missing_parent_references": (
                baseline_context["missing_parent_references"]
                + reranked_context["missing_parent_references"]
            ),
        }
    )

assert len(reranking_context_rows) == len(arvan_hybrid_query_records)
assert sum(
    row["missing_parent_references"]
    for row in reranking_context_rows
) == 0, "A selected Child referenced a missing Parent."

baseline_parent_retention = float(
    np.mean(
        [
            row["baseline_parent_evidence_retained"]
            for row in reranking_context_rows
        ]
    )
)

reranked_parent_retention = float(
    np.mean(
        [
            row["reranked_parent_evidence_retained"]
            for row in reranking_context_rows
        ]
    )
)

reranked_scope_rows = [
    row
    for row in reranking_context_rows
    if row["reranking_applied"]
]

baseline_scope_retention = float(
    np.mean(
        [
            row["baseline_parent_evidence_retained"]
            for row in reranked_scope_rows
        ]
    )
)

reranked_scope_retention = float(
    np.mean(
        [
            row["reranked_parent_evidence_retained"]
            for row in reranked_scope_rows
        ]
    )
)

improved_parent_retention_count = sum(
    (
        not row["baseline_parent_evidence_retained"]
        and row["reranked_parent_evidence_retained"]
    )
    for row in reranking_context_rows
)

worsened_parent_retention_count = sum(
    (
        row["baseline_parent_evidence_retained"]
        and not row["reranked_parent_evidence_retained"]
    )
    for row in reranking_context_rows
)

reranking_context_retention_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "reranker_model": arvan_chat_model,
    "context_policy": selected_context_policy,
    "query_count": len(reranking_context_rows),
    "reranked_query_count": len(reranked_scope_rows),
    "all_query_parent_evidence_retention": {
        "baseline": baseline_parent_retention,
        "reranked": reranked_parent_retention,
        "delta": reranked_parent_retention - baseline_parent_retention,
    },
    "reranked_scope_parent_evidence_retention": {
        "baseline": baseline_scope_retention,
        "reranked": reranked_scope_retention,
        "delta": reranked_scope_retention - baseline_scope_retention,
    },
    "improved_parent_retention_count": improved_parent_retention_count,
    "worsened_parent_retention_count": worsened_parent_retention_count,
    "rows": reranking_context_rows,
}

reranking_context_retention_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_parent_context_retention.json"
)

reranking_context_retention_report_path.write_text(
    json.dumps(
        reranking_context_retention_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== RERANKING IMPACT ON PARENT-CONTEXT RETENTION ===")
print("Context policy:", selected_context_policy["policy_id"])
print("Queries evaluated:", len(reranking_context_rows))
print("Queries reranked:", len(reranked_scope_rows))

print("\nAll-query parent-evidence retention:")
print(f"- baseline: {baseline_parent_retention:.4f}")
print(f"- reranked: {reranked_parent_retention:.4f}")
print(
    "- delta: "
    f"{reranked_parent_retention - baseline_parent_retention:+.4f}"
)

print("\nReranked-query scope parent-evidence retention:")
print(f"- baseline: {baseline_scope_retention:.4f}")
print(f"- reranked: {reranked_scope_retention:.4f}")
print(
    "- delta: "
    f"{reranked_scope_retention - baseline_scope_retention:+.4f}"
)

print("\nContext evidence changes:")
print("- improved:", improved_parent_retention_count)
print("- worsened:", worsened_parent_retention_count)

print("\nReport:", reranking_context_retention_report_path)
print("\nRERANKING IMPACT ON PARENT-CONTEXT RETENTION: PASS")

=== RERANKING IMPACT ON PARENT-CONTEXT RETENTION ===
Context policy: expansive
Queries evaluated: 3813
Queries reranked: 15

All-query parent-evidence retention:
- baseline: 0.6937
- reranked: 0.6937
- delta: +0.0000

Reranked-query scope parent-evidence retention:
- baseline: 0.9333
- reranked: 0.9333
- delta: +0.0000

Context evidence changes:
- improved: 1
- worsened: 1

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_reranking_parent_context_retention.json

RERANKING IMPACT ON PARENT-CONTEXT RETENTION: PASS


<div dir="rtl" align="right">

## امکان‌سنجی اجرایی Reranking

نتایج قبل نشان دادند که Reranking جایگاه شواهد درست را بهتر می‌کند و در بخشی از موارد باعث می‌شود Parent درست وارد کانتکست نهایی شود.

اما ارزیابی قبلی فقط روی پرسش‌هایی انجام شد که در دادهٔ ارزیابی می‌دانستیم شواهد درست از قبل در ده نتیجهٔ اول هستند. در محصول واقعی چنین اطلاعی نداریم؛ بنابراین اگر Reranking وارد محصول شود، باید برای همهٔ پرسش‌ها یا بر اساس یک قانون مستقل و قابل‌اجرا اعمال شود.

در این سل، با استفاده از زمان و هزینهٔ واقعی درخواست‌های ثبت‌شده، هزینه و زمان تقریبی اجرای کامل روی همهٔ پرسش‌های Golden Dataset تخمین زده می‌شود. این تخمین تصمیم اجرایی نیست و هیچ درخواست جدیدی به سرویس ارسال نمی‌کند.

</div>

In [73]:
# Code Cell 57
# NB13 — Reranking feasibility profile from cached live executions
# No Arvan API requests are made in this cell.


reranking_execution_rows = []

for result in parsed_conditional_results:
    attempt_seconds = [
        float(attempt["seconds"])
        for attempt in result["attempts"]
        if attempt.get("seconds") is not None
    ]
    attempt_costs = [
        float(attempt["cost"])
        for attempt in result["attempts"]
        if attempt.get("cost") is not None
    ]

    reranking_execution_rows.append(
        {
            "candidate_id": result["candidate_id"],
            "source_seed_id": result["source_seed_id"],
            "attempt_count": len(result["attempts"]),
            "total_seconds": float(sum(attempt_seconds)),
            "total_cost": float(sum(attempt_costs)),
            "response_parse_status": result["response_parse_status"],
        }
    )

assert all(
    row["response_parse_status"] == "PASS"
    for row in reranking_execution_rows
)

observed_seconds = np.array(
    [row["total_seconds"] for row in reranking_execution_rows],
    dtype=float,
)
observed_costs = np.array(
    [row["total_cost"] for row in reranking_execution_rows],
    dtype=float,
)

full_query_count = len(arvan_hybrid_query_records)

reranking_feasibility_profile = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "reranker_model": arvan_chat_model,
    "measured_query_count": len(reranking_execution_rows),
    "full_golden_query_count": full_query_count,
    "measured_execution": {
        "parse_success_rate": 1.0,
        "mean_seconds_per_query": float(np.mean(observed_seconds)),
        "p95_seconds_per_query": float(
            np.percentile(observed_seconds, 95)
        ),
        "mean_cost_per_query": float(np.mean(observed_costs)),
        "p95_cost_per_query": float(
            np.percentile(observed_costs, 95)
        ),
        "total_observed_cost": float(np.sum(observed_costs)),
    },
    "sequential_full_evaluation_estimate": {
        "estimated_seconds": float(
            np.mean(observed_seconds) * full_query_count
        ),
        "estimated_minutes": float(
            np.mean(observed_seconds) * full_query_count / 60
        ),
        "estimated_cost": float(
            np.mean(observed_costs) * full_query_count
        ),
    },
    "interpretation": [
        (
            "This is an estimate from cached live executions, not a "
            "guaranteed production latency or cost."
        ),
        (
            "The prior reranking evaluation was conditional on a "
            "positive Child already being in the top 10."
        ),
        (
            "A production policy cannot use Golden labels to decide "
            "whether reranking should run."
        ),
    ],
    "next_decision": (
        "Decide whether to run an unbiased full-query reranking "
        "evaluation on all active Golden Dataset questions."
    ),
}

reranking_feasibility_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_feasibility_profile.json"
)

reranking_feasibility_report_path.write_text(
    json.dumps(
        reranking_feasibility_profile,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

measured = reranking_feasibility_profile["measured_execution"]
estimated = reranking_feasibility_profile[
    "sequential_full_evaluation_estimate"
]

print("=== RERANKING FEASIBILITY PROFILE ===")
print("Measured cached queries:", len(reranking_execution_rows))
print("Full Golden query count:", full_query_count)

print("\nObserved per-query execution:")
print(f"- Parse success: {measured['parse_success_rate']:.4f}")
print(f"- Mean seconds: {measured['mean_seconds_per_query']:.2f}")
print(f"- P95 seconds: {measured['p95_seconds_per_query']:.2f}")
print(f"- Mean cost: {measured['mean_cost_per_query']:.6f}")
print(f"- P95 cost: {measured['p95_cost_per_query']:.6f}")
print(f"- Total observed cost: {measured['total_observed_cost']:.6f}")

print("\nSequential estimate for all Golden queries:")
print(f"- Estimated minutes: {estimated['estimated_minutes']:.1f}")
print(f"- Estimated cost: {estimated['estimated_cost']:.4f}")

print("\nImportant boundary:")
for item in reranking_feasibility_profile["interpretation"]:
    print("-", item)

print("\nReport:", reranking_feasibility_report_path)
print("\nRERANKING FEASIBILITY PROFILE: PASS")

=== RERANKING FEASIBILITY PROFILE ===
Measured cached queries: 15
Full Golden query count: 3813

Observed per-query execution:
- Parse success: 1.0000
- Mean seconds: 10.59
- P95 seconds: 16.50
- Mean cost: 0.001100
- P95 cost: 0.001486
- Total observed cost: 0.016493

Sequential estimate for all Golden queries:
- Estimated minutes: 673.1
- Estimated cost: 4.1924

Important boundary:
- This is an estimate from cached live executions, not a guaranteed production latency or cost.
- The prior reranking evaluation was conditional on a positive Child already being in the top 10.
- A production policy cannot use Golden labels to decide whether reranking should run.

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_reranking_feasibility_profile.json

RERANKING FEASIBILITY PROFILE: PASS


<div dir="rtl" align="right">

## ارزیابی کامل و قابل‌ادامهٔ Reranking

در این مرحله Reranking برای همهٔ پرسش‌های Golden Dataset ارزیابی می‌شود، نه فقط پرسش‌هایی که در ارزیابی قبلی می‌دانستیم شواهد درست در ده نتیجهٔ اول دارند.

نتیجهٔ هر پرسش بلافاصله در یک کش ذخیره می‌شود. شناسهٔ این کش بر اساس هش Golden Dataset، مدل، Chunk candidate و تنظیمات Reranking است؛ بنابراین تغییر نام Notebook یا Restart کردن Kernel باعث تکرار درخواست‌های انجام‌شده نمی‌شود. اگر اجرا قطع شود، اجرای بعدی فقط پرسش‌های باقی‌مانده را ارسال می‌کند.

</div>

In [74]:
# Code Cell 58
# NB13 — Cache-only full Arvan Chat reranking reconstruction
# FINAL RUN-ALL SAFETY: NO CHAT API CALLS ARE ALLOWED IN THIS CELL.

import json

full_reranker_top_k = 10
full_reranker_excerpt_char_limit = 1000

full_reranker_cache_path = (
    active_benchmark_run_directory
    / "arvan_chat_full_reranking_cache.json"
)

full_reranker_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_full_reranking_evaluation.json"
)

allowed_full_reranker_labels = {
    f"P{rank:02d}"
    for rank in range(1, full_reranker_top_k + 1)
}


def get_successful_reranked_labels(result):
    labels = result.get("reranked_labels")

    if (
        isinstance(labels, list)
        and len(labels) == full_reranker_top_k
        and set(labels) == allowed_full_reranker_labels
    ):
        return labels

    for attempt in result.get("attempts", []):
        if not attempt.get("parsed"):
            continue

        labels = extract_reranked_labels(
            attempt.get("raw_response", ""),
            allowed_labels=allowed_full_reranker_labels,
        )

        if labels is not None:
            return labels

    return None


# ------------------------------------------------------------
# Active Golden-v2 query universe
# ------------------------------------------------------------

full_reranker_candidate_ids = [
    query_record["candidate_id"]
    for query_record in arvan_hybrid_query_records
]

assert len(full_reranker_candidate_ids) == len(
    set(full_reranker_candidate_ids)
), "Duplicate candidate_id detected in full reranker query universe."

full_reranker_candidate_id_set = set(
    full_reranker_candidate_ids
)

print(
    "=== FULL ARVAN CHAT RERANKING — CACHE-ONLY RECONSTRUCTION ==="
)
print(
    "Golden queries:",
    len(full_reranker_candidate_ids),
)
print(
    "API execution:",
    "DISABLED",
)
print(
    "Full cache path:",
    full_reranker_cache_path,
)


# ------------------------------------------------------------
# Load existing FULL reranking cache only.
# Missing cache is valid and means zero full-cache results.
# ------------------------------------------------------------

if full_reranker_cache_path.is_file():
    full_reranker_cache = json.loads(
        full_reranker_cache_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        full_reranker_cache.get(
            "active_golden_dataset_sha256"
        )
        == golden_dataset_context["sha256"]
    ), (
        "The full reranking cache belongs to a different "
        "Golden Dataset."
    )

    assert (
        full_reranker_cache.get("model")
        == arvan_chat_model
    ), (
        "The full reranking cache belongs to a different "
        "chat model."
    )

    assert (
        full_reranker_cache.get("candidate")
        == best_candidate_name
    ), (
        "The full reranking cache belongs to a different "
        "Chunk candidate."
    )

    assert (
        int(
            full_reranker_cache.get(
                "top_k",
                full_reranker_top_k,
            )
        )
        == full_reranker_top_k
    ), (
        "The full reranking cache uses a different "
        "top-k setting."
    )

    full_results_by_candidate_id = (
        full_reranker_cache.get(
            "results_by_candidate_id",
            {},
        )
    )

    assert isinstance(
        full_results_by_candidate_id,
        dict,
    ), (
        "Full reranking cache has an invalid "
        "results_by_candidate_id structure."
    )

else:
    full_reranker_cache = {
        "active_golden_dataset_sha256": (
            golden_dataset_context["sha256"]
        ),
        "model": arvan_chat_model,
        "candidate": best_candidate_name,
        "top_k": full_reranker_top_k,
        "excerpt_char_limit": (
            full_reranker_excerpt_char_limit
        ),
        "results_by_candidate_id": {},
    }

    full_results_by_candidate_id = {}


# ------------------------------------------------------------
# Validate any pre-existing full-cache records.
# ------------------------------------------------------------

unknown_full_candidate_ids = sorted(
    set(full_results_by_candidate_id)
    - full_reranker_candidate_id_set
)

assert not unknown_full_candidate_ids, (
    "Full reranking cache contains candidate IDs outside "
    "the active Golden-v2 query universe. "
    f"Examples: {unknown_full_candidate_ids[:5]}"
)


# ------------------------------------------------------------
# Reuse already-paid CONDITIONAL results in memory.
#
# IMPORTANT:
# This performs ZERO API calls.
# It only copies valid cached conditional results into the
# in-memory full-result registry.
# ------------------------------------------------------------

conditional_reuse_count = 0

for conditional_result in parsed_conditional_results:
    candidate_id = conditional_result["candidate_id"]

    if candidate_id not in full_reranker_candidate_id_set:
        continue

    if candidate_id in full_results_by_candidate_id:
        continue

    reranked_labels = get_successful_reranked_labels(
        conditional_result
    )

    assert reranked_labels is not None, (
        "A cached conditional result marked as parsed "
        "has no usable reranked labels: "
        f"{candidate_id}"
    )

    reused_result = {
        **conditional_result,
        "reranked_labels": reranked_labels,
        "result_origin": (
            "REUSED_FROM_CONDITIONAL_CACHE"
        ),
    }

    full_results_by_candidate_id[
        candidate_id
    ] = reused_result

    conditional_reuse_count += 1


# ------------------------------------------------------------
# Persist ONLY reuse of already-paid cached results.
#
# No API-generated result is created here.
# ------------------------------------------------------------

full_reranker_cache[
    "results_by_candidate_id"
] = full_results_by_candidate_id

if conditional_reuse_count > 0:
    full_reranker_cache_path.write_text(
        json.dumps(
            full_reranker_cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


# ------------------------------------------------------------
# Reconstruct results in deterministic Golden query order.
# ------------------------------------------------------------

full_reranker_results = []

for candidate_id in full_reranker_candidate_ids:
    if candidate_id not in full_results_by_candidate_id:
        continue

    result = full_results_by_candidate_id[
        candidate_id
    ]

    assert (
        result.get("candidate_id")
        == candidate_id
    ), (
        "Cached full reranker result has a "
        f"candidate_id mismatch: {candidate_id}"
    )

    full_reranker_results.append(
        result
    )


completed_full_candidate_id_set = {
    result["candidate_id"]
    for result in full_reranker_results
}

missing_full_reranker_candidate_ids = [
    candidate_id
    for candidate_id in full_reranker_candidate_ids
    if candidate_id
    not in completed_full_candidate_id_set
]

full_reranker_completed_count = len(
    full_reranker_results
)

full_reranker_missing_count = len(
    missing_full_reranker_candidate_ids
)

assert (
    full_reranker_completed_count
    + full_reranker_missing_count
    == len(full_reranker_candidate_ids)
), (
    "Full reranker completion accounting does not match "
    "the active Golden query count."
)


# ------------------------------------------------------------
# Parse statistics ONLY for actually cached results.
# ------------------------------------------------------------

parsed_full_reranker_results = [
    result
    for result in full_reranker_results
    if (
        result.get("response_parse_status")
        == "PASS"
        and get_successful_reranked_labels(result)
        is not None
    )
]

failed_parse_full_reranker_results = [
    result
    for result in full_reranker_results
    if result not in parsed_full_reranker_results
]

full_reranker_parse_success_count = len(
    parsed_full_reranker_results
)

full_reranker_total_cost = float(
    np.sum(
        [
            attempt.get("cost") or 0.0
            for result in full_reranker_results
            for attempt in result.get(
                "attempts",
                [],
            )
        ]
    )
)


# ------------------------------------------------------------
# Explicit PARTIAL/COMPLETE state.
# A partial cache must never masquerade as full evaluation.
# ------------------------------------------------------------

full_reranker_evaluation_status = (
    "COMPLETE"
    if full_reranker_missing_count == 0
    else "PARTIAL_CACHE_ONLY"
)

full_reranker_report = {
    "active_golden_dataset_sha256": (
        golden_dataset_context["sha256"]
    ),
    "model": arvan_chat_model,
    "candidate": best_candidate_name,
    "top_k": full_reranker_top_k,
    "excerpt_char_limit": (
        full_reranker_excerpt_char_limit
    ),
    "execution_mode": (
        "CACHE_ONLY_API_DISABLED"
    ),
    "evaluation_status": (
        full_reranker_evaluation_status
    ),
    "eligible_query_count": len(
        full_reranker_candidate_ids
    ),
    "completed_query_count": (
        full_reranker_completed_count
    ),
    "missing_query_count": (
        full_reranker_missing_count
    ),
    "completion_rate": (
        full_reranker_completed_count
        / len(full_reranker_candidate_ids)
        if full_reranker_candidate_ids
        else 1.0
    ),
    "parse_success_count": (
        full_reranker_parse_success_count
    ),
    "parse_failure_count": len(
        failed_parse_full_reranker_results
    ),
    "parse_success_rate_among_completed": (
        full_reranker_parse_success_count
        / full_reranker_completed_count
        if full_reranker_completed_count
        else None
    ),
    "reused_from_conditional_cache_this_run": (
        conditional_reuse_count
    ),
    "total_observed_cost": (
        full_reranker_total_cost
    ),
    "missing_candidate_ids": (
        missing_full_reranker_candidate_ids
    ),
    "results": full_reranker_results,
}


# ------------------------------------------------------------
# Write report only.
# ------------------------------------------------------------

full_reranker_report_path.write_text(
    json.dumps(
        full_reranker_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# Human-readable status
# ------------------------------------------------------------

print(
    "Previously/full cached after reconstruction:",
    full_reranker_completed_count,
)

print(
    "Reused from conditional cache this run:",
    conditional_reuse_count,
)

print(
    "Missing / NOT executed:",
    full_reranker_missing_count,
)

print(
    "Completion rate:",
    f"{full_reranker_report['completion_rate']:.4f}",
)

print(
    "Parsed cached results:",
    full_reranker_parse_success_count,
)

print(
    "Failed-parse cached results:",
    len(failed_parse_full_reranker_results),
)

if (
    full_reranker_report[
        "parse_success_rate_among_completed"
    ]
    is None
):
    print(
        "Parse success among completed:",
        "N/A",
    )
else:
    print(
        "Parse success among completed:",
        f"{full_reranker_report['parse_success_rate_among_completed']:.4f}",
    )

print(
    "Total observed cached cost:",
    f"{full_reranker_total_cost:.6f}",
)

print(
    "Evaluation status:",
    full_reranker_evaluation_status,
)

print(
    "Cache:",
    full_reranker_cache_path,
)

print(
    "Report:",
    full_reranker_report_path,
)

if full_reranker_missing_count:
    print(
        "\nNOTE: Full reranking is intentionally incomplete. "
        "Missing Golden-v2 queries were NOT sent to the API."
    )

print(
    "\nFULL ARVAN CHAT RERANKING "
    "CACHE-ONLY RECONSTRUCTION: PASS"
)

=== FULL ARVAN CHAT RERANKING — CACHE-ONLY RECONSTRUCTION ===
Golden queries: 3813
API execution: DISABLED
Full cache path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_full_reranking_cache.json
Previously/full cached after reconstruction: 15
Reused from conditional cache this run: 0
Missing / NOT executed: 3798
Completion rate: 0.0039
Parsed cached results: 15
Failed-parse cached results: 0
Parse success among completed: 1.0000
Total observed cached cost: 0.016493
Evaluation status: PARTIAL_CACHE_ONLY
Cache: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_full_reranking_cache.json
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e7106

<div dir="rtl" align="right">

## ترمیم پاسخ‌های نامعتبر Reranking

اجرای کامل Reranking برای همهٔ پرسش‌ها انجام شد. اگر تعداد کمی پاسخ به‌علت ناقص یا نامعتبر بودن ساختار JSON قابل‌خواندن نباشند، فقط همان موارد با تنظیم خروجی بزرگ‌تر دوباره اجرا می‌شوند.

نتیجهٔ ترمیم در همان کش اصلی ثبت می‌شود. این مرحله هیچ پرسش موفقی را دوباره ارسال نمی‌کند.

</div>

In [75]:
# Code Cell 59
# NB13 — Full reranking cache-only validation and failed-response inspection
# RUN-ALL SAFE: ZERO API CALLS.

import json

# ------------------------------------------------------------
# 1. Required upstream state
# ------------------------------------------------------------

assert "arvan_hybrid_query_records" in globals(), (
    "arvan_hybrid_query_records is unavailable."
)

assert "active_benchmark_run_directory" in globals(), (
    "active_benchmark_run_directory is unavailable."
)

assert "golden_dataset_context" in globals(), (
    "golden_dataset_context is unavailable."
)

assert "arvan_chat_model" in globals(), (
    "arvan_chat_model is unavailable."
)

assert "best_candidate_name" in globals(), (
    "best_candidate_name is unavailable."
)

assert "extract_reranked_labels" in globals(), (
    "extract_reranked_labels is unavailable."
)


# ------------------------------------------------------------
# 2. Paths and fixed configuration
# ------------------------------------------------------------

full_reranker_top_k = 10

full_reranker_cache_path = (
    active_benchmark_run_directory
    / "arvan_chat_full_reranking_cache.json"
)

full_reranker_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_full_reranking_evaluation.json"
)

allowed_full_reranker_labels = {
    f"P{rank:02d}"
    for rank in range(
        1,
        full_reranker_top_k + 1,
    )
}


# ------------------------------------------------------------
# 3. Local parser helper
# ------------------------------------------------------------

def cell82_get_reranked_labels(result):
    labels = result.get("reranked_labels")

    if (
        isinstance(labels, list)
        and len(labels) == full_reranker_top_k
        and set(labels)
        == allowed_full_reranker_labels
    ):
        return labels

    for attempt in result.get("attempts", []):
        if not attempt.get("parsed"):
            continue

        labels = extract_reranked_labels(
            attempt.get("raw_response", ""),
            allowed_labels=(
                allowed_full_reranker_labels
            ),
        )

        if labels is not None:
            return labels

    return None


# ------------------------------------------------------------
# 4. Active Golden-v2 universe
# ------------------------------------------------------------

active_full_candidate_ids = [
    record["candidate_id"]
    for record in arvan_hybrid_query_records
]

assert len(active_full_candidate_ids) == len(
    set(active_full_candidate_ids)
), (
    "Duplicate candidate_id detected in "
    "arvan_hybrid_query_records."
)

active_full_candidate_id_set = set(
    active_full_candidate_ids
)

total_golden_query_count = len(
    active_full_candidate_ids
)


# ------------------------------------------------------------
# 5. Read paid-result cache DIRECTLY FROM DISK
#
# This deliberately ignores potentially stale in-memory
# full_reranker_report/full_reranker_results variables.
# ------------------------------------------------------------

if full_reranker_cache_path.is_file():
    cell82_cache = json.loads(
        full_reranker_cache_path.read_text(
            encoding="utf-8"
        )
    )
else:
    cell82_cache = {
        "active_golden_dataset_sha256": (
            golden_dataset_context["sha256"]
        ),
        "model": arvan_chat_model,
        "candidate": best_candidate_name,
        "top_k": full_reranker_top_k,
        "results_by_candidate_id": {},
    }


# ------------------------------------------------------------
# 6. Validate cache identity
# ------------------------------------------------------------

assert (
    cell82_cache.get(
        "active_golden_dataset_sha256"
    )
    == golden_dataset_context["sha256"]
), (
    "Full reranking cache belongs to a different "
    "Golden Dataset."
)

assert (
    cell82_cache.get("model")
    == arvan_chat_model
), (
    "Full reranking cache belongs to a different "
    "chat model."
)

assert (
    cell82_cache.get("candidate")
    == best_candidate_name
), (
    "Full reranking cache belongs to a different "
    "retrieval candidate."
)

assert (
    int(
        cell82_cache.get(
            "top_k",
            full_reranker_top_k,
        )
    )
    == full_reranker_top_k
), (
    "Full reranking cache uses a different top-k."
)

cell82_results_by_candidate_id = (
    cell82_cache.get(
        "results_by_candidate_id",
        {},
    )
)

assert isinstance(
    cell82_results_by_candidate_id,
    dict,
), (
    "Invalid results_by_candidate_id in "
    "full reranking cache."
)


# ------------------------------------------------------------
# 7. Validate cached candidate IDs
# ------------------------------------------------------------

unknown_cached_ids = sorted(
    set(cell82_results_by_candidate_id)
    - active_full_candidate_id_set
)

assert not unknown_cached_ids, (
    "Full reranking cache contains candidate IDs "
    "outside the active Golden-v2 universe. "
    f"Examples: {unknown_cached_ids[:5]}"
)


# ------------------------------------------------------------
# 8. Reconstruct cached results in deterministic order
# ------------------------------------------------------------

full_reranker_results = []

for candidate_id in active_full_candidate_ids:
    result = cell82_results_by_candidate_id.get(
        candidate_id
    )

    if result is None:
        continue

    assert (
        result.get("candidate_id")
        == candidate_id
    ), (
        "Cached result candidate_id mismatch: "
        f"{candidate_id}"
    )

    full_reranker_results.append(
        result
    )


# ------------------------------------------------------------
# 9. Separate successful and failed cached responses
# ------------------------------------------------------------

successful_full_reranker_results = []
failed_full_reranker_results = []

for result in full_reranker_results:
    reranked_labels = (
        cell82_get_reranked_labels(result)
    )

    if (
        result.get("response_parse_status")
        == "PASS"
        and reranked_labels is not None
    ):
        successful_full_reranker_results.append(
            result
        )
    else:
        failed_full_reranker_results.append(
            result
        )


# ------------------------------------------------------------
# 10. Missing queries
#
# Missing means NOT EXECUTED.
# It is NOT equivalent to parse failure.
# ------------------------------------------------------------

cached_candidate_id_set = {
    result["candidate_id"]
    for result in full_reranker_results
}

missing_full_reranker_candidate_ids = [
    candidate_id
    for candidate_id in active_full_candidate_ids
    if candidate_id
    not in cached_candidate_id_set
]

full_reranker_cached_count = len(
    full_reranker_results
)

full_reranker_parse_success_count = len(
    successful_full_reranker_results
)

full_reranker_parse_failure_count = len(
    failed_full_reranker_results
)

full_reranker_missing_count = len(
    missing_full_reranker_candidate_ids
)


# ------------------------------------------------------------
# 11. Accounting checks
# ------------------------------------------------------------

assert (
    full_reranker_cached_count
    == (
        full_reranker_parse_success_count
        + full_reranker_parse_failure_count
    )
), (
    "Cached parse accounting mismatch."
)

assert (
    total_golden_query_count
    == (
        full_reranker_cached_count
        + full_reranker_missing_count
    )
), (
    "Golden query completion accounting mismatch."
)


# ------------------------------------------------------------
# 12. Calculate observed cost from cached requests only
# ------------------------------------------------------------

full_reranker_total_cost = float(
    np.sum(
        [
            attempt.get("cost") or 0.0
            for result in full_reranker_results
            for attempt in result.get(
                "attempts",
                [],
            )
        ]
    )
)


# ------------------------------------------------------------
# 13. Explicit evaluation state
# ------------------------------------------------------------

if full_reranker_missing_count == 0:
    full_reranker_evaluation_status = (
        "COMPLETE"
    )
else:
    full_reranker_evaluation_status = (
        "PARTIAL_CACHE_ONLY"
    )

if full_reranker_parse_failure_count == 0:
    failed_response_repair_status = (
        "NOT_REQUIRED"
    )
else:
    failed_response_repair_status = (
        "UNRESOLVED_API_DISABLED"
    )

parse_success_rate_among_cached = (
    full_reranker_parse_success_count
    / full_reranker_cached_count
    if full_reranker_cached_count
    else None
)

completion_rate = (
    full_reranker_cached_count
    / total_golden_query_count
    if total_golden_query_count
    else 1.0
)


# ------------------------------------------------------------
# 14. Rebuild report from scratch
#
# Do not trust an old in-memory report object.
# ------------------------------------------------------------

full_reranker_report = {
    "active_golden_dataset_sha256": (
        golden_dataset_context["sha256"]
    ),
    "model": arvan_chat_model,
    "candidate": best_candidate_name,
    "top_k": full_reranker_top_k,

    "execution_mode": (
        "CACHE_ONLY_API_DISABLED"
    ),

    "evaluation_status": (
        full_reranker_evaluation_status
    ),

    "eligible_query_count": (
        total_golden_query_count
    ),

    "completed_query_count": (
        full_reranker_cached_count
    ),

    "missing_query_count": (
        full_reranker_missing_count
    ),

    "completion_rate": (
        completion_rate
    ),

    "parse_success_count": (
        full_reranker_parse_success_count
    ),

    "parse_failure_count": (
        full_reranker_parse_failure_count
    ),

    "parse_success_rate_among_completed": (
        parse_success_rate_among_cached
    ),

    "failed_response_repair_mode": (
        "DISABLED"
    ),

    "failed_response_repair_status": (
        failed_response_repair_status
    ),

    "total_observed_cost": (
        full_reranker_total_cost
    ),

    "missing_candidate_ids": (
        missing_full_reranker_candidate_ids
    ),

    "results": (
        full_reranker_results
    ),
}


# ------------------------------------------------------------
# 15. Write REPORT only.
#
# IMPORTANT:
# arvan_chat_full_reranking_cache.json is NOT modified.
# ------------------------------------------------------------

full_reranker_report_path.write_text(
    json.dumps(
        full_reranker_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 16. Human-readable final status
# ------------------------------------------------------------

print(
    "=== FULL RERANKING CACHE-ONLY "
    "VALIDATION ==="
)

print(
    "API execution:",
    "DISABLED",
)

print(
    "Golden queries:",
    total_golden_query_count,
)

print(
    "Cached results:",
    full_reranker_cached_count,
)

print(
    "Successful cached responses:",
    full_reranker_parse_success_count,
)

print(
    "Failed cached responses:",
    full_reranker_parse_failure_count,
)

print(
    "Missing / NOT executed:",
    full_reranker_missing_count,
)

print(
    "Completion rate:",
    f"{completion_rate:.4f}",
)

if parse_success_rate_among_cached is None:
    print(
        "Parse success among cached:",
        "N/A",
    )
else:
    print(
        "Parse success among cached:",
        f"{parse_success_rate_among_cached:.4f}",
    )

print(
    "Failed-response repair:",
    failed_response_repair_status,
)

print(
    "Total observed cached cost:",
    f"{full_reranker_total_cost:.6f}",
)

print(
    "Evaluation status:",
    full_reranker_evaluation_status,
)

print(
    "Cache:",
    full_reranker_cache_path,
)

print(
    "Report:",
    full_reranker_report_path,
)

if full_reranker_parse_failure_count:
    print(
        "\nNOTE: Cached parse failures remain "
        "unresolved because API execution is disabled."
    )

if full_reranker_missing_count:
    print(
        "\nNOTE: Missing Golden-v2 queries were NOT "
        "sent to the API."
    )

print(
    "\nFULL RERANKING CACHE-ONLY "
    "VALIDATION: PASS"
)

=== FULL RERANKING CACHE-ONLY VALIDATION ===
API execution: DISABLED
Golden queries: 3813
Cached results: 15
Successful cached responses: 15
Failed cached responses: 0
Missing / NOT executed: 3798
Completion rate: 0.0039
Parse success among cached: 1.0000
Failed-response repair: NOT_REQUIRED
Total observed cached cost: 0.016493
Evaluation status: PARTIAL_CACHE_ONLY
Cache: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_full_reranking_cache.json
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_full_reranking_evaluation.json

NOTE: Missing Golden-v2 queries were NOT sent to the API.

FULL RERANKING CACHE-ONLY VALIDATION: PASS


<div dir="rtl" align="right">

## اثر کامل Reranking بر بازیابی

اکنون برای همهٔ پرسش‌های Golden Dataset یک ترتیب جدید از ده Chunk اول داریم. در این سل، ترتیب اولیه و ترتیب پس از Reranking مقایسه می‌شوند تا مشخص شود شواهد درست چند بار به رتبه‌های بالاتر منتقل شده‌اند.

Reranking فقط ترتیب ده نتیجهٔ اول را تغییر می‌دهد؛ بنابراین انتظار نداریم Recall@10 افزایش یابد، اما معیارهایی مانند Recall@1، Recall@3 و MRR می‌توانند بهتر شوند.

</div>

In [76]:
# Code Cell 60
# NB13 — Partial cached reranking metric evaluation
# RUN-ALL SAFE: ZERO API CALLS.

import json
import numpy as np

# ------------------------------------------------------------
# 1. Required state
# ------------------------------------------------------------

required_names = [
    "full_reranker_results",
    "arvan_hybrid_query_records",
    "best_rrf_ranked_indexes",
    "best_positive_indexes_by_seed",
    "active_benchmark_run_directory",
    "golden_dataset_context",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, (
    "Missing required state: "
    + ", ".join(missing_names)
)


# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

top_k = 10

report_path = (
    active_benchmark_run_directory
    / "arvan_chat_full_reranking_metric_impact.json"
)

allowed_labels = {
    "P" + str(rank).zfill(2)
    for rank in range(1, top_k + 1)
}


# ------------------------------------------------------------
# 3. Helper for cached reranking labels
# ------------------------------------------------------------

def get_cell83_labels(result):
    labels = result.get("reranked_labels")

    if isinstance(labels, list):
        if (
            len(labels) == top_k
            and set(labels) == allowed_labels
        ):
            return labels

    if "extract_reranked_labels" not in globals():
        return None

    for attempt in result.get("attempts", []):
        if not attempt.get("parsed"):
            continue

        parsed = extract_reranked_labels(
            attempt.get("raw_response", ""),
            allowed_labels=allowed_labels,
        )

        if parsed is not None:
            return parsed

    return None


# ------------------------------------------------------------
# 4. Active query registry
# ------------------------------------------------------------

query_index_by_candidate_id = {}

for query_index, query_record in enumerate(
    arvan_hybrid_query_records
):
    candidate_id = query_record["candidate_id"]

    assert candidate_id not in query_index_by_candidate_id, (
        "Duplicate candidate_id: " + str(candidate_id)
    )

    query_index_by_candidate_id[
        candidate_id
    ] = query_index


total_query_count = len(
    arvan_hybrid_query_records
)


# ------------------------------------------------------------
# 5. Keep only usable cached reranking results
# ------------------------------------------------------------

usable_cached_results = []
unusable_cached_results = []

for result in full_reranker_results:
    candidate_id = result.get("candidate_id")

    if candidate_id not in query_index_by_candidate_id:
        unusable_cached_results.append(
            {
                "candidate_id": candidate_id,
                "reason": "NOT_IN_ACTIVE_QUERY_SET",
            }
        )
        continue

    labels = get_cell83_labels(result)

    if (
        result.get("response_parse_status") != "PASS"
        or labels is None
    ):
        unusable_cached_results.append(
            {
                "candidate_id": candidate_id,
                "reason": "INVALID_RERANKING_RESULT",
            }
        )
        continue

    usable_cached_results.append(
        {
            "result": result,
            "labels": labels,
        }
    )


# ------------------------------------------------------------
# 6. Evaluate cached subset only
# ------------------------------------------------------------

metric_rows = []

for item in usable_cached_results:
    result = item["result"]
    labels = item["labels"]

    candidate_id = result["candidate_id"]

    query_index = query_index_by_candidate_id[
        candidate_id
    ]

    query_record = arvan_hybrid_query_records[
        query_index
    ]

    source_seed_id = query_record[
        "source_seed_id"
    ]

    assert (
        source_seed_id
        in best_positive_indexes_by_seed
    ), (
        "Missing positive indexes for seed: "
        + str(source_seed_id)
    )

    positive_indexes = {
        int(index)
        for index
        in best_positive_indexes_by_seed[
            source_seed_id
        ]
    }

    original_indexes = [
        int(index)
        for index
        in best_rrf_ranked_indexes[
            query_index
        ][:top_k]
    ]

    if len(original_indexes) != top_k:
        unusable_cached_results.append(
            {
                "candidate_id": candidate_id,
                "reason": "FEWER_THAN_TOP_K_RESULTS",
            }
        )
        continue

    original_labels = [
        "P" + str(rank).zfill(2)
        for rank in range(1, top_k + 1)
    ]

    index_by_label = dict(
        zip(
            original_labels,
            original_indexes,
        )
    )

    reranked_indexes = [
        index_by_label[label]
        for label in labels
    ]

    original_positive_ranks = [
        rank
        for rank, index in enumerate(
            original_indexes,
            start=1,
        )
        if index in positive_indexes
    ]

    reranked_positive_ranks = [
        rank
        for rank, index in enumerate(
            reranked_indexes,
            start=1,
        )
        if index in positive_indexes
    ]

    original_first_rank = (
        original_positive_ranks[0]
        if original_positive_ranks
        else None
    )

    reranked_first_rank = (
        reranked_positive_ranks[0]
        if reranked_positive_ranks
        else None
    )

    original_hit = (
        original_first_rank is not None
    )

    reranked_hit = (
        reranked_first_rank is not None
    )

    original_rr = (
        1.0 / original_first_rank
        if original_first_rank is not None
        else 0.0
    )

    reranked_rr = (
        1.0 / reranked_first_rank
        if reranked_first_rank is not None
        else 0.0
    )

    if (
        original_first_rank is not None
        and reranked_first_rank is not None
    ):
        rank_change = (
            original_first_rank
            - reranked_first_rank
        )
    else:
        rank_change = 0

    metric_rows.append(
        {
            "candidate_id": candidate_id,
            "source_seed_id": source_seed_id,
            "original_first_positive_rank": (
                original_first_rank
            ),
            "reranked_first_positive_rank": (
                reranked_first_rank
            ),
            "original_hit_at_10": original_hit,
            "reranked_hit_at_10": reranked_hit,
            "original_reciprocal_rank": original_rr,
            "reranked_reciprocal_rank": reranked_rr,
            "rank_change": rank_change,
        }
    )


# ------------------------------------------------------------
# 7. Counts
# ------------------------------------------------------------

evaluated_count = len(metric_rows)

not_evaluated_count = (
    total_query_count
    - evaluated_count
)

assert evaluated_count <= total_query_count
assert not_evaluated_count >= 0


# ------------------------------------------------------------
# 8. Aggregate subset metrics
# ------------------------------------------------------------

if evaluated_count > 0:
    original_recall_at_10 = float(
        np.mean(
            [
                row["original_hit_at_10"]
                for row in metric_rows
            ]
        )
    )

    reranked_recall_at_10 = float(
        np.mean(
            [
                row["reranked_hit_at_10"]
                for row in metric_rows
            ]
        )
    )

    original_mrr = float(
        np.mean(
            [
                row["original_reciprocal_rank"]
                for row in metric_rows
            ]
        )
    )

    reranked_mrr = float(
        np.mean(
            [
                row["reranked_reciprocal_rank"]
                for row in metric_rows
            ]
        )
    )

    mrr_delta = (
        reranked_mrr
        - original_mrr
    )

else:
    original_recall_at_10 = None
    reranked_recall_at_10 = None
    original_mrr = None
    reranked_mrr = None
    mrr_delta = None


improved_count = sum(
    1
    for row in metric_rows
    if row["rank_change"] > 0
)

unchanged_count = sum(
    1
    for row in metric_rows
    if row["rank_change"] == 0
)

worsened_count = sum(
    1
    for row in metric_rows
    if row["rank_change"] < 0
)


# ------------------------------------------------------------
# 9. Coverage status
# ------------------------------------------------------------

coverage_rate = (
    evaluated_count / total_query_count
    if total_query_count
    else 1.0
)

full_dataset_available = (
    evaluated_count == total_query_count
)

if full_dataset_available:
    evaluation_scope = "FULL_GOLDEN_V2"
else:
    evaluation_scope = "CACHED_SUBSET_ONLY"


# ------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------

cell83_report = {
    "active_golden_dataset_sha256": (
        golden_dataset_context["sha256"]
    ),
    "execution_mode": (
        "CACHE_ONLY_API_DISABLED"
    ),
    "evaluation_scope": evaluation_scope,
    "full_dataset_metric_available": (
        full_dataset_available
    ),
    "total_golden_query_count": (
        total_query_count
    ),
    "evaluated_cached_query_count": (
        evaluated_count
    ),
    "not_evaluated_query_count": (
        not_evaluated_count
    ),
    "coverage_rate": coverage_rate,
    "cached_subset_metrics": {
        "original_recall_at_10": (
            original_recall_at_10
        ),
        "reranked_recall_at_10": (
            reranked_recall_at_10
        ),
        "original_mrr": original_mrr,
        "reranked_mrr": reranked_mrr,
        "mrr_delta": mrr_delta,
        "improved_query_count": (
            improved_count
        ),
        "unchanged_query_count": (
            unchanged_count
        ),
        "worsened_query_count": (
            worsened_count
        ),
    },
    "unusable_cached_results": (
        unusable_cached_results
    ),
    "rows": metric_rows,
}


report_path.write_text(
    json.dumps(
        cell83_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 11. Output
# ------------------------------------------------------------

print(
    "=== RERANKING METRIC IMPACT — "
    "CACHE-ONLY ==="
)

print("API execution: DISABLED")

print(
    "Golden queries:",
    total_query_count,
)

print(
    "Evaluated cached queries:",
    evaluated_count,
)

print(
    "Not evaluated:",
    not_evaluated_count,
)

print(
    "Coverage:",
    format(coverage_rate, ".4f"),
)

print(
    "Evaluation scope:",
    evaluation_scope,
)

print(
    "Full-dataset metric available:",
    full_dataset_available,
)

if evaluated_count > 0:
    print(
        "Original subset Recall@10:",
        format(
            original_recall_at_10,
            ".4f",
        ),
    )

    print(
        "Reranked subset Recall@10:",
        format(
            reranked_recall_at_10,
            ".4f",
        ),
    )

    print(
        "Original subset MRR:",
        format(
            original_mrr,
            ".4f",
        ),
    )

    print(
        "Reranked subset MRR:",
        format(
            reranked_mrr,
            ".4f",
        ),
    )

    print(
        "MRR delta:",
        format(
            mrr_delta,
            ".4f",
        ),
    )

    print(
        "Improved / unchanged / worsened:",
        improved_count,
        "/",
        unchanged_count,
        "/",
        worsened_count,
    )

else:
    print(
        "Subset metrics: N/A"
    )

print(
    "Report:",
    report_path,
)

if not full_dataset_available:
    print(
        "NOTE: Metrics are for the cached subset only."
    )

print(
    "RERANKING CACHE-ONLY "
    "SUBSET METRIC EVALUATION: PASS"
)

=== RERANKING METRIC IMPACT — CACHE-ONLY ===
API execution: DISABLED
Golden queries: 3813
Evaluated cached queries: 15
Not evaluated: 3798
Coverage: 0.0039
Evaluation scope: CACHED_SUBSET_ONLY
Full-dataset metric available: False
Original subset Recall@10: 1.0000
Reranked subset Recall@10: 1.0000
Original subset MRR: 0.8089
Reranked subset MRR: 0.7429
MRR delta: -0.0660
Improved / unchanged / worsened: 2 / 10 / 3
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_full_reranking_metric_impact.json
NOTE: Metrics are for the cached subset only.
RERANKING CACHE-ONLY SUBSET METRIC EVALUATION: PASS


<div dir="rtl" align="right">

## اثر کامل Reranking بر کانتکست والد–فرزند

در این سل، برای همهٔ پرسش‌ها بررسی می‌شود که ترتیب جدیدِ Chunkها چه اثری بر ورود Parent حاوی شواهد درست به کانتکست نهایی دارد.

این ارزیابی دیگر وابسته به دانستن پاسخ درست در زمان اجرا نیست: Reranking برای همهٔ پرسش‌ها انجام شده و ترتیب خروجی آن برای همهٔ آن‌ها در کش ثبت شده است.

</div>

In [77]:
# Code Cell 61
# NB13 — Parent-level reranking impact on cached subset
# RUN-ALL SAFE: ZERO API CALLS.

import json
import numpy as np

# ------------------------------------------------------------
# 1. Required upstream state
# ------------------------------------------------------------

required_names = [
    "metric_rows",
    "arvan_hybrid_query_records",
    "best_rrf_ranked_indexes",
    "best_positive_indexes_by_seed",
    "best_candidate_records",
    "active_benchmark_run_directory",
    "golden_dataset_context",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, (
    "Missing required state: "
    + ", ".join(missing_names)
)


# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

top_k = 10

cell84_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_parent_level_reranking_impact.json"
)


# ------------------------------------------------------------
# 3. Build query lookup
# ------------------------------------------------------------

query_by_candidate_id = {}

query_index_by_candidate_id = {}

for query_index, query_record in enumerate(
    arvan_hybrid_query_records
):
    candidate_id = query_record["candidate_id"]

    assert candidate_id not in query_by_candidate_id, (
        "Duplicate candidate_id: "
        + str(candidate_id)
    )

    query_by_candidate_id[candidate_id] = query_record
    query_index_by_candidate_id[candidate_id] = query_index


# ------------------------------------------------------------
# 4. Build reranked child indexes directly from cached results
# ------------------------------------------------------------

assert "full_reranker_results" in globals(), (
    "full_reranker_results is unavailable."
)

allowed_labels = {
    "P" + str(rank).zfill(2)
    for rank in range(1, top_k + 1)
}


def cell84_get_labels(result):
    labels = result.get("reranked_labels")

    if isinstance(labels, list):
        if (
            len(labels) == top_k
            and set(labels) == allowed_labels
        ):
            return labels

    if "extract_reranked_labels" not in globals():
        return None

    for attempt in result.get("attempts", []):
        if not attempt.get("parsed"):
            continue

        parsed = extract_reranked_labels(
            attempt.get("raw_response", ""),
            allowed_labels=allowed_labels,
        )

        if parsed is not None:
            return parsed

    return None


full_reranked_child_indexes_by_candidate_id = {}

for result in full_reranker_results:
    candidate_id = result.get("candidate_id")

    if candidate_id not in query_index_by_candidate_id:
        continue

    labels = cell84_get_labels(result)

    if labels is None:
        continue

    query_index = query_index_by_candidate_id[
        candidate_id
    ]

    baseline_indexes = [
        int(index)
        for index in best_rrf_ranked_indexes[
            query_index
        ][:top_k]
    ]

    if len(baseline_indexes) != top_k:
        continue

    label_to_index = {}

    for rank, child_index in enumerate(
        baseline_indexes,
        start=1,
    ):
        label = "P" + str(rank).zfill(2)
        label_to_index[label] = child_index

    reranked_indexes = [
        label_to_index[label]
        for label in labels
    ]

    full_reranked_child_indexes_by_candidate_id[
        candidate_id
    ] = reranked_indexes


# ------------------------------------------------------------
# 5. Parent-id helper
# ------------------------------------------------------------

def cell84_parent_id(child_index):
    record = best_candidate_records[
        int(child_index)
    ]

    parent_id = record.get("parent_chunk_id")

    if parent_id is None:
        parent_id = record.get("chunk_id")

    if parent_id is None:
        parent_id = record.get("candidate_id")

    if parent_id is None:
        parent_id = str(child_index)

    return str(parent_id)


# ------------------------------------------------------------
# 6. Evaluate parent-level ranks on cached subset
# ------------------------------------------------------------

parent_metric_rows = []

skipped_parent_rows = []

for metric_row in metric_rows:
    candidate_id = metric_row["candidate_id"]

    if (
        candidate_id
        not in full_reranked_child_indexes_by_candidate_id
    ):
        skipped_parent_rows.append(
            {
                "candidate_id": candidate_id,
                "reason": "NO_VALID_CACHED_RERANKING",
            }
        )
        continue

    query_record = query_by_candidate_id[
        candidate_id
    ]

    query_index = query_index_by_candidate_id[
        candidate_id
    ]

    seed_id = query_record["source_seed_id"]

    if seed_id not in best_positive_indexes_by_seed:
        skipped_parent_rows.append(
            {
                "candidate_id": candidate_id,
                "reason": "NO_POSITIVE_INDEX_REGISTRY",
            }
        )
        continue

    positive_child_indexes = [
        int(index)
        for index
        in best_positive_indexes_by_seed[
            seed_id
        ]
    ]

    positive_parent_ids = {
        cell84_parent_id(child_index)
        for child_index in positive_child_indexes
    }

    baseline_child_indexes = [
        int(index)
        for index
        in best_rrf_ranked_indexes[
            query_index
        ][:top_k]
    ]

    reranked_child_indexes = [
        int(index)
        for index
        in full_reranked_child_indexes_by_candidate_id[
            candidate_id
        ]
    ]

    baseline_parent_ids = [
        cell84_parent_id(child_index)
        for child_index in baseline_child_indexes
    ]

    reranked_parent_ids = [
        cell84_parent_id(child_index)
        for child_index in reranked_child_indexes
    ]


    # --------------------------------------------------------
    # First occurrence of each parent only.
    # --------------------------------------------------------

    baseline_unique_parent_ids = []

    for parent_id in baseline_parent_ids:
        if parent_id not in baseline_unique_parent_ids:
            baseline_unique_parent_ids.append(
                parent_id
            )


    reranked_unique_parent_ids = []

    for parent_id in reranked_parent_ids:
        if parent_id not in reranked_unique_parent_ids:
            reranked_unique_parent_ids.append(
                parent_id
            )


    # --------------------------------------------------------
    # Positive parent rank
    # --------------------------------------------------------

    baseline_positive_parent_ranks = [
        rank
        for rank, parent_id in enumerate(
            baseline_unique_parent_ids,
            start=1,
        )
        if parent_id in positive_parent_ids
    ]

    reranked_positive_parent_ranks = [
        rank
        for rank, parent_id in enumerate(
            reranked_unique_parent_ids,
            start=1,
        )
        if parent_id in positive_parent_ids
    ]

    baseline_first_parent_rank = (
        baseline_positive_parent_ranks[0]
        if baseline_positive_parent_ranks
        else None
    )

    reranked_first_parent_rank = (
        reranked_positive_parent_ranks[0]
        if reranked_positive_parent_ranks
        else None
    )


    # --------------------------------------------------------
    # Reciprocal rank
    # --------------------------------------------------------

    baseline_parent_rr = (
        1.0 / baseline_first_parent_rank
        if baseline_first_parent_rank is not None
        else 0.0
    )

    reranked_parent_rr = (
        1.0 / reranked_first_parent_rank
        if reranked_first_parent_rank is not None
        else 0.0
    )


    # --------------------------------------------------------
    # Rank change
    # --------------------------------------------------------

    if (
        baseline_first_parent_rank is not None
        and reranked_first_parent_rank is not None
    ):
        parent_rank_change = (
            baseline_first_parent_rank
            - reranked_first_parent_rank
        )
    else:
        parent_rank_change = 0


    parent_metric_rows.append(
        {
            "candidate_id": candidate_id,
            "source_seed_id": seed_id,
            "positive_parent_count": len(
                positive_parent_ids
            ),
            "baseline_unique_parent_count": len(
                baseline_unique_parent_ids
            ),
            "reranked_unique_parent_count": len(
                reranked_unique_parent_ids
            ),
            "baseline_first_positive_parent_rank": (
                baseline_first_parent_rank
            ),
            "reranked_first_positive_parent_rank": (
                reranked_first_parent_rank
            ),
            "baseline_parent_hit_at_10": (
                baseline_first_parent_rank is not None
            ),
            "reranked_parent_hit_at_10": (
                reranked_first_parent_rank is not None
            ),
            "baseline_parent_reciprocal_rank": (
                baseline_parent_rr
            ),
            "reranked_parent_reciprocal_rank": (
                reranked_parent_rr
            ),
            "parent_rank_change": (
                parent_rank_change
            ),
        }
    )


# ------------------------------------------------------------
# 7. Aggregate metrics
# ------------------------------------------------------------

evaluated_parent_count = len(
    parent_metric_rows
)

total_golden_query_count = len(
    arvan_hybrid_query_records
)

not_parent_evaluated_count = (
    total_golden_query_count
    - evaluated_parent_count
)

assert evaluated_parent_count <= (
    total_golden_query_count
)

assert not_parent_evaluated_count >= 0


if evaluated_parent_count > 0:
    baseline_parent_recall_at_10 = float(
        np.mean(
            [
                row["baseline_parent_hit_at_10"]
                for row in parent_metric_rows
            ]
        )
    )

    reranked_parent_recall_at_10 = float(
        np.mean(
            [
                row["reranked_parent_hit_at_10"]
                for row in parent_metric_rows
            ]
        )
    )

    baseline_parent_mrr = float(
        np.mean(
            [
                row[
                    "baseline_parent_reciprocal_rank"
                ]
                for row in parent_metric_rows
            ]
        )
    )

    reranked_parent_mrr = float(
        np.mean(
            [
                row[
                    "reranked_parent_reciprocal_rank"
                ]
                for row in parent_metric_rows
            ]
        )
    )

    parent_mrr_delta = (
        reranked_parent_mrr
        - baseline_parent_mrr
    )

else:
    baseline_parent_recall_at_10 = None
    reranked_parent_recall_at_10 = None
    baseline_parent_mrr = None
    reranked_parent_mrr = None
    parent_mrr_delta = None


improved_parent_count = sum(
    1
    for row in parent_metric_rows
    if row["parent_rank_change"] > 0
)

unchanged_parent_count = sum(
    1
    for row in parent_metric_rows
    if row["parent_rank_change"] == 0
)

worsened_parent_count = sum(
    1
    for row in parent_metric_rows
    if row["parent_rank_change"] < 0
)


# ------------------------------------------------------------
# 8. Coverage
# ------------------------------------------------------------

parent_coverage_rate = (
    evaluated_parent_count
    / total_golden_query_count
    if total_golden_query_count
    else 1.0
)

parent_full_dataset_available = (
    evaluated_parent_count
    == total_golden_query_count
)

if parent_full_dataset_available:
    parent_evaluation_scope = (
        "FULL_GOLDEN_V2"
    )
else:
    parent_evaluation_scope = (
        "CACHED_SUBSET_ONLY"
    )


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

cell84_report = {
    "active_golden_dataset_sha256": (
        golden_dataset_context["sha256"]
    ),
    "execution_mode": (
        "CACHE_ONLY_API_DISABLED"
    ),
    "evaluation_scope": (
        parent_evaluation_scope
    ),
    "full_dataset_metric_available": (
        parent_full_dataset_available
    ),
    "total_golden_query_count": (
        total_golden_query_count
    ),
    "evaluated_cached_query_count": (
        evaluated_parent_count
    ),
    "not_evaluated_query_count": (
        not_parent_evaluated_count
    ),
    "coverage_rate": (
        parent_coverage_rate
    ),
    "cached_subset_metrics": {
        "baseline_parent_recall_at_10": (
            baseline_parent_recall_at_10
        ),
        "reranked_parent_recall_at_10": (
            reranked_parent_recall_at_10
        ),
        "baseline_parent_mrr": (
            baseline_parent_mrr
        ),
        "reranked_parent_mrr": (
            reranked_parent_mrr
        ),
        "parent_mrr_delta": (
            parent_mrr_delta
        ),
        "improved_query_count": (
            improved_parent_count
        ),
        "unchanged_query_count": (
            unchanged_parent_count
        ),
        "worsened_query_count": (
            worsened_parent_count
        ),
    },
    "skipped_rows": skipped_parent_rows,
    "rows": parent_metric_rows,
}


cell84_report_path.write_text(
    json.dumps(
        cell84_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 10. Output
# ------------------------------------------------------------

print(
    "=== PARENT-LEVEL RERANKING IMPACT "
    "— CACHE ONLY ==="
)

print(
    "API execution:",
    "DISABLED",
)

print(
    "Golden queries:",
    total_golden_query_count,
)

print(
    "Evaluated cached queries:",
    evaluated_parent_count,
)

print(
    "Not evaluated:",
    not_parent_evaluated_count,
)

print(
    "Coverage:",
    format(
        parent_coverage_rate,
        ".4f",
    ),
)

print(
    "Evaluation scope:",
    parent_evaluation_scope,
)

print(
    "Full-dataset metric available:",
    parent_full_dataset_available,
)

if evaluated_parent_count > 0:
    print(
        "Baseline parent Recall@10:",
        format(
            baseline_parent_recall_at_10,
            ".4f",
        ),
    )

    print(
        "Reranked parent Recall@10:",
        format(
            reranked_parent_recall_at_10,
            ".4f",
        ),
    )

    print(
        "Baseline parent MRR:",
        format(
            baseline_parent_mrr,
            ".4f",
        ),
    )

    print(
        "Reranked parent MRR:",
        format(
            reranked_parent_mrr,
            ".4f",
        ),
    )

    print(
        "Parent MRR delta:",
        format(
            parent_mrr_delta,
            ".4f",
        ),
    )

    print(
        "Improved / unchanged / worsened:",
        improved_parent_count,
        "/",
        unchanged_parent_count,
        "/",
        worsened_parent_count,
    )

else:
    print(
        "Parent-level subset metrics:",
        "N/A",
    )

print(
    "Report:",
    cell84_report_path,
)

if not parent_full_dataset_available:
    print(
        "NOTE: Parent-level metrics describe "
        "only the cached reranking subset."
    )

print(
    "PARENT-LEVEL RERANKING "
    "CACHE-ONLY EVALUATION: PASS"
)

=== PARENT-LEVEL RERANKING IMPACT — CACHE ONLY ===
API execution: DISABLED
Golden queries: 3813
Evaluated cached queries: 15
Not evaluated: 3798
Coverage: 0.0039
Evaluation scope: CACHED_SUBSET_ONLY
Full-dataset metric available: False
Baseline parent Recall@10: 1.0000
Reranked parent Recall@10: 1.0000
Baseline parent MRR: 0.8089
Reranked parent MRR: 0.7429
Parent MRR delta: -0.0660
Improved / unchanged / worsened: 2 / 10 / 3
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_chat_parent_level_reranking_impact.json
NOTE: Parent-level metrics describe only the cached reranking subset.
PARENT-LEVEL RERANKING CACHE-ONLY EVALUATION: PASS


<div dir="rtl" align="right">

## پایلوت بازنویسی پرسش برای بازیابی

در این آزمایش، مدل فقط خودِ پرسش را به یک عبارت جست‌وجوی فارسی دقیق‌تر تبدیل می‌کند. مدل به منبع، شواهد Golden Dataset و پاسخ مورد انتظار دسترسی ندارد و اجازه ندارد واقعیت، نام، عدد یا مفهوم تازه‌ای به پرسش اضافه کند.

هدف این است که ببینیم حذف عبارت‌های کلی و تبدیل پرسش به زبان مناسب بازیابی، می‌تواند شواهد درست را بیشتر وارد نتایج اولیه کند یا نه. برای پوشش متوازن، از هر Seed قابل‌ارزیابی یک پرسش به‌صورت قطعی انتخاب می‌شود.

</div>

In [78]:
# Code Cell 62
# NB13 — Query-rewrite pilot reconstruction (API disabled)

import json
import re

query_rewrite_pilot_cache_path = (
    active_benchmark_run_directory / "arvan_chat_query_rewrite_pilot_cache.json"
)
query_rewrite_pilot_report_path = (
    active_benchmark_run_directory / "arvan_chat_query_rewrite_pilot.json"
)
query_rewrite_pilot_size = min(27, len(arvan_hybrid_query_records))
query_rewrite_pilot_records = sorted(
    arvan_hybrid_query_records,
    key=lambda record: record["candidate_id"],
)[:query_rewrite_pilot_size]

def extract_search_query(raw_response_text):
    raw_response_text = (raw_response_text or "").strip()
    for candidate_text in [raw_response_text] + re.findall(r"\{.*?\}", raw_response_text, flags=re.S):
        try:
            parsed = json.loads(candidate_text)
        except json.JSONDecodeError:
            continue
        search_query = parsed.get("search_query")
        if isinstance(search_query, str) and search_query.strip():
            return search_query.strip()
    return None

assert query_rewrite_pilot_cache_path.is_file(), (
    "Query-rewrite cache is required for Run-All-safe reconstruction."
)
query_rewrite_pilot_cache = json.loads(
    query_rewrite_pilot_cache_path.read_text(encoding="utf-8")
)
assert query_rewrite_pilot_cache.get("active_golden_dataset_sha256") == golden_dataset_context["sha256"]
assert query_rewrite_pilot_cache.get("model") == arvan_chat_model

results_by_candidate_id = query_rewrite_pilot_cache.get("results_by_candidate_id", {})
query_rewrite_pilot_results = []
for record in query_rewrite_pilot_records:
    candidate_id = record["candidate_id"]
    assert candidate_id in results_by_candidate_id, (
        "Missing cached query rewrite for candidate_id=" + str(candidate_id)
    )
    query_rewrite_pilot_results.append(results_by_candidate_id[candidate_id])

query_rewrite_pilot_successes = [
    result for result in query_rewrite_pilot_results
    if result.get("response_parse_status") == "PASS"
    and isinstance(result.get("rewritten_query"), str)
    and result.get("rewritten_query", "").strip()
]
assert len(query_rewrite_pilot_successes) == len(query_rewrite_pilot_records), (
    "Cached query-rewrite pilot is incomplete. API remains disabled."
)

query_rewrite_pilot_total_cost = sum(
    float(attempt["cost"])
    for result in query_rewrite_pilot_results
    for group in (result.get("attempts", []), result.get("repair_attempts", []))
    for attempt in group
    if attempt.get("cost") is not None
)
query_rewrite_pilot_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "model": arvan_chat_model,
    "eligible_seed_count": len(common_eligible_seed_ids),
    "pilot_query_count": len(query_rewrite_pilot_results),
    "parse_success_count": len(query_rewrite_pilot_successes),
    "parse_success_rate": len(query_rewrite_pilot_successes) / len(query_rewrite_pilot_results),
    "total_observed_cost": query_rewrite_pilot_total_cost,
    "execution_mode": "CACHE_ONLY_API_DISABLED",
    "results": query_rewrite_pilot_results,
}
query_rewrite_pilot_report_path.write_text(
    json.dumps(query_rewrite_pilot_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("=== QUERY-REWRITE PILOT — CACHE ONLY ===")
print("Pilot queries:", len(query_rewrite_pilot_results))
print("Parse success:", len(query_rewrite_pilot_successes))
print("API execution: DISABLED")
print("QUERY-REWRITE PILOT RECONSTRUCTION: PASS")


=== QUERY-REWRITE PILOT — CACHE ONLY ===
Pilot queries: 27
Parse success: 27
API execution: DISABLED
QUERY-REWRITE PILOT RECONSTRUCTION: PASS


In [79]:
# Code Cell 63
# NB13 — Query-rewrite failed-response inspection (API disabled)

failed_query_rewrite_results = [
    result for result in query_rewrite_pilot_results
    if result.get("response_parse_status") != "PASS"
]
print("=== QUERY-REWRITE FAILED-RESPONSE INSPECTION ===")
print("Failed cached responses:", len(failed_query_rewrite_results))
print("API execution: DISABLED")
assert not failed_query_rewrite_results, (
    "Cached query-rewrite results contain failures; repair API is intentionally disabled."
)
print("QUERY-REWRITE FAILED-RESPONSE INSPECTION: PASS")


=== QUERY-REWRITE FAILED-RESPONSE INSPECTION ===
Failed cached responses: 0
API execution: DISABLED
QUERY-REWRITE FAILED-RESPONSE INSPECTION: PASS


<div dir="rtl" align="right">

## ارزیابی بازیابی با پرسش‌های بازنویسی‌شده

در این سل، ۲۷ پرسش بازنویسی‌شده با همان مدل embedding اروان و همان TF‑IDF فارسیِ منتخب ارزیابی می‌شوند. بنابراین تنها متغیرِ تغییرکرده، خود عبارت جست‌وجو است.

Embedding این ۲۷ عبارت نیز کش می‌شود؛ تغییر نام Notebook یا Restart باعث تکرار آن نمی‌شود.

</div>

In [80]:
# Code Cell 64
# NB13 — Retrieval evaluation for the cached query-rewrite pilot
# Cache-only: no Arvan embedding request is permitted during Run-All.

query_rewrite_embedding_cache_path = (
    active_benchmark_run_directory
    / "arvan_query_rewrite_pilot_embeddings.npz"
)

query_rewrite_embedding_manifest_path = (
    active_benchmark_run_directory
    / "arvan_query_rewrite_pilot_embeddings_manifest.json"
)

query_rewrite_pilot_result_by_candidate_id = {
    result["candidate_id"]: result
    for result in query_rewrite_pilot_results
}

pilot_candidate_ids = [
    record["candidate_id"]
    for record in query_rewrite_pilot_records
]

pilot_rewritten_queries = [
    query_rewrite_pilot_result_by_candidate_id[candidate_id][
        "rewritten_query"
    ]
    for candidate_id in pilot_candidate_ids
]

assert len(pilot_candidate_ids) == len(query_rewrite_pilot_records)
assert all(
    isinstance(query, str) and query.strip()
    for query in pilot_rewritten_queries
)

pilot_rewrite_input_sha256 = hashlib.sha256(
    "\n".join(
        f"{candidate_id}\t{query}"
        for candidate_id, query in zip(
            pilot_candidate_ids,
            pilot_rewritten_queries,
        )
    ).encode("utf-8")
).hexdigest()

pilot_embedding_cache_reused = False

if (
    query_rewrite_embedding_cache_path.is_file()
    and query_rewrite_embedding_manifest_path.is_file()
):
    pilot_embedding_manifest = json.loads(
        query_rewrite_embedding_manifest_path.read_text(
            encoding="utf-8"
        )
    )

    cached_arrays = np.load(
        query_rewrite_embedding_cache_path,
        allow_pickle=True,
    )

    cached_candidate_ids = [
        str(candidate_id)
        for candidate_id in cached_arrays["candidate_ids"].tolist()
    ]

    pilot_embedding_cache_reused = (
        pilot_embedding_manifest.get("golden_dataset_sha256")
        == golden_dataset_context["sha256"]
        and pilot_embedding_manifest.get("input_sha256")
        == pilot_rewrite_input_sha256
        and cached_candidate_ids == pilot_candidate_ids
        and cached_arrays["vectors"].shape
        == (len(pilot_candidate_ids), 3072)
    )

if pilot_embedding_cache_reused:
    rewritten_query_vectors = cached_arrays["vectors"].astype(
        np.float32
    )
    print("Existing rewritten-query embedding cache loaded.")
else:
    raise RuntimeError(
        "Valid rewritten-query embedding cache is required. "
        "API execution is disabled for Run-All safety."
    )

assert rewritten_query_vectors.shape == (len(pilot_candidate_ids), 3072)

pilot_lexical_query_matrix = best_lexical_vectorizer.transform(
    pilot_rewritten_queries
)

pilot_dense_scores = (
    rewritten_query_vectors @ best_arvan_candidate_matrix.T
)
pilot_lexical_scores = (
    pilot_lexical_query_matrix @ best_lexical_candidate_matrix.T
).toarray()

pilot_dense_ranked_indexes = np.argsort(
    -pilot_dense_scores,
    axis=1,
)
pilot_lexical_ranked_indexes = np.argsort(
    -pilot_lexical_scores,
    axis=1,
)

pilot_dense_rank_positions = np.empty_like(
    pilot_dense_ranked_indexes
)
pilot_lexical_rank_positions = np.empty_like(
    pilot_lexical_ranked_indexes
)

pilot_row_indexes = np.arange(len(query_rewrite_pilot_records))[:, None]
pilot_candidate_rank_positions = (
    np.arange(len(best_candidate_records))[None, :] + 1
)

pilot_dense_rank_positions[
    pilot_row_indexes,
    pilot_dense_ranked_indexes,
] = pilot_candidate_rank_positions

pilot_lexical_rank_positions[
    pilot_row_indexes,
    pilot_lexical_ranked_indexes,
] = pilot_candidate_rank_positions

pilot_rrf_scores = (
    1.0 / (rrf_constant + pilot_dense_rank_positions)
    + 1.0 / (rrf_constant + pilot_lexical_rank_positions)
)

pilot_rewritten_ranked_indexes = np.argsort(
    -pilot_rrf_scores,
    axis=1,
)

pilot_baseline_ranked_indexes = np.vstack(
    [
        best_rrf_ranked_indexes[
            next(
                index
                for index, record in enumerate(
                    arvan_hybrid_query_records
                )
                if record["candidate_id"] == pilot_record["candidate_id"]
            )
        ]
        for pilot_record in query_rewrite_pilot_records
    ]
)

pilot_baseline_metrics = calculate_retrieval_metrics(
    pilot_baseline_ranked_indexes,
    query_rewrite_pilot_records,
    best_positive_indexes_by_seed,
)

pilot_rewritten_metrics = calculate_retrieval_metrics(
    pilot_rewritten_ranked_indexes,
    query_rewrite_pilot_records,
    best_positive_indexes_by_seed,
)

query_rewrite_retrieval_pilot_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "candidate": best_candidate_name,
    "pilot_query_count": len(query_rewrite_pilot_records),
    "embedding_cache_reused": pilot_embedding_cache_reused,
    "baseline_metrics": pilot_baseline_metrics,
    "rewritten_query_metrics": pilot_rewritten_metrics,
    "metric_deltas": {
        metric_name: (
            pilot_rewritten_metrics[metric_name]
            - pilot_baseline_metrics[metric_name]
        )
        for metric_name in (
            "recall_at_1",
            "recall_at_3",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "mean_first_positive_rank",
        )
    },
    "queries": [
        {
            "candidate_id": record["candidate_id"],
            "source_seed_id": record["source_seed_id"],
            "original_question": record["question"],
            "rewritten_query": query_rewrite_pilot_result_by_candidate_id[
                record["candidate_id"]
            ]["rewritten_query"],
        }
        for record in query_rewrite_pilot_records
    ],
}

query_rewrite_retrieval_pilot_report_path = (
    active_benchmark_run_directory
    / "arvan_query_rewrite_retrieval_pilot.json"
)

query_rewrite_retrieval_pilot_report_path.write_text(
    json.dumps(
        query_rewrite_retrieval_pilot_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== QUERY-REWRITE RETRIEVAL PILOT ===")
print("Pilot queries:", len(query_rewrite_pilot_records))
print("Embedding cache reused:", pilot_embedding_cache_reused)

for metric_name in (
    "recall_at_1",
    "recall_at_3",
    "recall_at_5",
    "recall_at_10",
    "mrr",
    "mean_first_positive_rank",
):
    print(
        f"\n{metric_name}"
        f"\n- baseline: {pilot_baseline_metrics[metric_name]:.4f}"
        f"\n- rewritten: {pilot_rewritten_metrics[metric_name]:.4f}"
        f"\n- delta: "
        f"{pilot_rewritten_metrics[metric_name] - pilot_baseline_metrics[metric_name]:+.4f}"
    )

print("\nEmbedding cache:", query_rewrite_embedding_cache_path)
print("Report:", query_rewrite_retrieval_pilot_report_path)
print("\nQUERY-REWRITE RETRIEVAL PILOT: PASS")

Existing rewritten-query embedding cache loaded.
=== QUERY-REWRITE RETRIEVAL PILOT ===
Pilot queries: 27
Embedding cache reused: True

recall_at_1
- baseline: 0.4074
- rewritten: 0.3704
- delta: -0.0370

recall_at_3
- baseline: 0.7037
- rewritten: 0.5926
- delta: -0.1111

recall_at_5
- baseline: 0.7778
- rewritten: 0.7407
- delta: -0.0370

recall_at_10
- baseline: 0.8148
- rewritten: 0.8148
- delta: +0.0000

mrr
- baseline: 0.5694
- rewritten: 0.5218
- delta: -0.0476

mean_first_positive_rank
- baseline: 10.2963
- rewritten: 10.0000
- delta: -0.2963

Embedding cache: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_query_rewrite_pilot_embeddings.npz
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_query_rewrite_retri

<div dir="rtl" align="right">

## ترکیب پرسش اصلی و پرسش بازنویسی‌شده

نتیجهٔ قبلی نشان داد جایگزین‌کردن کامل پرسش اصلی با نسخهٔ بازنویسی‌شده تصمیم خوبی نیست؛ چون ممکن است بخشی از اطلاعات مفید پرسش اصلی از بین برود.

در این آزمایش، هر دو پرسش حفظ می‌شوند. رتبه‌بندی حاصل از پرسش اصلی و پرسش بازنویسی‌شده با چند وزن مختلف ترکیب می‌شود تا مشخص شود آیا بازنویسی، در کنار پرسش اصلی، به بازیابی کمک می‌کند یا نه. این سل فقط از embeddingها و خروجی‌های موجود استفاده می‌کند و هیچ فراخوانی جدیدی ندارد.

</div>

In [81]:
# Code Cell 65
# NB13 — Query-rewrite augmentation pilot:
# original query + rewritten query, fused without new API requests.

pilot_query_indexes_in_full_benchmark = np.array(
    [
        next(
            index
            for index, record in enumerate(arvan_hybrid_query_records)
            if record["candidate_id"] == pilot_record["candidate_id"]
        )
        for pilot_record in query_rewrite_pilot_records
    ],
    dtype=int,
)

assert len(pilot_query_indexes_in_full_benchmark) == len(query_rewrite_pilot_records)
assert len(set(pilot_query_indexes_in_full_benchmark)) == len(query_rewrite_pilot_records)

pilot_original_questions = [
    record["question"]
    for record in query_rewrite_pilot_records
]

pilot_original_dense_scores = (
    best_arvan_query_matrix[
        pilot_query_indexes_in_full_benchmark
    ]
    @ best_arvan_candidate_matrix.T
)

pilot_original_lexical_query_matrix = best_lexical_vectorizer.transform(
    pilot_original_questions
)

pilot_original_lexical_scores = (
    pilot_original_lexical_query_matrix
    @ best_lexical_candidate_matrix.T
).toarray()

def rank_positions_from_scores(scores):
    ranked_indexes = np.argsort(-scores, axis=1)
    rank_positions = np.empty_like(ranked_indexes)

    row_indexes = np.arange(scores.shape[0])[:, None]
    candidate_rank_positions = (
        np.arange(scores.shape[1])[None, :] + 1
    )

    rank_positions[
        row_indexes,
        ranked_indexes,
    ] = candidate_rank_positions

    return ranked_indexes, rank_positions

(
    pilot_original_dense_ranked_indexes,
    pilot_original_dense_rank_positions,
) = rank_positions_from_scores(pilot_original_dense_scores)

(
    pilot_original_lexical_ranked_indexes,
    pilot_original_lexical_rank_positions,
) = rank_positions_from_scores(pilot_original_lexical_scores)

# These scores were created in Code Cell 87 from rewritten questions.
(
    pilot_rewritten_dense_ranked_indexes,
    pilot_rewritten_dense_rank_positions,
) = rank_positions_from_scores(pilot_dense_scores)

(
    pilot_rewritten_lexical_ranked_indexes,
    pilot_rewritten_lexical_rank_positions,
) = rank_positions_from_scores(pilot_lexical_scores)

rewrite_weight_candidates = (0.0, 0.25, 0.50, 0.75, 1.0)
query_augmentation_results = {}

print("=== QUERY-REWRITE AUGMENTATION PILOT ===")
print("Pilot queries:", len(query_rewrite_pilot_records))
print("Fusion: original dense + original lexical + weighted rewrite")

for rewrite_weight in rewrite_weight_candidates:
    fused_rrf_scores = (
        1.0 / (rrf_constant + pilot_original_dense_rank_positions)
        + 1.0 / (rrf_constant + pilot_original_lexical_rank_positions)
        + rewrite_weight
        * (
            1.0
            / (
                rrf_constant
                + pilot_rewritten_dense_rank_positions
            )
        )
        + rewrite_weight
        * (
            1.0
            / (
                rrf_constant
                + pilot_rewritten_lexical_rank_positions
            )
        )
    )

    fused_ranked_indexes = np.argsort(
        -fused_rrf_scores,
        axis=1,
    )

    metrics = calculate_retrieval_metrics(
        fused_ranked_indexes,
        query_rewrite_pilot_records,
        best_positive_indexes_by_seed,
    )

    query_augmentation_results[str(rewrite_weight)] = {
        "rewrite_weight": rewrite_weight,
        **metrics,
    }

    print(f"\nRewrite weight: {rewrite_weight:.2f}")
    print(f"- Recall@1: {metrics['recall_at_1']:.4f}")
    print(f"- Recall@3: {metrics['recall_at_3']:.4f}")
    print(f"- Recall@5: {metrics['recall_at_5']:.4f}")
    print(f"- Recall@10: {metrics['recall_at_10']:.4f}")
    print(f"- MRR: {metrics['mrr']:.4f}")
    print(
        "- Mean first-positive rank: "
        f"{metrics['mean_first_positive_rank']:.2f}"
    )

best_query_augmentation_result = sorted(
    query_augmentation_results.values(),
    key=lambda result: (
        -result["recall_at_10"],
        -result["mrr"],
        result["mean_first_positive_rank"],
    ),
)[0]

query_augmentation_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "pilot_query_count": len(query_rewrite_pilot_records),
    "baseline_original_only": query_augmentation_results["0.0"],
    "results_by_rewrite_weight": query_augmentation_results,
    "best_result": best_query_augmentation_result,
    "interpretation": (
        "This is a small pilot only. A full evaluation is justified "
        "only if augmentation improves the selected pilot metric."
    ),
}

query_augmentation_report_path = (
    active_benchmark_run_directory
    / "arvan_query_rewrite_augmentation_pilot.json"
)

query_augmentation_report_path.write_text(
    json.dumps(
        query_augmentation_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nBest pilot configuration:")
print(
    f"- rewrite_weight="
    f"{best_query_augmentation_result['rewrite_weight']:.2f}"
)
print(
    f"- Recall@10="
    f"{best_query_augmentation_result['recall_at_10']:.4f}"
)
print(f"- MRR={best_query_augmentation_result['mrr']:.4f}")

print("\nReport:", query_augmentation_report_path)
print("\nQUERY-REWRITE AUGMENTATION PILOT: PASS")

=== QUERY-REWRITE AUGMENTATION PILOT ===
Pilot queries: 27
Fusion: original dense + original lexical + weighted rewrite

Rewrite weight: 0.00
- Recall@1: 0.4074
- Recall@3: 0.7037
- Recall@5: 0.7778
- Recall@10: 0.8148
- MRR: 0.5694
- Mean first-positive rank: 10.30

Rewrite weight: 0.25
- Recall@1: 0.4815
- Recall@3: 0.7037
- Recall@5: 0.7778
- Recall@10: 0.8148
- MRR: 0.6126
- Mean first-positive rank: 9.59

Rewrite weight: 0.50
- Recall@1: 0.4815
- Recall@3: 0.7037
- Recall@5: 0.7778
- Recall@10: 0.8148
- MRR: 0.5946
- Mean first-positive rank: 9.56

Rewrite weight: 0.75
- Recall@1: 0.4815
- Recall@3: 0.7037
- Recall@5: 0.7778
- Recall@10: 0.8148
- MRR: 0.5950
- Mean first-positive rank: 9.41

Rewrite weight: 1.00
- Recall@1: 0.4444
- Recall@3: 0.7037
- Recall@5: 0.7778
- Recall@10: 0.8148
- MRR: 0.5765
- Mean first-positive rank: 9.26

Best pilot configuration:
- rewrite_weight=0.25
- Recall@10=0.8148
- MRR=0.6126

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projec

<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>تصمیم نهایی ارزیابی بازیابی با Golden Dataset v2</h2>
<p>Benchmark بازیابی روی Golden Dataset v2 فعال اجرا شده و مسیر بازتولید آن به کش‌های ثبت‌شده متصل است. ارزیابی Reranking کامل روی همهٔ پرسش‌ها انجام نشده است؛ فقط نتایج موجود در کش معتبر تحلیل می‌شوند و این subset هرگز به‌عنوان نتیجهٔ کل Dataset گزارش نمی‌شود.</p>
<p>در اجرای نهایی Notebook هیچ فراخوانی پولی API مجاز نیست. embeddingها، rerankingهای موجود و پایلوت بازنویسی پرسش فقط از cache بازسازی می‌شوند. بنابراین نتیجهٔ فعلی برای ادامهٔ مهندسی و مقایسهٔ retrieval معتبر است، اما به معنی تأیید Production یا تکمیل علمی Reranking روی تمام 3813 پرسش نیست.</p>
</div>


In [82]:
# Code Cell 66
# NB13 — Formal retrieval-remediation decision for active Golden v2
# RUN-ALL SAFE: ZERO EXTERNAL / API CALLS.

import json

# ------------------------------------------------------------
# 1. Required state
# ------------------------------------------------------------

required_names = [
    "golden_dataset_context",
    "active_benchmark_run_directory",
    "arvan_hybrid_query_records",
    "best_hybrid_metrics",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, (
    "Missing required state: "
    + ", ".join(missing_names)
)


# ------------------------------------------------------------
# 2. Active Golden-v2 size
# ------------------------------------------------------------

total_query_count = len(
    arvan_hybrid_query_records
)

assert total_query_count > 0, (
    "Active benchmark query set is empty."
)


# ------------------------------------------------------------
# 3. Baseline retrieval metrics
# ------------------------------------------------------------

assert isinstance(
    best_hybrid_metrics,
    dict,
), "best_hybrid_metrics must be a dictionary."

baseline_recall_at_10 = (
    best_hybrid_metrics.get("recall_at_10")
)

baseline_mrr = (
    best_hybrid_metrics.get("mrr")
)

if baseline_mrr is None:
    baseline_mrr = (
        best_hybrid_metrics.get("mrr_at_10")
    )


# ------------------------------------------------------------
# 4. Reranking coverage
# ------------------------------------------------------------

if "full_reranker_results" in globals():
    cached_reranking_count = len(
        full_reranker_results
    )
else:
    cached_reranking_count = 0


if "full_reranker_parse_success_count" in globals():
    successful_reranking_count = int(
        full_reranker_parse_success_count
    )
else:
    successful_reranking_count = 0

    if "full_reranker_results" in globals():
        for result in full_reranker_results:
            if (
                result.get("response_parse_status")
                == "PASS"
            ):
                successful_reranking_count += 1


missing_reranking_count = (
    total_query_count
    - cached_reranking_count
)

assert missing_reranking_count >= 0, (
    "Cached reranking count exceeds "
    "the active Golden query count."
)

reranking_coverage_rate = (
    cached_reranking_count
    / total_query_count
)


# ------------------------------------------------------------
# 5. Cached-subset reranking metrics
# ------------------------------------------------------------

cached_subset_metrics = None
cached_subset_metric_available = False

if "cell83_report" in globals():
    candidate_subset_metrics = (
        cell83_report.get(
            "cached_subset_metrics"
        )
    )

    if isinstance(
        candidate_subset_metrics,
        dict,
    ):
        cached_subset_metrics = (
            candidate_subset_metrics
        )
        cached_subset_metric_available = True


# ------------------------------------------------------------
# 6. Parent-level cached-subset metrics
# ------------------------------------------------------------

parent_subset_metrics = None
parent_subset_metric_available = False

if "cell84_report" in globals():
    candidate_parent_metrics = (
        cell84_report.get(
            "cached_subset_metrics"
        )
    )

    if isinstance(
        candidate_parent_metrics,
        dict,
    ):
        parent_subset_metrics = (
            candidate_parent_metrics
        )
        parent_subset_metric_available = True


# ------------------------------------------------------------
# 7. Determine whether a full reranking conclusion exists
# ------------------------------------------------------------

full_reranking_available = (
    cached_reranking_count
    == total_query_count
    and successful_reranking_count
    == total_query_count
)

if full_reranking_available:
    reranking_evidence_status = (
        "FULL_DATASET_AVAILABLE"
    )
else:
    reranking_evidence_status = (
        "PARTIAL_CACHE_ONLY"
    )


# ------------------------------------------------------------
# 8. Formal benchmark status
#
# IMPORTANT:
# We do NOT promote cached-subset reranking metrics to
# full-Golden-v2 metrics.
# ------------------------------------------------------------

production_approval = False

decision_status = (
    "NO_PRODUCTION_APPROVAL_WITH_ACTIVE_GOLDEN_DATASET"
)

decision_reason_codes = []

if baseline_recall_at_10 is None:
    decision_reason_codes.append(
        "BASELINE_RECALL_AT_10_UNAVAILABLE"
    )

if baseline_mrr is None:
    decision_reason_codes.append(
        "BASELINE_MRR_UNAVAILABLE"
    )

if not full_reranking_available:
    decision_reason_codes.append(
        "FULL_RERANKING_EVALUATION_INCOMPLETE"
    )

if cached_reranking_count < total_query_count:
    decision_reason_codes.append(
        "RERANKING_COVERAGE_PARTIAL"
    )

if successful_reranking_count < total_query_count:
    decision_reason_codes.append(
        "FULL_RERANKING_PARSE_COVERAGE_INCOMPLETE"
    )


# ------------------------------------------------------------
# 9. Evidence limitations
# ------------------------------------------------------------

evidence_limitations = []

if not full_reranking_available:
    evidence_limitations.append(
        "Reranking results are available only for "
        "a cached subset of the active Golden-v2 queries."
    )

    evidence_limitations.append(
        "Cached-subset reranking metrics must not be "
        "reported as full-dataset reranking metrics."
    )

if missing_reranking_count > 0:
    evidence_limitations.append(
        "Missing reranking queries were intentionally "
        "not sent to the API."
    )


# ------------------------------------------------------------
# 10. Recommended engineering next state
#
# This is a benchmark workflow state, not a claim that
# reranking has been evaluated on all Golden-v2 queries.
# ------------------------------------------------------------

recommended_next_state = (
    "RETRIEVAL_REMEDIATION_REQUIRED"
)

required_follow_up = [
    "Preserve the active Golden-v2 dataset unchanged.",
    "Preserve existing embedding and reranking caches.",
    "Do not interpret partial cached reranking metrics "
    "as full-dataset metrics.",
    "Continue retrieval remediation and benchmark "
    "validation before production approval.",
]


# ------------------------------------------------------------
# 11. Build decision artifact
# ------------------------------------------------------------

retrieval_remediation_decision = {
    "active_golden_dataset_sha256": (
        golden_dataset_context["sha256"]
    ),
    "execution_mode": (
        "CACHE_ONLY_API_DISABLED"
    ),
    "status": decision_status,
    "production_approval": (
        production_approval
    ),
    "recommended_next_state": (
        recommended_next_state
    ),
    "query_counts": {
        "active_golden_queries": (
            total_query_count
        ),
        "cached_reranking_results": (
            cached_reranking_count
        ),
        "successful_cached_reranking_results": (
            successful_reranking_count
        ),
        "missing_reranking_results": (
            missing_reranking_count
        ),
        "reranking_coverage_rate": (
            reranking_coverage_rate
        ),
    },
    "baseline_metrics": {
        "recall_at_10": (
            baseline_recall_at_10
        ),
        "mrr": (
            baseline_mrr
        ),
    },
    "reranking_evidence_status": (
        reranking_evidence_status
    ),
    "full_reranking_available": (
        full_reranking_available
    ),
    "cached_subset_metric_available": (
        cached_subset_metric_available
    ),
    "cached_subset_metrics": (
        cached_subset_metrics
    ),
    "parent_subset_metric_available": (
        parent_subset_metric_available
    ),
    "parent_cached_subset_metrics": (
        parent_subset_metrics
    ),
    "decision_reason_codes": (
        decision_reason_codes
    ),
    "evidence_limitations": (
        evidence_limitations
    ),
    "required_follow_up": (
        required_follow_up
    ),
}


# ------------------------------------------------------------
# 12. Write artifact
# ------------------------------------------------------------

decision_path = (
    active_benchmark_run_directory
    / "retrieval_remediation_decision.json"
)

decision_path.write_text(
    json.dumps(
        retrieval_remediation_decision,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 13. Output
# ------------------------------------------------------------

print(
    "=== FORMAL RETRIEVAL REMEDIATION DECISION ==="
)

print(
    "API execution:",
    "DISABLED",
)

print(
    "Golden queries:",
    total_query_count,
)

print(
    "Baseline Recall@10:",
    baseline_recall_at_10,
)

print(
    "Baseline MRR:",
    baseline_mrr,
)

print(
    "Cached reranking results:",
    cached_reranking_count,
)

print(
    "Successful cached reranking results:",
    successful_reranking_count,
)

print(
    "Missing reranking results:",
    missing_reranking_count,
)

print(
    "Reranking coverage:",
    format(
        reranking_coverage_rate,
        ".4f",
    ),
)

print(
    "Reranking evidence:",
    reranking_evidence_status,
)

print(
    "Full reranking available:",
    full_reranking_available,
)

print(
    "Production approval:",
    production_approval,
)

print(
    "Decision status:",
    decision_status,
)

print(
    "Recommended next state:",
    recommended_next_state,
)

print(
    "Decision artifact:",
    decision_path,
)

if not full_reranking_available:
    print(
        "NOTE: Partial cached reranking results "
        "were not promoted to full-dataset metrics."
    )

print(
    "FORMAL RETRIEVAL REMEDIATION DECISION: PASS"
)

=== FORMAL RETRIEVAL REMEDIATION DECISION ===
API execution: DISABLED
Golden queries: 3813
Baseline Recall@10: 0.7440335693679517
Baseline MRR: 0.5117593490850066
Cached reranking results: 15
Successful cached reranking results: 15
Missing reranking results: 3798
Reranking coverage: 0.0039
Reranking evidence: PARTIAL_CACHE_ONLY
Full reranking available: False
Production approval: False
Decision status: NO_PRODUCTION_APPROVAL_WITH_ACTIVE_GOLDEN_DATASET
Recommended next state: RETRIEVAL_REMEDIATION_REQUIRED
Decision artifact: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/retrieval_remediation_decision.json
NOTE: Partial cached reranking results were not promoted to full-dataset metrics.
FORMAL RETRIEVAL REMEDIATION DECISION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>Handoff نهایی NB13</h2>
<p>Notebook به Golden Dataset v2 با 3813 پرسش متصل است و artifactهای benchmark در namespace هش Dataset نگهداری می‌شوند. مسیر اجرای نهایی cache-only است و تغییر نام Notebook باعث تکرار API نمی‌شود.</p>
<p>Reranking موجود partial/cache-only است: نتایج cached حفظ می‌شوند و پرسش‌های فاقد cache در Run-All به API ارسال نمی‌شوند. این محدودیت بخشی از قرارداد بازتولیدپذیری نهایی Notebook است.</p>
</div>


In [83]:
# Code Cell 67
# NB13 — Final reproducibility handoff for active Golden v2
# RUN-ALL SAFE: ZERO API / EXTERNAL CALLS.

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path


# ------------------------------------------------------------
# 1. Required state
# ------------------------------------------------------------

required_names = [
    "active_benchmark_run_directory",
    "golden_dataset_context",
    "arvan_hybrid_query_records",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, (
    "Missing required state: "
    + ", ".join(missing_names)
)


active_benchmark_run_directory = Path(
    active_benchmark_run_directory
)

assert active_benchmark_run_directory.is_dir(), (
    "Active benchmark run directory does not exist: "
    + str(active_benchmark_run_directory)
)


# ------------------------------------------------------------
# 2. Canonical artifact paths
#
# Define paths directly instead of depending on stale
# variable names from the old NB13 architecture.
# ------------------------------------------------------------

nb13_artifact_paths = {
    "full_reranking_cache": (
        active_benchmark_run_directory
        / "arvan_chat_full_reranking_cache.json"
    ),
    "full_reranking_report": (
        active_benchmark_run_directory
        / "arvan_chat_full_reranking_evaluation.json"
    ),
    "full_reranking_metric_impact": (
        active_benchmark_run_directory
        / "arvan_chat_full_reranking_metric_impact.json"
    ),
    "full_reranking_context_impact": (
        active_benchmark_run_directory
        / "arvan_chat_parent_level_reranking_impact.json"
    ),
    "remediation_decision": (
        active_benchmark_run_directory
        / "retrieval_remediation_decision.json"
    ),
}


# ------------------------------------------------------------
# 3. Add known upstream artifacts when their path variables
#    are actually available.
# ------------------------------------------------------------

optional_path_variables = {
    "benchmark_manifest": (
        "embedding_benchmark_manifest_path"
    ),
    "e5_instructional_cache": (
        "e5_instructional_cache_path"
    ),
    "query_rewrite_pilot_report": (
        "query_rewrite_pilot_report_path"
    ),
    "query_rewrite_augmentation_report": (
        "query_augmentation_report_path"
    ),
}

for artifact_name, variable_name in (
    optional_path_variables.items()
):
    if variable_name not in globals():
        continue

    artifact_path = globals()[variable_name]

    if artifact_path is None:
        continue

    nb13_artifact_paths[
        artifact_name
    ] = Path(artifact_path)


# ------------------------------------------------------------
# 4. File SHA helper
# ------------------------------------------------------------

def cell90_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(1024 * 1024)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


# ------------------------------------------------------------
# 5. Audit every known artifact
# ------------------------------------------------------------

artifact_inventory = {}

for artifact_name, artifact_path in (
    nb13_artifact_paths.items()
):
    artifact_path = Path(artifact_path)

    exists = artifact_path.is_file()

    artifact_entry = {
        "path": str(artifact_path),
        "exists": exists,
        "sha256": None,
        "size_bytes": None,
    }

    if exists:
        artifact_entry["sha256"] = (
            cell90_sha256(artifact_path)
        )

        artifact_entry["size_bytes"] = (
            artifact_path.stat().st_size
        )

    artifact_inventory[
        artifact_name
    ] = artifact_entry


# ------------------------------------------------------------
# 6. Required artifacts for the CURRENT cache-only NB13 state
#
# These are the artifacts produced by Cells 81–89 after the
# Golden-v2 migration.
# ------------------------------------------------------------

required_current_artifacts = [
    "full_reranking_cache",
    "full_reranking_report",
    "full_reranking_metric_impact",
    "full_reranking_context_impact",
    "remediation_decision",
]

missing_required_artifacts = [
    artifact_name
    for artifact_name
    in required_current_artifacts
    if not artifact_inventory[
        artifact_name
    ]["exists"]
]


# ------------------------------------------------------------
# 7. Optional artifacts
# ------------------------------------------------------------

optional_artifact_names = [
    artifact_name
    for artifact_name
    in artifact_inventory
    if artifact_name
    not in required_current_artifacts
]

missing_optional_artifacts = [
    artifact_name
    for artifact_name
    in optional_artifact_names
    if not artifact_inventory[
        artifact_name
    ]["exists"]
]


# ------------------------------------------------------------
# 8. Read current reranking/remediation state
# ------------------------------------------------------------

reranking_report_path = (
    nb13_artifact_paths[
        "full_reranking_report"
    ]
)

if reranking_report_path.is_file():
    current_reranking_report = json.loads(
        reranking_report_path.read_text(
            encoding="utf-8"
        )
    )
else:
    current_reranking_report = {}


decision_path = (
    nb13_artifact_paths[
        "remediation_decision"
    ]
)

if decision_path.is_file():
    current_remediation_decision = json.loads(
        decision_path.read_text(
            encoding="utf-8"
        )
    )
else:
    current_remediation_decision = {}


# ------------------------------------------------------------
# 9. Golden-v2 identity
# ------------------------------------------------------------

active_golden_sha256 = (
    golden_dataset_context["sha256"]
)

active_query_count = len(
    arvan_hybrid_query_records
)

assert active_query_count > 0, (
    "Active Golden-v2 benchmark query set is empty."
)


# ------------------------------------------------------------
# 10. Reranking state
# ------------------------------------------------------------

cached_reranking_count = (
    current_reranking_report.get(
        "completed_query_count",
        0,
    )
)

missing_reranking_count = (
    current_reranking_report.get(
        "missing_query_count",
        active_query_count
        - cached_reranking_count,
    )
)

reranking_evaluation_status = (
    current_reranking_report.get(
        "evaluation_status",
        "UNKNOWN",
    )
)

reranking_execution_mode = (
    current_reranking_report.get(
        "execution_mode",
        "UNKNOWN",
    )
)


# ------------------------------------------------------------
# 11. Reproducibility status
#
# Missing optional historical artifacts do not invalidate the
# current cache-only handoff.
# ------------------------------------------------------------

if missing_required_artifacts:
    handoff_status = (
        "INCOMPLETE_REQUIRED_ARTIFACTS"
    )
else:
    handoff_status = (
        "CACHE_ONLY_HANDOFF_READY"
    )


# ------------------------------------------------------------
# 12. Explicit limitations
# ------------------------------------------------------------

limitations = []

if missing_reranking_count > 0:
    limitations.append(
        "Full reranking is incomplete for the active "
        "Golden-v2 query set."
    )

    limitations.append(
        "Missing reranking queries were intentionally "
        "not sent to the API."
    )

    limitations.append(
        "Cached-subset reranking metrics must not be "
        "reported as full-dataset metrics."
    )

if missing_optional_artifacts:
    limitations.append(
        "One or more optional historical artifacts "
        "were not available in the current runtime."
    )


# ------------------------------------------------------------
# 13. Handoff artifact
# ------------------------------------------------------------

nb13_handoff = {
    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "active_golden_dataset_sha256": (
        active_golden_sha256
    ),
    "active_golden_query_count": (
        active_query_count
    ),
    "execution_mode": (
        "CACHE_ONLY_API_DISABLED"
    ),
    "handoff_status": (
        handoff_status
    ),
    "reranking_state": {
        "execution_mode": (
            reranking_execution_mode
        ),
        "evaluation_status": (
            reranking_evaluation_status
        ),
        "cached_query_count": (
            cached_reranking_count
        ),
        "missing_query_count": (
            missing_reranking_count
        ),
    },
    "remediation_state": {
        "status": (
            current_remediation_decision.get(
                "status"
            )
        ),
        "production_approval": (
            current_remediation_decision.get(
                "production_approval"
            )
        ),
        "recommended_next_state": (
            current_remediation_decision.get(
                "recommended_next_state"
            )
        ),
    },
    "required_artifacts": (
        required_current_artifacts
    ),
    "missing_required_artifacts": (
        missing_required_artifacts
    ),
    "optional_artifacts": (
        optional_artifact_names
    ),
    "missing_optional_artifacts": (
        missing_optional_artifacts
    ),
    "artifact_inventory": (
        artifact_inventory
    ),
    "limitations": (
        limitations
    ),
}


# ------------------------------------------------------------
# 14. Write handoff
# ------------------------------------------------------------

nb13_handoff_path = (
    active_benchmark_run_directory
    / "nb13_reproducibility_handoff.json"
)

nb13_handoff_path.write_text(
    json.dumps(
        nb13_handoff,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 15. Verify written handoff
# ------------------------------------------------------------

assert nb13_handoff_path.is_file(), (
    "NB13 handoff artifact was not written."
)

written_handoff = json.loads(
    nb13_handoff_path.read_text(
        encoding="utf-8"
    )
)

assert (
    written_handoff[
        "active_golden_dataset_sha256"
    ]
    == active_golden_sha256
), (
    "Written handoff Golden SHA mismatch."
)

assert (
    written_handoff[
        "active_golden_query_count"
    ]
    == active_query_count
), (
    "Written handoff query-count mismatch."
)


# ------------------------------------------------------------
# 16. Output
# ------------------------------------------------------------

print(
    "=== NB13 REPRODUCIBILITY HANDOFF ==="
)

print(
    "API execution:",
    "DISABLED",
)

print(
    "Golden SHA256:",
    active_golden_sha256,
)

print(
    "Golden queries:",
    active_query_count,
)

print(
    "Cached reranking queries:",
    cached_reranking_count,
)

print(
    "Missing reranking queries:",
    missing_reranking_count,
)

print(
    "Reranking status:",
    reranking_evaluation_status,
)

print(
    "Required artifacts:",
    len(required_current_artifacts),
)

print(
    "Missing required artifacts:",
    missing_required_artifacts,
)

print(
    "Missing optional artifacts:",
    missing_optional_artifacts,
)

print(
    "Handoff status:",
    handoff_status,
)

print(
    "Handoff artifact:",
    nb13_handoff_path,
)

if missing_required_artifacts:
    print(
        "NOTE: Handoff is incomplete because one or "
        "more current required artifacts are missing."
    )

if missing_reranking_count > 0:
    print(
        "NOTE: Reranking remains partial/cache-only."
    )

print(
    "NB13 REPRODUCIBILITY HANDOFF: PASS"
)

=== NB13 REPRODUCIBILITY HANDOFF ===
API execution: DISABLED
Golden SHA256: b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Golden queries: 3813
Cached reranking queries: 15
Missing reranking queries: 3798
Reranking status: PARTIAL_CACHE_ONLY
Required artifacts: 5
Missing required artifacts: []
Missing optional artifacts: []
Handoff status: CACHE_ONLY_HANDOFF_READY
Handoff artifact: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/nb13_reproducibility_handoff.json
NOTE: Reranking remains partial/cache-only.
NB13 REPRODUCIBILITY HANDOFF: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>Closure رسمی NB13 — ممیزی Revision و قرارداد Freeze</h2>
<p>این بخش بدون API و بدون بازتولید Embedding اجرا می‌شود. هدف آن بستن Deliverableهای رسمی Notebook 13 طبق Roadmap است. ابتدا provenance مدل‌های محلی و Cacheها بررسی می‌شود. اگر exact revision مدل منتخب از شواهد موجود قابل اثبات نباشد، Notebook در Final Gate به‌صورت صریح متوقف می‌شود و هیچ Freeze جعلی تولید نمی‌شود.</p>
</div>


In [84]:
# Code Cell 68 — Resolve E5 Revision by Snapshot Content Equivalence

from pathlib import Path
import hashlib
import json

print("=== NB13 E5 REVISION RESOLUTION ===")

E5_MODEL_ID = "intfloat/multilingual-e5-small"

E5_HF_ROOT = (
    Path.home()
    / ".cache"
    / "huggingface"
    / "hub"
    / "models--intfloat--multilingual-e5-small"
)

E5_SNAPSHOTS_ROOT = E5_HF_ROOT / "snapshots"
E5_REFS_ROOT = E5_HF_ROOT / "refs"
E5_MAIN_REF = E5_REFS_ROOT / "main"


def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def _snapshot_inventory(snapshot_root: Path) -> dict:
    inventory = {}

    for path in sorted(snapshot_root.rglob("*")):
        if not path.is_file():
            continue

        rel = path.relative_to(snapshot_root).as_posix()
        resolved = path.resolve()

        inventory[rel] = {
            "size": int(resolved.stat().st_size),
            "sha256": _sha256_file(resolved),
        }

    return inventory


def _inventory_identity_sha256(inventory: dict) -> str:
    canonical = json.dumps(
        inventory,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(canonical).hexdigest()


# ------------------------------------------------------------
# 1. Discover locally available E5 snapshots
# ------------------------------------------------------------

assert E5_HF_ROOT.is_dir(), (
    f"E5 HuggingFace cache root not found: {E5_HF_ROOT}"
)

assert E5_SNAPSHOTS_ROOT.is_dir(), (
    f"E5 snapshots directory not found: {E5_SNAPSHOTS_ROOT}"
)

e5_snapshot_dirs = sorted(
    p
    for p in E5_SNAPSHOTS_ROOT.iterdir()
    if p.is_dir()
)

e5_snapshot_ids = [p.name for p in e5_snapshot_dirs]

assert e5_snapshot_ids, "No local E5 snapshots were found."

print("Model:", E5_MODEL_ID)
print("Local snapshots:", e5_snapshot_ids)


# ------------------------------------------------------------
# 2. Read current HuggingFace main reference
# ------------------------------------------------------------

e5_main_revision = None

if E5_MAIN_REF.is_file():
    e5_main_revision = E5_MAIN_REF.read_text(
        encoding="utf-8"
    ).strip()

print("refs/main:", e5_main_revision)

assert e5_main_revision, (
    "HuggingFace refs/main is unavailable; "
    "an exact release revision cannot be bound safely."
)

assert e5_main_revision in e5_snapshot_ids, (
    "refs/main points to a revision that is not present "
    "in the local snapshot cache."
)


# ------------------------------------------------------------
# 3. Build byte-level inventories for every local snapshot
# ------------------------------------------------------------

e5_snapshot_inventories = {}

for snapshot_dir in e5_snapshot_dirs:
    inventory = _snapshot_inventory(snapshot_dir)

    assert inventory, (
        f"Snapshot contains no files: {snapshot_dir.name}"
    )

    e5_snapshot_inventories[snapshot_dir.name] = inventory


# ------------------------------------------------------------
# 4. Verify complete content equivalence
# ------------------------------------------------------------

reference_snapshot_id = e5_snapshot_ids[0]
reference_inventory = e5_snapshot_inventories[
    reference_snapshot_id
]

content_equivalence_details = {}

all_snapshots_content_equivalent = True

for snapshot_id in e5_snapshot_ids:
    inventory = e5_snapshot_inventories[snapshot_id]

    reference_files = set(reference_inventory)
    candidate_files = set(inventory)

    only_reference = sorted(
        reference_files - candidate_files
    )

    only_candidate = sorted(
        candidate_files - reference_files
    )

    different_files = []

    for rel in sorted(
        reference_files & candidate_files
    ):
        if (
            reference_inventory[rel]["size"]
            != inventory[rel]["size"]
            or
            reference_inventory[rel]["sha256"]
            != inventory[rel]["sha256"]
        ):
            different_files.append(rel)

    equivalent = (
        not only_reference
        and not only_candidate
        and not different_files
    )

    content_equivalence_details[snapshot_id] = {
        "equivalent_to_reference": equivalent,
        "only_in_reference": only_reference,
        "only_in_candidate": only_candidate,
        "different_files": different_files,
        "file_count": len(inventory),
        "inventory_sha256": _inventory_identity_sha256(
            inventory
        ),
    }

    if not equivalent:
        all_snapshots_content_equivalent = False


print(
    "All local snapshots content-equivalent:",
    all_snapshots_content_equivalent,
)

for snapshot_id in e5_snapshot_ids:
    info = content_equivalence_details[snapshot_id]

    print(
        f"- {snapshot_id}: "
        f"files={info['file_count']} | "
        f"equivalent={info['equivalent_to_reference']} | "
        f"inventory_sha256={info['inventory_sha256']}"
    )


# ------------------------------------------------------------
# 5. Provenance decision
# ------------------------------------------------------------

assert all_snapshots_content_equivalent, (
    "Multiple E5 snapshots exist with different content. "
    "Historical cache revision therefore remains unresolved."
)

selected_inventory = e5_snapshot_inventories[
    e5_main_revision
]

e5_content_identity_sha256 = (
    _inventory_identity_sha256(selected_inventory)
)

resolved_e5_revision = e5_main_revision

e5_revision_resolution = {
    "model_id": E5_MODEL_ID,
    "release_revision": resolved_e5_revision,
    "resolution_status": "RESOLVED_BY_CONTENT_EQUIVALENCE",
    "historical_snapshot_name_identifiable": False,
    "historical_snapshot_name_claimed": False,
    "local_snapshot_ids": e5_snapshot_ids,
    "all_local_snapshots_content_equivalent": True,
    "complete_snapshot_file_count": len(
        selected_inventory
    ),
    "snapshot_content_identity_sha256": (
        e5_content_identity_sha256
    ),
    "release_binding_policy": (
        "The historical embedding cache does not record the "
        "snapshot commit name. All locally available E5 snapshots "
        "were independently verified to contain identical complete "
        "file payloads. The frozen release is therefore bound to "
        "the current local refs/main revision while preserving the "
        "content-equivalence evidence and without claiming that "
        "this commit name was historically recorded during embedding."
    ),
    "snapshot_comparison": (
        content_equivalence_details
    ),
}

assert (
    e5_revision_resolution["resolution_status"]
    == "RESOLVED_BY_CONTENT_EQUIVALENCE"
)

assert (
    e5_revision_resolution[
        "all_local_snapshots_content_equivalent"
    ]
    is True
)

assert len(resolved_e5_revision) == 40, (
    "Resolved E5 revision is not a 40-character commit hash."
)


# ------------------------------------------------------------
# 6. Persist reproducibility evidence
# ------------------------------------------------------------

revision_resolution_path = (
    active_benchmark_run_directory
    / "e5_revision_resolution.json"
)

revision_resolution_path.write_text(
    json.dumps(
        e5_revision_resolution,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print()
print(
    "Historical snapshot name:",
    "NOT RECORDED",
)
print(
    "Historical snapshot content ambiguity:",
    "NONE — all local snapshots are byte/content-equivalent",
)
print(
    "Frozen release revision:",
    resolved_e5_revision,
)
print(
    "Snapshot content identity SHA256:",
    e5_content_identity_sha256,
)
print(
    "Resolution status:",
    e5_revision_resolution["resolution_status"],
)
print(
    "Evidence artifact:",
    revision_resolution_path,
)

print()
print("NB13 E5 EXACT REVISION BINDING: PASS")

=== NB13 E5 REVISION RESOLUTION ===
Model: intfloat/multilingual-e5-small
Local snapshots: ['614241f622f53c4eeff9890bdc4f31cfecc418b3', 'c007d7ef6fd86656326059b28395a7a03a7c5846']
refs/main: 614241f622f53c4eeff9890bdc4f31cfecc418b3
All local snapshots content-equivalent: True
- 614241f622f53c4eeff9890bdc4f31cfecc418b3: files=10 | equivalent=True | inventory_sha256=0a487b28bf6bee275240c12ddc31ac3639ed367138e2382c113045eba10d217c
- c007d7ef6fd86656326059b28395a7a03a7c5846: files=10 | equivalent=True | inventory_sha256=0a487b28bf6bee275240c12ddc31ac3639ed367138e2382c113045eba10d217c

Historical snapshot name: NOT RECORDED
Historical snapshot content ambiguity: NONE — all local snapshots are byte/content-equivalent
Frozen release revision: 614241f622f53c4eeff9890bdc4f31cfecc418b3
Snapshot content identity SHA256: 0a487b28bf6bee275240c12ddc31ac3639ed367138e2382c113045eba10d217c
Resolution status: RESOLVED_BY_CONTENT_EQUIVALENCE
Evidence artifact: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJ

<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>تکمیل Metricهای مشتق‌شدنی Roadmap</h2>
<p>در این مرحله فقط از Golden Dataset، Chunk releaseها و بردارهای Cacheشده استفاده می‌شود. nDCG@k، breakdown منبع و تحلیل طول Query محاسبه می‌شوند. هیچ Embedding جدیدی تولید نمی‌شود. اگر برچسب معتبر محاوره‌ای در Golden موجود نباشد، این محدودیت به‌صورت NOT_AVAILABLE ثبت می‌شود و برچسب ساختگی تولید نمی‌شود.</p>
</div>


In [85]:
# Code Cell 69
# NB13 closure — derived roadmap metrics from existing vectors only
import math
from collections import defaultdict

closure_rank_details = {}

def ndcg_binary_at_k(ranked_indexes, positive_indexes, k):
    dcg = 0.0
    for rank0, idx in enumerate(ranked_indexes[:k]):
        if int(idx) in positive_indexes:
            dcg += 1.0 / math.log2(rank0 + 2.0)
    ideal_hits = min(len(positive_indexes), k)
    idcg = sum(1.0 / math.log2(i + 2.0) for i in range(ideal_hits))
    return 0.0 if idcg == 0 else dcg / idcg

def nested_first(record, keys):
    stack = [record]
    while stack:
        item = stack.pop()
        if isinstance(item, dict):
            for key, value in item.items():
                if key in keys and value not in (None, "", [], {}):
                    return value
                if isinstance(value, (dict, list)):
                    stack.append(value)
        elif isinstance(item, list):
            stack.extend(x for x in item if isinstance(x, (dict, list)))
    return None

raw_by_candidate = {r["candidate_id"]: r for r in raw_golden_dataset}
source_family_by_seed = {}
intent_by_seed = {}
for q in evaluation_query_records:
    sid = q["source_seed_id"]
    raw = raw_by_candidate.get(sid, {})
    source_value = nested_first(raw, {"source_family", "source_type", "source_kind", "corpus_type"})
    source_family_by_seed[sid] = str(source_value) if source_value is not None else "UNKNOWN"
    intent_value = q.get("question_intent") or nested_first(raw, {"question_intent", "query_type", "question_type"})
    intent_by_seed[sid] = str(intent_value) if intent_value is not None else "UNKNOWN"

question_lengths = np.asarray([len(q["question"].split()) for q in evaluation_query_records], dtype=int)
q33, q67 = np.quantile(question_lengths, [1/3, 2/3])
def length_class(n):
    if n <= q33: return "short"
    if n >= q67: return "long"
    return "medium"
length_class_by_seed = {q["source_seed_id"]: length_class(len(q["question"].split())) for q in evaluation_query_records}

closure_metric_results = {}
for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack([
        embedding_lookup["query_vectors_by_text_hash"][hashlib.sha256(q["question"].encode("utf-8")).hexdigest()]
        for q in evaluation_query_records
    ]).astype(np.float32)
    query_matrix = l2_normalize_rows(query_matrix)
    closure_metric_results[model_name] = {}
    for candidate_name, candidate_data in candidate_chunk_registry.items():
        passage_matrix = np.vstack([
            embedding_lookup["passage_vectors_by_text_hash"][h]
            for h in candidate_data["chunk_text_hashes"]
        ]).astype(np.float32)
        passage_matrix = l2_normalize_rows(passage_matrix)
        rows = []
        batch_size = 16
        for start in range(0, len(evaluation_query_records), batch_size):
            end = min(start + batch_size, len(evaluation_query_records))
            scores = query_matrix[start:end] @ passage_matrix.T
            for local_i, q in enumerate(evaluation_query_records[start:end]):
                ranked = np.argsort(scores[local_i])[::-1]
                positives = candidate_data["positive_chunk_indexes_by_seed"][q["source_seed_id"]]
                first_rank = next(i + 1 for i, idx in enumerate(ranked) if int(idx) in positives)
                rows.append({
                    "source_seed_id": q["source_seed_id"],
                    "first_positive_rank": int(first_rank),
                    "ndcg_at_5": float(ndcg_binary_at_k(ranked, positives, 5)),
                    "ndcg_at_10": float(ndcg_binary_at_k(ranked, positives, 10)),
                    "source_family": source_family_by_seed[q["source_seed_id"]],
                    "query_length_class": length_class_by_seed[q["source_seed_id"]],
                    "question_intent": intent_by_seed[q["source_seed_id"]],
                })
        def aggregate(subrows):
            ranks = np.asarray([r["first_positive_rank"] for r in subrows], dtype=float)
            return {
                "query_count": len(subrows),
                "recall_at_5": float(np.mean(ranks <= 5)),
                "recall_at_10": float(np.mean(ranks <= 10)),
                "mrr": float(np.mean(1.0 / ranks)),
                "ndcg_at_5": float(np.mean([r["ndcg_at_5"] for r in subrows])),
                "ndcg_at_10": float(np.mean([r["ndcg_at_10"] for r in subrows])),
            }
        by_source = {}
        for value in sorted(set(r["source_family"] for r in rows)):
            by_source[value] = aggregate([r for r in rows if r["source_family"] == value])
        by_length = {}
        for value in ("short", "medium", "long"):
            subset = [r for r in rows if r["query_length_class"] == value]
            if subset: by_length[value] = aggregate(subset)
        known_intents = sorted(set(r["question_intent"] for r in rows if r["question_intent"] != "UNKNOWN"))
        by_intent = {value: aggregate([r for r in rows if r["question_intent"] == value]) for value in known_intents}
        closure_metric_results[model_name][candidate_name] = {
            "overall": aggregate(rows),
            "by_source_family": by_source,
            "by_query_length": by_length,
            "by_question_intent": by_intent,
            "conversational_analysis_status": "AVAILABLE_FROM_EXISTING_LABELS" if any("convers" in x.lower() or "محاوره" in x for x in known_intents) else "NOT_AVAILABLE_NO_EXPLICIT_EXISTING_LABEL",
        }

closure_metrics_path = active_benchmark_run_directory / "embedding_benchmark_roadmap_metrics.json"
closure_metrics_path.write_text(json.dumps(closure_metric_results, ensure_ascii=False, indent=2), encoding="utf-8")
print("=== NB13 ROADMAP METRICS CLOSURE ===")
print("Models:", list(closure_metric_results))
print("Source families observed:", sorted(set(source_family_by_seed.values())))
print("Query length thresholds (words):", {"q33": float(q33), "q67": float(q67)})
print("Artifact:", closure_metrics_path)
print("NB13 ROADMAP METRICS CLOSURE: PASS")


=== NB13 ROADMAP METRICS CLOSURE ===
Models: ['arvan_text_embedding_3_large', 'multilingual_e5_small', 'all_minilm_l6_v2']
Source families observed: ['UNKNOWN']
Query length thresholds (words): {'q33': 7.0, 'q67': 11.0}
Artifact: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/embedding_benchmark_roadmap_metrics.json
NB13 ROADMAP METRICS CLOSURE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>ADR و انتخاب نهایی مبتنی بر شواهد</h2>
<p>این مرحله Candidateها را با Metricهای موجود و Metricهای تکمیلی مقایسه می‌کند. انتخاب فقط در صورتی قابل Freeze است که exact revision مدل منتخب از provenance موجود اثبات شده باشد. Latency/Cost تاریخیِ ثبت‌نشده جعل نمی‌شود و به‌عنوان NOT_MEASURED_HISTORICALLY ثبت خواهد شد.</p>
</div>


In [86]:
# Code Cell 70 — Final Embedding ADR with Exact Schema + Revision Binding

import json
import hashlib

print("=== NB13 EMBEDDING ADR ===")

# ------------------------------------------------------------
# 1. Require exact revision evidence from Cell 68
# ------------------------------------------------------------

assert "resolved_e5_revision" in globals(), (
    "Cell 68 has not resolved the E5 release revision."
)

assert "e5_revision_resolution" in globals(), (
    "Cell 68 revision-resolution evidence is unavailable."
)

assert (
    e5_revision_resolution.get("resolution_status")
    == "RESOLVED_BY_CONTENT_EQUIVALENCE"
), "E5 revision resolution is not PASS."

assert (
    e5_revision_resolution.get(
        "all_local_snapshots_content_equivalent"
    )
    is True
), "E5 snapshots are not content-equivalent."

assert (
    e5_revision_resolution.get("release_revision")
    == resolved_e5_revision
), "Cell 68 release revision mismatch."

assert len(resolved_e5_revision) == 40

resolved_e5_content_sha256 = (
    e5_revision_resolution[
        "snapshot_content_identity_sha256"
    ]
)


# ------------------------------------------------------------
# 2. Load exact existing artifacts
# ------------------------------------------------------------

tradeoff_path = (
    active_benchmark_run_directory
    / "embedding_benchmark_tradeoff.json"
)

roadmap_metrics_path = (
    active_benchmark_run_directory
    / "embedding_benchmark_roadmap_metrics.json"
)

assert tradeoff_path.is_file(), (
    f"Missing tradeoff artifact: {tradeoff_path}"
)

assert roadmap_metrics_path.is_file(), (
    f"Missing Roadmap metrics artifact: {roadmap_metrics_path}"
)

tradeoff_payload = json.loads(
    tradeoff_path.read_text(encoding="utf-8")
)

roadmap_metrics = json.loads(
    roadmap_metrics_path.read_text(encoding="utf-8")
)


# ------------------------------------------------------------
# 3. Recover the actual benchmark decision
# ------------------------------------------------------------

decision_basis = tradeoff_payload.get(
    "decision_basis",
    {},
)

best_question_retrieval = decision_basis.get(
    "best_question_retrieval"
)

assert isinstance(best_question_retrieval, dict), (
    "tradeoff artifact does not contain "
    "decision_basis.best_question_retrieval."
)

selected_model_key = best_question_retrieval.get(
    "model"
)

selected_chunk_strategy = best_question_retrieval.get(
    "candidate"
)

assert selected_model_key == "multilingual_e5_small", (
    "The recorded best question-retrieval model is not "
    "multilingual_e5_small."
)

assert selected_chunk_strategy, (
    "Selected chunk strategy is missing."
)

tradeoff_recall_at_10 = float(
    best_question_retrieval["recall_at_10"]
)

tradeoff_mrr = float(
    best_question_retrieval["mrr"]
)


# ------------------------------------------------------------
# 4. Bind to exact Roadmap metrics schema
# ------------------------------------------------------------

assert selected_model_key in roadmap_metrics, (
    f"{selected_model_key} missing from Roadmap metrics."
)

assert (
    selected_chunk_strategy
    in roadmap_metrics[selected_model_key]
), (
    f"{selected_chunk_strategy} missing from "
    f"{selected_model_key} Roadmap metrics."
)

selected_metrics_record = (
    roadmap_metrics[selected_model_key][
        selected_chunk_strategy
    ]
)

overall_metrics = selected_metrics_record.get(
    "overall"
)

assert isinstance(overall_metrics, dict), (
    "Selected Roadmap metrics record has no overall section."
)

roadmap_recall_at_10 = float(
    overall_metrics["recall_at_10"]
)

roadmap_mrr = float(
    overall_metrics["mrr"]
)

roadmap_ndcg_at_10 = float(
    overall_metrics["ndcg_at_10"]
)

roadmap_ndcg_at_5 = float(
    overall_metrics["ndcg_at_5"]
)

roadmap_recall_at_5 = float(
    overall_metrics["recall_at_5"]
)

query_count = int(
    overall_metrics["query_count"]
)


# ------------------------------------------------------------
# 5. Cross-check old and new metric artifacts
# ------------------------------------------------------------

TOL = 1e-12

assert abs(
    tradeoff_recall_at_10
    - roadmap_recall_at_10
) <= TOL, (
    "Recall@10 mismatch between tradeoff and "
    "Roadmap metrics artifacts."
)

assert abs(
    tradeoff_mrr
    - roadmap_mrr
) <= TOL, (
    "MRR mismatch between tradeoff and "
    "Roadmap metrics artifacts."
)

assert query_count == 3813, (
    f"Unexpected query count: {query_count}"
)


# ------------------------------------------------------------
# 6. Record Roadmap analysis availability honestly
# ------------------------------------------------------------

by_source_family = selected_metrics_record.get(
    "by_source_family",
    {},
)

observed_source_families = sorted(
    by_source_family.keys()
)

source_family_status = (
    "UNAVAILABLE_FROM_CURRENT_METADATA"
    if observed_source_families == ["UNKNOWN"]
    else "RECORDED"
)

by_query_length = selected_metrics_record.get(
    "by_query_length",
    {},
)

assert {
    "short",
    "medium",
    "long",
}.issubset(by_query_length.keys()), (
    "Short/medium/long query analysis is incomplete."
)

by_question_intent = selected_metrics_record.get(
    "by_question_intent",
    {}
)

assert isinstance(by_question_intent, dict)
assert by_question_intent, (
    "Question-intent analysis is unavailable."
)


# ------------------------------------------------------------
# 7. Preserve unresolved historical limitations
# ------------------------------------------------------------

historical_tradeoff_status = (
    tradeoff_payload.get("selection_status")
)

historical_blocking_findings = (
    tradeoff_payload.get(
        "blocking_findings",
        [],
    )
)

historical_required_next_work = (
    tradeoff_payload.get(
        "required_next_work",
        [],
    )
)


# ------------------------------------------------------------
# 8. Build canonical revision-bound ADR
# ------------------------------------------------------------

embedding_adr = {
    "schema_version": "1.1",
    "decision_type": "EMBEDDING_SELECTION_ADR",

    "decision_status": "SELECTED_REVISION_BOUND",

    "active_golden_dataset_sha256": (
        tradeoff_payload.get(
            "active_golden_dataset_sha256"
        )
    ),

    "selected_model_key": selected_model_key,
    "selected_model_id": (
        "intfloat/multilingual-e5-small"
    ),

    "selected_provider": (
        "local_huggingface_sentence_transformers"
    ),

    "selected_revision": resolved_e5_revision,

    "revision_binding": {
        "status": (
            e5_revision_resolution[
                "resolution_status"
            ]
        ),
        "historical_snapshot_name_recorded": False,
        "historical_snapshot_name_claimed": False,
        "all_local_snapshots_content_equivalent": True,
        "snapshot_content_identity_sha256": (
            resolved_e5_content_sha256
        ),
        "evidence_artifact": str(
            active_benchmark_run_directory
            / "e5_revision_resolution.json"
        ),
    },

    "embedding_dimension": 384,

    "query_prefix": "query:",
    "passage_prefix": "passage:",

    "normalization": (
        "normalized SentenceTransformer encoding "
        "with L2-normalized retrieval vectors"
    ),

    "selected_chunk_strategy": (
        selected_chunk_strategy
    ),

    "quality_metrics": {
        "query_count": query_count,
        "recall_at_5": roadmap_recall_at_5,
        "recall_at_10": roadmap_recall_at_10,
        "mrr": roadmap_mrr,
        "ndcg_at_5": roadmap_ndcg_at_5,
        "ndcg_at_10": roadmap_ndcg_at_10,
    },

    "roadmap_analysis": {
        "source_family_status": (
            source_family_status
        ),
        "observed_source_families": (
            observed_source_families
        ),
        "query_length_analysis": {
            "status": "RECORDED",
            "classes": [
                "short",
                "medium",
                "long",
            ],
        },
        "question_intent_analysis": {
            "status": "RECORDED",
            "intent_count": len(
                by_question_intent
            ),
        },
    },

    "historical_operational_measurements": {
        "embedding_generation_latency": (
            "NOT_MEASURED_HISTORICALLY"
        ),
        "embedding_generation_failure_rate": (
            "NOT_MEASURED_HISTORICALLY"
        ),
        "historical_exact_snapshot_name": (
            "NOT_RECORDED"
        ),
        "historical_snapshot_content_ambiguity": (
            "NONE_CONTENT_EQUIVALENCE_PROVEN"
        ),
    },

    "historical_tradeoff_context": {
        "previous_selection_status": (
            historical_tradeoff_status
        ),
        "blocking_findings": (
            historical_blocking_findings
        ),
        "required_next_work": (
            historical_required_next_work
        ),
        "interpretation": (
            "The historical tradeoff artifact remained "
            "NO_FINAL_SELECTION. NB13 closure does not "
            "rewrite that historical artifact. This ADR "
            "records the final embedding decision using "
            "the completed Roadmap metrics and the "
            "revision provenance evidence established "
            "during closure."
        ),
    },

    "selection_basis": {
        "source": (
            "embedding_benchmark_tradeoff.json:"
            "decision_basis.best_question_retrieval"
        ),
        "metric_consistency_check": "PASS",
        "roadmap_metrics_source": (
            "embedding_benchmark_roadmap_metrics.json"
        ),
    },

    "reranking_dependency": "NONE",
    "paid_api_required_for_closure": False,
}


# ------------------------------------------------------------
# 9. Final ADR assertions
# ------------------------------------------------------------

assert (
    embedding_adr[
        "active_golden_dataset_sha256"
    ]
    == "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2"
), "Golden v2 SHA mismatch."

assert (
    embedding_adr["selected_revision"]
    == resolved_e5_revision
)

assert (
    embedding_adr["revision_binding"]["status"]
    == "RESOLVED_BY_CONTENT_EQUIVALENCE"
)

assert (
    embedding_adr["decision_status"]
    == "SELECTED_REVISION_BOUND"
)


# ------------------------------------------------------------
# 10. Persist canonical ADR
# ------------------------------------------------------------

embedding_adr_path = (
    active_benchmark_run_directory
    / "embedding_selection_adr_v1.json"
)

embedding_adr_path.write_text(
    json.dumps(
        embedding_adr,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

embedding_adr_sha256 = hashlib.sha256(
    embedding_adr_path.read_bytes()
).hexdigest()


# ------------------------------------------------------------
# 11. Output
# ------------------------------------------------------------

print(
    "Selected:",
    f"{selected_model_key} × "
    f"{selected_chunk_strategy}",
)

print(
    "R@10:",
    roadmap_recall_at_10,
)

print(
    "MRR:",
    roadmap_mrr,
)

print(
    "nDCG@10:",
    roadmap_ndcg_at_10,
)

print(
    "Exact revision:",
    resolved_e5_revision,
)

print(
    "Revision binding:",
    e5_revision_resolution[
        "resolution_status"
    ],
)

print(
    "Content identity SHA256:",
    resolved_e5_content_sha256,
)

print(
    "Source-family status:",
    source_family_status,
)

print(
    "Historical tradeoff status preserved:",
    historical_tradeoff_status,
)

print(
    "ADR status:",
    embedding_adr[
        "decision_status"
    ],
)

print(
    "ADR SHA256:",
    embedding_adr_sha256,
)

print(
    "Artifact:",
    embedding_adr_path,
)

print()
print("NB13 EMBEDDING ADR: PASS")

=== NB13 EMBEDDING ADR ===
Selected: multilingual_e5_small × baseline_phase1_char_safe
R@10: 0.6113296616837136
MRR: 0.4112586794908517
nDCG@10: 0.3735597782433693
Exact revision: 614241f622f53c4eeff9890bdc4f31cfecc418b3
Revision binding: RESOLVED_BY_CONTENT_EQUIVALENCE
Content identity SHA256: 0a487b28bf6bee275240c12ddc31ac3639ed367138e2382c113045eba10d217c
Source-family status: UNAVAILABLE_FROM_CURRENT_METADATA
Historical tradeoff status preserved: NO_FINAL_SELECTION
ADR status: SELECTED_REVISION_BOUND
ADR SHA256: ea893234f36a136d173b64f473f6eb29c3b38bdb37126f51286fb51dac25b3a8
Artifact: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/embedding_selection_adr_v1.json

NB13 EMBEDDING ADR: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>Release و Handoff رسمی NB13 → NB14</h2>
<p>Release فقط زمانی با وضعیت FROZEN_FOR_NB14 نوشته می‌شود که مدل منتخب revision اثبات‌شده داشته باشد، Chunk artifact موجود باشد و SHA/Count آن از فایل واقعی محاسبه شود. در غیر این صورت Artifact با وضعیت BLOCKED نوشته می‌شود و Final Gate عمداً PASS نمی‌شود.</p>
</div>


In [87]:
# Code Cell 71 — Canonical Embedding/Chunk Release + NB14 Handoff

from pathlib import Path
import json
import hashlib

print("=== NB13 CANONICAL RELEASE / NB14 HANDOFF ===")


# ------------------------------------------------------------
# 1. Recover project paths safely
# ------------------------------------------------------------

if "project_root" not in globals():
    cwd = Path.cwd().resolve()

    project_root = next(
        (
            p
            for p in [cwd, *cwd.parents]
            if (p / "notebooks" / "phase2").is_dir()
        ),
        None,
    )

    assert project_root is not None, (
        "Could not recover project_root from current working directory."
    )

project_root = Path(project_root).resolve()

GOLDEN_SHA = (
    "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2"
)

if "active_benchmark_run_directory" not in globals():
    active_benchmark_run_directory = (
        project_root
        / "artifacts"
        / "phase2"
        / "embedding_benchmark"
        / GOLDEN_SHA
    )

active_benchmark_run_directory = Path(
    active_benchmark_run_directory
).resolve()

assert active_benchmark_run_directory.is_dir(), (
    "Active embedding benchmark directory is missing: "
    f"{active_benchmark_run_directory}"
)


# ------------------------------------------------------------
# 2. Load canonical persisted ADR from Cell 70
# ------------------------------------------------------------

embedding_adr_path = (
    active_benchmark_run_directory
    / "embedding_selection_adr_v1.json"
)

assert embedding_adr_path.is_file(), (
    f"Missing Cell 70 ADR artifact: {embedding_adr_path}"
)

embedding_adr = json.loads(
    embedding_adr_path.read_text(encoding="utf-8")
)

assert (
    embedding_adr.get("decision_status")
    == "SELECTED_REVISION_BOUND"
), "Embedding ADR is not revision-bound."

assert (
    embedding_adr.get("active_golden_dataset_sha256")
    == GOLDEN_SHA
), "Embedding ADR Golden SHA mismatch."

selected_model_key = embedding_adr["selected_model_key"]
selected_model_id = embedding_adr["selected_model_id"]
selected_revision = embedding_adr["selected_revision"]
selected_chunk_strategy = embedding_adr[
    "selected_chunk_strategy"
]

assert selected_model_key == "multilingual_e5_small"
assert selected_model_id == "intfloat/multilingual-e5-small"
assert len(selected_revision) == 40


# ------------------------------------------------------------
# 3. Resolve selected chunk artifact from persisted filesystem
# ------------------------------------------------------------

chunk_root = (
    project_root
    / "data"
    / "phase2"
    / "chunks"
    / "nb11_benchmark_v1"
)

assert chunk_root.is_dir(), (
    f"NB11 chunk directory missing: {chunk_root}"
)

chunk_matches = sorted(
    chunk_root.glob(
        f"{selected_chunk_strategy}*.jsonl"
    )
)

assert len(chunk_matches) == 1, (
    "Expected exactly one JSONL artifact for selected chunk "
    f"strategy {selected_chunk_strategy!r}; "
    f"found {[p.name for p in chunk_matches]}"
)

selected_chunk_path = chunk_matches[0].resolve()

assert selected_chunk_path.is_file()

selected_chunk_file_sha256 = hashlib.sha256(
    selected_chunk_path.read_bytes()
).hexdigest()

with selected_chunk_path.open(
    "r",
    encoding="utf-8",
) as f:
    selected_chunk_count = sum(
        1
        for line in f
        if line.strip()
    )

assert selected_chunk_count > 0


# ------------------------------------------------------------
# 4. Validate against recorded benchmark descriptor
# ------------------------------------------------------------

tradeoff_path = (
    active_benchmark_run_directory
    / "embedding_benchmark_tradeoff.json"
)

assert tradeoff_path.is_file()

tradeoff = json.loads(
    tradeoff_path.read_text(encoding="utf-8")
)

best_question_retrieval = (
    tradeoff
    .get("decision_basis", {})
    .get("best_question_retrieval")
)

assert isinstance(best_question_retrieval, dict)

assert (
    best_question_retrieval.get("model")
    == selected_model_key
)

assert (
    best_question_retrieval.get("candidate")
    == selected_chunk_strategy
)

expected_chunk_count = int(
    best_question_retrieval[
        "candidate_chunk_count"
    ]
)

assert (
    selected_chunk_count
    == expected_chunk_count
), (
    "Selected chunk artifact count mismatch: "
    f"disk={selected_chunk_count}, "
    f"benchmark={expected_chunk_count}"
)


# ------------------------------------------------------------
# 5. Validate revision-resolution artifact
# ------------------------------------------------------------

revision_resolution_path = (
    active_benchmark_run_directory
    / "e5_revision_resolution.json"
)

assert revision_resolution_path.is_file(), (
    "Missing E5 revision-resolution artifact."
)

revision_resolution = json.loads(
    revision_resolution_path.read_text(
        encoding="utf-8"
    )
)

assert (
    revision_resolution.get("resolution_status")
    == "RESOLVED_BY_CONTENT_EQUIVALENCE"
)

assert (
    revision_resolution.get("release_revision")
    == selected_revision
)

content_identity_sha256 = (
    revision_resolution[
        "snapshot_content_identity_sha256"
    ]
)


# ------------------------------------------------------------
# 6. Validate Roadmap metrics artifact
# ------------------------------------------------------------

roadmap_metrics_path = (
    active_benchmark_run_directory
    / "embedding_benchmark_roadmap_metrics.json"
)

assert roadmap_metrics_path.is_file()

roadmap_metrics = json.loads(
    roadmap_metrics_path.read_text(
        encoding="utf-8"
    )
)

assert selected_model_key in roadmap_metrics

assert (
    selected_chunk_strategy
    in roadmap_metrics[selected_model_key]
)

selected_metrics = (
    roadmap_metrics[selected_model_key][
        selected_chunk_strategy
    ]["overall"]
)

assert int(selected_metrics["query_count"]) == 3813
assert "ndcg_at_10" in selected_metrics


# ------------------------------------------------------------
# 7. Recover embedding cache evidence read-only
# ------------------------------------------------------------

selected_model_cache_root = (
    active_benchmark_run_directory
    / selected_model_key
)

assert selected_model_cache_root.is_dir(), (
    f"Selected model cache missing: {selected_model_cache_root}"
)

batch_files = sorted(
    selected_model_cache_root.rglob("*.npz")
)

# Some cache layouts may store batch metadata as JSON.
batch_json_files = sorted(
    selected_model_cache_root.rglob("*.json")
)

assert batch_files or batch_json_files, (
    "No persisted selected-model cache files were found."
)

cache_file_inventory = []

for path in batch_files:
    cache_file_inventory.append(
        {
            "path": str(
                path.relative_to(
                    active_benchmark_run_directory
                )
            ),
            "sha256": hashlib.sha256(
                path.read_bytes()
            ).hexdigest(),
            "size_bytes": path.stat().st_size,
        }
    )

cache_inventory_payload = json.dumps(
    cache_file_inventory,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

cache_inventory_sha256 = hashlib.sha256(
    cache_inventory_payload
).hexdigest()


# ------------------------------------------------------------
# 8. Hash acceptance / ADR / evidence artifacts
# ------------------------------------------------------------

acceptance_path = (
    project_root
    / "reports"
    / "phase2"
    / "embedding_benchmark_acceptance_v1.json"
)

acceptance_exists = acceptance_path.is_file()

acceptance_sha256 = (
    hashlib.sha256(
        acceptance_path.read_bytes()
    ).hexdigest()
    if acceptance_exists
    else None
)

embedding_adr_sha256 = hashlib.sha256(
    embedding_adr_path.read_bytes()
).hexdigest()

roadmap_metrics_sha256 = hashlib.sha256(
    roadmap_metrics_path.read_bytes()
).hexdigest()

revision_resolution_sha256 = hashlib.sha256(
    revision_resolution_path.read_bytes()
).hexdigest()


# ------------------------------------------------------------
# 9. Create canonical release directories
# ------------------------------------------------------------

release_dir = (
    project_root
    / "artifacts"
    / "phase2"
    / "embedding"
)

release_dir.mkdir(
    parents=True,
    exist_ok=True,
)

reports_dir = (
    project_root
    / "reports"
    / "phase2"
)

reports_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 10. Build/update acceptance artifact
# ------------------------------------------------------------

acceptance_payload = {
    "schema_version": "1.0",
    "artifact_type": (
        "NB13_EMBEDDING_BENCHMARK_ACCEPTANCE"
    ),
    "status": "PASS",
    "golden_dataset_sha256": GOLDEN_SHA,
    "selected_model_key": selected_model_key,
    "selected_model_id": selected_model_id,
    "selected_revision": selected_revision,
    "revision_binding_status": (
        revision_resolution[
            "resolution_status"
        ]
    ),
    "selected_chunk_strategy": (
        selected_chunk_strategy
    ),
    "selected_chunk_path": str(
        selected_chunk_path.relative_to(
            project_root
        )
    ),
    "selected_chunk_count": (
        selected_chunk_count
    ),
    "selected_chunk_sha256": (
        selected_chunk_file_sha256
    ),
    "metrics": selected_metrics,
    "source_family_analysis": (
        "UNAVAILABLE_FROM_CURRENT_METADATA"
    ),
    "historical_embedding_generation_latency": (
        "NOT_MEASURED_HISTORICALLY"
    ),
    "historical_embedding_generation_failure_rate": (
        "NOT_MEASURED_HISTORICALLY"
    ),
    "reranking_required_for_nb13_acceptance": False,
    "paid_api_used_for_closure": False,
}

acceptance_path.write_text(
    json.dumps(
        acceptance_payload,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

acceptance_sha256 = hashlib.sha256(
    acceptance_path.read_bytes()
).hexdigest()


# ------------------------------------------------------------
# 11. Build canonical frozen embedding release
# ------------------------------------------------------------

embedding_release = {
    "schema_version": "1.0",
    "artifact_type": "EMBEDDING_RELEASE",
    "release_version": "embedding_release_v1",
    "status": "FROZEN_FOR_NB14",

    "golden_dataset": {
        "sha256": GOLDEN_SHA,
        "record_count": 3813,
    },

    "chunking": {
        "strategy": selected_chunk_strategy,
        "path": str(
            selected_chunk_path.relative_to(
                project_root
            )
        ),
        "record_count": selected_chunk_count,
        "sha256": selected_chunk_file_sha256,
    },

    "embedding": {
        "model_key": selected_model_key,
        "model_id": selected_model_id,
        "provider": embedding_adr[
            "selected_provider"
        ],
        "revision": selected_revision,
        "dimension": int(
            embedding_adr[
                "embedding_dimension"
            ]
        ),
        "query_prefix": embedding_adr[
            "query_prefix"
        ],
        "passage_prefix": embedding_adr[
            "passage_prefix"
        ],
        "normalization": embedding_adr[
            "normalization"
        ],
        "revision_binding_status": (
            revision_resolution[
                "resolution_status"
            ]
        ),
        "snapshot_content_identity_sha256": (
            content_identity_sha256
        ),
    },

    "benchmark": {
        "query_count": int(
            selected_metrics[
                "query_count"
            ]
        ),
        "recall_at_5": float(
            selected_metrics[
                "recall_at_5"
            ]
        ),
        "recall_at_10": float(
            selected_metrics[
                "recall_at_10"
            ]
        ),
        "mrr": float(
            selected_metrics[
                "mrr"
            ]
        ),
        "ndcg_at_5": float(
            selected_metrics[
                "ndcg_at_5"
            ]
        ),
        "ndcg_at_10": float(
            selected_metrics[
                "ndcg_at_10"
            ]
        ),
    },

    "cache": {
        "root": str(
            selected_model_cache_root.relative_to(
                project_root
            )
        ),
        "npz_file_count": len(
            batch_files
        ),
        "inventory_sha256": (
            cache_inventory_sha256
        ),
        "recompute_required_for_nb14": False,
    },

    "evidence": {
        "embedding_adr_path": str(
            embedding_adr_path.relative_to(
                project_root
            )
        ),
        "embedding_adr_sha256": (
            embedding_adr_sha256
        ),
        "roadmap_metrics_path": str(
            roadmap_metrics_path.relative_to(
                project_root
            )
        ),
        "roadmap_metrics_sha256": (
            roadmap_metrics_sha256
        ),
        "revision_resolution_path": str(
            revision_resolution_path.relative_to(
                project_root
            )
        ),
        "revision_resolution_sha256": (
            revision_resolution_sha256
        ),
        "acceptance_path": str(
            acceptance_path.relative_to(
                project_root
            )
        ),
        "acceptance_sha256": (
            acceptance_sha256
        ),
    },

    "closure": {
        "api_enabled": False,
        "embedding_recompute_performed": False,
        "reranking_dependency": "NONE",
    },
}


# ------------------------------------------------------------
# 12. Freeze assertions
# ------------------------------------------------------------

assert (
    embedding_release["embedding"]["dimension"]
    == 384
)

assert (
    embedding_release["embedding"]["revision"]
    == selected_revision
)

assert (
    embedding_release["chunking"]["record_count"]
    == expected_chunk_count
)

assert (
    embedding_release["status"]
    == "FROZEN_FOR_NB14"
)

assert (
    embedding_release["closure"]["api_enabled"]
    is False
)


# ------------------------------------------------------------
# 13. Persist release
# ------------------------------------------------------------

release_path = (
    release_dir
    / "embedding_release_v1.json"
)

release_path.write_text(
    json.dumps(
        embedding_release,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

release_sha256 = hashlib.sha256(
    release_path.read_bytes()
).hexdigest()


# ------------------------------------------------------------
# 14. Build NB14 handoff
# ------------------------------------------------------------

nb14_handoff = {
    "schema_version": "1.0",
    "artifact_type": (
        "NB14_DENSE_INDEX_HANDOFF"
    ),
    "status": "READY_FOR_NB14",

    "embedding_release": {
        "path": str(
            release_path.relative_to(
                project_root
            )
        ),
        "sha256": release_sha256,
    },

    "golden_dataset_sha256": GOLDEN_SHA,

    "chunk_input": {
        "strategy": selected_chunk_strategy,
        "path": str(
            selected_chunk_path.relative_to(
                project_root
            )
        ),
        "record_count": selected_chunk_count,
        "sha256": selected_chunk_file_sha256,
    },

    "embedding_config": {
        "model_id": selected_model_id,
        "revision": selected_revision,
        "dimension": 384,
        "query_prefix": embedding_adr[
            "query_prefix"
        ],
        "passage_prefix": embedding_adr[
            "passage_prefix"
        ],
        "normalization": embedding_adr[
            "normalization"
        ],
        "snapshot_content_identity_sha256": (
            content_identity_sha256
        ),
    },

    "vector_cache": {
        "root": str(
            selected_model_cache_root.relative_to(
                project_root
            )
        ),
        "inventory_sha256": (
            cache_inventory_sha256
        ),
        "recompute_required": False,
    },

    "lineage": {
        "embedding_adr_sha256": (
            embedding_adr_sha256
        ),
        "acceptance_sha256": (
            acceptance_sha256
        ),
        "roadmap_metrics_sha256": (
            roadmap_metrics_sha256
        ),
        "revision_resolution_sha256": (
            revision_resolution_sha256
        ),
        "embedding_release_sha256": (
            release_sha256
        ),
    },

    "constraints": {
        "paid_api_required": False,
        "reranking_required": False,
        "golden_dataset_mutation_allowed": False,
        "embedding_config_mutation_allowed": False,
    },
}

assert (
    nb14_handoff["status"]
    == "READY_FOR_NB14"
)

assert (
    nb14_handoff[
        "embedding_config"
    ]["revision"]
    == selected_revision
)

assert (
    nb14_handoff[
        "chunk_input"
    ]["sha256"]
    == selected_chunk_file_sha256
)


# ------------------------------------------------------------
# 15. Persist NB14 handoff
# ------------------------------------------------------------

handoff_path = (
    release_dir
    / "nb14_dense_index_handoff_v1.json"
)

handoff_path.write_text(
    json.dumps(
        nb14_handoff,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

handoff_sha256 = hashlib.sha256(
    handoff_path.read_bytes()
).hexdigest()


# ------------------------------------------------------------
# 16. Export downstream variables
# ------------------------------------------------------------

embedding_release_path = release_path
embedding_release_sha256 = release_sha256

nb14_dense_index_handoff_path = handoff_path
nb14_dense_index_handoff_sha256 = handoff_sha256

nb13_release_status = embedding_release["status"]
nb14_handoff_status = nb14_handoff["status"]


# ------------------------------------------------------------
# 17. Final output
# ------------------------------------------------------------

print("Release status:", nb13_release_status)
print("Selected model:", selected_model_id)
print("Exact revision:", selected_revision)
print(
    "Selected chunk strategy:",
    selected_chunk_strategy,
)
print(
    "Chunk count:",
    selected_chunk_count,
)
print(
    "Chunk SHA256:",
    selected_chunk_file_sha256,
)
print(
    "Cache NPZ files:",
    len(batch_files),
)
print(
    "Cache inventory SHA256:",
    cache_inventory_sha256,
)
print(
    "Acceptance:",
    acceptance_path,
)
print(
    "Release:",
    release_path,
)
print(
    "Release SHA256:",
    release_sha256,
)
print(
    "Handoff:",
    handoff_path,
)
print(
    "Handoff SHA256:",
    handoff_sha256,
)
print(
    "NB14 status:",
    nb14_handoff_status,
)

print()
print(
    "NB13 CANONICAL RELEASE / NB14 HANDOFF: PASS"
)

=== NB13 CANONICAL RELEASE / NB14 HANDOFF ===
Release status: FROZEN_FOR_NB14
Selected model: intfloat/multilingual-e5-small
Exact revision: 614241f622f53c4eeff9890bdc4f31cfecc418b3
Selected chunk strategy: baseline_phase1_char_safe
Chunk count: 3488
Chunk SHA256: 3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50
Cache NPZ files: 90
Cache inventory SHA256: a3d29c5eadadca3a55c622a11271b6b0ffd492a1782e80a5704b0bdb3c5ea5f2
Acceptance: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/reports/phase2/embedding_benchmark_acceptance_v1.json
Release: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding/embedding_release_v1.json
Release SHA256: 8c48acc7c266dd153bce038ff6ec679b716e88679ccd865e15c333e4fbd7f429
Handoff: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding/nb14_dense_index_handoff_v1.json
Handoff SHA256: 77eb9f34e36f174cf748fd99caa1d507b5dc8e4abcb51

<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>Final Roadmap Gate و Final Health</h2>
<p>این Gate نهایی بین سلامت اجرایی و تکمیل مأموریت Roadmap تفاوت می‌گذارد. Notebook فقط زمانی COMPLETE/FROZEN محسوب می‌شود که هم اجرای Cache-only سالم باشد و هم ADR، revision binding، Embedding release و NB14 handoff همگی PASS باشند.</p>
</div>


In [89]:
# Code Cell 72 — NB13 Roadmap-Aware Final Health Check

from pathlib import Path
import json
import hashlib
import ast
import re

print("=== NB13 ROADMAP-AWARE FINAL HEALTH ===")


# ------------------------------------------------------------
# 1. Recover project root
# ------------------------------------------------------------

cwd = Path.cwd().resolve()

project_root = next(
    (
        p
        for p in [cwd, *cwd.parents]
        if (p / "notebooks" / "phase2").is_dir()
    ),
    None,
)

assert project_root is not None, (
    "Could not recover project_root."
)

project_root = project_root.resolve()

GOLDEN_SHA = (
    "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2"
)

benchmark_root = (
    project_root
    / "artifacts"
    / "phase2"
    / "embedding_benchmark"
    / GOLDEN_SHA
)

release_root = (
    project_root
    / "artifacts"
    / "phase2"
    / "embedding"
)


# ------------------------------------------------------------
# 2. Canonical artifact paths
# ------------------------------------------------------------

revision_path = (
    benchmark_root
    / "e5_revision_resolution.json"
)

metrics_path = (
    benchmark_root
    / "embedding_benchmark_roadmap_metrics.json"
)

adr_path = (
    benchmark_root
    / "embedding_selection_adr_v1.json"
)

acceptance_path = (
    project_root
    / "reports"
    / "phase2"
    / "embedding_benchmark_acceptance_v1.json"
)

release_path = (
    release_root
    / "embedding_release_v1.json"
)

handoff_path = (
    release_root
    / "nb14_dense_index_handoff_v1.json"
)

required_paths = [
    revision_path,
    metrics_path,
    adr_path,
    acceptance_path,
    release_path,
    handoff_path,
]

for path in required_paths:
    assert path.is_file(), (
        f"Required closure artifact missing: {path}"
    )


# ------------------------------------------------------------
# 3. Load persisted artifacts
# ------------------------------------------------------------

revision = json.loads(
    revision_path.read_text(encoding="utf-8")
)

metrics = json.loads(
    metrics_path.read_text(encoding="utf-8")
)

adr = json.loads(
    adr_path.read_text(encoding="utf-8")
)

acceptance = json.loads(
    acceptance_path.read_text(encoding="utf-8")
)

release = json.loads(
    release_path.read_text(encoding="utf-8")
)

handoff = json.loads(
    handoff_path.read_text(encoding="utf-8")
)


# ------------------------------------------------------------
# 4. Individual Roadmap gates
# ------------------------------------------------------------

gates = {}


# Golden v2 identity
gates["golden_v2_identity"] = (
    adr.get("active_golden_dataset_sha256")
    == GOLDEN_SHA
    and release.get(
        "golden_dataset",
        {},
    ).get("sha256")
    == GOLDEN_SHA
    and handoff.get("golden_dataset_sha256")
    == GOLDEN_SHA
)


# Main embedding caches
selected_model_key = adr.get(
    "selected_model_key"
)

selected_cache_root = (
    benchmark_root
    / str(selected_model_key)
)

cache_npz_files = (
    sorted(
        selected_cache_root.rglob("*.npz")
    )
    if selected_cache_root.is_dir()
    else []
)

gates["main_embedding_caches_complete"] = (
    selected_model_key
    == "multilingual_e5_small"
    and len(cache_npz_files) == 90
    and release.get(
        "cache",
        {},
    ).get("npz_file_count")
    == 90
    and release.get(
        "cache",
        {},
    ).get("recompute_required_for_nb14")
    is False
)


# nDCG
selected_chunk_strategy = adr.get(
    "selected_chunk_strategy"
)

selected_metric_record = (
    metrics
    .get(
        str(selected_model_key),
        {},
    )
    .get(
        str(selected_chunk_strategy),
        {},
    )
    .get(
        "overall",
        {},
    )
)

gates["ndcg_present"] = (
    "ndcg_at_5" in selected_metric_record
    and "ndcg_at_10" in selected_metric_record
)


# ADR
gates["adr_recorded"] = (
    adr.get("decision_status")
    == "SELECTED_REVISION_BOUND"
    and adr.get("selected_model_id")
    == "intfloat/multilingual-e5-small"
)


# Exact revision
resolved_revision = revision.get(
    "release_revision"
)

gates["exact_revision_bound"] = (
    revision.get("resolution_status")
    == "RESOLVED_BY_CONTENT_EQUIVALENCE"
    and isinstance(resolved_revision, str)
    and len(resolved_revision) == 40
    and adr.get("selected_revision")
    == resolved_revision
    and release.get(
        "embedding",
        {},
    ).get("revision")
    == resolved_revision
    and handoff.get(
        "embedding_config",
        {},
    ).get("revision")
    == resolved_revision
)


# Release frozen
gates["embedding_release_frozen"] = (
    release.get("status")
    == "FROZEN_FOR_NB14"
)


# NB14 handoff
gates["nb14_handoff_ready"] = (
    handoff.get("status")
    == "READY_FOR_NB14"
)


# Closure API disabled
gates["closure_api_disabled"] = (
    release.get(
        "closure",
        {},
    ).get("api_enabled")
    is False
    and release.get(
        "closure",
        {},
    ).get("embedding_recompute_performed")
    is False
    and handoff.get(
        "constraints",
        {},
    ).get("paid_api_required")
    is False
)


# Acceptance
gates["acceptance_pass"] = (
    acceptance.get("status") == "PASS"
    and acceptance.get(
        "selected_revision"
    )
    == resolved_revision
    and acceptance.get(
        "selected_chunk_strategy"
    )
    == selected_chunk_strategy
)


# Chunk lineage
chunk_relative_path = release.get(
    "chunking",
    {},
).get("path")

chunk_path = (
    project_root
    / str(chunk_relative_path)
)

chunk_sha = (
    hashlib.sha256(
        chunk_path.read_bytes()
    ).hexdigest()
    if chunk_path.is_file()
    else None
)

gates["chunk_lineage_bound"] = (
    chunk_path.is_file()
    and chunk_sha
    == release.get(
        "chunking",
        {},
    ).get("sha256")
    and chunk_sha
    == handoff.get(
        "chunk_input",
        {},
    ).get("sha256")
)


# ------------------------------------------------------------
# 5. Release ↔ handoff cryptographic binding
# ------------------------------------------------------------

actual_release_sha = hashlib.sha256(
    release_path.read_bytes()
).hexdigest()

gates["release_handoff_sha_bound"] = (
    handoff.get(
        "embedding_release",
        {},
    ).get("sha256")
    == actual_release_sha
)


# ------------------------------------------------------------
# 6. Notebook static health
# ------------------------------------------------------------

notebook_candidates = [
    project_root
    / "notebooks"
    / "phase2"
    / "13_embedding_benchmark_closure_candidate.ipynb",

    project_root
    / "13_embedding_benchmark_closure_candidate.ipynb",

    Path.cwd()
    / "13_embedding_benchmark_closure_candidate.ipynb",
]

notebook_path = next(
    (
        p.resolve()
        for p in notebook_candidates
        if p.is_file()
    ),
    None,
)

if notebook_path is None:
    matches = list(
        project_root.rglob(
            "13_embedding_benchmark_closure_candidate*.ipynb"
        )
    )

    notebook_path = (
        matches[0].resolve()
        if matches
        else None
    )

assert notebook_path is not None, (
    "Could not locate NB13 closure notebook "
    "for static health audit."
)

notebook_payload = json.loads(
    notebook_path.read_text(
        encoding="utf-8"
    )
)

code_cells = [
    cell
    for cell in notebook_payload.get(
        "cells",
        []
    )
    if cell.get("cell_type") == "code"
]

syntax_errors = []

for index, cell in enumerate(code_cells):
    source = "".join(
        cell.get("source", [])
    )

    try:
        ast.parse(source)
    except SyntaxError as exc:
        syntax_errors.append(
            {
                "cell_index": index,
                "error": str(exc),
            }
        )

gates["syntax_clean"] = (
    len(syntax_errors) == 0
)


# ------------------------------------------------------------
# 7. Code-cell numbering
# ------------------------------------------------------------

number_pattern = re.compile(
    r"^\s*#\s*Code Cell\s+(\d+)\b",
    re.IGNORECASE,
)

code_numbers = []

for cell in code_cells:
    source = "".join(
        cell.get("source", [])
    )

    match = number_pattern.search(source)

    if match:
        code_numbers.append(
            int(match.group(1))
        )

duplicate_numbers = sorted(
    {
        number
        for number in code_numbers
        if code_numbers.count(number) > 1
    }
)

expected_numbers = list(
    range(0, 73)
)

gates["code_numbers_unique"] = (
    not duplicate_numbers
)

gates["code_numbers_sequential"] = (
    sorted(code_numbers)
    == expected_numbers
)


# ------------------------------------------------------------
# 8. Saved notebook execution/error audit
# ------------------------------------------------------------

saved_error_cells = []
unexecuted_numbered_cells = []

for cell in code_cells:
    source = "".join(
        cell.get("source", [])
    )

    match = number_pattern.search(source)

    if not match:
        continue

    number = int(match.group(1))

    if cell.get("execution_count") is None:
        unexecuted_numbered_cells.append(
            number
        )

    for output in cell.get(
        "outputs",
        []
    ):
        if output.get("output_type") == "error":
            saved_error_cells.append(
                {
                    "code_cell": number,
                    "ename": output.get(
                        "ename"
                    ),
                    "evalue": output.get(
                        "evalue"
                    ),
                }
            )

gates["saved_notebook_no_errors"] = (
    len(saved_error_cells) == 0
)

gates["all_numbered_cells_executed"] = (
    set(unexecuted_numbered_cells).issubset({72})
)


# ------------------------------------------------------------
# 9. Duplicate-import static audit
# ------------------------------------------------------------

import_occurrences = {}

for cell in code_cells:
    source = "".join(
        cell.get("source", [])
    )

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in ast.walk(tree):
        if isinstance(
            node,
            (ast.Import, ast.ImportFrom),
        ):
            try:
                key = ast.unparse(node)
            except Exception:
                continue

            import_occurrences[key] = (
                import_occurrences.get(
                    key,
                    0,
                )
                + 1
            )

duplicate_imports = {
    key: count
    for key, count in import_occurrences.items()
    if count > 1
}

# Duplicate imports are reported for hygiene.
# They are not automatically treated as semantic failure,
# because historical notebook cells may intentionally
# re-import standard libraries for restart safety.
duplicate_import_audit_completed = True

gates["duplicate_import_audit_completed"] = (
    duplicate_import_audit_completed
)


# ------------------------------------------------------------
# 10. Paid/API activation static scan
# ------------------------------------------------------------

dangerous_true_patterns = [
    r"\bAPI_ENABLED\s*=\s*True\b",
    r"\bENABLE_API\s*=\s*True\b",
    r"\bPAID_API_ENABLED\s*=\s*True\b",
    r"\bALLOW_API_CALLS\s*=\s*True\b",
    r"\bENABLE_PAID_API\s*=\s*True\b",
]

api_enabled_hits = []

for index, cell in enumerate(code_cells):
    source = "".join(
        cell.get("source", [])
    )

    for pattern in dangerous_true_patterns:
        if re.search(
            pattern,
            source,
            flags=re.IGNORECASE,
        ):
            api_enabled_hits.append(
                {
                    "cell_index": index,
                    "pattern": pattern,
                }
            )

gates["no_api_enabled_true_assignments"] = (
    len(api_enabled_hits) == 0
)


# ------------------------------------------------------------
# 11. Final classification
# ------------------------------------------------------------

roadmap_gate_names = [
    "golden_v2_identity",
    "main_embedding_caches_complete",
    "ndcg_present",
    "adr_recorded",
    "exact_revision_bound",
    "embedding_release_frozen",
    "nb14_handoff_ready",
    "closure_api_disabled",
    "acceptance_pass",
    "chunk_lineage_bound",
    "release_handoff_sha_bound",
]

execution_gate_names = [
    "syntax_clean",
    "code_numbers_unique",
    "code_numbers_sequential",
    "saved_notebook_no_errors",
    "all_numbered_cells_executed",
    "duplicate_import_audit_completed",
    "no_api_enabled_true_assignments",
]

roadmap_failures = [
    name
    for name in roadmap_gate_names
    if not gates.get(name, False)
]

execution_failures = [
    name
    for name in execution_gate_names
    if not gates.get(name, False)
]


# ------------------------------------------------------------
# 12. Print health report
# ------------------------------------------------------------

for name, value in gates.items():
    print(
        f"- {name}:",
        "PASS" if value else "FAIL",
    )

print()
print(
    "Failed roadmap gates:",
    roadmap_failures,
)

print(
    "Failed execution gates:",
    execution_failures,
)

print(
    "Saved error cells:",
    saved_error_cells,
)

print(
    "Unexecuted numbered cells:",
    unexecuted_numbered_cells,
)

print(
    "Duplicate code numbers:",
    duplicate_numbers,
)

print(
    "Duplicate imports:",
    duplicate_imports,
)

print(
    "API-enabled True assignments:",
    api_enabled_hits,
)

execution_health = (
    "PASS"
    if not execution_failures
    else "FAIL"
)

roadmap_health = (
    "PASS"
    if not roadmap_failures
    else "FAIL"
)

final_status = (
    "PASS_COMPLETE_FROZEN"
    if (
        execution_health == "PASS"
        and roadmap_health == "PASS"
    )
    else "FAIL_NOT_FROZEN"
)

print()
print(
    "Execution health:",
    execution_health,
)

print(
    "Roadmap deliverables:",
    roadmap_health,
)

print(
    "Release status:",
    release.get("status"),
)

print(
    "NB14 handoff:",
    handoff.get("status"),
)

print(
    "Final status:",
    final_status,
)


# ------------------------------------------------------------
# 13. Hard final gate
# ------------------------------------------------------------

assert not roadmap_failures, (
    "NB13 Roadmap gates failed: "
    f"{roadmap_failures}"
)

assert not execution_failures, (
    "NB13 execution gates failed: "
    f"{execution_failures}"
)

assert final_status == "PASS_COMPLETE_FROZEN"

print()
print(
    "NB13 FINAL HEALTH: PASS — COMPLETE / FROZEN"
)

=== NB13 ROADMAP-AWARE FINAL HEALTH ===
- golden_v2_identity: PASS
- main_embedding_caches_complete: PASS
- ndcg_present: PASS
- adr_recorded: PASS
- exact_revision_bound: PASS
- embedding_release_frozen: PASS
- nb14_handoff_ready: PASS
- closure_api_disabled: PASS
- acceptance_pass: PASS
- chunk_lineage_bound: PASS
- release_handoff_sha_bound: PASS
- syntax_clean: PASS
- code_numbers_unique: PASS
- code_numbers_sequential: PASS
- saved_notebook_no_errors: PASS
- all_numbered_cells_executed: PASS
- duplicate_import_audit_completed: PASS
- no_api_enabled_true_assignments: PASS

Failed roadmap gates: []
Failed execution gates: []
Saved error cells: []
Unexecuted numbered cells: [72]
Duplicate code numbers: []
Duplicate imports: {'from pathlib import Path': 8, 'import hashlib': 8, 'import json': 14, 'import re': 5, 'from datetime import datetime, timezone': 3, 'import time': 4, 'import importlib.util': 3, 'import numpy as np': 4, 'import os': 3, 'from dotenv import load_dotenv': 2, 'from 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">
<h2>جمع‌بندی نهایی Notebook 13</h2>
<p>اگر Code Cell 72 با PASS پایان یابد، Notebook 13 هم از نظر Execution Health و هم از نظر Deliverableهای رسمی Roadmap بسته شده است و Handoff ماشین‌خوان برای Notebook 14 آماده است. اگر Gate نهایی FAIL شود، فقط همان Gate گزارش‌شده باید ترمیم شود؛ Cacheهای کامل، Golden Dataset و Benchmark اصلی نباید دوباره اجرا یا بازتولید شوند.</p>
</div>
